In [ ]:
import os, subprocess
from pathlib import Path
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["BLUFF_FAST_OBSERVATIONS"]="1"
work=Path("/kaggle/working/training");work.mkdir(exist_ok=True)
sources={'bluff_policy.py': '"""Compact learned Bluff action scorer using own cards and public evidence."""\nfrom collections import Counter, defaultdict\nimport json\nfrom pathlib import Path\n\nRANKS = ("A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K")\nPLACE_FEATURES = ["batch_size", "truthful", "bluff", "bluff_pile", "bluff_call_rate",\n                  "bluff_finish", "remaining_groups", "remaining_largest_group", "claimed_rank_held"]\nCHALLENGE_FEATURES = ["bias", "revealed_bluff_rate", "rank_held", "claim_count", "pile_size",\n                      "opponent_finishing", "own_small_hand", "opponent_small_hand", "samples"]\nINITIAL_WEIGHTS = [4., 5., -2., -4., -8., -8., -1., .2, .5,\n                   -3.5, 7., .7, -.3, -.5, 1., -1., .5, 0.]\n\n\nclass Policy:\n    def __init__(self, weights):\n        if len(weights) != len(INITIAL_WEIGHTS):\n            raise ValueError("wrong feature count")\n        self.weights = [float(v) for v in weights]\n        self.processed = 0\n        self.claims = {}\n        self.stats = defaultdict(lambda: [0, 0])\n        self.our_outcomes = [0, 0]\n\n    def update(self, observation):\n        history = observation.get("public_history", [])\n        if len(history) < self.processed:\n            self.processed, self.claims, self.stats, self.our_outcomes = 0, {}, defaultdict(lambda: [0, 0]), [0, 0]\n        for event in history[self.processed:]:\n            kind = event.get("type")\n            identity = event.get("claim_id")\n            if kind == "claim":\n                self.claims[identity] = (event.get("seat"), event.get("count", 1))\n            elif kind in ("claim_accepted", "challenge_resolved"):\n                placer, count = self.claims.pop(identity, (event.get("placer", event.get("seat")), event.get("count", 1)))\n                if placer == observation["seat"]:\n                    self.our_outcomes[0] += 1\n                    self.our_outcomes[1] += int(kind == "challenge_resolved")\n                if kind == "challenge_resolved" and isinstance(event.get("truthful"), bool):\n                    stats = self.stats[(placer, count)]\n                    stats[0] += 1\n                    stats[1] += int(not event["truthful"])\n        self.processed = len(history)\n\n    def candidates(self, observation):\n        hand = observation["hand"]\n        held = Counter(c["rank"] for c in hand)\n        pile = min(observation.get("pile_size", 0), 40) / 20.\n        if observation["phase"] == "CHALLENGE":\n            claim = observation["pending_claim"]\n            count, rank, placer = claim["count"], claim["claimed_rank"], claim["seat"]\n            copies = 4 * observation.get("game_config", {}).get("decks", 1)\n            if count > copies - held[rank]:\n                return [({"type": "challenge"}, [0.] * len(self.weights))]\n            samples, bluffs = self.stats[(placer, count)]\n            posterior = (bluffs + .6) / (samples + 5.)\n            sizes = observation["hand_sizes"]\n            vector = [1., posterior, held[rank] / copies, count / 4., pile,\n                      float(claim.get("provisional_finish", False)), float(len(hand) <= 3),\n                      1. / max(1, sizes[placer]), min(samples, 20) / 20.]\n            return [({"type": "accept"}, [0.] * len(self.weights)),\n                    ({"type": "challenge"}, [0.] * len(PLACE_FEATURES) + vector)]\n        groups = defaultdict(list)\n        for card in hand:\n            groups[card["rank"]].append(card)\n        actions = []\n        # Safe truthful groups are always available. Constructed bluff candidates\n        # use only currently owned references; the count is the actual batch size.\n        for rank, cards in groups.items():\n            actions.append({"type": "place", "card_refs": [c["card_ref"] for c in cards[:4]], "claimed_rank": rank})\n        ordered = sorted(hand, key=lambda c: (held[c["rank"]], RANKS.index(c["rank"])))\n        for count in (2, 3):\n            if len(ordered) >= count:\n                for rank in groups:\n                    actions.append({"type": "place", "card_refs": [c["card_ref"] for c in ordered[:count]], "claimed_rank": rank})\n        call_rate = (self.our_outcomes[1] + 1.) / (self.our_outcomes[0] + 4.)\n        by_ref = {c["card_ref"]: c for c in hand}\n        result = []\n        for action in actions:\n            refs = set(action["card_refs"])\n            rank = action["claimed_rank"]\n            truthful = all(by_ref[ref]["rank"] == rank for ref in refs)\n            remaining = Counter(c["rank"] for c in hand if c["card_ref"] not in refs)\n            vector = [len(refs), float(truthful), float(not truthful), float(not truthful) * pile,\n                      float(not truthful) * call_rate, float(not truthful and len(refs) == len(hand)),\n                      len(remaining) / 13., max(remaining.values(), default=0) / 4., held[rank] / 4.]\n            result.append((action, vector + [0.] * len(CHALLENGE_FEATURES)))\n        return result\n\n    def act(self, observation):\n        self.update(observation)\n        return max(self.candidates(observation), key=lambda candidate:\n                   sum(w * value for w, value in zip(self.weights, candidate[1])))[0]\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({"format": "triad-bluff-linear-v1", "features": PLACE_FEATURES + CHALLENGE_FEATURES,\n            "weights": self.weights, "training": metadata}, indent=2) + "\\n")\n', 'bluff_policy_v2.py': '"""Learned Bluff scorer with conservative rank bounds and public caller evidence.\n\nInference reads only own hand/private history and public events. Unknown discarded\ncards decrement every rank bound, ensuring no hidden-hand identity is assumed.\n"""\nfrom collections import Counter, defaultdict\nimport json\nimport math\nfrom pathlib import Path\n\nfrom bluff_policy import Policy as V1, PLACE_FEATURES, CHALLENGE_FEATURES, INITIAL_WEIGHTS as V1_INITIAL, RANKS\n\nEXTRA_PLACE = [\'bluff_any_call_probability\', \'bluff_impossible_for_caller\',\n               \'truthful_turns_removed\', \'bluff_known_pile_fraction\',\n               \'truthful_finish\', \'bluff_no_impossibility_call_possible\']\nEXTRA_CHALLENGE = [\'placer_global_revealed_bluff_rate\', \'known_outside_rank_fraction\',\n                   \'known_pile_fraction\', \'claim_fraction_of_original_hand\',\n                   \'finishing_large_batch\', \'known_placer_rank_before_claim\']\nFEATURE_NAMES = PLACE_FEATURES + CHALLENGE_FEATURES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = [.5, 1., -.5, -2., -8., -3., -1., .2, .5] + V1_INITIAL[len(PLACE_FEATURES):] + [-6., -12., 3., -1., 4., 3., 1., 1., -.5, .5, .5, -.5]\n\n\ndef identity(card):\n    return card.get(\'deck\', 0), card[\'rank\'], card[\'suit\']\n\n\nclass Policy(V1):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid model weights\')\n        super().__init__(weights[:len(V1_INITIAL)])\n        self.weights = list(map(float, weights))\n        self._reset_evidence()\n\n    def _reset_evidence(self):\n        self.folded_history = []\n        self.known = defaultdict(Counter)\n        self.pile_known = {}\n        self.calls = defaultdict(lambda: [0, 0])\n        self.global_bluffs = defaultdict(lambda: [0, 0])\n        self.pending_meta = {}\n        self.active = []\n        self.before_claim = {}\n\n    def update(self, observation):\n        history = observation.get(\'public_history\', [])\n        if len(history) < len(self.folded_history) or history[:len(self.folded_history)] != self.folded_history:\n            self.processed = 0\n            self.claims = {}\n            self.stats = defaultdict(lambda: [0, 0])\n            self.our_outcomes = [0, 0]\n            self._reset_evidence()\n        super().update(observation)\n        own_places = {event[\'claim_id\']: event[\'cards\'] for event in observation.get(\'private_history\', []) if event.get(\'type\') == \'own_placement\'}\n        for event in history[len(self.folded_history):]:\n            kind, cid = event.get(\'type\'), event.get(\'claim_id\')\n            if kind == \'game_start\':\n                self.active = list(event[\'active_seats\'])\n            elif kind == \'claim\':\n                placer, count = event[\'seat\'], event[\'count\']\n                self.before_claim[cid] = dict(self.known[placer])\n                # The rank in an unverified claim is not evidence of actual cards.\n                for rank in list(self.known[placer]):\n                    self.known[placer][rank] = max(0, self.known[placer][rank] - count)\n                priority = sorted((s for s in self.active if s != placer), key=lambda s: (s - placer) % observation[\'game_config\'][\'players\'])\n                self.pending_meta[cid] = (placer, count, priority, bool(event.get(\'provisional_finish\')))\n                for card in own_places.get(cid, []):\n                    self.pile_known[identity(card)] = card\n            elif kind in (\'claim_accepted\', \'challenge_resolved\'):\n                placer, count, priority, finish = self.pending_meta.pop(cid)\n                selected = event.get(\'challenger\') if kind == \'challenge_resolved\' else None\n                # The first selected challenger is revealed. Earlier seats must\n                # have accepted; later choices remain unknown and are not counted.\n                for seat in priority:\n                    record = self.calls[(seat, count, finish)]\n                    record[0] += 1\n                    record[1] += int(seat == selected)\n                    if seat == selected:\n                        break\n                if kind == \'challenge_resolved\':\n                    record = self.global_bluffs[placer]\n                    record[0] += 1\n                    record[1] += int(not event[\'truthful\'])\n                    for card in event[\'revealed_batch\']:\n                        self.pile_known[identity(card)] = card\n            elif kind == \'pickup\':\n                recipient = event[\'recipient\']\n                self.known[recipient].update(card[\'rank\'] for card in self.pile_known.values())\n                self.pile_known = {}\n            elif kind == \'finish\':\n                self.active = [s for s in self.active if s != event[\'seat\']]\n        self.folded_history = list(history)\n\n    def caller_probability(self, observation, count, finish):\n        escape = 1.\n        for seat in observation[\'active_seats\']:\n            if seat == observation[\'seat\']:\n                continue\n            samples, calls = self.calls.get((seat, count, finish), (0, 0))\n            # Beta prior avoids certainties from small or selectively observed samples.\n            escape *= 1. - (calls + 1.) / (samples + 5.)\n        return 1. - escape\n\n    def candidates(self, observation):\n        hand = observation[\'hand\']\n        held = Counter(c[\'rank\'] for c in hand)\n        copies = 4 * observation.get(\'game_config\', {}).get(\'decks\', 1)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            claim = observation[\'pending_claim\']\n            placer, rank, count = claim[\'seat\'], claim[\'claimed_rank\'], claim[\'count\']\n            outside = held[rank] + sum(counts[rank] for seat, counts in self.known.items() if seat not in (observation[\'seat\'], placer))\n            if count > copies - outside:\n                return [({\'type\': \'challenge\'}, [0.] * len(self.weights))]\n            base = super().candidates(observation)\n            samples, bluffs = self.global_bluffs.get(placer, (0, 0))\n            extra = [(bluffs + .6) / (samples + 5.), outside / copies,\n                     len(self.pile_known) / max(1, observation[\'pile_size\']),\n                     count / max(1, observation[\'hand_sizes\'][placer] + count),\n                     float(claim.get(\'provisional_finish\', False)) * count / 4.,\n                     self.before_claim.get(claim[\'claim_id\'], {}).get(rank, 0) / copies]\n            return [(a, v + [0.] * len(EXTRA_PLACE) + (extra if a[\'type\'] == \'challenge\' else [0.] * len(EXTRA_CHALLENGE))) for a, v in base]\n        groups = defaultdict(list)\n        for card in hand:\n            groups[card[\'rank\']].append(card)\n        actions = [action for action, _ in super().candidates(observation)]\n        # Full truthful batches plus partial truthful batches and mixed shedding\n        # batches of up to four cards, including a possible finishing bluff.\n        for rank, cards in groups.items():\n            for count in range(1, min(4, len(cards)) + 1):\n                actions.append({\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in cards[:count]], \'claimed_rank\': rank})\n        ordered = sorted(hand, key=lambda c: (held[c[\'rank\']], RANKS.index(c[\'rank\'])))\n        for count in range(2, min(4, len(hand)) + 1):\n            for rank in RANKS:\n                actions.append({\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in ordered[:count]], \'claimed_rank\': rank})\n        by_ref, seen, result = {c[\'card_ref\']: c for c in hand}, set(), []\n        pile = min(observation.get(\'pile_size\', 0), 40) / 20.\n        call_rate = (self.our_outcomes[1] + 1.) / (self.our_outcomes[0] + 4.)\n        for action in actions:\n            refs, rank = set(action[\'card_refs\']), action[\'claimed_rank\']\n            key = tuple(sorted(refs)), rank\n            if key in seen:\n                continue\n            seen.add(key)\n            truthful = all(by_ref[r][\'rank\'] == rank for r in refs)\n            bluff = float(not truthful)\n            remaining = Counter(c[\'rank\'] for c in hand if c[\'card_ref\'] not in refs)\n            finish = not remaining\n            vector = [len(refs), float(truthful), bluff, bluff * pile, bluff * call_rate,\n                      float(bluff and finish), len(remaining) / 13., max(remaining.values(), default=0) / 4., held[rank] / 4.]\n            impossible = any(self.known.get(s, {}).get(rank, 0) + len(refs) > copies for s in observation[\'active_seats\'] if s != observation[\'seat\'])\n            extra = [bluff * self.caller_probability(observation, len(refs), finish),\n                     bluff * impossible, sum((n + 3) // 4 for n in held.values()) - sum((n + 3) // 4 for n in remaining.values()),\n                     bluff * len(self.pile_known) / max(1, observation[\'pile_size\']),\n                     float(truthful and finish), bluff * float(held[rank] >= len(refs))]\n            result.append((action, vector + [0.] * len(CHALLENGE_FEATURES) + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return result\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v2\', \'features\': FEATURE_NAMES,\n                                        \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_policy_v3.py': '"""Bluff v2 model with additional, sound public/own pile rank deductions.\n\nOlder unchallenged batches are never assumed truthful. These deductions use\nonly own placement cards and the last publicly revealed batch before pickup.\n"""\nfrom collections import Counter\nfrom bluff_policy_v2 import Policy as V2, FEATURE_NAMES, INITIAL_WEIGHTS\n\n\nclass Policy(V2):\n    def candidates(self, observation):\n        if observation[\'phase\'] == \'CHALLENGE\':\n            claim = observation[\'pending_claim\']\n            rank, count, placer = claim[\'claimed_rank\'], claim[\'count\'], claim[\'seat\']\n            # Known old pile cards, current own cards and lower bounds on other\n            # players\' hands are disjoint, so their sum is a conservative count.\n            pile_count = Counter(card[\'rank\'] for card in self.pile_known.values())[rank]\n            held_count = sum(card[\'rank\'] == rank for card in observation[\'hand\'])\n            other_count = sum(counts[rank] for seat, counts in self.known.items()\n                              if seat not in (observation[\'seat\'], placer))\n            copies = 4 * observation[\'game_config\'][\'decks\']\n            if count > copies - held_count - other_count - pile_count:\n                return [({\'type\': \'challenge\'}, [0.] * len(self.weights))]\n        return super().candidates(observation)\n', 'bluff_policy_v4.py': '"""Linear Bluff scorer with learned interactions for public call-risk adaptation."""\nfrom collections import Counter\nimport json\nimport math\nfrom pathlib import Path\nfrom bluff_policy_v3 import Policy as V3\nfrom bluff_policy_v2 import FEATURE_NAMES as V2_NAMES, INITIAL_WEIGHTS as V2_INITIAL\n\nEXTRA_PLACE = [\'bluff_call_probability_batch\', \'bluff_call_probability_pile\',\n               \'bluff_call_probability_finish\', \'bluff_nearest_opponent_finish\',\n               \'truthful_call_probability_batch\', \'bluff_remaining_truthful_turns\']\nEXTRA_CHALLENGE = [\'global_revealed_bluff_finishing\', \'count_revealed_bluff_pile\',\n                   \'global_revealed_bluff_pile\', \'count_revealed_bluff_small_own_hand\',\n                   \'global_revealed_bluff_count\', \'opponent_hand_pace_relative_to_own\']\nFEATURE_NAMES = V2_NAMES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = V2_INITIAL + [0.] * 12\n\n\nclass Policy(V3):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid v4 model weights\')\n        super().__init__(weights[:len(V2_NAMES)])\n        self.weights = list(map(float, weights))\n\n    def candidates(self, observation):\n        candidates = super().candidates(observation)\n        hand = observation[\'hand\']\n        pile = min(observation[\'pile_size\'], 40) / 20.\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if len(candidates) == 1:\n                return [(candidates[0][0], [0.] * len(self.weights))]\n            claim = observation[\'pending_claim\']\n            samples, bluffs = self.global_bluffs.get(claim[\'seat\'], (0, 0))\n            global_rate = (bluffs + .6) / (samples + 5.)\n            samples, bluffs = self.stats.get((claim[\'seat\'], claim[\'count\']), (0, 0))\n            count_rate = (bluffs + .6) / (samples + 5.)\n            extra = [global_rate * float(claim.get(\'provisional_finish\', False)),\n                     count_rate * pile, global_rate * pile,\n                     count_rate * float(len(hand) <= 3), global_rate * claim[\'count\'] / 4.,\n                     (len(hand) - observation[\'hand_sizes\'][claim[\'seat\']]) / max(1, len(hand))]\n            return [(action, vector[:len(V2_NAMES)] + [0.] * len(EXTRA_PLACE) +\n                     (extra if action[\'type\'] == \'challenge\' else [0.] * len(EXTRA_CHALLENGE)))\n                    for action, vector in candidates]\n        by_ref = {card[\'card_ref\']: card for card in hand}\n        held = Counter(card[\'rank\'] for card in hand)\n        nearest = min(observation[\'hand_sizes\'][seat] for seat in observation[\'active_seats\']\n                      if seat != observation[\'seat\'])\n        results = []\n        risks = {}\n        for action, vector in candidates:\n            count = len(action[\'card_refs\']); finish = count == len(hand)\n            if count not in risks:\n                risks[count] = self.caller_probability(observation, count, finish)\n            risk = risks[count]\n            bluff = float(any(by_ref[ref][\'rank\'] != action[\'claimed_rank\'] for ref in action[\'card_refs\']))\n            remaining = held.copy()\n            for ref in action[\'card_refs\']:\n                remaining[by_ref[ref][\'rank\']] -= 1\n            turns = sum((n + 3) // 4 for n in remaining.values() if n > 0)\n            extra = [bluff * risk * count / 4., bluff * risk * pile,\n                     bluff * risk * float(finish), bluff / max(1, nearest),\n                     (1. - bluff) * risk * count / 4., bluff * turns / 13.]\n            results.append((action, vector[:len(V2_NAMES)] + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return results\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v4\',\n            \'features\': FEATURE_NAMES, \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_policy_v5.py': '"""Public caller pooling and uncertainty features for a learned Bluff scorer.\n\nBeta summaries are modeling features, not calibrated certainty claims. Observed\nselected/earlier caller outcomes are pooled; later sealed votes stay unknown.\n"""\nimport json\nimport math\nfrom pathlib import Path\nfrom bluff_policy_v4 import Policy as V4, FEATURE_NAMES as V4_NAMES, INITIAL_WEIGHTS as V4_INITIAL\n\nEXTRA_PLACE = [\'bluff_pooled_caller_probability\', \'bluff_pooled_caller_uncertainty\',\n               \'bluff_specific_caller_uncertainty\', \'bluff_sparse_own_outcomes\',\n               \'truthful_pooled_caller_probability_batch\', \'bluff_pooled_caller_probability_pile\']\nEXTRA_CHALLENGE = [\'global_revealed_bluff_uncertainty\', \'count_revealed_bluff_uncertainty\']\nFEATURE_NAMES = V4_NAMES + EXTRA_PLACE + EXTRA_CHALLENGE\nINITIAL_WEIGHTS = V4_INITIAL + [0.] * 8\n\n\ndef beta_summary(samples, successes, prior_success=1., prior_total=5.):\n    a = successes + prior_success\n    b = samples - successes + prior_total - prior_success\n    return a / (a + b), math.sqrt(a * b / ((a + b) ** 2 * (a + b + 1.)))\n\n\nclass Policy(V4):\n    def __init__(self, weights):\n        if len(weights) != len(FEATURE_NAMES) or not all(math.isfinite(float(w)) for w in weights):\n            raise ValueError(\'invalid v5 model weights\')\n        super().__init__(weights[:len(V4_NAMES)])\n        self.weights = list(map(float, weights))\n\n    def caller_summaries(self, observation, count=None, finish=None):\n        escape_mean = escape_upper = 1.\n        for seat in observation[\'active_seats\']:\n            if seat == observation[\'seat\']:\n                continue\n            if count is None:\n                samples = calls = 0\n                for (caller, _, _), record in self.calls.items():\n                    if caller == seat:\n                        samples += record[0]; calls += record[1]\n            else:\n                samples, calls = self.calls.get((seat, count, finish), (0, 0))\n            mean, deviation = beta_summary(samples, calls)\n            escape_mean *= 1. - mean\n            escape_upper *= 1. - min(1., mean + deviation)\n        return 1. - escape_mean, escape_mean - escape_upper\n\n    def candidates(self, observation):\n        candidates = super().candidates(observation)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if len(candidates) == 1:\n                return [(candidates[0][0], [0.] * len(self.weights))]\n            claim = observation[\'pending_claim\']\n            samples, bluffs = self.global_bluffs.get(claim[\'seat\'], (0, 0))\n            _, global_uncertainty = beta_summary(samples, bluffs, .6, 5.)\n            samples, bluffs = self.stats.get((claim[\'seat\'], claim[\'count\']), (0, 0))\n            _, count_uncertainty = beta_summary(samples, bluffs, .6, 5.)\n            return [(action, vector[:len(V4_NAMES)] + [0.] * len(EXTRA_PLACE) +\n                     ([global_uncertainty, count_uncertainty] if action[\'type\'] == \'challenge\' else [0., 0.]))\n                    for action, vector in candidates]\n        by_ref = {card[\'card_ref\']: card for card in observation[\'hand\']}\n        pooled, pooled_uncertainty = self.caller_summaries(observation)\n        sparse = float(self.our_outcomes[0] <= 2)\n        pile = min(observation[\'pile_size\'], 40) / 20.\n        risks = {}\n        results = []\n        for action, vector in candidates:\n            count = len(action[\'card_refs\'])\n            if count not in risks:\n                risks[count] = self.caller_summaries(observation, count, count == len(observation[\'hand\']))\n            _, specific_uncertainty = risks[count]\n            bluff = float(any(by_ref[ref][\'rank\'] != action[\'claimed_rank\'] for ref in action[\'card_refs\']))\n            extra = [bluff * pooled, bluff * pooled_uncertainty, bluff * specific_uncertainty,\n                     bluff * sparse, (1. - bluff) * pooled * count / 4., bluff * pooled * pile]\n            results.append((action, vector[:len(V4_NAMES)] + extra + [0.] * len(EXTRA_CHALLENGE)))\n        return results\n\n    def save(self, path, metadata):\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(json.dumps({\'format\': \'triad-bluff-linear-v5\',\n            \'features\': FEATURE_NAMES, \'weights\': self.weights, \'training\': metadata}, indent=2) + \'\\n\')\n', 'bluff_mlp_policy.py': '"""Tiny residual neural Bluff scorer with standard-library inference."""\nimport math\nfrom bluff_policy_v5 import Policy as Features, FEATURE_NAMES\n\n\nclass Policy(Features):\n    def __init__(self, document):\n        if document.get(\'format\') != \'triad-bluff-residual-mlp-v1\' or document.get(\'features\') != FEATURE_NAMES:\n            raise ValueError(\'incompatible neural Bluff schema\')\n        base = document[\'base_weights\']\n        super().__init__(base)\n        self.hidden = document[\'hidden_weight\']; self.bias = document[\'hidden_bias\']\n        self.output = document[\'output_weight\']; self.output_bias = float(document[\'output_bias\'])\n        self.means = document[\'feature_mean\']; self.scales = document[\'feature_scale\']\n        width = len(self.hidden)\n        if not 1 <= width <= 128 or len(self.bias) != width or len(self.output) != width:\n            raise ValueError(\'invalid hidden layer shape\')\n        if any(len(row) != len(FEATURE_NAMES) for row in self.hidden) or len(self.means) != len(FEATURE_NAMES) or len(self.scales) != len(FEATURE_NAMES):\n            raise ValueError(\'invalid neural input shape\')\n        parameters = base + self.bias + self.output + self.means + self.scales + [self.output_bias]\n        parameters += [weight for row in self.hidden for weight in row]\n        if not all(math.isfinite(float(weight)) for weight in parameters) or any(scale <= 0. for scale in self.scales):\n            raise ValueError(\'nonfinite neural parameters or invalid feature scale\')\n\n    def score_features(self, features):\n        normalized = [(value - mean) / scale for value, mean, scale in zip(features, self.means, self.scales)]\n        residual = self.output_bias\n        for row, bias, output in zip(self.hidden, self.bias, self.output):\n            hidden = max(0., bias + sum(weight * value for weight, value in zip(row, normalized)))\n            residual += output * hidden\n        return sum(weight * value for weight, value in zip(self.weights, features)) + residual\n\n    def act(self, observation):\n        self.update(observation)\n        return max(self.candidates(observation), key=lambda item: self.score_features(item[1]))[0]\n\n\nclass FastPolicy(Policy):\n    """Training-only NumPy batching; near ties use exact deployment scoring."""\n    def __init__(self, document):\n        super().__init__(document)\n        import numpy as np\n        self.np = np\n        self.matrix = np.asarray(self.hidden, dtype=np.float64)\n        self.hidden_bias = np.asarray(self.bias, dtype=np.float64)\n        self.output_array = np.asarray(self.output, dtype=np.float64)\n        self.base_array = np.asarray(self.weights, dtype=np.float64)\n        self.mean_array = np.asarray(self.means, dtype=np.float64)\n        self.scale_array = np.asarray(self.scales, dtype=np.float64)\n        # These exported parameters are immutable configuration, never actor\n        # memory. Freezing them allows exact snapshot sharing during rollouts.\n        self.weights = tuple(self.weights)\n        self.hidden = tuple(tuple(row) for row in self.hidden)\n        self.bias = tuple(self.bias); self.output = tuple(self.output)\n        self.means = tuple(self.means); self.scales = tuple(self.scales)\n        for matrix in (self.matrix, self.hidden_bias, self.output_array,\n                       self.base_array, self.mean_array, self.scale_array):\n            matrix.flags.writeable = False\n\n\n    def __deepcopy__(self, memo):\n        # Copy public/private actor memory, and share only frozen configuration.\n        # This avoids repeatedly copying thousands of constant network weights.\n        import copy\n        clone = type(self).__new__(type(self))\n        memo[id(self)] = clone\n        configuration = {\'np\', \'weights\', \'hidden\', \'bias\', \'output\', \'output_bias\',\n                         \'means\', \'scales\', \'matrix\', \'hidden_bias\', \'output_array\',\n                         \'base_array\', \'mean_array\', \'scale_array\'}\n        for name, value in self.__dict__.items():\n            setattr(clone, name, value if name in configuration else copy.deepcopy(value, memo))\n        return clone\n\n    def act(self, observation):\n        self.update(observation)\n        candidates = self.candidates(observation)\n        matrix = self.np.asarray([vector for _, vector in candidates], dtype=self.np.float64)\n        scores = matrix @ self.base_array + self.output_bias\n        scores += self.np.maximum(0., ((matrix - self.mean_array) / self.scale_array) @ self.matrix.T + self.hidden_bias) @ self.output_array\n        largest = float(scores.max())\n        band = 1e-8 * max(1., abs(largest))\n        finalists = [index for index, value in enumerate(scores) if value >= largest - band]\n        chosen = max(finalists, key=lambda index: self.score_features(candidates[index][1]))\n        return candidates[chosen][0]\n', 'bluff_training_observation.py': '"""Training-only observation shortcut for verified read-only simulator actors.\n\nAll fields still come from the SDK\'s filtered observe(). The engine is not\nadvanced while histories are temporarily detached. Only the requested seat\'s\nprivate history is returned. Never import this helper into an inference image.\n"""\nfrom triad_bluff import engine\n\n\ndef observe_read_only(state, seat):\n    public = state.public_history\n    private = state.private_history[seat]\n    try:\n        state.public_history = []\n        state.private_history[seat] = []\n        observation = engine.observe(state, seat)\n    finally:\n        state.public_history = public\n        state.private_history[seat] = private\n    observation[\'public_history\'] = public\n    observation[\'private_history\'] = private\n    return observation\n', 'train_bluff_league.py': '"""Train against diverse Bluff policies with paired seat blocks and held-out tests."""\nimport argparse\nimport concurrent.futures\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport sys\nimport time\nimport numpy as np\n\nROOT = Path(__file__).resolve().parent\nSDK = Path(os.environ.get(\'BLUFF_SDK\', ROOT.parent / \'bluffSDK\'))\nsys.path.insert(0, str(SDK / \'local_evaluator/src\'))\nfrom triad_bluff import engine\nfrom triad_bluff.baselines import HeuristicAgent, RandomAgent, TruthfulAgent, AggressiveChallengeAgent\nfrom bluff_policy import Policy as V1, INITIAL_WEIGHTS as V1_INITIAL, RANKS\nfrom bluff_policy_v2 import Policy as V2, INITIAL_WEIGHTS, FEATURE_NAMES\nfrom bluff_policy_v3 import Policy as V3\nfrom bluff_policy_v4 import Policy as V4, INITIAL_WEIGHTS as V4_INITIAL\nfrom bluff_policy_v5 import Policy as V5, INITIAL_WEIGHTS as V5_INITIAL\nfrom bluff_training_observation import observe_read_only\n\n\nclass Starter(TruthfulAgent):\n    def act(self, observation):\n        if observation[\'phase\'] == \'CHALLENGE\':\n            return super().act(observation)\n        from collections import defaultdict\n        groups = defaultdict(list)\n        for card in observation[\'hand\']:\n            groups[card[\'rank\']].append(card)\n        rank, cards = max(groups.items(), key=lambda item: (len(item[1]), item[0]))\n        return {\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in cards[:4]], \'claimed_rank\': rank}\n\n\nclass SDKCompetitive:\n    """The SDK\'s supplied adaptive strategy, used only as a development rival."""\n    def __init__(self):\n        import importlib.util\n        path = SDK / \'agents/bluff-agent/src/strategy.py\'\n        spec = importlib.util.spec_from_file_location(\'sdk_competitive_strategy\', path)\n        module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(module)\n        self.select_action = module.select_action\n        self.state = None\n\n    def act(self, observation):\n        action, self.state = self.select_action(observation, self.state)\n        return action\n\n\nclass StochasticBluffer(TruthfulAgent):\n    def __init__(self, seed, bluff_rate, call_rate):\n        self.rng = random.Random(seed)\n        self.bluff_rate, self.call_rate = bluff_rate, call_rate\n\n    def act(self, observation):\n        truthful = super().act(observation)\n        if observation[\'phase\'] == \'CHALLENGE\':\n            if truthful[\'type\'] == \'challenge\':\n                return truthful\n            finishing = observation[\'pending_claim\'].get(\'provisional_finish\', False)\n            probability = min(.95, self.call_rate + .2 * finishing)\n            return {\'type\': \'challenge\' if self.rng.random() < probability else \'accept\'}\n        hand = observation[\'hand\']\n        if len(hand) > 1 and self.rng.random() < self.bluff_rate:\n            from collections import Counter\n            held = Counter(c[\'rank\'] for c in hand)\n            count = min(4, len(hand)) if len(hand) <= 4 else self.rng.choice((2, 3, 4))\n            chosen = sorted(hand, key=lambda c: (held[c[\'rank\']], RANKS.index(c[\'rank\'])))[:count]\n            # Owning copies of the claimed rank makes impossible calls less likely.\n            claimed = max(RANKS, key=lambda rank: held[rank])\n            return {\'type\': \'place\', \'card_refs\': [c[\'card_ref\'] for c in chosen], \'claimed_rank\': claimed}\n        return truthful\n\n\ndef make_policy(spec, seed):\n    kind = spec[\'kind\']\n    if kind == \'mlp\':\n        from bluff_mlp_policy import FastPolicy\n        return FastPolicy(spec[\'model\'])\n    if kind == \'v1\':\n        return V1(spec[\'weights\'])\n    if kind == \'v2\':\n        return V2(spec[\'weights\'])\n    if kind == \'v3\':\n        return V3(spec[\'weights\'])\n    if kind == \'v4\':\n        return V4(spec[\'weights\'])\n    if kind == \'v5\':\n        return V5(spec[\'weights\'])\n    if kind == \'bluffer\':\n        return StochasticBluffer(seed, spec[\'bluff_rate\'], spec[\'call_rate\'])\n    if kind == \'starter\':\n        return Starter()\n    if kind == \'competitive\':\n        return SDKCompetitive()\n    if kind == \'truthful\':\n        return TruthfulAgent()\n    if kind == \'random\':\n        return RandomAgent(seed)\n    if kind == \'aggressive\':\n        return AggressiveChallengeAgent(seed)\n    return HeuristicAgent(seed)\n\n\ndef score_game(seed, seat, candidate, rivals, horizon=1000):\n    actors = [None] * 4\n    actors[seat] = make_policy(candidate, seed * 10 + 9)\n    for offset, rival in enumerate(rivals, 1):\n        actors[(seat + offset) % 4] = make_policy(rival, seed * 10 + offset)\n    state = engine.new_game(engine.GameConfig(max_turns=horizon), seed)\n    while state.outcome is None and seat not in state.finished_set:\n        decision = engine.next_decision(state)\n        observation = observe_read_only(state, decision.seat) if os.environ.get(\'BLUFF_FAST_OBSERVATIONS\') == \'1\' else engine.observe(state, decision.seat)\n        payload = actors[decision.seat].act(observation)\n        action = engine.decode_action(state, decision, payload)\n        if not engine.step(state, decision, action).ok:\n            raise RuntimeError(\'policy produced an illegal action\')\n    if seat in state.finished_order:\n        placement = state.finished_order.index(seat)\n        return state.config.position_points[placement], float(placement == 0)\n    return state.outcome.points[seat], 0.\n\n\ndef evaluate_job(job):\n    candidate, seeds, pool, mode, *options = job\n    horizon = options[0] if options else 1000\n    points, wins = [], []\n    for seed in seeds:\n        rng = random.Random(seed ^ 0x424C554646)\n        if mode == \'league\':\n            rivals = [rng.choice(pool) for _ in range(3)]\n        elif mode == \'bluffer\':\n            rivals = [{\'kind\': \'bluffer\', \'bluff_rate\': .35, \'call_rate\': .1}] * 3\n        else:\n            rivals = [{\'kind\': mode}] * 3\n        outcomes = [score_game(seed, seat, candidate, rivals, horizon) for seat in range(4)]\n        points.append(sum(p for p, _ in outcomes) / 4.)\n        wins.append(sum(w for _, w in outcomes) / 4.)\n    return {\'points\': float(np.mean(points)), \'win_rate\': float(np.mean(wins)),\n            \'games\': 4 * len(seeds), \'block_points\': points, \'block_wins\': wins}\n\n\ndef summarize(result):\n    return {k: v for k, v in result.items() if not k.startswith(\'block_\')}\n\n\ndef parallel_evaluate(pool, candidate, seeds, reference, mode):\n    """Distribute validation seed blocks instead of leaving other workers idle."""\n    chunks = [seeds[i:i + 16] for i in range(0, len(seeds), 16)]\n    parts = list(pool.map(evaluate_job, [(candidate, chunk, reference, mode) for chunk in chunks]))\n    points = [point for part in parts for point in part[\'block_points\']]\n    wins = [win for part in parts for win in part[\'block_wins\']]\n    return {\'points\': float(np.mean(points)), \'win_rate\': float(np.mean(wins)),\n            \'games\': 4 * len(points), \'block_points\': points, \'block_wins\': wins}\n\n\ndef paired_interval(candidate, reference, seed=20261014):\n    differences = np.asarray(candidate[\'block_points\']) - np.asarray(reference[\'block_points\'])\n    rng = np.random.default_rng(seed)\n    means = differences[rng.integers(0, len(differences), (5000, len(differences)))].mean(axis=1)\n    return {\'mean_gain\': float(differences.mean()), \'ci95\': [float(v) for v in np.quantile(means, [.025, .975])],\n            \'unit\': \'points per game\', \'sampling_unit\': \'four-seat seed block\'}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--incumbent\', type=Path)\n    parser.add_argument(\'--reference-policy\', type=Path)\n    parser.add_argument(\'--policy-kind\', choices=(\'v2\', \'v3\', \'v4\', \'v5\'), default=\'v2\')\n    parser.add_argument(\'--fast-observations\', action=\'store_true\')\n    parser.add_argument(\'--stress-opponents\', action=\'store_true\')\n    parser.add_argument(\'--selection-seed-start\', type=int, default=8_000_000)\n    parser.add_argument(\'--test-seed-start\', type=int, default=9_000_000)\n    parser.add_argument(\'--train-seed-start\', type=int, default=50_000)\n    parser.add_argument(\'--iterations\', type=int, default=30)\n    parser.add_argument(\'--population\', type=int, default=32)\n    parser.add_argument(\'--train-blocks\', type=int, default=8)\n    parser.add_argument(\'--train-horizon\', type=int, default=200)\n    parser.add_argument(\'--validation-blocks\', type=int, default=64)\n    parser.add_argument(\'--test-blocks\', type=int, default=256)\n    parser.add_argument(\'--workers\', type=int, default=min(4, os.cpu_count() or 1))\n    parser.add_argument(\'--seed\', type=int, default=20261016)\n    parser.add_argument(\'--output\', type=Path, default=ROOT / \'bluff-league-output\')\n    args = parser.parse_args()\n    if min(args.iterations, args.population, args.train_blocks, args.validation_blocks, args.test_blocks, args.workers, args.train_horizon) < 1 or args.train_horizon > 1000:\n        parser.error(\'counts must be positive\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    if args.fast_observations:\n        os.environ[\'BLUFF_FAST_OBSERVATIONS\'] = \'1\'\n    incumbent_document = json.loads(args.incumbent.read_text()) if args.incumbent else {\'weights\': V1_INITIAL}\n    weights = incumbent_document[\'weights\']\n    incumbent_kind = incumbent_document.get(\'training\', {}).get(\'policy_kind\', \'v2\' if len(weights) == len(FEATURE_NAMES) else \'v4\' if len(weights) == len(V4_INITIAL) else \'v5\' if len(weights) == len(V5_INITIAL) else \'v1\')\n    incumbent = {\'kind\': incumbent_kind, \'weights\': weights}\n    initial_weights = V5_INITIAL if args.policy_kind == \'v5\' else V4_INITIAL if args.policy_kind == \'v4\' else INITIAL_WEIGHTS\n    mean = np.asarray(weights + initial_weights[len(weights):], dtype=float)\n    best, sigma = mean.copy(), np.full(len(mean), 1.5)\n    rng = np.random.default_rng(args.seed)\n    cautious = list(INITIAL_WEIGHTS); cautious[2] -= 8.; cautious[4] -= 8.\n    daring = list(INITIAL_WEIGHTS); daring[2] += 4.; daring[5] += 4.\n    reference = [{\'kind\': k} for k in (\'starter\', \'heuristic\', \'truthful\', \'aggressive\', \'random\', \'competitive\')] + [\n        incumbent, {\'kind\': \'v2\', \'weights\': INITIAL_WEIGHTS}, {\'kind\': \'v2\', \'weights\': cautious},\n        {\'kind\': \'v2\', \'weights\': daring}] + [\n        {\'kind\': \'bluffer\', \'bluff_rate\': b, \'call_rate\': c} for b,c in ((.15,.03),(.35,.1),(.5,.35))]\n    if args.reference_policy:\n        reference_document = json.loads(args.reference_policy.read_text())\n        reference.append({\'kind\': reference_document.get(\'training\', {}).get(\'policy_kind\', \'v1\' if len(reference_document[\'weights\']) == len(V1_INITIAL) else \'v2\'),\n                          \'weights\': reference_document[\'weights\']})\n    if args.stress_opponents:\n        reference.extend({\'kind\': \'bluffer\', \'bluff_rate\': b, \'call_rate\': c}\n                         for b, c in ((.1, .35), (.65, .35), (.95, .0), (.95, .35)))\n    archive = list(reference)\n    selection_seeds = list(range(args.selection_seed_start, args.selection_seed_start + args.validation_blocks))\n    history, started = [], time.time()\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers, mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        baseline = parallel_evaluate(pool, incumbent, selection_seeds, reference, \'league\')\n        initial = parallel_evaluate(pool, {\'kind\': args.policy_kind, \'weights\': best.tolist()}, selection_seeds, reference, \'league\')\n        selected_score = initial[\'points\']\n        for generation in range(args.iterations):\n            if generation and generation % 10 == 0:\n                mean, sigma = best.copy(), np.full(len(mean), 1.5)\n            population = rng.normal(mean, sigma, (args.population, len(mean)))\n            probes = max(1, args.population // 5)\n            population[-probes:] = rng.normal(best, 3., (probes, len(mean)))\n            population = np.clip(population, -20., 20.)\n            population[0] = mean\n            if args.population > 1:\n                population[1] = best\n            seeds = list(range(args.train_seed_start + generation * args.train_blocks, args.train_seed_start + (generation + 1) * args.train_blocks))\n            # Fast training probes use a shorter cap; model selection and final\n            # reporting always use the real 1,000-turn cap. Short-cap results are\n            # approximate training rewards, never evidence of event performance.\n            scores = list(pool.map(evaluate_job, [({\'kind\':args.policy_kind,\'weights\':w.tolist()},seeds,archive,\'league\',args.train_horizon) for w in population]))\n            ranked = sorted(range(len(scores)), key=lambda i: (scores[i][\'points\'], scores[i][\'win_rate\']), reverse=True)\n            elite = population[ranked[:max(2, args.population // 5)]]\n            mean = .35 * mean + .65 * elite.mean(axis=0)\n            sigma = np.maximum(.3, .35 * sigma + .65 * elite.std(axis=0))\n            candidate = population[ranked[0]]\n            spec = {\'kind\':args.policy_kind,\'weights\':candidate.tolist()}\n            validation = parallel_evaluate(pool, spec, selection_seeds, reference, \'league\')\n            if validation[\'points\'] > selected_score:\n                best, selected_score = candidate.copy(), validation[\'points\']\n            if generation % 4 == 3:\n                archive.append(spec)\n                archive = reference + archive[len(reference):][-12:]\n            record = {\'generation\':generation+1,\'train\':summarize(scores[ranked[0]]),\'validation\':summarize(validation),\n                      \'paired_validation_gain\':paired_interval(validation,baseline),\'selected_points\':selected_score,\n                      \'archive_size\':len(archive),\'elapsed_seconds\':round(time.time()-started,1)}\n            history.append(record); print(json.dumps(record),flush=True)\n            policy_class = V5 if args.policy_kind == \'v5\' else V4 if args.policy_kind == \'v4\' else V2\n            policy_class(best).save(args.output/\'bluff-policy-v2.json\',{\'method\':\'archive-league cross-entropy policy optimization\',\n                \'fast_observations\': args.fast_observations, \'stress_opponents\': args.stress_opponents,\n                \'policy_kind\': args.policy_kind, \'selection_seed_start\': args.selection_seed_start,\n                \'test_seed_start\': args.test_seed_start, \'train_seed_start\': args.train_seed_start,\n                \'seed\':args.seed,\'generations\':generation+1,\'population\':args.population,\n                \'train_blocks\':args.train_blocks,\'train_horizon\':args.train_horizon,\'evaluation_horizon\':1000,\'validation_blocks\':args.validation_blocks,\n                \'selected_validation_points\':selected_score,\'incumbent_validation_points\':baseline[\'points\'],\n                \'sdk_commit\':\'96da8042c285af4e4f43655f2ad99592bd49b8e6\'})\n            (args.output/\'training-progress.json\').write_text(json.dumps(history,indent=2)+\'\\n\')\n            (args.output/\'archive.json\').write_text(json.dumps(archive,indent=2)+\'\\n\')\n        seeds = list(range(args.test_seed_start,args.test_seed_start+args.test_blocks))\n        chosen = {\'kind\':args.policy_kind,\'weights\':best.tolist()}\n        modes = (\'league\',\'starter\',\'truthful\',\'bluffer\',\'aggressive\',\'competitive\')\n        results = list(pool.map(evaluate_job,[(spec,seeds,reference,mode) for mode in modes for spec in (chosen,incumbent)]))\n        report = {mode:{\'candidate\':summarize(results[2*i]),\'incumbent\':summarize(results[2*i+1]),\n                        \'paired_gain\':paired_interval(results[2*i],results[2*i+1])} for i,mode in enumerate(modes)}\n        (args.output/\'evaluation.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n        print(json.dumps({\'held_out_evaluation\':report}),flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'collect_bluff_counterfactual.py': '"""Paired offline policy-improvement returns; deployment features are own/public.\n\nThe dealt simulator hands are held fixed within each labeled state. Opponent\nrandom choices are averaged using common trial seeds across candidate actions.\nWhole-game, phase-stratified reservoir sampling covers later/endgame decisions.\nThere is no draw deck to shuffle in Bluff. Hidden simulator state is confined to\ntraining returns and never appears in the exported feature vectors.\n"""\nimport argparse\nimport concurrent.futures\nimport copy\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport time\nfrom collections import Counter\nimport numpy as np\nfrom train_bluff_league import engine, make_policy\nfrom bluff_policy_v5 import Policy as Features, FEATURE_NAMES\nfrom bluff_training_observation import observe_read_only\n\n\ndef reward(state, seat):\n    if seat in state.finished_order:\n        return float(state.config.position_points[state.finished_order.index(seat)])\n    return float(state.outcome.points[seat])\n\n\ndef advance(state, actors):\n    decision = engine.next_decision(state)\n    observation = observe_read_only(state, decision.seat)\n    action = actors[decision.seat].act(observation)\n    if not engine.step(state, decision, engine.decode_action(state, decision, action)).ok:\n        raise RuntimeError(\'rollout actor produced an illegal action\')\n\n\ndef copy_snapshot(snapshot):\n    if os.environ.get(\'BLUFF_FAST_SNAPSHOTS\') == \'1\':\n        from bluff_training_snapshot import clone_snapshot\n        return clone_snapshot(snapshot)\n    return copy.deepcopy(snapshot)\n\n\ndef rollout(snapshot, seat, selected, trial_seed):\n    state, actors = copy_snapshot(snapshot)\n    # Preserve each actor\'s public/private memory at the sampled decision while\n    # integrating future stochastic choices. Each action gets the same streams.\n    for player, actor in enumerate(actors):\n        if hasattr(actor, \'rng\'):\n            actor.rng = random.Random(trial_seed * 10 + player)\n    decision = engine.next_decision(state)\n    if decision.seat != seat:\n        raise RuntimeError(\'sampled decision is for a different player\')\n    if not engine.step(state, decision, engine.decode_action(state, decision, selected)).ok:\n        raise RuntimeError(\'counterfactual action is not legal at sampled state\')\n    while state.outcome is None and seat not in state.finished_set:\n        advance(state, actors)\n    return reward(state, seat)\n\n\ndef checkpoint_spec(document):\n    if document.get(\'format\') == \'triad-bluff-residual-mlp-v1\':\n        return {\'kind\': \'mlp\', \'model\': document}\n    kind = document.get(\'training\', {}).get(\'policy_kind\',\n        \'v2\' if len(document[\'weights\']) == 30 else \'v1\')\n    return {\'kind\': kind, \'weights\': document[\'weights\']}\n\n\ndef pool_specs(incumbent, previous):\n    return [checkpoint_spec(incumbent), checkpoint_spec(previous),\n            *({\'kind\': kind} for kind in (\'starter\', \'competitive\', \'random\', \'aggressive\', \'heuristic\')),\n            *({\'kind\': \'bluffer\', \'bluff_rate\': bluff, \'call_rate\': call}\n              for bluff, call in ((.1, .0), (.1, .35), (.35, .1), (.65, .35), (.95, .0), (.95, .35)))]\n\n\nSOURCE_REGIMES = (\'league\', \'neural_selfplay\', \'v2_selfplay\', \'competitive\',\n                  \'truthful\', \'bluffer\', \'frequent_call\')\n\n\ndef source_specs(regime, rng, incumbent, previous):\n    if regime == \'league\':\n        return [rng.choice(pool_specs(incumbent, previous)) for _ in range(4)]\n    if regime == \'neural_selfplay\':\n        rival = checkpoint_spec(incumbent)\n    elif regime == \'v2_selfplay\':\n        rival = checkpoint_spec(previous)\n    elif regime in (\'competitive\', \'truthful\'):\n        rival = {\'kind\': regime}\n    elif regime == \'bluffer\':\n        rival = {\'kind\': \'bluffer\', \'bluff_rate\': .35, \'call_rate\': .1}\n    elif regime == \'frequent_call\':\n        rival = {\'kind\': \'bluffer\', \'bluff_rate\': .65, \'call_rate\': .35}\n    else:\n        raise ValueError(\'unknown source composition\')\n    return [rival] * 4\n\n\ndef collect_job(job):\n    seed, incumbent, previous, samples_per_phase, trials, max_actions, regime = job\n    rng = random.Random(seed ^ 0x424C554646)\n    own = seed % 4\n    specs = source_specs(regime, rng, incumbent, previous)\n    specs[own] = checkpoint_spec(incumbent)\n    actors = [make_policy(spec, seed * 10 + player) for player, spec in enumerate(specs)]\n    state = engine.new_game(engine.GameConfig(max_turns=1000), seed)\n    reservoirs = {\'PLACE\': [], \'CHALLENGE\': []}\n    eligible = Counter()\n    initial = incumbent.get(\'base_weights\', incumbent.get(\'weights\'))\n    feature_weights = initial + [0.] * (len(FEATURE_NAMES) - len(initial))\n    while state.outcome is None and own not in state.finished_set:\n        decision = engine.next_decision(state)\n        if decision.seat == own:\n            observation = observe_read_only(state, own)\n            policy = Features(feature_weights); policy.update(observation)\n            if len(policy.candidates(observation)) > 1:\n                phase = observation[\'phase\']\n                eligible[phase] += 1\n                index = rng.randrange(eligible[phase])\n                if index < samples_per_phase:\n                    snapshot = copy_snapshot((state, actors))\n                    if len(reservoirs[phase]) < samples_per_phase:\n                        reservoirs[phase].append(snapshot)\n                    else:\n                        reservoirs[phase][index] = snapshot\n        advance(state, actors)\n    rows = []\n    for phase in (\'PLACE\', \'CHALLENGE\'):\n        for snapshot in reservoirs[phase]:\n            original, _ = snapshot\n            observation = observe_read_only(original, own)\n            policy = Features(feature_weights); policy.update(observation)\n            candidates = policy.candidates(observation)\n            baseline = make_policy(specs[own], seed * 10 + own).act(observation)\n            preferred = next(item for item in candidates if item[0] == baseline) if any(item[0] == baseline for item in candidates) else candidates[0]\n            # Identical model inputs cannot be distinguished by the scorer;\n            # avoid spending rollout compute on exact feature aliases.\n            representatives = {tuple(preferred[1]): preferred}\n            for candidate in candidates:\n                representatives.setdefault(tuple(candidate[1]), candidate)\n            selected = list(representatives.values())\n            if len(selected) > max_actions:\n                selected = [preferred] + rng.sample([item for item in selected if item is not preferred], max_actions - 1)\n            trial_values = [[rollout(snapshot, own, action, seed * 100_000 + len(rows) * 1000 + trial)\n                             for trial in range(trials)] for action, _ in selected]\n            features = [vector for _, vector in selected]\n            targets = [sum(values) / trials for values in trial_values]\n            rows.append((features, targets, trial_values, seed, phase, len(observation[\'hand\']), regime))\n    return rows\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--incumbent\', type=Path, required=True)\n    parser.add_argument(\'--previous\', type=Path, required=True)\n    parser.add_argument(\'--games\', type=int, default=2000)\n    parser.add_argument(\'--samples-per-phase\', type=int, default=2)\n    parser.add_argument(\'--trials\', type=int, default=2)\n    parser.add_argument(\'--max-actions\', type=int, default=10)\n    parser.add_argument(\'--workers\', type=int, default=4)\n    parser.add_argument(\'--seed-start\', type=int, default=50_000_000)\n    parser.add_argument(\'--source-compositions\', choices=(\'league\', \'balanced\'), default=\'league\')\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args()\n    if min(args.games, args.samples_per_phase, args.trials, args.workers) < 1 or args.max_actions < 2:\n        parser.error(\'positive counts and at least two candidate actions required\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    incumbent = json.loads(args.incumbent.read_text()); previous = json.loads(args.previous.read_text())\n    features, targets, trial_targets, offsets, game_ids = [], [], [], [0], []\n    phases = Counter(); hands = Counter(); compositions = Counter(); source_regimes = []; started = time.time()\n    jobs = [(seed, incumbent, previous, args.samples_per_phase, args.trials, args.max_actions,\n             SOURCE_REGIMES[(seed - args.seed_start) % len(SOURCE_REGIMES)] if args.source_compositions == \'balanced\' else \'league\')\n            for seed in range(args.seed_start, args.seed_start + args.games)]\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers, mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        for index, rows in enumerate(pool.map(collect_job, jobs)):\n            for vectors, values, trials, seed, phase, hand, regime in rows:\n                features.extend(vectors); targets.extend(values); trial_targets.extend(trials)\n                offsets.append(len(features)); game_ids.append(seed); phases[phase] += 1; hands[hand] += 1\n                source_regimes.append(SOURCE_REGIMES.index(regime)); compositions[regime] += 1\n            if (index + 1) % 100 == 0:\n                print(json.dumps({\'games\': index + 1, \'states\': len(game_ids), \'actions\': len(features),\n                                  \'seconds\': round(time.time() - started, 1)}), flush=True)\n    np.savez_compressed(args.output / \'counterfactual.npz\', features=np.asarray(features, dtype=np.float32),\n                        targets=np.asarray(targets, dtype=np.float32), trial_targets=np.asarray(trial_targets, dtype=np.float32),\n                        offsets=np.asarray(offsets, dtype=np.int64), game_ids=np.asarray(game_ids, dtype=np.int64),\n                        source_regimes=np.asarray(source_regimes, dtype=np.int8))\n    metadata = {\'method\': \'paired one-step policy improvement, phase-stratified whole-game reservoirs\',\n                \'input_features\': FEATURE_NAMES, \'feature_policy\': \'v5\', \'games\': args.games,\n                \'states\': len(game_ids), \'actions\': len(features), \'trials\': args.trials,\n                \'samples_per_phase\': args.samples_per_phase, \'max_actions\': args.max_actions,\n                \'seed_start\': args.seed_start, \'incumbent\': incumbent, \'opponent_pool\': pool_specs(incumbent, previous),\n                \'sdk_game_commit\': \'96da8042c285af4e4f43655f2ad99592bd49b8e6\',\n                \'incumbent_policy_kind\': checkpoint_spec(incumbent)[\'kind\'],\n                \'source_compositions\': args.source_compositions, \'source_regime_names\': SOURCE_REGIMES,\n                \'source_game_composition_histogram\': dict(Counter(job[-1] for job in jobs)),\n                \'sampled_state_composition_histogram\': dict(compositions),\n                \'fast_snapshots\': os.environ.get(\'BLUFF_FAST_SNAPSHOTS\') == \'1\',\n                \'hidden_state_use\': \'training returns only; dealt hands fixed within state, common future actor seeds across actions\',\n                \'future_draw_pile_shuffle\': False, \'own_hand_size_histogram\': dict(sorted(hands.items())),\n                \'phase_histogram\': dict(phases), \'seconds\': round(time.time() - started, 1)}\n    (args.output / \'metadata.json\').write_text(json.dumps(metadata, indent=2) + \'\\n\')\n    print(json.dumps({key: value for key, value in metadata.items() if key not in (\'incumbent\', \'input_features\', \'opponent_pool\')}), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'select_bluff_residual_cpu.py': '"""CPU actual-game selection for GPU-exported neural candidates."""\nimport argparse\nimport concurrent.futures\nimport json\nimport multiprocessing\nimport os\nfrom pathlib import Path\nfrom train_bluff_league import evaluate_job, paired_interval, summarize\nfrom evaluate_bluff_checkpoint import combine\nfrom collect_bluff_counterfactual import checkpoint_spec\nfrom bluff_mlp_policy import Policy\n\n\ndef evaluate(pool, specifications, seeds, opponents, mode):\n    chunks=[seeds[start:start+16] for start in range(0,len(seeds),16)]\n    jobs=[(spec,chunk,opponents,mode) for spec in specifications for chunk in chunks]\n    raw=list(pool.map(evaluate_job,jobs));n=len(chunks)\n    return [combine(raw[index*n:(index+1)*n]) for index in range(len(specifications))]\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--candidates\',type=Path,required=True)\n    parser.add_argument(\'--linear-backup\',type=Path,required=True)\n    parser.add_argument(\'--previous-v1\',type=Path)\n    parser.add_argument(\'--archive\',type=Path,action=\'append\',default=[],help=\'additional fixed learned rivals\')\n    parser.add_argument(\'--validation-blocks\',type=int,default=128)\n    parser.add_argument(\'--test-blocks\',type=int,default=256)\n    parser.add_argument(\'--validation-seed-start\',type=int,default=65_000_000)\n    parser.add_argument(\'--test-seed-start\',type=int,default=66_000_000)\n    parser.add_argument(\'--workers\',type=int,default=4)\n    parser.add_argument(\'--robust-selection\',action=\'store_true\',help=\'select using five fixed rival compositions with a per-regime loss guard\')\n    parser.add_argument(\'--robust-calling-styles\',action=\'store_true\',help=\'also guard both fixed frequent-calling bluff styles during robust selection\')\n    parser.add_argument(\'--max-selection-regression\',type=float,default=.025)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    args=parser.parse_args()\n    if min(args.validation_blocks,args.test_blocks,args.workers)<1:parser.error(\'positive counts required\')\n    os.environ[\'BLUFF_FAST_OBSERVATIONS\']=\'1\';os.environ[\'OPENBLAS_NUM_THREADS\']=\'1\'\n    args.output.mkdir(parents=True,exist_ok=True)\n    fit=json.loads((args.candidates/\'fit-summary.json\').read_text())\n    incumbent_document=fit[\'baseline_checkpoint\'];incumbent=checkpoint_spec(incumbent_document)\n    linear_document=json.loads(args.linear_backup.read_text());linear=checkpoint_spec(linear_document)\n    documents=[json.loads((args.candidates/f\'candidate-{index}.json\').read_text()) for index in range(fit[\'candidate_count\'])]\n    for document in documents:Policy(document)\n    reference=[{\'kind\':kind} for kind in (\'starter\',\'competitive\',\'heuristic\',\'random\',\'truthful\',\'aggressive\')]+[incumbent,linear]\n    if args.previous_v1:reference.append(checkpoint_spec(json.loads(args.previous_v1.read_text())))\n    reference += [checkpoint_spec(json.loads(path.read_text())) for path in args.archive]\n    reference += [{\'kind\':\'bluffer\',\'bluff_rate\':bluff,\'call_rate\':call} for bluff,call in ((.1,.0),(.1,.35),(.35,.1),(.65,.35),(.95,.0),(.95,.35))]\n    specs=[incumbent]+[checkpoint_spec(document) for document in documents]\n    seeds=list(range(args.validation_seed_start,args.validation_seed_start+args.validation_blocks))\n    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers,mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        results=evaluate(pool,specs,seeds,reference,\'league\')\n        chosen=max(range(len(results)),key=lambda index:results[index][\'points\'])\n        selection={\'selected\':chosen,\'seed_start\':args.validation_seed_start,\'scores\':[summarize(row) for row in results],\'paired_gains\':[paired_interval(row,results[0]) for row in results[1:]],\'opponent_pool\':reference}\n        if args.robust_selection:\n            # These seeds select a fixed checkpoint only. Fresh final seeds are\n            # never consulted during candidate or objective selection.\n            regime_results={\'league\':results}\n            for label,opponents,mode in [(\'neural_selfplay\',[incumbent],\'league\'),\n                                         (\'competitive\',reference,\'competitive\'),\n                                         (\'truthful\',reference,\'truthful\'),\n                                         (\'bluffer\',reference,\'bluffer\')]:\n                regime_results[label]=evaluate(pool,specs,seeds,opponents,mode)\n                print(json.dumps({\'selection_regime\':label,\'scores\':[summarize(row) for row in regime_results[label]]}),flush=True)\n            if args.robust_calling_styles:\n                for label,bluff in [(\'weakspot_bluff01_call035\',.1),(\'weakspot_bluff065_call035\',.65)]:\n                    regime_results[label]=evaluate(pool,specs,seeds,[{\'kind\':\'bluffer\',\'bluff_rate\':bluff,\'call_rate\':.35}],\'league\')\n                    print(json.dumps({\'selection_regime\':label,\'scores\':[summarize(row) for row in regime_results[label]]}),flush=True)\n            gains={label:[summarize(row)[\'points\']-summarize(rows[0])[\'points\'] for row in rows]\n                   for label,rows in regime_results.items()}\n            utilities=[sum(values[index] for values in gains.values())/len(gains) for index in range(len(specs))]\n            feasible=[index for index in range(len(specs))\n                      if min(values[index] for values in gains.values())>=-args.max_selection_regression]\n            chosen=max(feasible,key=lambda index:utilities[index])\n            selection.update(selected=chosen,selection_rule=\'equal mean gain across predeclared regimes, subject to per-regime mean regression guard; baseline wins ties\',\n                             selection_regimes=list(regime_results),\n                             max_selection_regression=args.max_selection_regression,\n                             regime_scores={label:[summarize(row) for row in rows] for label,rows in regime_results.items()},\n                             regime_paired_gains={label:[paired_interval(row,rows[0]) for row in rows[1:]] for label,rows in regime_results.items()},\n                             feasible_candidates=feasible,mean_gain_utilities=utilities)\n        (args.output/\'selection.json\').write_text(json.dumps(selection,indent=2)+\'\\n\')\n        print(json.dumps({\'actual_game_selection\':{key:value for key,value in selection.items() if key!=\'opponent_pool\'}}),flush=True)\n        selected_document=documents[chosen-1] if chosen else incumbent_document\n        (args.output/\'bluff-policy-mlp.json\').write_text(json.dumps(selected_document,indent=2)+\'\\n\')\n        selected=specs[chosen];final_seeds=list(range(args.test_seed_start,args.test_seed_start+args.test_blocks));report={}\n        regimes=[(\'league\',reference,\'league\'),(\'competitive\',reference,\'competitive\'),(\'starter\',reference,\'starter\'),(\'truthful\',reference,\'truthful\'),(\'bluffer\',reference,\'bluffer\'),(\'aggressive\',reference,\'aggressive\'),(\'incumbent_selfplay\',[incumbent],\'league\')]\n        regimes += [(label,[{\'kind\':\'bluffer\',\'bluff_rate\':bluff,\'call_rate\':.35}],\'league\') for label,bluff in ((\'weakspot_bluff01_call035\',.1),(\'weakspot_bluff065_call035\',.65))]\n        for label,opponents,mode in regimes:\n            rows=evaluate(pool,[selected,incumbent,linear],final_seeds,opponents,mode)\n            report[label]={\'candidate\':summarize(rows[0]),\'incumbent_neural\':summarize(rows[1]),\'retained_v2\':summarize(rows[2]),\'paired_gain_vs_neural\':paired_interval(rows[0],rows[1]),\'paired_gain_vs_v2\':paired_interval(rows[0],rows[2])}\n            print(json.dumps({\'final_regime\':label,**report[label]}),flush=True)\n        (args.output/\'evaluation.json\').write_text(json.dumps({\'seed_start\':args.test_seed_start,\'candidate_training\':selected_document[\'training\'],\'baseline_training\':incumbent_document[\'training\'],\'evaluation\':report},indent=2)+\'\\n\')\n\n\nif __name__==\'__main__\':main()\n', 'evaluate_bluff_checkpoint.py': '"""Benchmark trained Bluff against actual SDK starter and adaptive mixtures."""\nimport argparse\nimport concurrent.futures\nimport json\nimport multiprocessing\nfrom pathlib import Path\nfrom train_bluff_league import evaluate_job, paired_interval, summarize\nfrom bluff_policy import INITIAL_WEIGHTS\nfrom bluff_policy_v2 import INITIAL_WEIGHTS as V2_INITIAL\n\n\ndef combine(parts):\n points=[v for p in parts for v in p[\'block_points\']]\n wins=[v for p in parts for v in p[\'block_wins\']]\n return {\'points\':sum(points)/len(points),\'win_rate\':sum(wins)/len(wins),\'games\':4*len(points),\'block_points\':points,\'block_wins\':wins}\n\n\ndef main():\n parser=argparse.ArgumentParser()\n parser.add_argument(\'--model\',type=Path,required=True)\n parser.add_argument(\'--blocks\',type=int,default=256)\n parser.add_argument(\'--workers\',type=int,default=4)\n parser.add_argument(\'--output\',type=Path,default=Path(__file__).parent/\'bluff-v1-verification.json\')\n args=parser.parse_args()\n document=json.loads(args.model.read_text())\n candidate={\'kind\':\'v1\',\'weights\':document[\'weights\']};starter={\'kind\':\'starter\'}\n reference=[{\'kind\':k} for k in (\'starter\',\'heuristic\',\'truthful\',\'aggressive\',\'random\')]+[\n  {\'kind\':\'v1\',\'weights\':INITIAL_WEIGHTS},{\'kind\':\'v2\',\'weights\':V2_INITIAL},candidate]+[\n  {\'kind\':\'bluffer\',\'bluff_rate\':b,\'call_rate\':c} for b,c in ((.15,.03),(.35,.1),(.5,.35))]\n modes=(\'league\',\'starter\',\'truthful\',\'bluffer\',\'aggressive\')\n seeds=list(range(13_000_000,13_000_000+args.blocks));chunks=[seeds[i:i+32] for i in range(0,len(seeds),32)]\n jobs=[(spec,chunk,reference,mode) for mode in modes for spec in (candidate,starter) for chunk in chunks]\n with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers,mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n  results=list(pool.map(evaluate_job,jobs))\n report={}\n for index,mode in enumerate(modes):\n  size=len(chunks);a=combine(results[(2*index)*size:(2*index+1)*size]);b=combine(results[(2*index+1)*size:(2*index+2)*size])\n  report[mode]={\'trained\':summarize(a),\'sdk_starter\':summarize(b),\'paired_gain\':paired_interval(a,b)}\n payload={\'model_training\':document[\'training\'],\'seed_start\':seeds[0],\'sampling_unit\':\'four-seat seed block\',\'evaluation\':report}\n args.output.write_text(json.dumps(payload,indent=2)+\'\\n\');print(json.dumps(payload),flush=True)\n\n\nif __name__==\'__main__\':main()\n'}
for name,text in sources.items(): (work/name).write_text(text)
sdk=Path("/kaggle/working/bluffSDK")
subprocess.run(["git","clone","--quiet","https://github.com/Sir-Nike/bluffSDK",str(sdk)],check=True)
subprocess.run(["git","-C",str(sdk),"checkout","--quiet","96da8042c285af4e4f43655f2ad99592bd49b8e6"],check=True)
os.environ["BLUFF_SDK"]=str(sdk)
(work/'linear.json').write_text('{\n  "format": "triad-bluff-linear-v2",\n  "features": [\n    "batch_size",\n    "truthful",\n    "bluff",\n    "bluff_pile",\n    "bluff_call_rate",\n    "bluff_finish",\n    "remaining_groups",\n    "remaining_largest_group",\n    "claimed_rank_held",\n    "bias",\n    "revealed_bluff_rate",\n    "rank_held",\n    "claim_count",\n    "pile_size",\n    "opponent_finishing",\n    "own_small_hand",\n    "opponent_small_hand",\n    "samples",\n    "bluff_any_call_probability",\n    "bluff_impossible_for_caller",\n    "truthful_turns_removed",\n    "bluff_known_pile_fraction",\n    "truthful_finish",\n    "bluff_no_impossibility_call_possible",\n    "placer_global_revealed_bluff_rate",\n    "known_outside_rank_fraction",\n    "known_pile_fraction",\n    "claim_fraction_of_original_hand",\n    "finishing_large_batch",\n    "known_placer_rank_before_claim"\n  ],\n  "weights": [\n    0.4755931924211193,\n    5.820676470779726,\n    -3.445920468550993,\n    0.5025301512894197,\n    -10.204982725387449,\n    -10.63672367105182,\n    1.9122839363624424,\n    6.692718730179745,\n    3.3398384247884447,\n    -3.967913294968817,\n    7.18093879068249,\n    -8.248647532051631,\n    -4.252488827029067,\n    -5.717390427387336,\n    -10.297751467694116,\n    -4.172529040138095,\n    3.643993344291045,\n    4.785033532440487,\n    -8.90940656773177,\n    -9.18268199666696,\n    3.5590525036076692,\n    -1.533025684746697,\n    5.957038223067108,\n    9.714173152584005,\n    3.5582377937169647,\n    -0.47803421898358756,\n    -1.2639464162072949,\n    0.6168380283723138,\n    2.861924827690982,\n    -7.115720155350792\n  ],\n  "training": {\n    "method": "archive-league cross-entropy policy optimization",\n    "seed": 20261016,\n    "generations": 30,\n    "population": 32,\n    "train_blocks": 8,\n    "train_horizon": 200,\n    "evaluation_horizon": 1000,\n    "validation_blocks": 64,\n    "selected_validation_points": 4.0859375,\n    "incumbent_validation_points": 3.62109375,\n    "sdk_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6"\n  }\n}\n')
(work/'previous.json').write_text('{\n  "format": "triad-bluff-linear-v1",\n  "features": [\n    "batch_size",\n    "truthful",\n    "bluff",\n    "bluff_pile",\n    "bluff_call_rate",\n    "bluff_finish",\n    "remaining_groups",\n    "remaining_largest_group",\n    "claimed_rank_held",\n    "bias",\n    "revealed_bluff_rate",\n    "rank_held",\n    "claim_count",\n    "pile_size",\n    "opponent_finishing",\n    "own_small_hand",\n    "opponent_small_hand",\n    "samples"\n  ],\n  "weights": [\n    4.351868833686492,\n    5.6322712663248415,\n    -2.6260873498626167,\n    0.7063932700439369,\n    -10.84185815552825,\n    -8.300750779890834,\n    -0.7486711333834275,\n    1.2260583122844886,\n    -1.5575700886723332,\n    -4.558178606791033,\n    8.25750430388175,\n    0.43873901259948533,\n    -1.3556124209301408,\n    -1.1333529834957075,\n    0.14766041336362645,\n    -1.5734330345566885,\n    -0.4730921181099686,\n    -0.5811911507935986\n  ],\n  "training": {\n    "method": "cross-entropy policy optimization",\n    "seed": 20261014,\n    "generations": 25,\n    "population": 32,\n    "games_per_candidate": 64,\n    "validation": 3.86328125,\n    "sdk_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6"\n  }\n}\n')
(work/'archive-v3.json').write_text('{\n  "format": "triad-bluff-linear-v2",\n  "features": [\n    "batch_size",\n    "truthful",\n    "bluff",\n    "bluff_pile",\n    "bluff_call_rate",\n    "bluff_finish",\n    "remaining_groups",\n    "remaining_largest_group",\n    "claimed_rank_held",\n    "bias",\n    "revealed_bluff_rate",\n    "rank_held",\n    "claim_count",\n    "pile_size",\n    "opponent_finishing",\n    "own_small_hand",\n    "opponent_small_hand",\n    "samples",\n    "bluff_any_call_probability",\n    "bluff_impossible_for_caller",\n    "truthful_turns_removed",\n    "bluff_known_pile_fraction",\n    "truthful_finish",\n    "bluff_no_impossibility_call_possible",\n    "placer_global_revealed_bluff_rate",\n    "known_outside_rank_fraction",\n    "known_pile_fraction",\n    "claim_fraction_of_original_hand",\n    "finishing_large_batch",\n    "known_placer_rank_before_claim"\n  ],\n  "weights": [\n    0.46906600062556736,\n    5.822992586451061,\n    -4.256151645404109,\n    0.11487356850784729,\n    -10.156848911305236,\n    -11.271340302891854,\n    3.3683578234527403,\n    6.2496185698779305,\n    3.1161980935637152,\n    -5.29939452905386,\n    6.593515895093171,\n    -7.663298336619744,\n    -3.343927012803054,\n    -5.123458812730883,\n    -11.107209975043212,\n    -3.3036174880859708,\n    2.1294430146208745,\n    4.554786083442725,\n    -9.228369107321132,\n    -9.767303499785623,\n    3.351150539292533,\n    -1.2999972896640095,\n    5.953639248627494,\n    11.752300427552342,\n    3.55635817044417,\n    -0.394400350462636,\n    -2.8038328960517234,\n    -0.23216308592893242,\n    1.7768406523782079,\n    -7.06630558838421\n  ],\n  "training": {\n    "method": "archive-league cross-entropy policy optimization",\n    "policy_kind": "v3",\n    "selection_seed_start": 25000000,\n    "test_seed_start": 26000000,\n    "train_seed_start": 24000000,\n    "seed": 20261016,\n    "generations": 40,\n    "population": 32,\n    "train_blocks": 16,\n    "train_horizon": 200,\n    "evaluation_horizon": 1000,\n    "validation_blocks": 128,\n    "selected_validation_points": 3.923828125,\n    "incumbent_validation_points": 3.884765625,\n    "sdk_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6"\n  }\n}\n')
(work/'archive-v4.json').write_text('{\n  "format": "triad-bluff-linear-v4",\n  "features": [\n    "batch_size",\n    "truthful",\n    "bluff",\n    "bluff_pile",\n    "bluff_call_rate",\n    "bluff_finish",\n    "remaining_groups",\n    "remaining_largest_group",\n    "claimed_rank_held",\n    "bias",\n    "revealed_bluff_rate",\n    "rank_held",\n    "claim_count",\n    "pile_size",\n    "opponent_finishing",\n    "own_small_hand",\n    "opponent_small_hand",\n    "samples",\n    "bluff_any_call_probability",\n    "bluff_impossible_for_caller",\n    "truthful_turns_removed",\n    "bluff_known_pile_fraction",\n    "truthful_finish",\n    "bluff_no_impossibility_call_possible",\n    "placer_global_revealed_bluff_rate",\n    "known_outside_rank_fraction",\n    "known_pile_fraction",\n    "claim_fraction_of_original_hand",\n    "finishing_large_batch",\n    "known_placer_rank_before_claim",\n    "bluff_call_probability_batch",\n    "bluff_call_probability_pile",\n    "bluff_call_probability_finish",\n    "bluff_nearest_opponent_finish",\n    "truthful_call_probability_batch",\n    "bluff_remaining_truthful_turns",\n    "global_revealed_bluff_finishing",\n    "count_revealed_bluff_pile",\n    "global_revealed_bluff_pile",\n    "count_revealed_bluff_small_own_hand",\n    "global_revealed_bluff_count",\n    "opponent_hand_pace_relative_to_own"\n  ],\n  "weights": [\n    0.7320463963289565,\n    3.7830344288585662,\n    -3.6643202745799996,\n    -0.6598575631829904,\n    -8.151139035254458,\n    -10.728635245656662,\n    5.53738880277373,\n    8.060614730559548,\n    3.980677726135072,\n    -6.3708300110301845,\n    8.820562181095452,\n    -11.194059164972861,\n    -2.2577537506142997,\n    -6.384975685154542,\n    -12.388615378209488,\n    -4.6687249187167374,\n    4.012963158958541,\n    3.5295502265837815,\n    -11.260811388804578,\n    -7.472099740678461,\n    3.5443459856372246,\n    -4.384346513374403,\n    4.442050737495265,\n    8.885460766633978,\n    1.169375994958378,\n    -0.5663349802695352,\n    -2.956319661396954,\n    -3.7975219601212844,\n    1.2373143583851205,\n    -9.91620089119303,\n    -2.3230584735194504,\n    -1.0326765648018956,\n    0.2687146698298245,\n    -1.498495087585903,\n    -5.241823377828466,\n    3.643585967235043,\n    0.8155443445928052,\n    1.0520029352791984,\n    -0.17768354106413486,\n    -1.7661001974864652,\n    -0.376138628965242,\n    1.5732760222813011\n  ],\n  "training": {\n    "method": "archive-league cross-entropy policy optimization",\n    "fast_observations": true,\n    "stress_opponents": true,\n    "policy_kind": "v4",\n    "selection_seed_start": 32000000,\n    "test_seed_start": 33000000,\n    "train_seed_start": 31000000,\n    "seed": 20261016,\n    "generations": 80,\n    "population": 48,\n    "train_blocks": 24,\n    "train_horizon": 400,\n    "evaluation_horizon": 1000,\n    "validation_blocks": 256,\n    "selected_validation_points": 4.126953125,\n    "incumbent_validation_points": 4.0615234375,\n    "sdk_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6"\n  }\n}\n')
(work/'archive-ranking.json').write_text('{\n  "format": "triad-bluff-residual-mlp-v1",\n  "features": [\n    "batch_size",\n    "truthful",\n    "bluff",\n    "bluff_pile",\n    "bluff_call_rate",\n    "bluff_finish",\n    "remaining_groups",\n    "remaining_largest_group",\n    "claimed_rank_held",\n    "bias",\n    "revealed_bluff_rate",\n    "rank_held",\n    "claim_count",\n    "pile_size",\n    "opponent_finishing",\n    "own_small_hand",\n    "opponent_small_hand",\n    "samples",\n    "bluff_any_call_probability",\n    "bluff_impossible_for_caller",\n    "truthful_turns_removed",\n    "bluff_known_pile_fraction",\n    "truthful_finish",\n    "bluff_no_impossibility_call_possible",\n    "placer_global_revealed_bluff_rate",\n    "known_outside_rank_fraction",\n    "known_pile_fraction",\n    "claim_fraction_of_original_hand",\n    "finishing_large_batch",\n    "known_placer_rank_before_claim",\n    "bluff_call_probability_batch",\n    "bluff_call_probability_pile",\n    "bluff_call_probability_finish",\n    "bluff_nearest_opponent_finish",\n    "truthful_call_probability_batch",\n    "bluff_remaining_truthful_turns",\n    "global_revealed_bluff_finishing",\n    "count_revealed_bluff_pile",\n    "global_revealed_bluff_pile",\n    "count_revealed_bluff_small_own_hand",\n    "global_revealed_bluff_count",\n    "opponent_hand_pace_relative_to_own",\n    "bluff_pooled_caller_probability",\n    "bluff_pooled_caller_uncertainty",\n    "bluff_specific_caller_uncertainty",\n    "bluff_sparse_own_outcomes",\n    "truthful_pooled_caller_probability_batch",\n    "bluff_pooled_caller_probability_pile",\n    "global_revealed_bluff_uncertainty",\n    "count_revealed_bluff_uncertainty"\n  ],\n  "base_weights": [\n    0.029724573716521263,\n    0.3637922704219818,\n    -0.2153700292110443,\n    0.03140813484787941,\n    -0.6378114223480225,\n    -0.6647952198982239,\n    0.1195177435874939,\n    0.41829490661621094,\n    0.20873990654945374,\n    -0.2479945868253708,\n    0.4488086700439453,\n    -0.5155404806137085,\n    -0.2657805383205414,\n    -0.3573369085788727,\n    -0.6436094641685486,\n    -0.2607830762863159,\n    0.22774958610534668,\n    0.29906460642814636,\n    -0.5568379163742065,\n    -0.5739176273345947,\n    0.22244077920913696,\n    -0.09581410884857178,\n    0.3723149001598358,\n    0.6071358323097229,\n    0.22238986194133759,\n    -0.029877139255404472,\n    -0.07899665087461472,\n    0.03855237737298012,\n    0.1788703054189682,\n    -0.44473251700401306,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0\n  ],\n  "feature_mean": [\n    2.1395998001098633,\n    0.23461349308490753,\n    0.5844796895980835,\n    0.1395501345396042,\n    0.212456613779068,\n    0.05323559418320656,\n    0.23304255306720734,\n    0.4180925786495209,\n    0.28125354647636414,\n    0.09045340120792389,\n    0.012826855294406414,\n    0.0059831151738762856,\n    0.05501168221235275,\n    0.032889094203710556,\n    0.0025440019089728594,\n    0.020031658932566643,\n    0.01566975563764572,\n    0.0028436535503715277,\n    0.290571004152298,\n    0.07750725746154785,\n    1.648437738418579,\n    0.02482268027961254,\n    0.007594316732138395,\n    0.08890815079212189,\n    0.016572389751672745,\n    0.008206761442124844,\n    0.008885364979505539,\n    0.02076456882059574,\n    0.0012484453618526459,\n    0.007570760790258646,\n    0.21950757503509521,\n    0.05633711814880371,\n    0.024478575214743614,\n    0.11529732495546341,\n    0.04431131109595299,\n    0.1452786922454834,\n    0.000323282991303131,\n    0.004547847434878349,\n    0.005558989476412535,\n    0.0030804327689111233,\n    0.010789883323013783,\n    -0.17579041421413422,\n    0.3107447326183319,\n    0.0922621339559555,\n    0.1262330412864685,\n    0.25931859016418457,\n    0.04479999840259552,\n    0.0446401983499527,\n    0.011760496534407139,\n    0.011893984861671925\n  ],\n  "feature_scale": [\n    1.3648862838745117,\n    0.4237686097621918,\n    0.49293312430381775,\n    0.31981444358825684,\n    0.22281301021575928,\n    0.22452004253864288,\n    0.20768232643604279,\n    0.2998815178871155,\n    0.28568950295448303,\n    0.2869352102279663,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.19105571508407593,\n    0.15773427486419678,\n    0.10000000149011612,\n    0.14014413952827454,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.26270508766174316,\n    0.2674539387226105,\n    1.3663667440414429,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.284549742937088,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.2118988186120987,\n    0.12224618345499039,\n    0.1045146957039833,\n    0.1629907488822937,\n    0.10000000149011612,\n    0.17951466143131256,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    1.656874656677246,\n    0.30376046895980835,\n    0.10000000149011612,\n    0.11033657938241959,\n    0.43835052847862244,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612,\n    0.10000000149011612\n  ],\n  "hidden_weight": [\n    [\n      -0.19202958047389984,\n      0.15125629305839539,\n      -0.10062059015035629,\n      0.19136938452720642,\n      0.26632729172706604,\n      0.16580453515052795,\n      0.39158135652542114,\n      0.4364096224308014,\n      -0.2203340232372284,\n      0.02731265500187874,\n      -0.021941421553492546,\n      0.004426548257470131,\n      0.04168559983372688,\n      -0.06025722995400429,\n      0.0010170531459152699,\n      -0.09620510041713715,\n      0.03122447244822979,\n      0.04913467541337013,\n      0.017747806385159492,\n      0.07794612646102905,\n      0.07376284152269363,\n      -0.6565521359443665,\n      0.22910697758197784,\n      -0.1319442242383957,\n      0.22352753579616547,\n      0.07389921694993973,\n      -0.021464945748448372,\n      -0.07632283121347427,\n      -0.0017953533679246902,\n      -0.15402238070964813,\n      -0.23365654051303864,\n      0.2789778709411621,\n      -0.048941243439912796,\n      0.3127325177192688,\n      0.1430678814649582,\n      -0.03163750469684601,\n      0.11275077611207962,\n      -0.04455846548080444,\n      -0.19067446887493134,\n      -0.02123730443418026,\n      0.15277132391929626,\n      -0.07499479502439499,\n      0.1938226819038391,\n      0.005214580800384283,\n      -0.16072870790958405,\n      0.4290143847465515,\n      -0.30015406012535095,\n      0.03683564066886902,\n      0.17921903729438782,\n      0.09538088738918304\n    ],\n    [\n      -0.3496174216270447,\n      -0.033204078674316406,\n      0.08492206782102585,\n      -0.14852312207221985,\n      0.24244695901870728,\n      -0.017229780554771423,\n      0.6605890393257141,\n      0.09992722421884537,\n      0.2635844349861145,\n      -0.1324535757303238,\n      -0.2760956585407257,\n      0.004355510231107473,\n      0.12184537947177887,\n      -0.0032640453428030014,\n      0.1219732016324997,\n      -0.11211785674095154,\n      -0.027204176411032677,\n      -0.41678473353385925,\n      0.26545149087905884,\n      -0.012706512585282326,\n      0.05574443191289902,\n      0.28975367546081543,\n      0.049956005066633224,\n      -0.5164822340011597,\n      -0.10123363882303238,\n      0.03145816922187805,\n      0.08416414260864258,\n      0.2687167227268219,\n      0.013803120702505112,\n      0.27541759610176086,\n      -0.3425333499908447,\n      0.1877971738576889,\n      0.17152805626392365,\n      -0.006548540201038122,\n      -0.13940852880477905,\n      0.1660059541463852,\n      -0.009708402678370476,\n      0.07059653103351593,\n      -0.1445803940296173,\n      -0.09821895509958267,\n      -0.19294242560863495,\n      0.028265466913580894,\n      0.31468135118484497,\n      -0.35061442852020264,\n      0.04629687964916229,\n      -0.021647922694683075,\n      -0.398400217294693,\n      -0.184276282787323,\n      -0.04051925987005234,\n      -0.076309435069561\n    ],\n    [\n      -0.44181036949157715,\n      -0.037182264029979706,\n      0.024050530046224594,\n      0.0012640011264011264,\n      -0.1819583624601364,\n      0.05889825522899628,\n      -0.05000065267086029,\n      0.17747391760349274,\n      0.35310065746307373,\n      0.16161592304706573,\n      0.08273530751466751,\n      0.11687079071998596,\n      -0.032777655869722366,\n      -0.10338082164525986,\n      -0.044549256563186646,\n      -0.08521641045808792,\n      0.10152506083250046,\n      0.42922788858413696,\n      0.24677245318889618,\n      0.275145560503006,\n      0.21537190675735474,\n      -0.06754159182310104,\n      0.10187806934118271,\n      -0.3514699637889862,\n      0.03684944659471512,\n      -0.011080002412199974,\n      -0.17129723727703094,\n      -0.07273056358098984,\n      0.012710507027804852,\n      0.0012870589271187782,\n      0.0401742048561573,\n      0.0069806226529181,\n      -0.14962238073349,\n      0.6312869191169739,\n      -0.019833341240882874,\n      0.07261871546506882,\n      -0.1414639800786972,\n      -0.014410188421607018,\n      0.0591629222035408,\n      -0.02386775054037571,\n      0.1051902174949646,\n      0.1169673427939415,\n      -0.1473882794380188,\n      0.2279590368270874,\n      -0.06350817531347275,\n      0.19910752773284912,\n      -0.13421642780303955,\n      0.04369090497493744,\n      -0.09575971961021423,\n      0.10184834897518158\n    ],\n    [\n      -0.8072140216827393,\n      0.0415179543197155,\n      0.25485357642173767,\n      -0.03188749775290489,\n      0.20593594014644623,\n      -0.3928118348121643,\n      -0.08975344151258469,\n      0.25190234184265137,\n      -0.2309490591287613,\n      -0.25593388080596924,\n      -0.3336438238620758,\n      -0.49595001339912415,\n      0.03515753895044327,\n      0.13897287845611572,\n      -0.5763046741485596,\n      -0.5007966160774231,\n      -0.17515550553798676,\n      0.08869050443172455,\n      0.2672591507434845,\n      -0.352481484413147,\n      -0.24845293164253235,\n      0.0012413441436365247,\n      0.10645341128110886,\n      0.049007534980773926,\n      -0.24921815097332,\n      -0.4750996530056,\n      0.48253777623176575,\n      -0.027838952839374542,\n      -0.38995078206062317,\n      -0.31328147649765015,\n      -0.3578946888446808,\n      -0.19243939220905304,\n      -0.5677703022956848,\n      -0.10083092004060745,\n      0.1341666430234909,\n      -0.11133559793233871,\n      -0.37774690985679626,\n      0.022098375484347343,\n      -0.13915301859378815,\n      -0.4281318783760071,\n      -0.2979890704154968,\n      0.17594309151172638,\n      0.3478923439979553,\n      0.13586987555027008,\n      0.13435274362564087,\n      -0.06514236330986023,\n      0.1479862779378891,\n      0.3291209638118744,\n      -0.15946771204471588,\n      -0.14842531085014343\n    ],\n    [\n      -0.4703312814235687,\n      -0.20633761584758759,\n      0.1900738924741745,\n      0.06791234016418457,\n      0.22767139971256256,\n      0.34287816286087036,\n      0.2780137062072754,\n      0.490080326795578,\n      0.23260803520679474,\n      0.026921316981315613,\n      0.1187470406293869,\n      -0.14169251918792725,\n      -0.08541760593652725,\n      0.12375392764806747,\n      0.05708643049001694,\n      -0.2624114453792572,\n      0.0009216552716679871,\n      -0.23335391283035278,\n      -0.033321600407361984,\n      0.40369388461112976,\n      0.24260860681533813,\n      -0.15176722407341003,\n      0.1375347226858139,\n      -0.06033790856599808,\n      0.1969938725233078,\n      0.33162984251976013,\n      -0.10945877432823181,\n      0.34658071398735046,\n      0.11664565652608871,\n      0.10710567981004715,\n      -0.36301830410957336,\n      0.27906519174575806,\n      0.09325948357582092,\n      0.29925283789634705,\n      0.033401526510715485,\n      0.18618789315223694,\n      -0.005079449154436588,\n      0.3451724946498871,\n      0.2539703845977783,\n      0.31034937500953674,\n      0.21049605309963226,\n      0.966443657875061,\n      -0.04152418300509453,\n      0.14757639169692993,\n      0.46571651101112366,\n      0.05958429351449013,\n      0.16158311069011688,\n      -0.25173884630203247,\n      0.18668290972709656,\n      0.14818932116031647\n    ],\n    [\n      -0.7125051617622375,\n      0.10390605032444,\n      -0.194485604763031,\n      0.04475666210055351,\n      0.37523153424263,\n      0.22402340173721313,\n      -0.45863744616508484,\n      0.298410564661026,\n      -0.03355305269360542,\n      0.20834043622016907,\n      0.12322287261486053,\n      0.025812890380620956,\n      0.033923838287591934,\n      -0.08452843129634857,\n      0.015942800790071487,\n      0.19076460599899292,\n      0.23285695910453796,\n      0.048181504011154175,\n      0.04661276191473007,\n      0.08323735743761063,\n      -0.17819494009017944,\n      -0.21954193711280823,\n      0.04759179800748825,\n      -0.42464905977249146,\n      0.10293136537075043,\n      -0.044991955161094666,\n      0.09410923719406128,\n      -0.0233745314180851,\n      -0.08290598541498184,\n      0.06586933881044388,\n      -0.1354251503944397,\n      0.05420057848095894,\n      0.30689606070518494,\n      -0.40738025307655334,\n      0.024982737377285957,\n      -0.3125905394554138,\n      0.10703320801258087,\n      -0.00779837416484952,\n      0.07616543024778366,\n      -0.020140083506703377,\n      0.06963186711072922,\n      -0.05971169099211693,\n      0.28734686970710754,\n      -0.1399356573820114,\n      0.4501485824584961,\n      0.43907997012138367,\n      -0.324989378452301,\n      -0.09979131817817688,\n      0.018416909500956535,\n      0.01727212220430374\n    ],\n    [\n      -0.035752419382333755,\n      0.04138004034757614,\n      -0.1337522566318512,\n      0.2738141715526581,\n      -0.18725088238716125,\n      0.15974830090999603,\n      0.24043655395507812,\n      0.6550118923187256,\n      -0.2806554138660431,\n      0.08063450455665588,\n      0.10613710433244705,\n      0.015439657494425774,\n      0.15130652487277985,\n      0.09189396351575851,\n      -0.12247393280267715,\n      0.2512091100215912,\n      0.07602106779813766,\n      0.29813095927238464,\n      -0.30767568945884705,\n      0.16544009745121002,\n      -0.27705997228622437,\n      -0.09668433666229248,\n      0.14222769439220428,\n      0.1231452226638794,\n      0.23493953049182892,\n      0.06945419311523438,\n      -0.07678230106830597,\n      0.11114998906850815,\n      0.059269193559885025,\n      0.13170039653778076,\n      -0.3228725492954254,\n      0.2168671041727066,\n      0.2696554958820343,\n      -0.16142678260803223,\n      0.5859753489494324,\n      0.09715444594621658,\n      0.002875487320125103,\n      0.031517673283815384,\n      -0.012696266174316406,\n      0.12384146451950073,\n      0.10011958330869675,\n      -0.23598912358283997,\n      -0.06339891999959946,\n      0.12267380952835083,\n      0.07089338451623917,\n      -0.039904601871967316,\n      0.3116064965724945,\n      0.17285385727882385,\n      0.34713345766067505,\n      0.10579388588666916\n    ],\n    [\n      -0.37698793411254883,\n      -0.13932669162750244,\n      0.1363975703716278,\n      0.21555204689502716,\n      0.7801560163497925,\n      -0.5264711976051331,\n      -0.6611112356185913,\n      0.008036970160901546,\n      0.03451893478631973,\n      0.0691150352358818,\n      0.09124680608510971,\n      0.20227183401584625,\n      -0.15124240517616272,\n      -0.10571960359811783,\n      -0.19745688140392303,\n      -0.3578275740146637,\n      0.0642014816403389,\n      0.09790126234292984,\n      -0.35584965348243713,\n      -0.15276634693145752,\n      -0.34679844975471497,\n      -0.461421936750412,\n      0.0122278006747365,\n      0.06871720403432846,\n      0.07319998741149902,\n      0.06467419862747192,\n      -0.02730027213692665,\n      0.0177640188485384,\n      0.009762308560311794,\n      0.18767277896404266,\n      -0.25133925676345825,\n      0.004508877173066139,\n      -0.4527457356452942,\n      0.5941516757011414,\n      0.09652741998434067,\n      0.1230541467666626,\n      -0.19630247354507446,\n      0.03672018647193909,\n      0.006273719482123852,\n      -0.27636194229125977,\n      0.10056620836257935,\n      -0.4671321213245392,\n      -0.2727147340774536,\n      0.11487728357315063,\n      -0.21639122068881989,\n      -0.08801549673080444,\n      0.17564719915390015,\n      0.3868160545825958,\n      -0.11838560551404953,\n      0.10246492177248001\n    ],\n    [\n      -0.1574113816022873,\n      -0.023291047662496567,\n      0.04730834811925888,\n      0.010046691633760929,\n      0.11161240190267563,\n      -0.01599869318306446,\n      -0.06460732221603394,\n      -0.0318559855222702,\n      0.031422652304172516,\n      0.09394171833992004,\n      -0.11481371521949768,\n      -0.15526078641414642,\n      -0.023349516093730927,\n      -0.16090510785579681,\n      0.05545642971992493,\n      0.2415606677532196,\n      -0.08094839006662369,\n      0.4150295555591583,\n      0.03846528381109238,\n      0.15705974400043488,\n      -0.10539405792951584,\n      0.05836347118020058,\n      -0.07604450732469559,\n      -0.30944785475730896,\n      0.1665249615907669,\n      -0.04150903970003128,\n      -0.05239039659500122,\n      -0.08471309393644333,\n      0.121266208589077,\n      -0.0014270353130996227,\n      -0.058652620762586594,\n      -0.02062084712088108,\n      0.06442729383707047,\n      0.1074879914522171,\n      -0.09955383092164993,\n      0.12186262011528015,\n      -0.14621612429618835,\n      0.04299991577863693,\n      0.04783204570412636,\n      0.15346753597259521,\n      0.014543632045388222,\n      0.18142564594745636,\n      -0.03303435444831848,\n      -0.003854575101286173,\n      -0.19295023381710052,\n      -0.03655415400862694,\n      -0.04338475316762924,\n      0.10026992112398148,\n      0.007083422504365444,\n      -0.1050834059715271\n    ],\n    [\n      -0.23585237562656403,\n      -0.12744158506393433,\n      0.012418247759342194,\n      0.08363228291273117,\n      -0.04309118166565895,\n      0.22508226335048676,\n      0.20685668289661407,\n      0.26138344407081604,\n      0.307697594165802,\n      0.08421118557453156,\n      0.04421668499708176,\n      -0.015539539977908134,\n      0.007419257890433073,\n      0.18849684298038483,\n      -0.26119303703308105,\n      -0.49621549248695374,\n      0.03169839456677437,\n      -0.3552597463130951,\n      0.11708559840917587,\n      -0.11115245521068573,\n      0.2439606934785843,\n      -0.25279805064201355,\n      0.05636773258447647,\n      -0.3035143315792084,\n      0.014462308958172798,\n      -0.00023079095990397036,\n      0.1388983428478241,\n      0.12865278124809265,\n      -0.12762616574764252,\n      0.005441761109977961,\n      -0.15054699778556824,\n      -0.05141157656908035,\n      0.1790742725133896,\n      0.5898136496543884,\n      -0.009311581030488014,\n      0.040430307388305664,\n      -0.1642482876777649,\n      0.1801123023033142,\n      0.03890753909945488,\n      -0.14680218696594238,\n      0.06145242601633072,\n      1.7709522247314453,\n      -0.04212886840105057,\n      -0.15215317904949188,\n      -0.24795854091644287,\n      0.19612069427967072,\n      -0.1281392127275467,\n      0.26828792691230774,\n      0.23018144071102142,\n      0.10932588577270508\n    ],\n    [\n      0.3348250985145569,\n      -0.18283122777938843,\n      0.07617422938346863,\n      0.22960039973258972,\n      0.3661072254180908,\n      0.20202983915805817,\n      0.3252094089984894,\n      -0.8164780735969543,\n      0.10693570971488953,\n      -0.03622877225279808,\n      -0.0836518257856369,\n      0.19181115925312042,\n      0.22558051347732544,\n      -0.025103483349084854,\n      -0.21364426612854004,\n      0.04919802397489548,\n      -0.02714921161532402,\n      -0.6437059640884399,\n      0.10068847984075546,\n      0.25471022725105286,\n      0.2781190574169159,\n      -0.2091243714094162,\n      0.011492776684463024,\n      -0.017406295984983444,\n      -0.09150644391775131,\n      0.1643635779619217,\n      -0.02910521812736988,\n      0.11285676062107086,\n      -0.033838752657175064,\n      -0.22987478971481323,\n      0.1301073282957077,\n      0.11931592971086502,\n      0.025617925450205803,\n      -0.2182742804288864,\n      -0.18915890157222748,\n      0.3615805506706238,\n      -0.10269317030906677,\n      0.15696042776107788,\n      -0.32188767194747925,\n      -0.09561910480260849,\n      0.060371577739715576,\n      0.13910731673240662,\n      -0.8219180107116699,\n      -0.2580358386039734,\n      0.3703741729259491,\n      -0.4993675947189331,\n      -0.04419863224029541,\n      0.29896190762519836,\n      -0.026430204510688782,\n      0.008099472150206566\n    ],\n    [\n      -0.02974196895956993,\n      -0.3326263129711151,\n      0.1647481620311737,\n      -0.010314425453543663,\n      0.04047681763768196,\n      0.11611238867044449,\n      0.044334352016448975,\n      0.02860240638256073,\n      0.022992627695202827,\n      -0.01952977105975151,\n      -0.16761919856071472,\n      -0.09857246279716492,\n      -0.23320281505584717,\n      0.14091981947422028,\n      -0.2308560311794281,\n      -0.2733481228351593,\n      -0.06087779253721237,\n      0.06693939119577408,\n      0.17333954572677612,\n      0.1657271385192871,\n      -0.13418972492218018,\n      0.49474406242370605,\n      -0.14715692400932312,\n      -0.312465637922287,\n      -0.20615436136722565,\n      -0.21729524433612823,\n      0.20773786306381226,\n      -0.3822798728942871,\n      -0.04398162290453911,\n      -0.1281927078962326,\n      0.007536695338785648,\n      -0.6053846478462219,\n      -0.05449367314577103,\n      -0.08574168384075165,\n      -0.6154975295066833,\n      0.24029196798801422,\n      -0.2639612555503845,\n      0.13328424096107483,\n      -0.06009092554450035,\n      -0.1239374503493309,\n      -0.3297577500343323,\n      0.17891216278076172,\n      0.23084509372711182,\n      -0.09761741012334824,\n      -0.3517642319202423,\n      -0.19414548575878143,\n      -0.26442575454711914,\n      -0.056469522416591644,\n      -0.10573189705610275,\n      -0.14346741139888763\n    ],\n    [\n      -0.2693448066711426,\n      0.08634926378726959,\n      0.008070897310972214,\n      -0.2684096395969391,\n      0.19476810097694397,\n      -0.04464145377278328,\n      -0.22594881057739258,\n      -0.7943860292434692,\n      0.3539235293865204,\n      -0.05916186422109604,\n      0.08153904974460602,\n      -0.007255932316184044,\n      -0.10079783946275711,\n      0.057758934795856476,\n      -0.0735824853181839,\n      -0.30621540546417236,\n      0.017771027982234955,\n      -0.08869937062263489,\n      -0.005812968593090773,\n      -0.27144044637680054,\n      -0.09996244311332703,\n      -0.33106791973114014,\n      -0.06385211646556854,\n      -0.12285376340150833,\n      -0.08270307630300522,\n      -0.046221524477005005,\n      -0.09148037433624268,\n      0.07982458919286728,\n      0.03203823044896126,\n      0.016645997762680054,\n      0.03970375284552574,\n      0.20412880182266235,\n      0.11975476145744324,\n      0.15956144034862518,\n      -0.02168131247162819,\n      -0.51045161485672,\n      -0.09138567000627518,\n      -0.004584163893014193,\n      0.05923668295145035,\n      -0.29008257389068604,\n      0.10724376887083054,\n      -0.12970346212387085,\n      -0.23237860202789307,\n      0.05910341441631317,\n      0.20426706969738007,\n      -0.10093513876199722,\n      0.22740091383457184,\n      0.09849297255277634,\n      -0.24182625114917755,\n      -0.07155266404151917\n    ],\n    [\n      -0.3225018382072449,\n      -0.04762255772948265,\n      -0.10957353562116623,\n      0.13319268822669983,\n      -0.17298497259616852,\n      0.1077594980597496,\n      0.14437638223171234,\n      -0.20987685024738312,\n      -0.48280054330825806,\n      -0.0852629691362381,\n      -0.2530125081539154,\n      -0.3614713251590729,\n      -0.05385148525238037,\n      -0.10216335207223892,\n      -0.22220778465270996,\n      0.3658044636249542,\n      0.0799582377076149,\n      0.15970627963542938,\n      0.07897891849279404,\n      0.29231953620910645,\n      -0.04558447375893593,\n      -0.18729595839977264,\n      -0.1059475988149643,\n      -0.3644512891769409,\n      -0.4145146608352661,\n      -0.056077029556035995,\n      -0.14889715611934662,\n      -0.11386126279830933,\n      -0.4162250757217407,\n      -0.043115030974149704,\n      -0.08218993991613388,\n      -0.02164817787706852,\n      0.010626472532749176,\n      0.3074485659599304,\n      0.09666881710290909,\n      0.380744069814682,\n      -0.3396758437156677,\n      -0.25534388422966003,\n      -0.4077712297439575,\n      0.06318295001983643,\n      -0.32920631766319275,\n      -0.13388368487358093,\n      -0.00998294074088335,\n      0.07424140721559525,\n      0.05949144810438156,\n      -0.2540847361087799,\n      0.04241947829723358,\n      0.019476082175970078,\n      -0.2556335926055908,\n      -0.15605509281158447\n    ],\n    [\n      0.6390224099159241,\n      0.04027320072054863,\n      -0.07253879308700562,\n      0.07125227153301239,\n      -0.07943342626094818,\n      0.04650329053401947,\n      0.5723587274551392,\n      0.23380215466022491,\n      0.43154025077819824,\n      0.006312540266662836,\n      0.06210228055715561,\n      -0.2050963193178177,\n      0.03479475900530815,\n      -0.02254035510122776,\n      0.08456797897815704,\n      0.08675093203783035,\n      0.10751130431890488,\n      -0.08908049017190933,\n      -0.4440874457359314,\n      -0.3069297671318054,\n      0.34359899163246155,\n      -0.272480845451355,\n      0.0885443314909935,\n      0.3405865430831909,\n      0.2651411294937134,\n      0.4373738169670105,\n      0.3549002408981323,\n      0.08648732304573059,\n      0.14063279330730438,\n      -0.06875208020210266,\n      0.15909093618392944,\n      0.04063528776168823,\n      0.05050201714038849,\n      0.11834593862295151,\n      -0.15896445512771606,\n      -0.2111312747001648,\n      0.04089963436126709,\n      -0.06695489585399628,\n      0.12484778463840485,\n      0.03442877158522606,\n      0.15138018131256104,\n      -0.5691663026809692,\n      -0.20928575098514557,\n      0.008606968447566032,\n      -0.06405515968799591,\n      -0.3298901915550232,\n      0.03272620216012001,\n      0.0718257948756218,\n      0.05229468643665314,\n      0.10798918455839157\n    ],\n    [\n      -0.491869181394577,\n      -0.28201091289520264,\n      0.18722136318683624,\n      -0.06772541254758835,\n      0.19725312292575836,\n      -0.2898535132408142,\n      0.32033488154411316,\n      -0.5915459394454956,\n      0.47859618067741394,\n      0.102713443338871,\n      0.14594915509223938,\n      0.10385118424892426,\n      -0.31587648391723633,\n      -0.16959750652313232,\n      -0.1857483834028244,\n      -0.5293170213699341,\n      -0.09148602187633514,\n      0.9138979911804199,\n      0.055135760456323624,\n      -0.047678906470537186,\n      -0.0790688544511795,\n      -0.08538047969341278,\n      -0.10432614386081696,\n      -0.17596939206123352,\n      -0.0110703120008111,\n      -0.2346513569355011,\n      0.6007078886032104,\n      -0.4091869592666626,\n      -0.10134308785200119,\n      0.08310243487358093,\n      -0.33455008268356323,\n      0.2771172523498535,\n      -0.03541961684823036,\n      -0.19754666090011597,\n      -0.21528184413909912,\n      0.13968715071678162,\n      -0.15148119628429413,\n      -0.11658339947462082,\n      -0.3361017405986786,\n      -0.3612913191318512,\n      -0.09712904691696167,\n      0.04137229919433594,\n      -0.09592539072036743,\n      -0.24035996198654175,\n      0.1065882220864296,\n      0.10164756327867508,\n      -0.4884163439273834,\n      -0.13870614767074585,\n      -0.027438627555966377,\n      0.026141872629523277\n    ],\n    [\n      0.07286754995584488,\n      0.033862899988889694,\n      -0.19253945350646973,\n      0.2913863956928253,\n      0.13169725239276886,\n      0.11510128527879715,\n      0.0933607667684555,\n      0.3138195276260376,\n      -0.13243186473846436,\n      0.2422548234462738,\n      0.276653528213501,\n      -0.03847532346844673,\n      0.3336777687072754,\n      0.05701414495706558,\n      0.09893937408924103,\n      0.16475585103034973,\n      0.029836229979991913,\n      0.2778482437133789,\n      -0.34936758875846863,\n      0.09510138630867004,\n      0.4257201552391052,\n      0.028854133561253548,\n      -0.10380593687295914,\n      0.053038425743579865,\n      0.22193656861782074,\n      -0.0042461915872991085,\n      0.030554668977856636,\n      0.16198888421058655,\n      -0.035144224762916565,\n      0.05954234302043915,\n      -0.6244704127311707,\n      0.18219521641731262,\n      0.2002418488264084,\n      -0.48137781023979187,\n      -0.19433249533176422,\n      -0.37679848074913025,\n      -0.005100379232317209,\n      0.22080577909946442,\n      0.07437482476234436,\n      0.16885313391685486,\n      0.27040478587150574,\n      -0.5074959397315979,\n      0.17990642786026,\n      0.2769649624824524,\n      0.05152503401041031,\n      0.051985349506139755,\n      0.4129163324832916,\n      0.24487924575805664,\n      0.14120706915855408,\n      0.2307230830192566\n    ],\n    [\n      -0.28624361753463745,\n      -0.30115148425102234,\n      -0.012694929726421833,\n      0.1387231945991516,\n      0.4286758601665497,\n      0.12692229449748993,\n      0.11035708338022232,\n      -0.619459867477417,\n      0.03378047049045563,\n      -0.14750291407108307,\n      -0.022354193031787872,\n      -0.04214084893465042,\n      -0.0415353886783123,\n      -0.03186623379588127,\n      0.058338381350040436,\n      -0.08181926608085632,\n      -0.16142955422401428,\n      -0.12001245468854904,\n      0.3380204141139984,\n      0.003465621965005994,\n      -0.36985522508621216,\n      -0.22993430495262146,\n      -0.06682021915912628,\n      -0.7731394171714783,\n      -0.10942716896533966,\n      -0.10159999877214432,\n      -0.10182453691959381,\n      -0.23560519516468048,\n      -0.14857430756092072,\n      0.0991126224398613,\n      0.0844532698392868,\n      0.12584654986858368,\n      -0.10633882135152817,\n      0.010240205563604832,\n      -0.0013407535152509809,\n      0.20805864036083221,\n      0.03138815984129906,\n      -0.18153460323810577,\n      -0.03544573113322258,\n      -0.20842579007148743,\n      -0.1990332305431366,\n      0.02942427434027195,\n      0.08941752463579178,\n      -0.29164257645606995,\n      -0.05479874834418297,\n      0.07783886790275574,\n      0.11599145084619522,\n      0.11354754865169525,\n      -0.11951977014541626,\n      -0.007311002351343632\n    ],\n    [\n      -0.31642815470695496,\n      0.02299356646835804,\n      -0.2362602800130844,\n      0.19294442236423492,\n      -0.3963755965232849,\n      0.37425100803375244,\n      0.20514032244682312,\n      0.21957765519618988,\n      -0.3362193703651428,\n      0.03631548956036568,\n      0.12130099534988403,\n      0.04169800877571106,\n      0.2558051645755768,\n      -0.03753611817955971,\n      0.1514752060174942,\n      -0.013219548389315605,\n      0.17202650010585785,\n      0.10219906270503998,\n      -0.43784651160240173,\n      0.21528437733650208,\n      -0.1388106495141983,\n      0.0338115394115448,\n      0.0718521997332573,\n      -0.4223823547363281,\n      0.19151411950588226,\n      0.06156781688332558,\n      -0.09438305348157883,\n      -0.032099466770887375,\n      0.12792031466960907,\n      -0.09593029320240021,\n      -0.14070561528205872,\n      0.21440736949443817,\n      0.2312585413455963,\n      -0.08131906390190125,\n      -0.08770885318517685,\n      -0.09653178602457047,\n      -0.11562037467956543,\n      0.21669070422649384,\n      0.2042226493358612,\n      0.10017234086990356,\n      0.0006136708543635905,\n      0.4325507581233978,\n      -0.24258072674274445,\n      -0.17294526100158691,\n      -0.013003125786781311,\n      0.195865198969841,\n      -0.061512336134910583,\n      0.09324324131011963,\n      0.1519479602575302,\n      0.11566980183124542\n    ],\n    [\n      -0.49828293919563293,\n      -0.3679612874984741,\n      0.12620051205158234,\n      -0.028123287484049797,\n      0.17352886497974396,\n      -0.1413561999797821,\n      0.20805388689041138,\n      -0.17863255739212036,\n      0.4204421639442444,\n      -0.1188741996884346,\n      -0.129194974899292,\n      -0.04541458934545517,\n      0.164279967546463,\n      -0.1619877964258194,\n      -0.04632454365491867,\n      -0.012672634795308113,\n      -0.13659162819385529,\n      -1.0188244581222534,\n      0.3739631175994873,\n      -0.19627021253108978,\n      0.0807306244969368,\n      0.1309960037469864,\n      -0.027644991874694824,\n      -0.2706710994243622,\n      -0.004682627506554127,\n      -0.030437076464295387,\n      -0.015656590461730957,\n      0.21033556759357452,\n      -0.1351899802684784,\n      -0.2936747372150421,\n      -0.439717561006546,\n      0.14963743090629578,\n      -0.22791548073291779,\n      -0.2977641820907593,\n      -0.002648756606504321,\n      0.11589837819337845,\n      -0.13817018270492554,\n      -0.11875613778829575,\n      -0.05769561976194382,\n      0.12040862441062927,\n      0.00770455552265048,\n      0.12820078432559967,\n      -0.208010733127594,\n      -0.1579517424106598,\n      0.3553411662578583,\n      -0.30912479758262634,\n      -0.12986424565315247,\n      -0.41573572158813477,\n      -0.1754993051290512,\n      -0.25732412934303284\n    ],\n    [\n      -0.5889866948127747,\n      0.022347690537571907,\n      -0.2853038012981415,\n      0.1920410543680191,\n      -0.2617713212966919,\n      -0.5179654955863953,\n      0.16290953755378723,\n      -0.5766183137893677,\n      0.26073157787323,\n      0.16202455759048462,\n      0.15952976047992706,\n      -0.4202682077884674,\n      -0.06313320994377136,\n      0.007500753737986088,\n      -0.14205460250377655,\n      0.14177697896957397,\n      0.10434304177761078,\n      0.33649593591690063,\n      0.006157818716019392,\n      0.39248237013816833,\n      -0.11903931200504303,\n      0.2969197630882263,\n      -0.1365073174238205,\n      -0.04290064424276352,\n      0.33930695056915283,\n      -0.3059912323951721,\n      -0.26080235838890076,\n      -0.08195023238658905,\n      -0.2389908730983734,\n      0.2225254625082016,\n      -0.020095668733119965,\n      0.0793214812874794,\n      -0.21756179630756378,\n      0.45377299189567566,\n      -0.23883163928985596,\n      -0.2405686378479004,\n      -0.4278993308544159,\n      -0.19001170992851257,\n      -0.06070360913872719,\n      0.01738264597952366,\n      -0.032029006630182266,\n      -0.5436313152313232,\n      0.732271134853363,\n      -0.35473912954330444,\n      0.04085606709122658,\n      0.1090874969959259,\n      0.37809082865715027,\n      0.4338505268096924,\n      0.2428039014339447,\n      0.15285803377628326\n    ],\n    [\n      0.32782307267189026,\n      0.06805059313774109,\n      0.164974182844162,\n      0.14158117771148682,\n      -0.11635490506887436,\n      -0.19637177884578705,\n      0.24395506083965302,\n      -0.5414725542068481,\n      0.04529007524251938,\n      -0.05476704239845276,\n      0.0914592370390892,\n      -0.03039310686290264,\n      -0.21819594502449036,\n      -0.2204119861125946,\n      0.24239540100097656,\n      -0.19282439351081848,\n      0.15428747236728668,\n      -1.0806773900985718,\n      -0.1393842101097107,\n      -0.28230562806129456,\n      0.09717746078968048,\n      0.576619029045105,\n      0.019800182431936264,\n      -0.29769036173820496,\n      0.029294926673173904,\n      0.30035170912742615,\n      0.19895866513252258,\n      -0.058695610612630844,\n      -0.013687225058674812,\n      0.09398889541625977,\n      -0.0004477549809962511,\n      0.04694584384560585,\n      -0.23674467206001282,\n      -0.42084288597106934,\n      -0.2173892706632614,\n      0.4112442433834076,\n      0.3221066892147064,\n      -0.03893650323152542,\n      0.18594510853290558,\n      -0.13303305208683014,\n      -0.03647095710039139,\n      0.12323474884033203,\n      -0.1367596536874771,\n      0.11072687059640884,\n      0.3003343641757965,\n      -0.043354254215955734,\n      -0.06701420247554779,\n      0.08294575661420822,\n      -0.058195117861032486,\n      0.07449038326740265\n    ],\n    [\n      -0.5274540185928345,\n      0.0008424089173786342,\n      -0.03786041960120201,\n      0.20802471041679382,\n      0.23122110962867737,\n      0.04570475593209267,\n      0.5382443070411682,\n      0.2421300709247589,\n      0.10757364332675934,\n      0.1445462554693222,\n      0.09491016715765,\n      0.18101878464221954,\n      0.08369506895542145,\n      0.11288034915924072,\n      -0.04997217655181885,\n      0.1543227732181549,\n      0.03232493996620178,\n      0.08581267297267914,\n      -0.0780368372797966,\n      0.18868184089660645,\n      -0.05946898087859154,\n      0.03136714547872543,\n      0.11256765574216843,\n      -0.5110180974006653,\n      0.10215844959020615,\n      0.031690146774053574,\n      -0.12930142879486084,\n      0.03609846532344818,\n      -0.02158038690686226,\n      0.051236532628536224,\n      -0.3602043092250824,\n      0.3114044964313507,\n      0.03166310116648674,\n      -0.21081118285655975,\n      0.022850561887025833,\n      -0.02768973633646965,\n      0.05107489600777626,\n      0.12503203749656677,\n      0.02067023143172264,\n      0.1583503931760788,\n      0.07459188997745514,\n      -0.1422303467988968,\n      -0.13173282146453857,\n      0.08350452780723572,\n      0.02853366918861866,\n      0.2560180425643921,\n      0.06665043532848358,\n      -0.0393829308450222,\n      0.16417503356933594,\n      0.03753133490681648\n    ],\n    [\n      -0.261016845703125,\n      0.024807047098875046,\n      0.05647578462958336,\n      -0.038194119930267334,\n      0.33809322118759155,\n      0.0058400859124958515,\n      0.15075398981571198,\n      -0.769255518913269,\n      0.32690465450286865,\n      -0.04608921706676483,\n      -0.00858503021299839,\n      -0.0998515784740448,\n      -0.15798179805278778,\n      -0.04737548157572746,\n      -0.12248252332210541,\n      -0.1795286387205124,\n      -0.09450267255306244,\n      0.09489660710096359,\n      0.18453851342201233,\n      0.1102704405784607,\n      -0.29388442635536194,\n      -0.33180755376815796,\n      -0.19467265903949738,\n      -0.0666903704404831,\n      -0.03837069496512413,\n      -0.02239038050174713,\n      -0.07939201593399048,\n      0.040755368769168854,\n      -0.04678730294108391,\n      -0.01703268103301525,\n      0.20496462285518646,\n      0.2020181119441986,\n      -0.12239402532577515,\n      0.5147801637649536,\n      -0.09148281067609787,\n      0.1324116587638855,\n      -0.05574595183134079,\n      -0.08588626980781555,\n      0.07564660161733627,\n      0.09508813172578812,\n      -0.07919439673423767,\n      0.12740088999271393,\n      -0.13756464421749115,\n      -0.08790223300457001,\n      0.18223954737186432,\n      0.014516650699079037,\n      0.2951229512691498,\n      0.15746422111988068,\n      -0.0012312111211940646,\n      -0.01717819832265377\n    ],\n    [\n      -0.04972575232386589,\n      -0.24363544583320618,\n      -0.09232550859451294,\n      0.0075925844721496105,\n      0.18644389510154724,\n      0.014780901372432709,\n      -0.10244834423065186,\n      -0.4580252170562744,\n      -0.09054479002952576,\n      -0.02345651015639305,\n      0.00871738325804472,\n      -0.2401668131351471,\n      -0.2995228171348572,\n      0.033403053879737854,\n      0.08899916708469391,\n      0.07311347872018814,\n      0.22819466888904572,\n      0.8446778059005737,\n      -0.003050765721127391,\n      -0.0703945904970169,\n      0.005050210747867823,\n      -0.47813647985458374,\n      -0.018671231344342232,\n      -0.6352458000183105,\n      -0.10394883900880814,\n      -0.23606987297534943,\n      -0.04755371809005737,\n      -0.3163021504878998,\n      0.0927969366312027,\n      -0.1444014608860016,\n      -0.15529879927635193,\n      0.12507364153862,\n      -0.023921415209770203,\n      0.20518258213996887,\n      -0.4144035577774048,\n      0.039500508457422256,\n      -0.2587064206600189,\n      -0.10681041330099106,\n      -0.27944910526275635,\n      -0.012667844071984291,\n      -0.04578428342938423,\n      0.11939243227243423,\n      0.002906305016949773,\n      -0.2608352601528168,\n      -0.14843575656414032,\n      0.20016378164291382,\n      -0.041379205882549286,\n      0.35504114627838135,\n      -0.16476699709892273,\n      0.10348972678184509\n    ],\n    [\n      -1.0341516733169556,\n      0.02665666863322258,\n      0.09478233754634857,\n      0.2962835133075714,\n      0.1908331960439682,\n      0.12934879958629608,\n      0.06857629120349884,\n      0.07498569041490555,\n      0.2019418627023697,\n      0.11029865592718124,\n      0.17000837624073029,\n      -0.4611847996711731,\n      0.08915544301271439,\n      -0.35103854537010193,\n      0.04933488368988037,\n      -0.04510995373129845,\n      -0.13147708773612976,\n      -0.04960452765226364,\n      0.14614497125148773,\n      0.5034998655319214,\n      0.2058747559785843,\n      0.049444835633039474,\n      -0.04659866914153099,\n      0.19822745025157928,\n      -0.049859073013067245,\n      0.19830238819122314,\n      -0.13150238990783691,\n      -0.1400122493505478,\n      -0.07341067492961884,\n      -0.14692482352256775,\n      -0.3269965648651123,\n      -0.18324625492095947,\n      0.2806042432785034,\n      -0.24424055218696594,\n      0.012850800529122353,\n      0.30601775646209717,\n      0.026300951838493347,\n      -0.06482067704200745,\n      -0.15643450617790222,\n      0.08075056970119476,\n      -0.15444286167621613,\n      0.16166044771671295,\n      0.2664753496646881,\n      -0.19517827033996582,\n      0.12971378862857819,\n      -0.17971952259540558,\n      -0.13540828227996826,\n      0.24183572828769684,\n      -0.054356083273887634,\n      -0.08053381741046906\n    ],\n    [\n      0.21856734156608582,\n      -0.19569134712219238,\n      0.1932661384344101,\n      -0.7120504975318909,\n      -0.2611786127090454,\n      0.14662888646125793,\n      0.16870395839214325,\n      0.050919100642204285,\n      -0.43599051237106323,\n      -0.01934799924492836,\n      -0.20496296882629395,\n      -0.27425745129585266,\n      -0.12715455889701843,\n      -0.042450375854969025,\n      -0.20894964039325714,\n      -0.2886899411678314,\n      -0.11864986270666122,\n      -0.22503788769245148,\n      0.416649729013443,\n      -0.02681145630776882,\n      -0.19657938182353973,\n      0.36040639877319336,\n      -0.05886327475309372,\n      0.0469992570579052,\n      -0.27169284224510193,\n      -0.13051527738571167,\n      -0.1766655296087265,\n      -0.25108131766319275,\n      -0.11183443665504456,\n      -0.2144920825958252,\n      0.46550852060317993,\n      -0.21205151081085205,\n      -0.03737577423453331,\n      0.22859391570091248,\n      -0.16448211669921875,\n      0.25836381316185,\n      -0.01632564887404442,\n      0.014032711274921894,\n      -0.20321834087371826,\n      -0.026093054562807083,\n      -0.20576433837413788,\n      0.2915514409542084,\n      -0.38074707984924316,\n      0.174935981631279,\n      -0.03453240916132927,\n      0.1508437991142273,\n      -0.05093148723244667,\n      -0.08195150643587112,\n      -0.0025741150602698326,\n      -0.16669988632202148\n    ],\n    [\n      0.2332446426153183,\n      -0.06864478439092636,\n      0.1714843511581421,\n      0.29004964232444763,\n      -0.1934589296579361,\n      0.04819100722670555,\n      0.2817428708076477,\n      0.02499600499868393,\n      -0.2759215235710144,\n      -0.049062591046094894,\n      -0.0011683930642902851,\n      -0.36487382650375366,\n      0.17626969516277313,\n      -0.337523490190506,\n      0.25553280115127563,\n      -0.236192986369133,\n      0.0665774717926979,\n      -0.41052478551864624,\n      -0.4100237190723419,\n      -0.019987458363175392,\n      0.3328666687011719,\n      -0.0759248211979866,\n      0.025317898020148277,\n      -0.6418779492378235,\n      -0.013546440750360489,\n      0.06632640957832336,\n      -0.3221264183521271,\n      0.2857840955257416,\n      -0.14754991233348846,\n      -0.03256852552294731,\n      -0.1443142592906952,\n      0.09014347195625305,\n      0.1633278727531433,\n      -0.18079210817813873,\n      -0.17719636857509613,\n      -0.07165199518203735,\n      0.2840081751346588,\n      -0.20591585338115692,\n      -0.2587043344974518,\n      -0.15888836979866028,\n      0.21626555919647217,\n      0.38320526480674744,\n      -0.6337767839431763,\n      0.25216415524482727,\n      0.407599538564682,\n      -0.07987558096647263,\n      -0.17660130560398102,\n      0.10293310135602951,\n      0.054722681641578674,\n      0.006313395220786333\n    ],\n    [\n      0.10293683409690857,\n      0.09244608879089355,\n      -0.050596028566360474,\n      -0.6368060111999512,\n      -0.009728374890983105,\n      0.22210001945495605,\n      0.4731529653072357,\n      -0.5407556295394897,\n      -0.45591017603874207,\n      -0.03582585230469704,\n      -0.034196190536022186,\n      -0.3148801326751709,\n      0.19140876829624176,\n      -0.31478726863861084,\n      -0.1983237862586975,\n      0.2532680630683899,\n      -0.14915519952774048,\n      -0.1021757423877716,\n      -0.011906340718269348,\n      0.12707871198654175,\n      -0.03453279286623001,\n      -0.10986010730266571,\n      0.21013735234737396,\n      0.196903795003891,\n      0.023383690044283867,\n      -0.024683650583028793,\n      -0.2989889085292816,\n      0.098348468542099,\n      -0.17126165330410004,\n      0.03279683366417885,\n      -0.28496959805488586,\n      -0.03776625543832779,\n      -0.20988790690898895,\n      -0.682373583316803,\n      -0.32984036207199097,\n      0.17175011336803436,\n      0.4668349623680115,\n      -0.136211559176445,\n      -0.044954363256692886,\n      0.3486177921295166,\n      -0.04184315726161003,\n      0.2218329906463623,\n      -0.019577952101826668,\n      0.1324857473373413,\n      0.3425091803073883,\n      0.10196127742528915,\n      -0.014197664335370064,\n      -0.389157235622406,\n      -0.10209089517593384,\n      0.14743943512439728\n    ],\n    [\n      -0.45702388882637024,\n      0.2858600616455078,\n      -0.20799002051353455,\n      0.17796678841114044,\n      -0.2767857611179352,\n      0.27277258038520813,\n      0.293668270111084,\n      0.43145477771759033,\n      0.35420820116996765,\n      -0.0944724753499031,\n      0.0426195003092289,\n      0.34546083211898804,\n      0.18080897629261017,\n      -0.05823073908686638,\n      -0.02116667665541172,\n      -0.1027803122997284,\n      -0.06086895242333412,\n      0.08966435492038727,\n      0.10035502910614014,\n      0.05500742048025131,\n      0.5306140184402466,\n      -0.22100670635700226,\n      0.13661669194698334,\n      -0.15382198989391327,\n      0.22065068781375885,\n      -0.04606064409017563,\n      -0.41266435384750366,\n      0.22386234998703003,\n      -0.17480097711086273,\n      0.04691757261753082,\n      -0.21521799266338348,\n      0.0317896232008934,\n      0.19608590006828308,\n      0.7198136448860168,\n      -0.12194477021694183,\n      0.46207258105278015,\n      0.042582012712955475,\n      0.16312097012996674,\n      0.3006892800331116,\n      -0.2718108594417572,\n      0.3281973600387573,\n      0.33100348711013794,\n      -0.18335245549678802,\n      -0.20376981794834137,\n      -0.29940688610076904,\n      0.12832029163837433,\n      -0.014466722495853901,\n      0.1662474125623703,\n      -0.10616820305585861,\n      -0.06032990664243698\n    ],\n    [\n      0.23451977968215942,\n      -0.4317513704299927,\n      0.20944185554981232,\n      0.2703356146812439,\n      0.3652176260948181,\n      0.0023920440580695868,\n      0.3730126917362213,\n      -0.1258680671453476,\n      0.11888574063777924,\n      -0.3392515480518341,\n      -0.27682510018348694,\n      -0.19002771377563477,\n      -0.40600985288619995,\n      -0.30819833278656006,\n      -0.3354426920413971,\n      -0.30264806747436523,\n      -0.35868096351623535,\n      -0.21975313127040863,\n      0.18676863610744476,\n      -0.06689847260713577,\n      0.3228631317615509,\n      0.228838250041008,\n      -0.40933188796043396,\n      -0.3791525363922119,\n      -0.22596678137779236,\n      -0.39264315366744995,\n      -0.3945613205432892,\n      -0.18992874026298523,\n      -0.23827767372131348,\n      -0.4351137578487396,\n      0.14544512331485748,\n      0.25990208983421326,\n      0.05854906141757965,\n      0.14807455241680145,\n      -0.4123094975948334,\n      0.2573210895061493,\n      -0.3620324432849884,\n      -0.2059416025876999,\n      -0.38578566908836365,\n      -0.3957683742046356,\n      -0.37723562121391296,\n      0.7308197617530823,\n      0.36188003420829773,\n      0.19895757734775543,\n      0.3714800477027893,\n      -0.06653132289648056,\n      -0.2212270051240921,\n      0.3493569493293762,\n      -0.308956503868103,\n      -0.29132598638534546\n    ],\n    [\n      0.024111876264214516,\n      -0.21821823716163635,\n      0.0507587194442749,\n      -0.05318260192871094,\n      0.6482554078102112,\n      0.16753409802913666,\n      0.27439624071121216,\n      0.43919211626052856,\n      0.20977558195590973,\n      -0.028151510283350945,\n      0.14328183233737946,\n      0.28424784541130066,\n      0.2987067997455597,\n      -0.2177303582429886,\n      -0.21891926229000092,\n      -0.23576006293296814,\n      -0.06515808403491974,\n      -0.45035234093666077,\n      0.34021008014678955,\n      -0.011921596713364124,\n      -0.5790233612060547,\n      0.5686787366867065,\n      0.1797482669353485,\n      0.21059221029281616,\n      -0.054653093218803406,\n      0.14171342551708221,\n      0.25563979148864746,\n      0.33700719475746155,\n      -0.11873351037502289,\n      -0.41830453276634216,\n      -0.0067972829565405846,\n      -0.052817750722169876,\n      0.08304918557405472,\n      0.3600437045097351,\n      -0.1528582125902176,\n      0.11624883860349655,\n      -0.1934702843427658,\n      0.06295482814311981,\n      -0.2973204255104065,\n      -0.24784885346889496,\n      0.21634313464164734,\n      -0.11483702808618546,\n      -0.06612847000360489,\n      0.34584861993789673,\n      0.15913565456867218,\n      -0.47181665897369385,\n      0.18123012781143188,\n      -0.18294475972652435,\n      0.16255755722522736,\n      0.11765027046203613\n    ],\n    [\n      0.14570292830467224,\n      0.22612415254116058,\n      -0.1877748668193817,\n      0.34022170305252075,\n      0.041422635316848755,\n      -0.3506368100643158,\n      -0.0014862247044220567,\n      -0.11388111114501953,\n      0.1741809993982315,\n      -0.0035023498348891735,\n      0.20990003645420074,\n      0.13823778927326202,\n      0.07559081166982651,\n      -0.07469211518764496,\n      -0.004729114938527346,\n      0.06328536570072174,\n      0.09177210181951523,\n      0.18506179749965668,\n      -0.27014589309692383,\n      -0.09474056959152222,\n      -0.19133833050727844,\n      -0.5123598575592041,\n      0.06547874957323074,\n      -0.05840158462524414,\n      0.08442848920822144,\n      0.04776329547166824,\n      0.190947026014328,\n      0.022976884618401527,\n      -0.08878858387470245,\n      0.061760641634464264,\n      0.27622199058532715,\n      0.12002182751893997,\n      -0.08499860018491745,\n      -0.6779487729072571,\n      0.03175099566578865,\n      -0.15046390891075134,\n      -0.0301596038043499,\n      -0.01838349923491478,\n      -0.004320202860981226,\n      0.1694989651441574,\n      0.0779040977358818,\n      -0.0650029331445694,\n      0.9728092551231384,\n      0.21260057389736176,\n      0.26559317111968994,\n      0.3780447840690613,\n      0.08787669241428375,\n      0.6356847882270813,\n      -0.08115611970424652,\n      -0.01330023817718029\n    ],\n    [\n      -0.07237344235181808,\n      0.10562606155872345,\n      -0.32154425978660583,\n      -0.09527838230133057,\n      0.11432189494371414,\n      0.11993439495563507,\n      -0.23047184944152832,\n      -0.652035117149353,\n      0.33657512068748474,\n      0.11953189969062805,\n      0.05820620432496071,\n      -0.09232763200998306,\n      0.0856214240193367,\n      -0.15519943833351135,\n      0.06318897008895874,\n      -0.14008790254592896,\n      -0.21589307487010956,\n      0.1472158432006836,\n      0.038088563829660416,\n      -0.05401105806231499,\n      0.3238195478916168,\n      -0.460136741399765,\n      0.09338053315877914,\n      -0.5336851477622986,\n      -0.09198839217424393,\n      0.019890688359737396,\n      0.10566385835409164,\n      -0.21438159048557281,\n      -0.16338294744491577,\n      -0.1149531677365303,\n      0.1435464322566986,\n      0.3260265588760376,\n      0.1857248842716217,\n      -0.040751613676548004,\n      -0.05743829905986786,\n      -0.3462681770324707,\n      0.07428231835365295,\n      -0.06336923688650131,\n      -0.06709934771060944,\n      0.231694757938385,\n      0.15215325355529785,\n      -0.3186857998371124,\n      -0.03424634039402008,\n      -0.28921061754226685,\n      -0.11073508858680725,\n      -0.2841932773590088,\n      -0.13264629244804382,\n      0.13366009294986725,\n      -0.10490302741527557,\n      0.03806979954242706\n    ],\n    [\n      -0.5463849902153015,\n      -0.08882702887058258,\n      0.0648236945271492,\n      0.14196942746639252,\n      -0.05845453590154648,\n      -0.20522470772266388,\n      0.19079658389091492,\n      0.20443758368492126,\n      0.19925634562969208,\n      0.09254951775074005,\n      0.07479687035083771,\n      -0.0952373743057251,\n      0.01445843931287527,\n      0.13086827099323273,\n      -0.025724386796355247,\n      -0.09979406744241714,\n      0.001289311796426773,\n      0.055360712110996246,\n      -0.08639182150363922,\n      0.08958149701356888,\n      0.1478147804737091,\n      -0.014574314467608929,\n      0.13247400522232056,\n      -0.5133284330368042,\n      0.1834602653980255,\n      0.06558286398649216,\n      -0.15803036093711853,\n      0.12210821360349655,\n      -0.05941721424460411,\n      -0.0416509173810482,\n      -0.4978678822517395,\n      0.18076245486736298,\n      0.013868234120309353,\n      0.08206693083047867,\n      -0.11546862870454788,\n      0.0463792085647583,\n      -0.06809011846780777,\n      0.17947758734226227,\n      -0.026595115661621094,\n      0.0039175753481686115,\n      0.03470650687813759,\n      0.19014006853103638,\n      0.04760660603642464,\n      -0.02402217872440815,\n      -0.03130790963768959,\n      0.18789556622505188,\n      -0.18670322000980377,\n      -0.04465540125966072,\n      0.01718798093497753,\n      0.008290152065455914\n    ],\n    [\n      -0.7404513955116272,\n      -0.6393402218818665,\n      0.20353810489177704,\n      0.07309754937887192,\n      -0.13419131934642792,\n      0.07597585022449493,\n      -0.2503192126750946,\n      0.11585275083780289,\n      0.7801663875579834,\n      -0.0417105108499527,\n      -0.21861222386360168,\n      -0.09075810015201569,\n      0.024236690253019333,\n      0.185333713889122,\n      -0.04334521293640137,\n      -0.15511207282543182,\n      0.07880350947380066,\n      -0.5477664470672607,\n      0.31979480385780334,\n      0.2220022827386856,\n      0.2467932254076004,\n      0.0010369870578870177,\n      0.0758650079369545,\n      -0.7008332014083862,\n      -0.018193775787949562,\n      0.14304082095623016,\n      -0.5774063467979431,\n      0.04663686826825142,\n      -0.38176771998405457,\n      -0.09514370560646057,\n      -0.2675936222076416,\n      -0.1742323786020279,\n      0.11005979031324387,\n      -0.006519821938127279,\n      -0.2783626914024353,\n      -0.1342814415693283,\n      -0.340509295463562,\n      0.31726163625717163,\n      0.16130729019641876,\n      0.4510969817638397,\n      -0.1292114555835724,\n      0.9920069575309753,\n      -0.13144664466381073,\n      0.135287806391716,\n      -0.2687465250492096,\n      0.040328409522771835,\n      0.2574154734611511,\n      0.1366054266691208,\n      0.17256863415241241,\n      -0.053452786058187485\n    ],\n    [\n      -0.3993047773838043,\n      -0.4472319781780243,\n      0.2185404896736145,\n      0.049584660679101944,\n      -0.3210127651691437,\n      0.29071688652038574,\n      0.048752401024103165,\n      -0.017607785761356354,\n      0.2558842897415161,\n      -0.09893877059221268,\n      0.1641201674938202,\n      0.10482963919639587,\n      0.22726145386695862,\n      0.10902833938598633,\n      -0.10824785381555557,\n      -0.02194862626492977,\n      -0.014227773062884808,\n      -0.020141875371336937,\n      -0.1364593356847763,\n      0.06048610061407089,\n      -0.23515792191028595,\n      -0.18884046375751495,\n      0.192330002784729,\n      -0.2804768681526184,\n      0.0446658581495285,\n      0.07818875461816788,\n      -0.19028626382350922,\n      0.13077665865421295,\n      0.08423267304897308,\n      0.008018983528017998,\n      -0.5158740282058716,\n      0.2169240117073059,\n      0.05654805526137352,\n      0.21795536577701569,\n      -0.19728632271289825,\n      0.23588678240776062,\n      -0.1427524983882904,\n      0.13327495753765106,\n      -0.05179793760180473,\n      0.006791737861931324,\n      0.09608040004968643,\n      0.352778822183609,\n      0.24248619377613068,\n      0.1281476765871048,\n      -0.026269838213920593,\n      0.0413728766143322,\n      -0.22248660027980804,\n      0.0083148293197155,\n      0.2173263132572174,\n      -0.05482521653175354\n    ],\n    [\n      -0.4726678133010864,\n      0.1884298026561737,\n      -0.032123737037181854,\n      0.31379759311676025,\n      -0.12840212881565094,\n      0.09116927534341812,\n      0.0712210014462471,\n      -0.5943979024887085,\n      0.15290339291095734,\n      0.02541860193014145,\n      0.10951870679855347,\n      -0.06244724616408348,\n      -0.029538040980696678,\n      0.07531997561454773,\n      0.10975819081068039,\n      0.027556391432881355,\n      0.14591236412525177,\n      0.18429380655288696,\n      -0.3336612284183502,\n      0.35364168882369995,\n      -0.25286218523979187,\n      -0.4387824237346649,\n      0.08082722127437592,\n      -0.03153035789728165,\n      0.14091207087039948,\n      -0.004438221920281649,\n      0.06744446605443954,\n      -0.06671259552240372,\n      -0.0782000795006752,\n      0.03303195908665657,\n      -0.2570938169956207,\n      0.08971381932497025,\n      0.129582017660141,\n      0.8097479343414307,\n      0.03202498331665993,\n      0.10126882046461105,\n      -0.0036100498400628567,\n      -0.09164755046367645,\n      0.10436680167913437,\n      0.0855497345328331,\n      0.10866373777389526,\n      0.1029772236943245,\n      -0.1571117490530014,\n      0.04706179350614548,\n      -0.1864275336265564,\n      0.3364744484424591,\n      0.012293729931116104,\n      0.2760608196258545,\n      -0.08011768013238907,\n      0.15603870153427124\n    ],\n    [\n      0.06581577658653259,\n      -0.18827734887599945,\n      -0.06755959242582321,\n      0.1655181497335434,\n      0.5695211887359619,\n      -0.05792021378874779,\n      -0.027904348447918892,\n      -0.6260024309158325,\n      -0.03253551572561264,\n      -0.07724105566740036,\n      -0.1571633219718933,\n      0.3467240035533905,\n      -0.0911177396774292,\n      -0.17030230164527893,\n      0.024456139653921127,\n      -0.28682994842529297,\n      -0.31211522221565247,\n      0.08157378435134888,\n      -0.09719515591859818,\n      -0.13252213597297668,\n      -0.2570953667163849,\n      -0.15055960416793823,\n      -0.1259966939687729,\n      0.06809551268815994,\n      0.01431370247155428,\n      0.10147762298583984,\n      0.08022072166204453,\n      -0.18964862823486328,\n      -0.10489154607057571,\n      -0.31562647223472595,\n      -0.2025877833366394,\n      0.09128223359584808,\n      -0.06587096303701401,\n      0.3457831144332886,\n      -0.1673191636800766,\n      0.059061404317617416,\n      -0.13730579614639282,\n      -0.10423211753368378,\n      -0.15937544405460358,\n      -0.046164944767951965,\n      0.04765899106860161,\n      0.10642270743846893,\n      -0.42377617955207825,\n      -0.019328445196151733,\n      0.23423634469509125,\n      0.05923926830291748,\n      -0.14192305505275726,\n      0.5786423087120056,\n      0.02490766905248165,\n      -0.13667245209217072\n    ],\n    [\n      -0.2899729013442993,\n      0.11231617629528046,\n      0.1297016441822052,\n      0.05842675641179085,\n      0.22800591588020325,\n      0.2642139196395874,\n      -0.23374362289905548,\n      -0.2411717027425766,\n      0.2473653107881546,\n      0.009148780256509781,\n      0.042949821799993515,\n      -0.13164646923542023,\n      0.01828075386583805,\n      0.032944414764642715,\n      0.08964651823043823,\n      -0.30144837498664856,\n      -0.2937764823436737,\n      -0.04746067896485329,\n      -0.05572081729769707,\n      -0.6959691047668457,\n      0.1253555417060852,\n      0.2353164255619049,\n      -0.10763247311115265,\n      0.09943883121013641,\n      -0.02287227101624012,\n      -0.11630058288574219,\n      0.10504776239395142,\n      -0.0014501643599942327,\n      -0.007164937909692526,\n      -0.022275321185588837,\n      -0.1642189472913742,\n      0.022345518693327904,\n      0.2727527916431427,\n      -0.6202316880226135,\n      -0.21537509560585022,\n      -0.6138015389442444,\n      -0.0035360658075660467,\n      -0.1185218021273613,\n      0.07292405515909195,\n      -0.2265658974647522,\n      -0.05880394205451012,\n      0.3018207848072052,\n      0.5687532424926758,\n      0.14834612607955933,\n      -0.011930413544178009,\n      0.1670478880405426,\n      -0.2517857253551483,\n      0.14223423600196838,\n      -0.11073827743530273,\n      -0.14627468585968018\n    ],\n    [\n      0.04114236310124397,\n      -0.19583795964717865,\n      -0.055413998663425446,\n      0.004671877715736628,\n      -0.5439082980155945,\n      -0.05514035001397133,\n      0.17144247889518738,\n      -0.19161802530288696,\n      -0.46398526430130005,\n      -0.17973530292510986,\n      -0.22411267459392548,\n      -0.047452736645936966,\n      -0.1587119698524475,\n      -0.11605357378721237,\n      -0.08002974838018417,\n      -0.13026608526706696,\n      -0.1606924831867218,\n      -0.21562559902668,\n      0.23327526450157166,\n      0.1618257611989975,\n      -0.2405116856098175,\n      0.008862434886395931,\n      -0.043698299676179886,\n      -0.27666839957237244,\n      -0.0220014750957489,\n      -0.07863172143697739,\n      -0.0968034639954567,\n      -0.028949622064828873,\n      -0.12328556925058365,\n      -0.17934443056583405,\n      -0.11222530156373978,\n      -0.11456257104873657,\n      0.023475298658013344,\n      0.2926361858844757,\n      -0.23260433971881866,\n      -0.17777258157730103,\n      -0.13378208875656128,\n      -0.21247243881225586,\n      -0.1951640099287033,\n      -0.15454521775245667,\n      0.04504670202732086,\n      0.16751927137374878,\n      0.13792264461517334,\n      0.3617118299007416,\n      -0.053597331047058105,\n      0.34196579456329346,\n      0.28757065534591675,\n      0.26012060046195984,\n      -0.01748303696513176,\n      -0.1285756379365921\n    ],\n    [\n      -0.3639151155948639,\n      -0.22649618983268738,\n      -0.0005233726697042584,\n      0.22674226760864258,\n      0.42065009474754333,\n      -0.07658745348453522,\n      -0.3162126839160919,\n      0.4435114562511444,\n      -0.20287536084651947,\n      -0.00397238926962018,\n      -0.01840481534600258,\n      0.14063875377178192,\n      -0.08513700217008591,\n      -0.16610054671764374,\n      0.004022188019007444,\n      -0.08643744140863419,\n      -0.03963321074843407,\n      0.6156153082847595,\n      0.11785836517810822,\n      -0.6028597950935364,\n      -0.3599906265735626,\n      -0.36379000544548035,\n      0.22392651438713074,\n      0.22166240215301514,\n      0.300642192363739,\n      0.08068472146987915,\n      -0.08311239629983902,\n      -0.2587467133998871,\n      0.12286464869976044,\n      -0.05061759427189827,\n      0.014952282421290874,\n      0.30812183022499084,\n      -0.15013085305690765,\n      0.2163490206003189,\n      -0.0540919154882431,\n      -0.2894705832004547,\n      -0.12411349266767502,\n      -0.02942589484155178,\n      -0.05207731947302818,\n      0.2911379337310791,\n      0.0683738961815834,\n      0.189646378159523,\n      -0.09781955182552338,\n      0.12265720218420029,\n      -0.08470438420772552,\n      -0.1662529706954956,\n      -0.09975920617580414,\n      0.33031022548675537,\n      -0.09275522083044052,\n      -0.035640649497509\n    ],\n    [\n      0.35380494594573975,\n      -0.034501928836107254,\n      0.14327821135520935,\n      -0.006438256707042456,\n      -0.1932838261127472,\n      -0.09070822596549988,\n      0.22571001946926117,\n      -0.8081300854682922,\n      -0.04116988927125931,\n      0.13970406353473663,\n      0.07134454697370529,\n      -0.5139862298965454,\n      0.021008899435400963,\n      -0.03319491446018219,\n      0.17269454896450043,\n      0.24028219282627106,\n      0.16204194724559784,\n      0.23592056334018707,\n      -0.16141211986541748,\n      0.0718686506152153,\n      0.32145190238952637,\n      0.14992383122444153,\n      0.0331338606774807,\n      -0.38394853472709656,\n      0.03696933388710022,\n      -0.1786099076271057,\n      -0.07471805810928345,\n      0.2389640063047409,\n      0.05816379189491272,\n      0.05141192674636841,\n      0.09160733222961426,\n      0.07774859666824341,\n      0.0027920096181333065,\n      -0.31789159774780273,\n      -0.18012261390686035,\n      -0.564876914024353,\n      0.1133439689874649,\n      -0.09709855169057846,\n      -0.10296037048101425,\n      0.26518115401268005,\n      0.16124682128429413,\n      -0.008628986775875092,\n      -0.6101848483085632,\n      0.16156505048274994,\n      0.3478695750236511,\n      0.18347054719924927,\n      -0.3454362750053406,\n      -0.5983367562294006,\n      -0.026411525905132294,\n      0.07318852841854095\n    ],\n    [\n      -0.7557329535484314,\n      0.2046063095331192,\n      -0.20456169545650482,\n      0.11834855377674103,\n      -0.04409688711166382,\n      0.2786159813404083,\n      0.21220922470092773,\n      -0.0617036297917366,\n      0.18101103603839874,\n      0.007628143765032291,\n      -0.08987760543823242,\n      -0.027995649725198746,\n      0.060265254229307175,\n      -0.07315652072429657,\n      -0.06576356291770935,\n      -0.08070582896471024,\n      -0.0628618597984314,\n      0.4630751609802246,\n      0.12331189960241318,\n      -0.12226780503988266,\n      0.09388118982315063,\n      -0.2538411319255829,\n      -0.02967400848865509,\n      0.08273287117481232,\n      -0.14069928228855133,\n      0.0226445235311985,\n      0.20883843302726746,\n      0.042885906994342804,\n      0.0958562046289444,\n      -0.0032906290143728256,\n      -0.26401346921920776,\n      0.040751248598098755,\n      0.4002375304698944,\n      0.0622052326798439,\n      0.08808048069477081,\n      0.13606813549995422,\n      -0.09330447763204575,\n      -0.33125561475753784,\n      -0.2283739298582077,\n      0.09303531050682068,\n      -0.18821220099925995,\n      0.11134716123342514,\n      0.340419739484787,\n      0.1392085701227188,\n      -0.03423663228750229,\n      0.2828396260738373,\n      0.13364717364311218,\n      0.3203960657119751,\n      -0.17096266150474548,\n      -0.010720270685851574\n    ],\n    [\n      0.2768948972225189,\n      0.07388105243444443,\n      0.052071988582611084,\n      0.12665824592113495,\n      0.2625283896923065,\n      -0.14916135370731354,\n      0.20064248144626617,\n      -0.15287983417510986,\n      0.4079132676124573,\n      0.015923116356134415,\n      0.1377006471157074,\n      -0.05734637379646301,\n      -0.015167105942964554,\n      -0.2161913961172104,\n      -0.08140499144792557,\n      0.09461503475904465,\n      0.09046290814876556,\n      -0.19140201807022095,\n      0.07475432008504868,\n      0.1589885950088501,\n      0.2517320513725281,\n      0.15308083593845367,\n      -0.11797293275594711,\n      0.1728641241788864,\n      0.25310906767845154,\n      -0.24070844054222107,\n      -0.010410037823021412,\n      0.06309834122657776,\n      -0.1297897845506668,\n      -0.16657744348049164,\n      0.37176263332366943,\n      0.2150847613811493,\n      -0.2652903199195862,\n      -0.755254864692688,\n      -0.07729964703321457,\n      -0.16562719643115997,\n      0.10895555466413498,\n      0.008314157836139202,\n      0.14638255536556244,\n      0.072003573179245,\n      0.6190196871757507,\n      0.04547493904829025,\n      0.2683342397212982,\n      -0.19142070412635803,\n      -0.2603810131549835,\n      -0.2659975290298462,\n      -0.2348342090845108,\n      -0.1489066481590271,\n      -0.0432773195207119,\n      -0.08126294612884521\n    ],\n    [\n      -0.6692041754722595,\n      0.08556678891181946,\n      -0.005721402354538441,\n      -0.1891971230506897,\n      0.2960665822029114,\n      -0.09747500717639923,\n      -0.019946685060858727,\n      -0.1553369015455246,\n      -0.15115711092948914,\n      -0.011417150497436523,\n      -0.11821027845144272,\n      -0.30145376920700073,\n      0.006259533576667309,\n      -0.15725859999656677,\n      -0.5127982497215271,\n      0.4643363952636719,\n      0.13342377543449402,\n      -0.04711974039673805,\n      0.2764546275138855,\n      -0.27472636103630066,\n      -0.1076204776763916,\n      -0.041911106556653976,\n      0.03554655984044075,\n      -0.19672590494155884,\n      0.05327753722667694,\n      -0.1296263337135315,\n      -0.32234781980514526,\n      0.0389278419315815,\n      -0.40813302993774414,\n      -0.02087150141596794,\n      -0.19803479313850403,\n      -0.07817266136407852,\n      -0.26311609148979187,\n      -0.1797085553407669,\n      0.13434351980686188,\n      -0.04427332803606987,\n      -0.42443978786468506,\n      -0.4097912311553955,\n      -0.04090137407183647,\n      -0.15951122343540192,\n      0.04449190944433212,\n      0.36492350697517395,\n      0.1204155832529068,\n      0.09004763513803482,\n      -0.00642409035935998,\n      -0.08240476995706558,\n      0.17297160625457764,\n      0.683956503868103,\n      -0.11428768187761307,\n      -0.10317038744688034\n    ],\n    [\n      -0.596125066280365,\n      -0.4539159834384918,\n      0.12050693482160568,\n      0.00020178512204438448,\n      -0.25953835248947144,\n      0.2887876033782959,\n      0.11215037852525711,\n      -0.21568867564201355,\n      0.16515378654003143,\n      0.08308178931474686,\n      0.001051013940013945,\n      0.0034190842416137457,\n      0.13665175437927246,\n      -0.5135810971260071,\n      -0.006541278213262558,\n      0.09124242514371872,\n      -0.005550340749323368,\n      -0.04771978780627251,\n      -0.20901226997375488,\n      0.05574951693415642,\n      0.2879886031150818,\n      0.15709182620048523,\n      -0.12951600551605225,\n      -0.46869632601737976,\n      0.15144944190979004,\n      -0.10234226286411285,\n      0.2640151381492615,\n      0.18310357630252838,\n      0.17227308452129364,\n      0.111046701669693,\n      -0.502553403377533,\n      0.07096965610980988,\n      0.025906644761562347,\n      -0.038808293640613556,\n      -0.2680417597293854,\n      -0.29808294773101807,\n      -0.22533218562602997,\n      -0.33518844842910767,\n      -0.46407124400138855,\n      0.41094958782196045,\n      0.031681012362241745,\n      0.3253796398639679,\n      0.22301101684570312,\n      0.21299022436141968,\n      0.07566091418266296,\n      0.3116750717163086,\n      0.3880297541618347,\n      0.170183464884758,\n      0.19991670548915863,\n      0.0319303423166275\n    ],\n    [\n      -0.2159595787525177,\n      -0.19462651014328003,\n      0.07753821462392807,\n      -0.14035706222057343,\n      0.2434714138507843,\n      0.4175080358982086,\n      0.4764839708805084,\n      0.36730894446372986,\n      -0.3480021357536316,\n      -0.18038807809352875,\n      -0.19077561795711517,\n      -0.2539205253124237,\n      -0.1980958878993988,\n      -0.29846954345703125,\n      -0.08020146191120148,\n      -0.13899675011634827,\n      -0.09612061083316803,\n      -0.08487864583730698,\n      0.030546197667717934,\n      0.14246825873851776,\n      -0.016987888142466545,\n      -0.23079997301101685,\n      -0.2903965711593628,\n      0.25121819972991943,\n      -0.09485653042793274,\n      -0.09040284156799316,\n      -0.31883135437965393,\n      -0.09886917471885681,\n      0.06538306176662445,\n      -0.15228867530822754,\n      -0.16328532993793488,\n      -0.31544917821884155,\n      0.1393311321735382,\n      0.08505311608314514,\n      -0.06813608109951019,\n      0.294460266828537,\n      -0.2196052223443985,\n      -0.3272660970687866,\n      -0.08426976948976517,\n      -0.20013859868049622,\n      -0.3208335340023041,\n      0.024309493601322174,\n      -0.2223830372095108,\n      0.12013121694326401,\n      -0.0037796024698764086,\n      0.33305037021636963,\n      -0.19772754609584808,\n      0.1737927347421646,\n      -0.24851001799106598,\n      -0.2322719395160675\n    ],\n    [\n      -0.11383021622896194,\n      0.21637263894081116,\n      -0.23179763555526733,\n      -0.7196720838546753,\n      0.45287850499153137,\n      -0.08642453700304031,\n      0.7513107657432556,\n      0.1292053461074829,\n      0.020178761333227158,\n      0.37042471766471863,\n      0.23946106433868408,\n      0.02072736620903015,\n      0.2615445852279663,\n      -0.05008074268698692,\n      -0.3307410478591919,\n      0.20578043162822723,\n      0.028270814567804337,\n      0.015179718844592571,\n      -0.12236128002405167,\n      -0.07217706739902496,\n      0.11568822711706161,\n      0.3111850619316101,\n      0.14832419157028198,\n      -0.03449722006917,\n      -0.15355663001537323,\n      -0.08854296803474426,\n      -0.1556462049484253,\n      0.13265253603458405,\n      -0.1301099956035614,\n      0.1300511360168457,\n      0.05629629269242287,\n      0.05570657551288605,\n      0.02707444317638874,\n      0.352376788854599,\n      -0.04316830635070801,\n      0.4691530168056488,\n      0.3546171188354492,\n      -0.13299521803855896,\n      -0.1892523318529129,\n      0.25428497791290283,\n      0.04711531475186348,\n      0.03584319353103638,\n      0.6117398142814636,\n      -0.22931945323944092,\n      -0.16232986748218536,\n      -0.11547771841287613,\n      0.08624336868524551,\n      -0.1307058334350586,\n      -0.049249663949012756,\n      0.32638129591941833\n    ],\n    [\n      -0.469477117061615,\n      0.0304791908711195,\n      -0.12124215066432953,\n      0.09660918265581131,\n      -0.00907145906239748,\n      -0.12918537855148315,\n      0.7300756573677063,\n      -0.7319830060005188,\n      -0.06930951029062271,\n      -0.011383617296814919,\n      0.04569047689437866,\n      0.04369886964559555,\n      -0.033264949917793274,\n      0.09870805591344833,\n      -0.32328906655311584,\n      0.13296082615852356,\n      0.12185964733362198,\n      0.19422496855258942,\n      0.16924366354942322,\n      -0.003278864547610283,\n      -0.55771404504776,\n      0.12102081626653671,\n      0.025927942246198654,\n      0.16784563660621643,\n      0.055674947798252106,\n      -0.13497957587242126,\n      0.17319534718990326,\n      -0.07802064716815948,\n      -0.19643627107143402,\n      0.04205043241381645,\n      0.21209818124771118,\n      0.07349347323179245,\n      0.026995519176125526,\n      0.38879433274269104,\n      -0.021716896444559097,\n      0.39063236117362976,\n      -0.4091492295265198,\n      -0.015331956557929516,\n      -0.05918412283062935,\n      -0.06600836664438248,\n      0.14513570070266724,\n      -0.059843383729457855,\n      0.398749977350235,\n      -0.20591595768928528,\n      -0.23202389478683472,\n      -0.08211151510477066,\n      -0.14469973742961884,\n      0.0754992663860321,\n      0.07525797933340073,\n      0.09106867760419846\n    ],\n    [\n      0.3415257930755615,\n      -0.153831347823143,\n      0.11485022306442261,\n      -0.30242475867271423,\n      -0.34699368476867676,\n      -0.44560158252716064,\n      -0.1764184832572937,\n      -0.5676378607749939,\n      0.1699093133211136,\n      0.05208507925271988,\n      0.25689783692359924,\n      -0.07145075500011444,\n      -0.4037784934043884,\n      0.13513149321079254,\n      -0.047830283641815186,\n      -0.4039551913738251,\n      -0.1249995082616806,\n      0.33150044083595276,\n      0.05314765125513077,\n      0.06158838048577309,\n      -0.15486305952072144,\n      0.1805446445941925,\n      -0.2663029432296753,\n      -0.05184081196784973,\n      -0.14440882205963135,\n      0.1770683228969574,\n      0.24545757472515106,\n      -0.026452191174030304,\n      -0.008723985403776169,\n      0.038196153938770294,\n      0.031409453600645065,\n      0.13587580621242523,\n      -0.43303075432777405,\n      0.34160497784614563,\n      -0.06604503095149994,\n      -0.12783390283584595,\n      -0.16646811366081238,\n      0.06960832327604294,\n      0.03154929354786873,\n      -0.123887799680233,\n      -0.46353864669799805,\n      -0.5340407490730286,\n      -0.42372673749923706,\n      0.5393332242965698,\n      0.2504104673862457,\n      -0.1323717087507248,\n      -0.06791354715824127,\n      -0.5405345559120178,\n      -0.08851554989814758,\n      0.03100821003317833\n    ],\n    [\n      -0.9812238812446594,\n      -0.040189482271671295,\n      0.34942948818206787,\n      -0.10027527064085007,\n      -0.02517566829919815,\n      -0.06290184706449509,\n      -0.19312864542007446,\n      0.1601618230342865,\n      0.12549884617328644,\n      0.07163868844509125,\n      -0.006204509641975164,\n      0.11568878591060638,\n      -0.08441077172756195,\n      0.07095397263765335,\n      0.09325511008501053,\n      0.03283608332276344,\n      0.15887966752052307,\n      0.10187280178070068,\n      0.15058138966560364,\n      0.37690049409866333,\n      -0.31451913714408875,\n      -0.1548262983560562,\n      0.1617109626531601,\n      0.26778900623321533,\n      0.003711925121024251,\n      -0.06057242304086685,\n      0.02376418001949787,\n      -0.05478191003203392,\n      -0.1576126664876938,\n      -0.2858216464519501,\n      -0.2780504822731018,\n      0.24791008234024048,\n      0.24800069630146027,\n      -0.21810151636600494,\n      -0.22577863931655884,\n      -0.564604640007019,\n      -0.04497148469090462,\n      0.07133816182613373,\n      0.0013389517553150654,\n      0.08474922180175781,\n      0.08366569876670837,\n      -0.0014679296873509884,\n      -0.08943085372447968,\n      0.12731218338012695,\n      0.24777041375637054,\n      -0.1055961400270462,\n      0.27318114042282104,\n      -0.06534984707832336,\n      -0.04846072942018509,\n      -0.01469405833631754\n    ],\n    [\n      0.40711545944213867,\n      -0.13086047768592834,\n      0.027484063059091568,\n      -0.034118086099624634,\n      0.16797634959220886,\n      0.09621519595384598,\n      -0.6257727742195129,\n      -0.6410952210426331,\n      0.17562253773212433,\n      0.04293148219585419,\n      -0.024463780224323273,\n      0.05978408828377724,\n      0.03841114416718483,\n      -0.029377024620771408,\n      0.009013879112899303,\n      -0.09484588354825974,\n      -0.22251936793327332,\n      -0.14870254695415497,\n      0.1473485231399536,\n      -0.21280832588672638,\n      0.1659867763519287,\n      -0.7619535326957703,\n      0.18007928133010864,\n      0.2634440064430237,\n      -0.023325514048337936,\n      -0.2032531052827835,\n      0.08467560261487961,\n      -0.2319343388080597,\n      -0.05421333387494087,\n      -0.14198003709316254,\n      -0.12579049170017242,\n      0.19478848576545715,\n      -0.040117163211107254,\n      0.26067042350769043,\n      0.19933588802814484,\n      0.25160160660743713,\n      0.010480734519660473,\n      -0.13123713433742523,\n      0.09066292643547058,\n      0.013406556099653244,\n      -0.14184661209583282,\n      -0.013591059483587742,\n      -0.487160861492157,\n      -0.42149728536605835,\n      0.050670281052589417,\n      0.17482592165470123,\n      -0.3086634874343872,\n      0.38461145758628845,\n      0.04986937716603279,\n      0.01650077849626541\n    ],\n    [\n      -0.12378466874361038,\n      -0.06046200916171074,\n      0.019548697397112846,\n      0.03617611899971962,\n      0.01643041893839836,\n      0.09493271261453629,\n      0.026497427374124527,\n      -0.17440447211265564,\n      -0.013315346091985703,\n      -0.0501890555024147,\n      -0.01390866655856371,\n      0.2167406976222992,\n      -0.016911398619413376,\n      -0.14071500301361084,\n      -0.12483185529708862,\n      -0.06333956122398376,\n      -0.11738940328359604,\n      0.02215925231575966,\n      0.11155501753091812,\n      0.1606231927871704,\n      -0.054391048848629,\n      -0.027850231155753136,\n      -0.02941415086388588,\n      -0.013678143732249737,\n      0.09084217250347137,\n      -0.006216452457010746,\n      -0.2358647584915161,\n      -0.15631726384162903,\n      0.015916109085083008,\n      -0.16677819192409515,\n      -0.017154738306999207,\n      -0.09949915111064911,\n      0.027231737971305847,\n      0.31949862837791443,\n      -0.3044946491718292,\n      0.18994103372097015,\n      0.05813062936067581,\n      -0.014103861525654793,\n      -0.04529847577214241,\n      -0.09722699970006943,\n      0.11625651270151138,\n      0.32086312770843506,\n      -0.2680007517337799,\n      -0.05369789898395538,\n      -0.11219187825918198,\n      0.007391939405351877,\n      -0.08856309205293655,\n      0.24546149373054504,\n      -0.06613007187843323,\n      -0.05349305644631386\n    ],\n    [\n      -0.1079045832157135,\n      -0.014098262414336205,\n      -0.01641385443508625,\n      -0.02195592410862446,\n      -0.23005801439285278,\n      0.1616675853729248,\n      0.05978211387991905,\n      -0.1353285014629364,\n      -0.4981476664543152,\n      -0.23770292103290558,\n      -0.19684772193431854,\n      0.08985518664121628,\n      -0.0923638865351677,\n      -0.16626796126365662,\n      -0.16400064527988434,\n      -0.2006738930940628,\n      -0.1495702862739563,\n      0.04063289985060692,\n      0.018038908019661903,\n      0.07207071036100388,\n      -0.17114020884037018,\n      -0.036891527473926544,\n      -0.10025763511657715,\n      -0.3757186233997345,\n      -0.17704632878303528,\n      -0.1166594922542572,\n      0.012740793637931347,\n      -0.14439982175827026,\n      -0.1996820718050003,\n      0.10451915115118027,\n      -0.03446890786290169,\n      -0.0004504455719143152,\n      -0.08308357000350952,\n      -0.03359827771782875,\n      -0.15017184615135193,\n      -0.1432216614484787,\n      -0.1047859638929367,\n      0.033395104110240936,\n      -0.1391402930021286,\n      -0.11430581659078598,\n      -0.16138480603694916,\n      -0.02296171337366104,\n      0.161920428276062,\n      0.06026143580675125,\n      -0.08917563408613205,\n      0.2838346064090729,\n      -0.651764452457428,\n      0.12981179356575012,\n      -0.21602566540241241,\n      -0.03437846899032593\n    ],\n    [\n      -0.0724293664097786,\n      -0.05406210571527481,\n      0.022960957139730453,\n      -0.013924119994044304,\n      0.3601144552230835,\n      0.17021197080612183,\n      0.5350490808486938,\n      -0.05730922147631645,\n      -0.11911367624998093,\n      0.03157370164990425,\n      -0.07598546147346497,\n      -0.046080268919467926,\n      0.28460660576820374,\n      -0.012411903589963913,\n      -0.2595909833908081,\n      -0.1523023545742035,\n      0.05720413103699684,\n      -0.5329489707946777,\n      0.11780048906803131,\n      0.30986955761909485,\n      -0.11931538581848145,\n      0.443386971950531,\n      0.10941300541162491,\n      -0.37997981905937195,\n      0.0910022184252739,\n      0.07382900267839432,\n      -0.02331758663058281,\n      0.2986155152320862,\n      -0.17429904639720917,\n      -0.02758517861366272,\n      -0.005708273034542799,\n      -0.06594708561897278,\n      -0.03319891169667244,\n      0.4405989348888397,\n      0.04386722669005394,\n      0.4329594373703003,\n      -0.2300683706998825,\n      -0.01011301577091217,\n      0.21833020448684692,\n      -0.052029334008693695,\n      0.1804760843515396,\n      0.6851625442504883,\n      0.25747278332710266,\n      -0.2084108591079712,\n      0.06812310218811035,\n      -0.13633348047733307,\n      0.12934842705726624,\n      0.017271699383854866,\n      -0.05304720252752304,\n      -0.057500772178173065\n    ],\n    [\n      -0.6656326055526733,\n      -0.06786555051803589,\n      0.1500479280948639,\n      0.21808698773384094,\n      0.3565209209918976,\n      -0.2642064094543457,\n      -0.2695239782333374,\n      0.9619693756103516,\n      -0.06292236596345901,\n      -0.0492720790207386,\n      -0.09979092329740524,\n      0.19463306665420532,\n      -0.28115010261535645,\n      0.135065957903862,\n      -0.37881940603256226,\n      0.11466693878173828,\n      0.13876503705978394,\n      -0.447234183549881,\n      -0.1335126906633377,\n      0.1342458575963974,\n      -0.2558201849460602,\n      -0.035014744848012924,\n      -0.038042210042476654,\n      -0.20137827098369598,\n      0.20277711749076843,\n      0.2858479917049408,\n      0.08572372794151306,\n      0.036232221871614456,\n      -0.36387720704078674,\n      0.06315044313669205,\n      -0.304229736328125,\n      -0.06671184301376343,\n      -0.31812211871147156,\n      0.08485091477632523,\n      0.20492403209209442,\n      0.4225458800792694,\n      -0.43515026569366455,\n      -0.157973974943161,\n      0.3636869788169861,\n      -0.24309775233268738,\n      -0.09181814640760422,\n      0.4596962034702301,\n      0.3584781885147095,\n      -0.036217812448740005,\n      -0.08027371019124985,\n      0.08222344517707825,\n      0.4066476821899414,\n      0.07486847043037415,\n      -0.024540740996599197,\n      -0.03841240331530571\n    ],\n    [\n      -0.310202419757843,\n      -0.03307098150253296,\n      -0.0005241789622232318,\n      -0.18973788619041443,\n      -0.025619158521294594,\n      0.19787085056304932,\n      -0.17174725234508514,\n      -0.096193827688694,\n      -0.1128743588924408,\n      0.1025427058339119,\n      0.11351918429136276,\n      -0.13518212735652924,\n      -0.09242846816778183,\n      -0.01899116113781929,\n      -0.14594362676143646,\n      -0.10651588439941406,\n      0.06628540903329849,\n      0.16020053625106812,\n      -0.08773788809776306,\n      0.34463590383529663,\n      -0.07163900882005692,\n      -0.07898686081171036,\n      -0.010547901503741741,\n      0.14906278252601624,\n      -0.056911613792181015,\n      -0.23423999547958374,\n      0.18207210302352905,\n      -0.22104330360889435,\n      -0.017040472477674484,\n      -0.3210408091545105,\n      -0.1582254320383072,\n      -0.18857455253601074,\n      0.19910481572151184,\n      -0.09129940718412399,\n      -0.022739620879292488,\n      -0.12447832524776459,\n      -0.17946694791316986,\n      -0.14818662405014038,\n      -0.16118058562278748,\n      -0.08301952481269836,\n      0.09531703591346741,\n      0.23505371809005737,\n      -0.06017303839325905,\n      -0.032108038663864136,\n      0.1209372878074646,\n      0.0489724762737751,\n      -0.01313747651875019,\n      -0.08050220459699631,\n      0.07315975427627563,\n      -0.046004388481378555\n    ],\n    [\n      -0.29210418462753296,\n      -0.08708672970533371,\n      0.10035543888807297,\n      0.04332204908132553,\n      0.3647075295448303,\n      -0.09270184487104416,\n      0.36686068773269653,\n      -0.7486045956611633,\n      0.21914063394069672,\n      0.05744710937142372,\n      -0.07454803586006165,\n      -0.24541698396205902,\n      -0.14902180433273315,\n      0.14240305125713348,\n      0.048876725137233734,\n      -0.14127632975578308,\n      0.15423622727394104,\n      0.20786859095096588,\n      0.2372860163450241,\n      0.267595499753952,\n      -0.651083767414093,\n      0.03661400079727173,\n      -0.08233238756656647,\n      -0.4253382682800293,\n      0.15542654693126678,\n      -0.15350627899169922,\n      -0.0007989830919541419,\n      0.07778619229793549,\n      0.030498310923576355,\n      -0.1517181098461151,\n      -0.06136256456375122,\n      0.07338765263557434,\n      0.10521666705608368,\n      0.5403516888618469,\n      -0.2513545751571655,\n      0.05711081251502037,\n      0.08913308382034302,\n      -0.08907200396060944,\n      0.11066343635320663,\n      -0.4256574809551239,\n      -0.09891491383314133,\n      0.18530671298503876,\n      -0.2609924376010895,\n      -0.2248126119375229,\n      0.0387667641043663,\n      0.10249046236276627,\n      0.15382498502731323,\n      -0.025798451155424118,\n      -0.09317366033792496,\n      -0.05684979632496834\n    ],\n    [\n      -0.38186508417129517,\n      -0.161854550242424,\n      0.0783638060092926,\n      0.3266356289386749,\n      0.32366517186164856,\n      0.045807547867298126,\n      0.06875194609165192,\n      0.5086491703987122,\n      -0.1841115802526474,\n      -0.17451585829257965,\n      0.11606128513813019,\n      -0.03831028565764427,\n      0.06962727010250092,\n      -0.2330610752105713,\n      0.17431028187274933,\n      0.1692020446062088,\n      0.162456214427948,\n      -0.2882015109062195,\n      -0.26588669419288635,\n      0.04539645090699196,\n      -0.40316376090049744,\n      -0.2358098179101944,\n      0.28061413764953613,\n      0.1503080576658249,\n      0.1423434019088745,\n      0.25796201825141907,\n      0.2916123569011688,\n      0.13723468780517578,\n      0.15418045222759247,\n      -0.2943059206008911,\n      -0.4818754196166992,\n      0.34253209829330444,\n      0.15703651309013367,\n      -0.3720393478870392,\n      0.12533169984817505,\n      0.2779051959514618,\n      -0.11738845705986023,\n      0.1978311687707901,\n      -0.19749420881271362,\n      0.06924550980329514,\n      0.07652833312749863,\n      0.6869409680366516,\n      0.5675890445709229,\n      0.07547017186880112,\n      -0.2992567718029022,\n      0.02261737734079361,\n      -0.33519017696380615,\n      0.25161999464035034,\n      -0.08938631415367126,\n      -0.03211525082588196\n    ],\n    [\n      -0.4964461326599121,\n      -0.0911947637796402,\n      0.0012285845587030053,\n      0.17744344472885132,\n      0.001088830758817494,\n      -0.2255207896232605,\n      0.20612721145153046,\n      0.4131866693496704,\n      -0.06502936780452728,\n      -0.05863253027200699,\n      -0.10307089239358902,\n      -0.10048248618841171,\n      -0.05168253183364868,\n      0.12736469507217407,\n      -0.4556204080581665,\n      -0.5288934111595154,\n      -0.0772886872291565,\n      -0.6759362816810608,\n      -0.18438668549060822,\n      0.1194375604391098,\n      -0.15955834090709686,\n      -0.16388441622257233,\n      0.22186605632305145,\n      -0.3690111041069031,\n      -0.06667602807283401,\n      0.01988230086863041,\n      0.21008752286434174,\n      0.27807095646858215,\n      -0.41078582406044006,\n      -0.022135812789201736,\n      -0.34491637349128723,\n      0.2048071324825287,\n      -0.08459106832742691,\n      -0.19134005904197693,\n      -0.0865306705236435,\n      0.07059844583272934,\n      -0.25230348110198975,\n      -0.0014619325520470738,\n      0.08896080404520035,\n      -0.3568987548351288,\n      0.11678487062454224,\n      2.1783952713012695,\n      -0.014260544441640377,\n      0.08429359644651413,\n      -0.09230770915746689,\n      0.077999047935009,\n      -0.036873769015073776,\n      0.11294715851545334,\n      0.24544107913970947,\n      -0.042737267911434174\n    ],\n    [\n      -0.4845077693462372,\n      -0.18271681666374207,\n      0.037262625992298126,\n      -0.05627450346946716,\n      -0.012320628389716148,\n      0.15234775841236115,\n      -0.0034987488761544228,\n      0.11176908761262894,\n      0.3271728754043579,\n      -0.15384134650230408,\n      -0.005189506337046623,\n      -0.04472450166940689,\n      -0.1015457957983017,\n      -0.04033057391643524,\n      0.16836167871952057,\n      0.15849150717258453,\n      -0.1028418242931366,\n      -0.30095595121383667,\n      0.4777080714702606,\n      0.22925704717636108,\n      0.10110879689455032,\n      0.05301765352487564,\n      -0.005597384180873632,\n      -0.4177002012729645,\n      0.0865410789847374,\n      0.0398264117538929,\n      -0.16981326043605804,\n      0.038346730172634125,\n      -0.23007746040821075,\n      0.2817489802837372,\n      -0.09236140549182892,\n      0.10943818092346191,\n      -0.005522370338439941,\n      0.3543744385242462,\n      -0.06669067591428757,\n      0.03688985854387283,\n      -0.05617225542664528,\n      -0.08823215961456299,\n      0.14528052508831024,\n      0.020949963480234146,\n      0.12183349579572678,\n      -0.23474465310573578,\n      -0.13625608384609222,\n      0.0687415823340416,\n      -0.14089366793632507,\n      0.16399326920509338,\n      0.17151618003845215,\n      -0.033355679363012314,\n      0.000742139876820147,\n      0.12967880070209503\n    ],\n    [\n      0.3202631175518036,\n      0.03485877439379692,\n      -0.09346461296081543,\n      0.12723493576049805,\n      0.3234465420246124,\n      0.09631389379501343,\n      -0.32207730412483215,\n      -0.21358433365821838,\n      0.10509505867958069,\n      -0.20443570613861084,\n      -0.15028421580791473,\n      0.10194536298513412,\n      0.036916110664606094,\n      0.133888378739357,\n      -0.0156784076243639,\n      -0.13010047376155853,\n      -0.10350515693426132,\n      -0.6414148807525635,\n      -0.047336969524621964,\n      0.20788955688476562,\n      -0.45630085468292236,\n      0.03627097234129906,\n      -0.175856351852417,\n      -0.17063723504543304,\n      -0.13331592082977295,\n      0.1563485711812973,\n      -0.35932809114456177,\n      0.2402045577764511,\n      0.11359019577503204,\n      -0.3508605360984802,\n      0.1626836508512497,\n      0.08251789957284927,\n      -0.007251400034874678,\n      0.1242891326546669,\n      -0.05476716160774231,\n      0.04078057408332825,\n      0.10900809615850449,\n      0.07841342687606812,\n      0.005295031704008579,\n      -0.15297219157218933,\n      0.081606924533844,\n      1.4746936559677124,\n      -0.3614385426044464,\n      0.00883118249475956,\n      0.24193428456783295,\n      -0.36414411664009094,\n      -0.005029651336371899,\n      0.14590124785900116,\n      0.13412189483642578,\n      -0.20834799110889435\n    ],\n    [\n      0.0017286190995946527,\n      -0.29636669158935547,\n      0.12899118661880493,\n      0.021900804713368416,\n      -0.8205260038375854,\n      0.03870779275894165,\n      0.3082137405872345,\n      -0.27012842893600464,\n      -0.4601005017757416,\n      -0.1473998725414276,\n      -0.20546512305736542,\n      -0.06842319667339325,\n      -0.09020210057497025,\n      0.07849380373954773,\n      -0.0466180220246315,\n      -0.1208835169672966,\n      0.10383602976799011,\n      -0.07716105133295059,\n      -0.0670761689543724,\n      0.3651956021785736,\n      0.339857816696167,\n      0.1663070172071457,\n      0.16690407693386078,\n      -0.1033751592040062,\n      -0.15468557178974152,\n      -0.029798904433846474,\n      0.16725780069828033,\n      -0.048754703253507614,\n      -0.1269257366657257,\n      0.2151537537574768,\n      0.03523479402065277,\n      0.032580748200416565,\n      0.05485229939222336,\n      0.09916841983795166,\n      -0.4070702791213989,\n      0.07546021044254303,\n      0.07813485711812973,\n      -0.15219397842884064,\n      -0.1822328418493271,\n      -0.040962252765893936,\n      0.016940148547291756,\n      -0.13055910170078278,\n      0.023234253749251366,\n      0.32189542055130005,\n      0.13369740545749664,\n      -0.15954746305942535,\n      0.39861297607421875,\n      -0.3328644931316376,\n      -0.19362740218639374,\n      -0.08363735675811768\n    ],\n    [\n      0.5043299794197083,\n      0.02933916263282299,\n      -0.05473920330405235,\n      -0.16243843734264374,\n      -0.0033856581430882215,\n      -0.008918536826968193,\n      -0.37827372550964355,\n      -0.28094935417175293,\n      -0.6361277103424072,\n      -0.04514237120747566,\n      0.19721893966197968,\n      -0.1097516119480133,\n      -0.061492957174777985,\n      -0.037738874554634094,\n      -0.08123360574245453,\n      -0.04192535579204559,\n      -0.06722385436296463,\n      0.06183044612407684,\n      -0.17703767120838165,\n      -0.04018424078822136,\n      0.1477927267551422,\n      0.10745926946401596,\n      0.1298968642950058,\n      0.054513223469257355,\n      -0.055118344724178314,\n      -0.23359888792037964,\n      -0.049808070063591,\n      -0.03921375051140785,\n      0.09473969787359238,\n      -0.13521595299243927,\n      0.22287286818027496,\n      -0.14509360492229462,\n      -0.08668286353349686,\n      0.09165522456169128,\n      0.039056647568941116,\n      -0.1710348129272461,\n      -0.026342878118157387,\n      -0.15309864282608032,\n      -0.05485689267516136,\n      0.10827630013227463,\n      -0.21201424300670624,\n      0.09953511506319046,\n      -0.11895919591188431,\n      -0.19564299285411835,\n      -0.3011750280857086,\n      -0.03419423848390579,\n      0.26703891158103943,\n      0.22840015590190887,\n      0.03957481309771538,\n      0.008850247599184513\n    ],\n    [\n      0.014539148658514023,\n      -0.19415642321109772,\n      0.038521599024534225,\n      -0.10992012172937393,\n      -0.12053506076335907,\n      0.07606638967990875,\n      -0.4863470196723938,\n      -0.07122555375099182,\n      0.1989845186471939,\n      0.05736089497804642,\n      -0.10156740248203278,\n      0.10494445264339447,\n      0.1390063762664795,\n      0.15094423294067383,\n      -0.20805954933166504,\n      -0.09563680738210678,\n      -0.13577350974082947,\n      -0.39974910020828247,\n      0.3944534659385681,\n      -0.2543533444404602,\n      0.4441564381122589,\n      0.1579088568687439,\n      -0.015148086473345757,\n      -0.09533890336751938,\n      -0.10510647296905518,\n      0.06381350010633469,\n      0.08867611736059189,\n      0.12296668440103531,\n      -0.13843034207820892,\n      -0.2037200629711151,\n      0.2831701338291168,\n      0.18194197118282318,\n      -0.024112846702337265,\n      -0.15392379462718964,\n      -0.14190664887428284,\n      -0.4160538911819458,\n      -0.24859222769737244,\n      0.05844549089670181,\n      0.08600461483001709,\n      -0.1156730130314827,\n      -0.0439031608402729,\n      0.34434834122657776,\n      0.037723857909440994,\n      0.23521187901496887,\n      0.29830050468444824,\n      0.13074417412281036,\n      0.0344058983027935,\n      -0.0719732716679573,\n      0.14540372788906097,\n      0.08780737966299057\n    ],\n    [\n      0.1948910653591156,\n      0.10733214765787125,\n      0.07530472427606583,\n      -0.05624249204993248,\n      0.05128409340977669,\n      0.03389812260866165,\n      -0.3410155475139618,\n      0.040449075400829315,\n      -0.7724848985671997,\n      -0.043709542602300644,\n      0.0667811706662178,\n      0.11910329759120941,\n      -0.03705789893865585,\n      0.038933269679546356,\n      -0.036780595779418945,\n      -0.03514090180397034,\n      -0.0718931034207344,\n      -0.07081571221351624,\n      -0.13524779677391052,\n      0.18792875111103058,\n      0.10655559599399567,\n      -0.08006305992603302,\n      -0.07910136133432388,\n      -0.0822816789150238,\n      -0.1847568154335022,\n      0.1027187779545784,\n      0.093699149787426,\n      0.006496126297861338,\n      0.034104686230421066,\n      0.06638153642416,\n      0.19322116672992706,\n      0.11322712153196335,\n      0.03854367509484291,\n      0.35975709557533264,\n      0.22882188856601715,\n      -0.02174290083348751,\n      -0.042293619364500046,\n      -0.07074176520109177,\n      -0.09086727350950241,\n      -0.009322679601609707,\n      -0.1463681012392044,\n      -0.061317622661590576,\n      -0.20099478960037231,\n      0.018240826204419136,\n      0.28795740008354187,\n      -0.13272546231746674,\n      0.24151937663555145,\n      0.07994650304317474,\n      0.07418253272771835,\n      -0.04687792807817459\n    ],\n    [\n      0.14904893934726715,\n      0.03294271230697632,\n      -0.05604713410139084,\n      0.12445636838674545,\n      -0.16884562373161316,\n      0.09113889187574387,\n      -0.2547721564769745,\n      0.3371550738811493,\n      -0.020004047080874443,\n      -0.11367616802453995,\n      0.009321744553744793,\n      0.052388351410627365,\n      0.01977723464369774,\n      0.09822718799114227,\n      -0.18978305160999298,\n      0.08961889892816544,\n      0.043629005551338196,\n      0.6816402673721313,\n      0.012297078967094421,\n      0.1364172101020813,\n      -0.1822156310081482,\n      -0.30174773931503296,\n      0.1627763956785202,\n      0.2689543664455414,\n      -0.07061351090669632,\n      0.021682893857359886,\n      -0.00846355501562357,\n      -0.12919069826602936,\n      -0.150018110871315,\n      0.0002967237960547209,\n      -0.0769563838839531,\n      0.27735134959220886,\n      -0.11921149492263794,\n      -0.08938027918338776,\n      0.054333824664354324,\n      -0.28033092617988586,\n      -0.05925486609339714,\n      -0.18247118592262268,\n      -0.04613292217254639,\n      -0.0913863256573677,\n      -0.08701692521572113,\n      0.05033797025680542,\n      -0.019510582089424133,\n      -0.03627089411020279,\n      -0.11111953854560852,\n      -0.014850755222141743,\n      0.006325642112642527,\n      -0.1459978073835373,\n      -0.11259117722511292,\n      -0.004364136140793562\n    ],\n    [\n      -0.24406494200229645,\n      0.1273249089717865,\n      -0.1886400580406189,\n      0.27847766876220703,\n      -0.1110217347741127,\n      0.10779301822185516,\n      -0.06219610944390297,\n      0.07278867065906525,\n      -0.4061797857284546,\n      0.08739645779132843,\n      -0.04865654557943344,\n      0.16185183823108673,\n      0.19424790143966675,\n      0.12576375901699066,\n      -0.009495723061263561,\n      0.15745417773723602,\n      -0.03710554540157318,\n      -0.05435856431722641,\n      -0.05476585775613785,\n      0.13533836603164673,\n      0.0037718971725553274,\n      0.24694278836250305,\n      0.1179020032286644,\n      -0.3288687765598297,\n      -0.02020018734037876,\n      0.1355099081993103,\n      -0.0046472265385091305,\n      0.03521103039383888,\n      0.05094588175415993,\n      -0.08995381742715836,\n      -0.2566852271556854,\n      0.08522021770477295,\n      -0.018191896378993988,\n      -0.034994401037693024,\n      -0.02247491292655468,\n      0.047683797776699066,\n      -0.01267529558390379,\n      -0.09098123013973236,\n      0.08542497456073761,\n      0.1833869367837906,\n      0.014979297295212746,\n      0.14905743300914764,\n      0.09230607002973557,\n      -0.05548985302448273,\n      -0.00448051979765296,\n      -0.1330123245716095,\n      0.1836729794740677,\n      -0.10402466356754303,\n      0.12758910655975342,\n      -0.07325763255357742\n    ],\n    [\n      -0.34983691573143005,\n      0.029349736869335175,\n      0.015143786557018757,\n      -0.024000581353902817,\n      -0.041852738708257675,\n      -0.17630594968795776,\n      0.2558605372905731,\n      0.6658105850219727,\n      -0.13473697006702423,\n      -0.010789003223180771,\n      -0.012910774908959866,\n      0.34538307785987854,\n      0.04805923253297806,\n      -0.3819877803325653,\n      -0.13596820831298828,\n      -0.07875337451696396,\n      -0.35911571979522705,\n      -0.1290648728609085,\n      0.11668400466442108,\n      -0.06212998926639557,\n      0.07615309208631516,\n      0.20379452407360077,\n      0.0906694084405899,\n      -0.37926220893859863,\n      0.30074504017829895,\n      0.27890488505363464,\n      0.17186929285526276,\n      -0.27450427412986755,\n      -0.05733219534158707,\n      0.08596591651439667,\n      -0.02495640143752098,\n      0.0465351939201355,\n      -0.18718832731246948,\n      0.020646261051297188,\n      -0.11993958801031113,\n      -0.03527653589844704,\n      -0.14927884936332703,\n      -0.2353937178850174,\n      0.13740003108978271,\n      -0.1291528344154358,\n      0.04958930239081383,\n      -0.3061542809009552,\n      -0.04082923009991646,\n      -0.030613334849476814,\n      0.14896270632743835,\n      0.14972452819347382,\n      0.2523791790008545,\n      0.06030091270804405,\n      0.2840847373008728,\n      0.18372304737567902\n    ],\n    [\n      -0.640332043170929,\n      0.20779047906398773,\n      -0.08499431610107422,\n      0.10804305970668793,\n      -0.048400089144706726,\n      -0.1458330601453781,\n      -0.1191936805844307,\n      0.20485572516918182,\n      0.5685217380523682,\n      -0.06178558990359306,\n      -0.08551926165819168,\n      0.0342237614095211,\n      -0.1439550220966339,\n      -0.03734762594103813,\n      0.032042328268289566,\n      -0.029078390449285507,\n      -0.10644522309303284,\n      0.003994960803538561,\n      -0.04053428769111633,\n      -0.008517459966242313,\n      -0.20613455772399902,\n      -7.540907245129347e-05,\n      0.1142420843243599,\n      0.15708111226558685,\n      0.033335164189338684,\n      -0.0532553531229496,\n      -0.0877905786037445,\n      0.023365048691630363,\n      0.025555633008480072,\n      -0.03495530039072037,\n      -0.028069598600268364,\n      -0.10584601014852524,\n      0.08135060966014862,\n      -0.0490114688873291,\n      0.15328824520111084,\n      -0.10053591430187225,\n      0.08681013435125351,\n      0.09900889545679092,\n      -0.03704693913459778,\n      -0.0005907650920562446,\n      -0.06875500828027725,\n      -0.05879843607544899,\n      0.09450006484985352,\n      -0.12530623376369476,\n      0.026538193225860596,\n      0.032347675412893295,\n      -0.04629650339484215,\n      -0.002947817323729396,\n      0.02928619086742401,\n      0.09912247955799103\n    ],\n    [\n      0.08140002191066742,\n      0.11167503148317337,\n      -0.16508185863494873,\n      -0.016303610056638718,\n      -0.28787121176719666,\n      -0.004064563661813736,\n      -0.18876129388809204,\n      0.5135671496391296,\n      -0.11337043344974518,\n      0.23118965327739716,\n      0.09550630301237106,\n      -0.22093386948108673,\n      0.12129199504852295,\n      -0.021307149901986122,\n      -0.12802977859973907,\n      -0.04884404316544533,\n      0.058712977916002274,\n      0.4270690977573395,\n      -0.3236040771007538,\n      -0.011830085888504982,\n      0.2583455443382263,\n      -0.059441644698381424,\n      0.13529072701931,\n      -0.28530755639076233,\n      0.24729283154010773,\n      -0.2194400429725647,\n      -0.1225767657160759,\n      -0.022761158645153046,\n      -0.11532733589410782,\n      0.10048070549964905,\n      0.21185053884983063,\n      0.19006019830703735,\n      0.041902948170900345,\n      -0.23339366912841797,\n      -0.0668245330452919,\n      -0.019187025725841522,\n      -0.09128252416849136,\n      -0.014576097950339317,\n      -0.008117320016026497,\n      0.021607769653201103,\n      0.1765657365322113,\n      -0.15659305453300476,\n      -0.3847407102584839,\n      0.0013599067460745573,\n      0.21256044507026672,\n      -0.005353284999728203,\n      -0.011456020176410675,\n      -0.019491344690322876,\n      0.19788216054439545,\n      0.17237816751003265\n    ],\n    [\n      0.28953632712364197,\n      0.13925378024578094,\n      0.08205343782901764,\n      -0.27908602356910706,\n      0.2428979128599167,\n      0.22484800219535828,\n      -0.000367651111446321,\n      0.14206978678703308,\n      -0.15443964302539825,\n      -0.13654489815235138,\n      -0.12298039346933365,\n      -0.30113714933395386,\n      -0.12072452157735825,\n      -0.22744733095169067,\n      -0.0940217450261116,\n      -0.0856209546327591,\n      -0.08591378480195999,\n      -0.327876478433609,\n      0.05327128991484642,\n      -0.15295302867889404,\n      0.07787638902664185,\n      -0.38026657700538635,\n      0.09214664995670319,\n      -0.16331323981285095,\n      -0.2683504819869995,\n      -0.2307659089565277,\n      -0.2738270163536072,\n      -0.3057094216346741,\n      -0.05567522346973419,\n      -0.1377234309911728,\n      0.000619811937212944,\n      -0.08049922436475754,\n      0.1183396652340889,\n      0.21825575828552246,\n      0.036955755203962326,\n      0.012632995843887329,\n      -0.012604935094714165,\n      -0.25250035524368286,\n      -0.2135806381702423,\n      -0.2747129201889038,\n      -0.18288195133209229,\n      0.2562117874622345,\n      0.06176943704485893,\n      -0.022785678505897522,\n      0.019122710451483727,\n      -0.15597209334373474,\n      0.036147069185972214,\n      -0.13895003497600555,\n      -0.31651827692985535,\n      -0.293058842420578\n    ],\n    [\n      -0.40370023250579834,\n      -0.26545995473861694,\n      0.07702212780714035,\n      -0.03483361378312111,\n      -0.04057028144598007,\n      0.009049269370734692,\n      -0.41229772567749023,\n      -0.10301266610622406,\n      0.6426098346710205,\n      -0.1343693733215332,\n      -0.11030065268278122,\n      -0.08711303025484085,\n      0.04561755433678627,\n      -0.01477182935923338,\n      0.07685907930135727,\n      0.013666828162968159,\n      0.030446909368038177,\n      0.3382207155227661,\n      0.26781556010246277,\n      -0.12564462423324585,\n      0.13443240523338318,\n      0.010743866674602032,\n      0.1676478087902069,\n      0.023253431543707848,\n      0.037907376885414124,\n      0.04349902644753456,\n      -0.00830759759992361,\n      -0.0734035074710846,\n      -0.07857763767242432,\n      0.15108488500118256,\n      -0.3077767491340637,\n      0.0904880240559578,\n      -0.13301825523376465,\n      -0.014929281547665596,\n      0.21297699213027954,\n      0.04177762195467949,\n      -0.13798144459724426,\n      -0.1464461386203766,\n      -0.12354304641485214,\n      0.13135926425457,\n      0.09041213244199753,\n      0.02548542618751526,\n      0.06575259566307068,\n      0.14348329603672028,\n      0.1455567479133606,\n      0.02482803538441658,\n      -0.10804501175880432,\n      -0.10096610337495804,\n      -0.08574295789003372,\n      -0.11648634821176529\n    ],\n    [\n      0.15047970414161682,\n      -0.027716053649783134,\n      -0.0006615031743422151,\n      0.27561378479003906,\n      -0.5912758708000183,\n      -0.1774587780237198,\n      -0.09519919753074646,\n      -0.022754359990358353,\n      0.08943246304988861,\n      0.1652323305606842,\n      0.24393446743488312,\n      0.19348347187042236,\n      0.16190621256828308,\n      -0.012713060714304447,\n      0.03360253944993019,\n      0.09779774397611618,\n      0.04535717889666557,\n      0.3193015158176422,\n      -0.13000644743442535,\n      0.3585076332092285,\n      0.024427462369203568,\n      0.1089487075805664,\n      0.177158385515213,\n      0.09568452835083008,\n      0.15790292620658875,\n      0.1327485591173172,\n      0.10913743078708649,\n      0.14023606479167938,\n      -0.023236848413944244,\n      0.26020923256874084,\n      0.03127178177237511,\n      0.018175894394516945,\n      -0.10059888660907745,\n      0.18312789499759674,\n      0.21881099045276642,\n      -0.014637072570621967,\n      0.02370334230363369,\n      0.15286704897880554,\n      0.11374657601118088,\n      -0.025766681879758835,\n      0.184790700674057,\n      -0.21694129705429077,\n      0.21411442756652832,\n      -0.01428288035094738,\n      -0.14639964699745178,\n      -0.41226255893707275,\n      -0.056105516850948334,\n      0.1187678873538971,\n      0.030638612806797028,\n      0.15861815214157104\n    ],\n    [\n      0.2719388008117676,\n      0.051114700734615326,\n      -0.021590935066342354,\n      0.11037538200616837,\n      0.005218899343162775,\n      -0.24564507603645325,\n      -0.21391674876213074,\n      -0.21522453427314758,\n      -0.002770385704934597,\n      -0.12921330332756042,\n      0.08126945048570633,\n      -0.011634992435574532,\n      -0.06803472340106964,\n      -0.08509702235460281,\n      0.07274820655584335,\n      0.006345059722661972,\n      -0.1391957849264145,\n      0.02819894813001156,\n      0.09393993765115738,\n      0.004709532484412193,\n      -0.27863526344299316,\n      0.023264659568667412,\n      0.05659421160817146,\n      -0.001945538679137826,\n      0.19896823167800903,\n      0.01705237850546837,\n      -0.08771692216396332,\n      0.11755639314651489,\n      -0.04157475009560585,\n      -0.005129705183207989,\n      0.22630387544631958,\n      0.1541735678911209,\n      -0.2246512919664383,\n      -0.42359286546707153,\n      -0.0958336815237999,\n      -0.3300692141056061,\n      -0.056562066078186035,\n      -0.008985496126115322,\n      -0.017234142869710922,\n      -0.03963775187730789,\n      0.10739342868328094,\n      -0.0009831369388848543,\n      0.41310617327690125,\n      0.052054263651371,\n      0.1778654009103775,\n      -0.017430130392313004,\n      -0.03303205966949463,\n      -0.05539882928133011,\n      0.01042916439473629,\n      0.0007579390658065677\n    ],\n    [\n      0.27128544449806213,\n      -0.034505296498537064,\n      -0.040213242173194885,\n      0.11318036168813705,\n      -0.13346385955810547,\n      -0.12872856855392456,\n      0.3434050381183624,\n      0.18842801451683044,\n      0.2228400558233261,\n      -0.06021462753415108,\n      -0.18834863603115082,\n      -0.19642165303230286,\n      -0.12409616261720657,\n      0.056384842842817307,\n      0.10052890330553055,\n      -0.16269949078559875,\n      -0.05096128210425377,\n      0.05027107149362564,\n      -0.043893761932849884,\n      -0.02944754809141159,\n      -0.29064738750457764,\n      0.08529970794916153,\n      0.2068440467119217,\n      0.24067099392414093,\n      -0.19966141879558563,\n      -0.14617855846881866,\n      -0.1142144575715065,\n      0.032301779836416245,\n      -0.10453588515520096,\n      0.043026331812143326,\n      -0.07375939935445786,\n      -0.0004318213032092899,\n      -0.19142428040504456,\n      0.12415096908807755,\n      0.11825786530971527,\n      0.14684507250785828,\n      0.11072612553834915,\n      -0.1917625218629837,\n      0.018479736521840096,\n      -0.12214773148298264,\n      -0.1168082058429718,\n      0.008978615514934063,\n      -0.12634414434432983,\n      0.034716419875621796,\n      0.012990223243832588,\n      0.1821407526731491,\n      -0.006598673760890961,\n      -0.09375396370887756,\n      -0.08710478246212006,\n      -0.04356458783149719\n    ],\n    [\n      -0.14505217969417572,\n      -0.08822640031576157,\n      -0.09047786891460419,\n      -0.05736172944307327,\n      -0.10070823132991791,\n      0.04783453047275543,\n      -0.13246966898441315,\n      0.09166441112756729,\n      0.05616271495819092,\n      -0.08501573652029037,\n      -0.1131853237748146,\n      -0.031512800604104996,\n      0.019897066056728363,\n      -0.06316599994897842,\n      0.017173796892166138,\n      0.017270075157284737,\n      -0.1585969477891922,\n      0.14174430072307587,\n      -0.05511895567178726,\n      -0.28183305263519287,\n      0.35783934593200684,\n      -0.1131209284067154,\n      0.0715881735086441,\n      0.019376620650291443,\n      -0.07771976292133331,\n      -0.12327621877193451,\n      0.029345929622650146,\n      -0.003469866933301091,\n      -0.004970467183738947,\n      -0.13360603153705597,\n      -0.17727480828762054,\n      0.243547722697258,\n      -0.13105089962482452,\n      0.2520577907562256,\n      0.04241148382425308,\n      0.17847082018852234,\n      0.10364316403865814,\n      -0.10709940642118454,\n      0.03209512680768967,\n      -0.04149705916643143,\n      -0.10761497914791107,\n      0.01688421703875065,\n      -0.12918278574943542,\n      -0.07636406272649765,\n      -0.008100989274680614,\n      -0.008867583237588406,\n      0.0695391297340393,\n      -0.08652527630329132,\n      -0.0012031991500407457,\n      0.09891438484191895\n    ],\n    [\n      0.09270425140857697,\n      0.01849387213587761,\n      0.07326843589544296,\n      0.2807095944881439,\n      0.32248198986053467,\n      -0.1775675117969513,\n      0.2282128632068634,\n      0.13353268802165985,\n      -0.5970945954322815,\n      0.14227305352687836,\n      0.2758938670158386,\n      0.016873233020305634,\n      0.18764472007751465,\n      -0.1807778924703598,\n      -0.1359180063009262,\n      -0.060975007712841034,\n      0.03175679221749306,\n      0.17449341714382172,\n      -0.03478042408823967,\n      -0.10046610981225967,\n      -0.05259787291288376,\n      -0.20447556674480438,\n      0.029074056074023247,\n      -0.029397545382380486,\n      0.3199222981929779,\n      0.1782207489013672,\n      0.10035109519958496,\n      0.20349876582622528,\n      0.05247572436928749,\n      0.3554273247718811,\n      -0.07660800963640213,\n      0.09147784113883972,\n      -0.14886680245399475,\n      0.04057469964027405,\n      -0.09090473502874374,\n      -0.37720999121665955,\n      0.01284798700362444,\n      -0.09535309672355652,\n      0.10586939007043839,\n      0.22367918491363525,\n      0.26421523094177246,\n      -0.3054308593273163,\n      -0.06356818228960037,\n      -0.01886872760951519,\n      0.061030007898807526,\n      -0.24965900182724,\n      0.44639110565185547,\n      0.09032124280929565,\n      0.06997677683830261,\n      -0.030646275728940964\n    ],\n    [\n      0.040634579956531525,\n      -0.041786983609199524,\n      -0.004726939834654331,\n      -0.2711052894592285,\n      -0.08454452455043793,\n      -0.19860659539699554,\n      0.1057829037308693,\n      0.2179684340953827,\n      0.05724979192018509,\n      -0.1391294151544571,\n      -0.09115983545780182,\n      0.03180735558271408,\n      -0.08666892349720001,\n      -0.10232199728488922,\n      -0.062443170696496964,\n      0.0823306143283844,\n      -0.18210220336914062,\n      -0.05019431561231613,\n      0.04339571297168732,\n      -0.2370556741952896,\n      0.2606472074985504,\n      0.030337508767843246,\n      -0.026345720514655113,\n      0.05619506910443306,\n      -0.015786558389663696,\n      -0.06574410200119019,\n      -0.11704316735267639,\n      0.030230991542339325,\n      -0.08904655277729034,\n      -0.15951402485370636,\n      -0.08493602275848389,\n      -0.054335422813892365,\n      -0.023891029879450798,\n      0.008293717168271542,\n      0.03272848203778267,\n      0.2292482852935791,\n      -0.009982968680560589,\n      0.1896837204694748,\n      0.15538348257541656,\n      0.016106735914945602,\n      -0.06298046559095383,\n      -0.2612428665161133,\n      -0.21094749867916107,\n      0.10689125210046768,\n      0.19038864970207214,\n      0.05840088054537773,\n      0.029079223051667213,\n      -0.3017039895057678,\n      -0.07343736290931702,\n      0.07662153989076614\n    ],\n    [\n      0.4725506901741028,\n      -0.0707649514079094,\n      -0.0010563131654635072,\n      0.012278818525373936,\n      0.0006784925935789943,\n      -0.0761190801858902,\n      -0.032154519110918045,\n      -0.14003491401672363,\n      -0.4124128818511963,\n      -0.005391787737607956,\n      0.08450250327587128,\n      0.12559248507022858,\n      -0.08261049538850784,\n      -0.057461515069007874,\n      -0.06466091424226761,\n      0.12772221863269806,\n      -0.04195520654320717,\n      -0.16468581557273865,\n      -0.2486467957496643,\n      -0.1561962366104126,\n      0.034249451011419296,\n      0.08950048685073853,\n      0.12543636560440063,\n      -0.1998712569475174,\n      -0.0057386658154428005,\n      0.23718200623989105,\n      -0.11690044403076172,\n      -0.03930367901921272,\n      -0.13919174671173096,\n      -0.0647430419921875,\n      0.017890939489006996,\n      -0.08938387036323547,\n      0.0003753116470761597,\n      0.08660537749528885,\n      -0.08902590721845627,\n      -0.13831213116645813,\n      -0.11644105613231659,\n      -0.009857592172920704,\n      -0.1723719984292984,\n      -0.12096800655126572,\n      -0.11862426996231079,\n      0.02769295871257782,\n      -0.10325272381305695,\n      0.2045702040195465,\n      0.10770023614168167,\n      0.01699814759194851,\n      0.07911045104265213,\n      0.12524709105491638,\n      0.12873625755310059,\n      0.18945471942424774\n    ],\n    [\n      0.2773967683315277,\n      0.13559141755104065,\n      -0.04928886145353317,\n      0.06578843295574188,\n      -0.07673446089029312,\n      0.15375176072120667,\n      0.017515920102596283,\n      0.3560907244682312,\n      0.009264444001019001,\n      0.09053517878055573,\n      -0.03693915903568268,\n      0.09463252872228622,\n      0.07256320863962173,\n      -0.05176682025194168,\n      0.07373547554016113,\n      0.08917587250471115,\n      -0.00418639462441206,\n      -0.06478241086006165,\n      0.09732762724161148,\n      0.10644177347421646,\n      0.5426039099693298,\n      -0.2068176567554474,\n      0.07323566824197769,\n      -0.12867364287376404,\n      0.09803669154644012,\n      0.19145753979682922,\n      0.219893217086792,\n      -0.09613620489835739,\n      -0.035784944891929626,\n      0.13890103995800018,\n      0.001283115241676569,\n      0.03863733261823654,\n      0.2056814730167389,\n      -0.31846553087234497,\n      0.12756827473640442,\n      0.12951341271400452,\n      -0.07387100905179977,\n      0.10622318834066391,\n      -0.10908738523721695,\n      0.16063781082630157,\n      0.09744976460933685,\n      -0.07971087843179703,\n      0.33566004037857056,\n      -0.030874568969011307,\n      -0.040626101195812225,\n      -0.10153461992740631,\n      0.14139534533023834,\n      0.17626211047172546,\n      0.04627908021211624,\n      0.16908465325832367\n    ],\n    [\n      -0.029644805938005447,\n      0.039922554045915604,\n      -0.009453613311052322,\n      0.10858519375324249,\n      -0.15345945954322815,\n      -0.08221570402383804,\n      0.0035270245280116796,\n      0.25997811555862427,\n      -0.12129528820514679,\n      0.030629996210336685,\n      0.36211124062538147,\n      0.007379202172160149,\n      0.2127171754837036,\n      -0.33370524644851685,\n      -0.057397570461034775,\n      0.11017102003097534,\n      -0.14327389001846313,\n      0.1713346242904663,\n      -0.0006836273241788149,\n      0.1010264977812767,\n      -0.05652341619133949,\n      0.10622099786996841,\n      -0.06417026370763779,\n      0.15002961456775665,\n      0.05596431344747543,\n      0.01955343782901764,\n      0.06558211147785187,\n      -0.02647310122847557,\n      0.12263336032629013,\n      0.21030369400978088,\n      -0.31074997782707214,\n      0.09510210156440735,\n      -0.023042267188429832,\n      -0.4396923780441284,\n      0.0303414985537529,\n      0.0929565280675888,\n      0.22393615543842316,\n      -0.09100788086652756,\n      -0.0017129839397966862,\n      0.18795344233512878,\n      0.06402703374624252,\n      -0.35313957929611206,\n      -0.14971859753131866,\n      0.043002840131521225,\n      0.20444394648075104,\n      -0.15163937211036682,\n      0.20621109008789062,\n      0.028167013078927994,\n      0.10050041973590851,\n      0.30244317650794983\n    ],\n    [\n      0.456930547952652,\n      0.1221766397356987,\n      -0.08020742982625961,\n      0.061357054859399796,\n      0.23213061690330505,\n      -0.1395331472158432,\n      -0.08100398629903793,\n      -0.05509378761053085,\n      0.2811223268508911,\n      0.11370626837015152,\n      0.03217261657118797,\n      -0.058322541415691376,\n      0.2134818285703659,\n      0.1010127067565918,\n      0.09025060385465622,\n      -0.024388650432229042,\n      -0.220816507935524,\n      0.5650851130485535,\n      -0.1168704479932785,\n      0.01818452961742878,\n      0.23495641350746155,\n      0.03327227756381035,\n      0.2393622249364853,\n      -0.14648421108722687,\n      0.17737682163715363,\n      0.04337174817919731,\n      0.24274784326553345,\n      0.02937503531575203,\n      0.05534271523356438,\n      0.11269079893827438,\n      0.13742701709270477,\n      0.10822659730911255,\n      -0.16574904322624207,\n      -0.1457739621400833,\n      0.09865637868642807,\n      -0.09658871591091156,\n      0.11719716340303421,\n      0.16998955607414246,\n      -0.09988456964492798,\n      0.018834955990314484,\n      0.11984183639287949,\n      -0.3916129171848297,\n      -0.07522542029619217,\n      -0.04351285845041275,\n      0.07041560113430023,\n      0.008226844482123852,\n      0.24201121926307678,\n      0.008937366306781769,\n      0.22946585714817047,\n      0.1061464473605156\n    ],\n    [\n      -0.6226834654808044,\n      0.1577286720275879,\n      0.09149634093046188,\n      0.06977380067110062,\n      -0.18204942345619202,\n      0.0026791475247591734,\n      -0.031615253537893295,\n      0.34477415680885315,\n      0.5521295666694641,\n      0.01754019595682621,\n      0.10202956944704056,\n      0.08277824521064758,\n      -0.013213773258030415,\n      -0.05033215507864952,\n      0.185551717877388,\n      -0.11133843660354614,\n      0.09491710364818573,\n      -0.0716095045208931,\n      -0.02940540574491024,\n      0.01840464025735855,\n      -0.30513039231300354,\n      -0.06284163147211075,\n      -0.15082934498786926,\n      0.2214691936969757,\n      -0.046946991235017776,\n      0.31403499841690063,\n      -0.2720049321651459,\n      -0.08731499314308167,\n      0.15515387058258057,\n      -0.12236860394477844,\n      0.061632946133613586,\n      -0.1314220428466797,\n      -0.04200713708996773,\n      0.04582309350371361,\n      0.18102200329303741,\n      -0.08186706155538559,\n      -0.17333298921585083,\n      -0.0437382273375988,\n      -0.11306498944759369,\n      0.09991008788347244,\n      -0.27315473556518555,\n      -0.08518975228071213,\n      -0.03632856905460358,\n      -0.04184344410896301,\n      -0.16806446015834808,\n      0.06292176991701126,\n      -0.04861863702535629,\n      -0.001719761174172163,\n      -0.057388678193092346,\n      0.0659494698047638\n    ],\n    [\n      0.2884177565574646,\n      -0.010983631946146488,\n      -0.1404755562543869,\n      0.033394746482372284,\n      0.22174537181854248,\n      -0.1724023073911667,\n      -0.11924140155315399,\n      0.029763113707304,\n      0.31867626309394836,\n      0.33455321192741394,\n      0.3054446876049042,\n      -0.09692668169736862,\n      0.0720156878232956,\n      0.16218630969524384,\n      0.016010409221053123,\n      0.03132975473999977,\n      0.17855195701122284,\n      0.17052291333675385,\n      -0.5121533870697021,\n      -0.07381009310483932,\n      0.07293107360601425,\n      -0.12198085337877274,\n      0.06550239771604538,\n      -0.395210325717926,\n      0.3121144771575928,\n      0.005597841460257769,\n      0.1475629359483719,\n      0.1215367540717125,\n      -0.05871628224849701,\n      0.22869478166103363,\n      0.1438530832529068,\n      0.08861490339040756,\n      -0.1385779231786728,\n      -0.03857434168457985,\n      -0.3949616849422455,\n      -0.047372158616781235,\n      -0.03692879527807236,\n      -0.02800326608121395,\n      0.11376316845417023,\n      0.07964926958084106,\n      -0.028248030692338943,\n      -0.3887813091278076,\n      -0.5342361330986023,\n      -0.014697138220071793,\n      0.15319187939167023,\n      0.0678306519985199,\n      -0.15860189497470856,\n      0.14472848176956177,\n      0.164354145526886,\n      0.0969284325838089\n    ],\n    [\n      0.2190181165933609,\n      -0.1425599455833435,\n      0.12541070580482483,\n      0.0755985826253891,\n      0.10162222385406494,\n      -0.1610725224018097,\n      -0.20899319648742676,\n      -0.40275782346725464,\n      -0.05114948749542236,\n      0.007114431355148554,\n      0.06046966463327408,\n      -0.01772220805287361,\n      -0.015035121701657772,\n      0.08273761719465256,\n      -0.03072594851255417,\n      0.13723507523536682,\n      0.015117775648832321,\n      0.13686159253120422,\n      -0.5433467030525208,\n      0.017892425879836082,\n      0.07612787187099457,\n      -0.04838920384645462,\n      0.09207193553447723,\n      -0.1475532203912735,\n      -0.08544520288705826,\n      0.061323169618844986,\n      0.16742439568042755,\n      0.04234403744339943,\n      -0.028445107862353325,\n      -0.06999465823173523,\n      -0.09956683963537216,\n      -0.051237888634204865,\n      -0.10635238140821457,\n      0.12748008966445923,\n      -0.19070930778980255,\n      -0.04349978268146515,\n      -0.13375826179981232,\n      0.07114973664283752,\n      -0.0781472772359848,\n      -0.028947139158844948,\n      -0.06280547380447388,\n      -0.19341371953487396,\n      -0.3609750270843506,\n      -0.13872505724430084,\n      0.20092220604419708,\n      0.15623413026332855,\n      -0.06721185892820358,\n      -0.0020517848897725344,\n      0.055891603231430054,\n      0.11883547902107239\n    ],\n    [\n      -0.47828030586242676,\n      -0.08302593231201172,\n      0.0524149164557457,\n      0.003554358845576644,\n      0.18951033055782318,\n      -0.12029864639043808,\n      -0.05645671859383583,\n      0.43363115191459656,\n      -0.019281376153230667,\n      -0.08534195274114609,\n      -0.1865358054637909,\n      -0.08901272714138031,\n      -0.14880213141441345,\n      0.051652658730745316,\n      -0.09417679905891418,\n      0.007637328468263149,\n      -0.19189170002937317,\n      -0.13050860166549683,\n      -0.11254993826150894,\n      -0.32405033707618713,\n      -0.2777194082736969,\n      -0.14185889065265656,\n      0.20999757945537567,\n      0.2172175943851471,\n      -0.127560555934906,\n      -0.014814051799476147,\n      -0.15998123586177826,\n      -0.09628836065530777,\n      -0.029319796711206436,\n      -0.007068714592605829,\n      -0.153651162981987,\n      -0.11215078830718994,\n      -0.08092724531888962,\n      -0.08844387531280518,\n      0.17556941509246826,\n      0.050010375678539276,\n      -0.10287365317344666,\n      0.03412827104330063,\n      -0.13877913355827332,\n      0.06416868418455124,\n      -0.08528616279363632,\n      0.03641097620129585,\n      0.3598376512527466,\n      0.10457005351781845,\n      -0.14764903485774994,\n      0.292182594537735,\n      -0.1419627070426941,\n      0.24759948253631592,\n      -0.10592738538980484,\n      -0.007059503812342882\n    ],\n    [\n      -0.45163148641586304,\n      -0.1018272265791893,\n      -0.07289356738328934,\n      0.04459352046251297,\n      -0.15772487223148346,\n      -0.012633118778467178,\n      -0.1643056571483612,\n      0.08351481705904007,\n      0.3059665560722351,\n      0.07810242474079132,\n      -0.04822159186005592,\n      -0.09633099287748337,\n      -0.2707177698612213,\n      0.02860104665160179,\n      0.014985285699367523,\n      0.056927286088466644,\n      -0.03858556970953941,\n      0.38550958037376404,\n      -0.0025921252090483904,\n      -0.09847022593021393,\n      0.06606054306030273,\n      -0.10275039076805115,\n      -0.1335868090391159,\n      -0.01984809711575508,\n      0.044521264731884,\n      -0.033239562064409256,\n      -0.05125300586223602,\n      -0.14364424347877502,\n      -0.007887989282608032,\n      -0.019558945670723915,\n      -0.2727149426937103,\n      0.19749951362609863,\n      -0.01765814609825611,\n      0.13079717755317688,\n      -0.09112612158060074,\n      0.13669128715991974,\n      -0.045642510056495667,\n      0.05701948329806328,\n      -0.08645081520080566,\n      -0.17412710189819336,\n      -0.1071171686053276,\n      -0.2340719997882843,\n      -0.004226475488394499,\n      0.08209216594696045,\n      -0.021204860880970955,\n      0.0793803483247757,\n      0.10126471519470215,\n      -0.2067374289035797,\n      -0.034180812537670135,\n      0.03728773444890976\n    ],\n    [\n      0.30459949374198914,\n      0.05883834511041641,\n      -0.0035978492815047503,\n      -0.011735348962247372,\n      0.19201916456222534,\n      -0.046286582946777344,\n      0.017796853557229042,\n      0.24453838169574738,\n      -0.028066683560609818,\n      0.24635471403598785,\n      0.18341492116451263,\n      0.003382560797035694,\n      -0.16677629947662354,\n      0.06935539841651917,\n      -0.05163224786520004,\n      0.07643570005893707,\n      0.18089838325977325,\n      -0.12008261680603027,\n      -0.07525505870580673,\n      -0.255641907453537,\n      0.16987915337085724,\n      0.0033322139643132687,\n      0.013302769511938095,\n      -0.2779732048511505,\n      0.05220600962638855,\n      0.1224128007888794,\n      0.2072216421365738,\n      0.04005645215511322,\n      -0.06515853106975555,\n      -0.02490244060754776,\n      0.2200509011745453,\n      0.005016815382987261,\n      0.10448118299245834,\n      -0.11825264990329742,\n      0.06118488311767578,\n      -0.014172430150210857,\n      -0.022695256397128105,\n      0.010374571196734905,\n      -0.06743790954351425,\n      -0.020810319110751152,\n      -0.2599695324897766,\n      0.05792487785220146,\n      0.20769283175468445,\n      0.10088524222373962,\n      0.2654789686203003,\n      -0.24583955109119415,\n      0.06660150736570358,\n      -0.13160103559494019,\n      0.21885038912296295,\n      0.1315593421459198\n    ],\n    [\n      -0.3543088734149933,\n      0.04008827731013298,\n      -0.18533161282539368,\n      0.07098817080259323,\n      -0.19713909924030304,\n      -0.05629609525203705,\n      -0.07199406623840332,\n      0.1401071697473526,\n      -0.08555614203214645,\n      0.11671272665262222,\n      -0.01530794519931078,\n      -0.042058587074279785,\n      0.0999850258231163,\n      0.12289270758628845,\n      -0.15461696684360504,\n      -0.02898641675710678,\n      0.15530988574028015,\n      0.11316968500614166,\n      0.11738578975200653,\n      0.0020700471941381693,\n      -0.12447109073400497,\n      -0.015546406619250774,\n      0.07057557255029678,\n      -0.10753141343593597,\n      0.19772911071777344,\n      -0.012123038060963154,\n      0.06125253438949585,\n      -0.02063896134495735,\n      0.07075339555740356,\n      0.12840400636196136,\n      -0.21250072121620178,\n      0.30692172050476074,\n      -0.08622489869594574,\n      -0.46613162755966187,\n      0.04178687930107117,\n      0.1406787931919098,\n      0.11538715660572052,\n      0.05313187465071678,\n      0.0874624103307724,\n      0.18925710022449493,\n      0.13958491384983063,\n      0.08948597311973572,\n      -0.17451021075248718,\n      0.20644332468509674,\n      -0.04740868881344795,\n      0.06372709572315216,\n      0.07060345262289047,\n      -0.02246815711259842,\n      -0.08433465659618378,\n      -0.12315847724676132\n    ],\n    [\n      0.5794162750244141,\n      -0.017537502571940422,\n      0.014955430291593075,\n      -0.256772518157959,\n      0.12504388391971588,\n      -0.03435146436095238,\n      -0.03349125757813454,\n      0.343310683965683,\n      -0.24639485776424408,\n      -0.030131259933114052,\n      -0.14476749300956726,\n      -0.033926885575056076,\n      0.19736431539058685,\n      -0.0783672034740448,\n      0.05690150335431099,\n      -0.01157072652131319,\n      0.04712950810790062,\n      -0.30123627185821533,\n      0.054461557418107986,\n      0.14809204638004303,\n      0.03179607167840004,\n      0.02668347954750061,\n      0.0726996511220932,\n      -0.14425580203533173,\n      -0.032050784677267075,\n      0.10808844864368439,\n      -0.03246728703379631,\n      0.2442215383052826,\n      -0.1532435417175293,\n      -0.07659578323364258,\n      0.15857437252998352,\n      -0.12527328729629517,\n      0.14313435554504395,\n      -0.3494451940059662,\n      0.15489529073238373,\n      -0.2879970669746399,\n      0.03632771968841553,\n      0.10455991327762604,\n      0.1275724172592163,\n      -0.028510723263025284,\n      0.14504985511302948,\n      0.551957368850708,\n      -0.0684952586889267,\n      -0.16835109889507294,\n      -0.1312812715768814,\n      -0.1684080958366394,\n      0.15567755699157715,\n      -0.11753445118665695,\n      -0.16592387855052948,\n      -0.07338932901620865\n    ],\n    [\n      0.23991668224334717,\n      -0.0180264450609684,\n      -0.014837702736258507,\n      0.1301812380552292,\n      0.14607544243335724,\n      -0.07277470082044601,\n      0.08912700414657593,\n      -0.33732837438583374,\n      0.10037631541490555,\n      0.07985477894544601,\n      0.18808946013450623,\n      0.05012630298733711,\n      -0.014887124300003052,\n      0.06604204326868057,\n      0.0005500817205756903,\n      0.028741538524627686,\n      0.08529388159513474,\n      0.2604596018791199,\n      -0.2808876931667328,\n      -0.07850474864244461,\n      0.3873696029186249,\n      0.18533414602279663,\n      0.13631664216518402,\n      -0.12459415197372437,\n      0.19013993442058563,\n      -0.022525604814291,\n      -0.012276661582291126,\n      0.020731650292873383,\n      -0.11688441038131714,\n      0.05490725114941597,\n      0.13886836171150208,\n      -0.031336404383182526,\n      0.1942378729581833,\n      -0.036322128027677536,\n      -0.22164380550384521,\n      -0.2681078314781189,\n      -0.07622399181127548,\n      -0.023452598601579666,\n      0.022461216896772385,\n      -0.031164981424808502,\n      0.055314552038908005,\n      -0.20361272990703583,\n      -0.23492689430713654,\n      -0.06851640343666077,\n      0.1641259342432022,\n      -0.04192860424518585,\n      0.12114401906728745,\n      -0.31064170598983765,\n      0.09836088865995407,\n      0.22191976010799408\n    ],\n    [\n      0.1236395537853241,\n      0.12666933238506317,\n      -0.09197709709405899,\n      -0.005839409306645393,\n      0.011701659299433231,\n      0.134878009557724,\n      -0.1280127316713333,\n      0.02251252345740795,\n      -0.1244189664721489,\n      0.05282459408044815,\n      0.16619917750358582,\n      0.005332404747605324,\n      0.0457490049302578,\n      -0.03407835587859154,\n      -0.02787940949201584,\n      -0.006282927468419075,\n      -0.0372762605547905,\n      -0.0198616161942482,\n      0.000190340681001544,\n      0.05301978811621666,\n      0.024076322093605995,\n      0.2614472508430481,\n      0.028134966269135475,\n      0.12228650599718094,\n      -0.0431290864944458,\n      0.025440899655222893,\n      -0.07766345888376236,\n      0.044842224568128586,\n      0.0281049981713295,\n      0.0070953709073364735,\n      0.11333548277616501,\n      -0.060029346495866776,\n      0.1472339928150177,\n      -0.4472377300262451,\n      -0.006866642739623785,\n      0.011948881670832634,\n      -0.13811399042606354,\n      -0.005673236213624477,\n      0.1646929830312729,\n      0.06167198345065117,\n      0.1773473620414734,\n      0.008727982640266418,\n      0.37058451771736145,\n      0.054284363985061646,\n      0.22545263171195984,\n      0.24153722822666168,\n      -0.030681466683745384,\n      0.03457941114902496,\n      -0.002807611832395196,\n      -0.054477985948324203\n    ],\n    [\n      0.27277615666389465,\n      0.022845840081572533,\n      -0.01073552668094635,\n      -0.20833027362823486,\n      0.0759144201874733,\n      0.23329727351665497,\n      -0.19601508975028992,\n      0.17056621611118317,\n      -0.2747786343097687,\n      -0.3002575635910034,\n      -0.12134093046188354,\n      0.04868050664663315,\n      0.02015797235071659,\n      0.06344202905893326,\n      -0.05661065876483917,\n      0.00929902121424675,\n      0.047792792320251465,\n      -0.03304164111614227,\n      0.1069868803024292,\n      0.01130998320877552,\n      0.14880141615867615,\n      -0.2790745496749878,\n      -0.27151018381118774,\n      -0.06316789984703064,\n      -0.09916515648365021,\n      0.0038567387964576483,\n      -0.09402293711900711,\n      0.13603158295154572,\n      0.06031173840165138,\n      -0.282517671585083,\n      -0.06581494957208633,\n      -0.2219838947057724,\n      0.10996493697166443,\n      -0.018384091556072235,\n      0.06628625839948654,\n      -0.23241345584392548,\n      0.07930579036474228,\n      0.18168683350086212,\n      0.027870088815689087,\n      -0.07929727435112,\n      0.2921501100063324,\n      0.41576024889945984,\n      0.0745079442858696,\n      -0.0613350011408329,\n      0.07358646392822266,\n      -0.4220126271247864,\n      0.02249801717698574,\n      -0.16258007287979126,\n      -0.2000391036272049,\n      -0.22319571673870087\n    ],\n    [\n      -0.0385502353310585,\n      -0.05226923152804375,\n      -0.036529090255498886,\n      0.1236613318324089,\n      0.2681025564670563,\n      -0.03224341198801994,\n      -0.09992127120494843,\n      0.0010361898457631469,\n      -0.06564231961965561,\n      0.041852813214063644,\n      0.020085396245121956,\n      -0.23751047253608704,\n      -0.10153575241565704,\n      0.03695661947131157,\n      -0.011831986717879772,\n      0.15727394819259644,\n      -0.14977559447288513,\n      -0.12742877006530762,\n      -0.0973849669098854,\n      -0.025743264704942703,\n      0.1915695071220398,\n      0.264739453792572,\n      0.10179785639047623,\n      -0.003365443553775549,\n      -0.01918058656156063,\n      -0.16486136615276337,\n      0.14065031707286835,\n      0.03669837489724159,\n      -0.003221075050532818,\n      0.04809710755944252,\n      0.06384709477424622,\n      -0.10834203660488129,\n      0.020608432590961456,\n      -0.22896350920200348,\n      0.020371047779917717,\n      -0.18653810024261475,\n      0.0662861317396164,\n      -0.031302113085985184,\n      0.05160418525338173,\n      -0.03221552073955536,\n      -0.030375374481081963,\n      0.02165369875729084,\n      -0.028407655656337738,\n      -0.08941957354545593,\n      0.14514479041099548,\n      0.09012299031019211,\n      0.08053499460220337,\n      0.0006052384851500392,\n      0.11163946986198425,\n      -0.004945351276546717\n    ],\n    [\n      -0.14397424459457397,\n      -0.19338250160217285,\n      0.16507773101329803,\n      0.02060086652636528,\n      -0.0672735646367073,\n      0.08524637669324875,\n      -0.43584194779396057,\n      -0.2681632339954376,\n      0.12399973720312119,\n      -0.1765618771314621,\n      0.0504734180867672,\n      0.020824618637561798,\n      -0.1845197081565857,\n      -0.08728909492492676,\n      0.20699849724769592,\n      -0.11006063222885132,\n      -0.0947813168168068,\n      0.12152741849422455,\n      0.1773013174533844,\n      -0.2885051965713501,\n      0.07327599823474884,\n      -0.3464727997779846,\n      0.11940550804138184,\n      -0.08096971362829208,\n      0.004756953567266464,\n      -0.0885610580444336,\n      -0.2522056996822357,\n      0.030362000688910484,\n      0.14581802487373352,\n      -0.10453055053949356,\n      -0.07506604492664337,\n      0.03471910580992699,\n      0.010410751216113567,\n      0.07905561476945877,\n      -0.22440244257450104,\n      -0.01174938678741455,\n      0.1319950819015503,\n      -0.021664917469024658,\n      -0.1837579309940338,\n      -0.17247429490089417,\n      -0.11014974117279053,\n      -0.020763011649250984,\n      0.0014410024741664529,\n      -0.01305319182574749,\n      0.17210228741168976,\n      -0.14940354228019714,\n      -0.2272445559501648,\n      -0.03693901747465134,\n      0.024418162181973457,\n      -0.034573744982481\n    ],\n    [\n      0.014346597716212273,\n      0.18488182127475739,\n      -0.058648545295000076,\n      -0.006606512703001499,\n      0.14180584251880646,\n      -0.223921537399292,\n      -0.42887046933174133,\n      0.446142315864563,\n      -0.2935766279697418,\n      -0.04133259132504463,\n      -0.1076042577624321,\n      0.10927433520555496,\n      -0.1408437043428421,\n      0.09148303419351578,\n      0.17592363059520721,\n      -0.11831328272819519,\n      0.02745344117283821,\n      -0.10052713006734848,\n      -0.10411763936281204,\n      -0.11875776201486588,\n      -0.08508162945508957,\n      0.4191172420978546,\n      -0.040117692202329636,\n      0.03148714825510979,\n      -0.22620074450969696,\n      0.03932109847664833,\n      0.28412190079689026,\n      -0.14472991228103638,\n      0.057003919035196304,\n      -0.011241498403251171,\n      0.2455122172832489,\n      -0.015811925753951073,\n      0.11038870364427567,\n      0.08340095728635788,\n      -0.022736288607120514,\n      0.015321166254580021,\n      0.02704452909529209,\n      -0.14583352208137512,\n      -0.030600043013691902,\n      0.007112834602594376,\n      -0.14603166282176971,\n      0.035307396203279495,\n      0.035654474049806595,\n      -0.2787499725818634,\n      0.0013386644423007965,\n      -0.3366869390010834,\n      -0.1019175797700882,\n      0.006605697330087423,\n      0.08988703042268753,\n      0.14832091331481934\n    ],\n    [\n      -0.4803024232387543,\n      -0.07053821533918381,\n      0.11276406794786453,\n      -0.07292688637971878,\n      -0.1313030868768692,\n      0.0546669103205204,\n      -0.18746498227119446,\n      0.042623765766620636,\n      0.825219452381134,\n      0.05689175799489021,\n      -0.021084100008010864,\n      0.11139362305402756,\n      0.02997627668082714,\n      -0.1355852335691452,\n      -0.08049681037664413,\n      0.04298706725239754,\n      -0.17423643171787262,\n      -0.10579095035791397,\n      0.17299574613571167,\n      -0.09018343687057495,\n      -0.12347360700368881,\n      -0.039774518460035324,\n      0.11407832056283951,\n      -0.06652989238500595,\n      -0.02845418080687523,\n      0.09406835585832596,\n      0.05258952081203461,\n      -0.22122757136821747,\n      0.034523963928222656,\n      -0.0654832050204277,\n      -0.44894760847091675,\n      0.1378968060016632,\n      -0.10184305906295776,\n      -0.02881873957812786,\n      0.015057938173413277,\n      -0.020076779648661613,\n      0.06598792225122452,\n      -0.18970127403736115,\n      -0.32221418619155884,\n      -0.0692475363612175,\n      -0.035558298230171204,\n      0.12313206493854523,\n      -0.015238964930176735,\n      0.07299274951219559,\n      0.08260896056890488,\n      -0.03829437866806984,\n      -0.4512038826942444,\n      -0.08345114439725876,\n      -0.11887004971504211,\n      -0.046204544603824615\n    ],\n    [\n      0.7204809188842773,\n      0.011776936240494251,\n      0.07280578464269638,\n      0.0931386947631836,\n      -0.0435587503015995,\n      0.12382116168737411,\n      -0.3792595863342285,\n      0.0694858506321907,\n      -0.06762221455574036,\n      0.14782758057117462,\n      0.17117077112197876,\n      -0.035250939428806305,\n      0.24480105936527252,\n      -0.3396846652030945,\n      0.013523180969059467,\n      -0.05839085951447487,\n      -0.25541332364082336,\n      -0.4379868805408478,\n      -0.1285203993320465,\n      0.001346062170341611,\n      0.056247297674417496,\n      -0.3134918212890625,\n      0.03009544126689434,\n      0.1740824580192566,\n      0.3489476442337036,\n      0.0932265892624855,\n      0.061328306794166565,\n      0.0831252783536911,\n      0.3190075755119324,\n      0.012210302986204624,\n      0.25015789270401,\n      -0.04923205450177193,\n      0.05214247107505798,\n      -0.1424858719110489,\n      0.24285735189914703,\n      -0.09917695075273514,\n      0.2665071189403534,\n      -0.17913758754730225,\n      -0.24088694155216217,\n      0.19283975660800934,\n      0.08210967481136322,\n      -0.3165081739425659,\n      -0.04801516234874725,\n      0.10164310038089752,\n      -0.23676712810993195,\n      -0.0016613529296591878,\n      0.05920364335179329,\n      0.1994168758392334,\n      0.35624372959136963,\n      0.2958475351333618\n    ],\n    [\n      0.010231226682662964,\n      -0.011317485012114048,\n      -0.039703160524368286,\n      0.10590970516204834,\n      0.16849812865257263,\n      -0.016755517572164536,\n      0.24889744818210602,\n      -0.16539263725280762,\n      -0.5940143465995789,\n      -0.029377687722444534,\n      0.17717905342578888,\n      -0.1738501340150833,\n      -0.06737609207630157,\n      0.1960020661354065,\n      0.01135212555527687,\n      0.022043345496058464,\n      0.0201167743653059,\n      0.26913684606552124,\n      0.020225726068019867,\n      -0.010020019486546516,\n      0.16997984051704407,\n      0.09704004228115082,\n      -0.014946337789297104,\n      0.5791301131248474,\n      0.14442528784275055,\n      -0.11439576745033264,\n      -0.06722160428762436,\n      0.2073768824338913,\n      0.0399557463824749,\n      0.0967492163181305,\n      -0.13437828421592712,\n      0.22274018824100494,\n      0.043753303587436676,\n      0.04735367372632027,\n      0.013933319598436356,\n      -0.004410421941429377,\n      0.1004231721162796,\n      0.22464850544929504,\n      0.1003904640674591,\n      0.0895513966679573,\n      -0.08527716249227524,\n      -0.11862854659557343,\n      -0.20925790071487427,\n      0.058147624135017395,\n      -0.004563036374747753,\n      -0.02078866958618164,\n      0.37877511978149414,\n      -0.16176870465278625,\n      -0.016034558415412903,\n      -0.07944884151220322\n    ],\n    [\n      -0.07337212562561035,\n      -0.21858149766921997,\n      -0.09257995337247849,\n      0.1612076610326767,\n      -0.09416960179805756,\n      -0.1683948040008545,\n      -0.18884879350662231,\n      0.659869909286499,\n      -0.05941752344369888,\n      0.014336599968373775,\n      -0.06218842789530754,\n      0.2476852387189865,\n      -0.049744073301553726,\n      -0.2581755220890045,\n      0.0835624560713768,\n      -0.04546076059341431,\n      -0.3261212706565857,\n      -0.3956606686115265,\n      -0.0482466034591198,\n      -0.002340744948014617,\n      -0.13290491700172424,\n      -0.1389538198709488,\n      0.11236041784286499,\n      0.12881268560886383,\n      -0.09690424054861069,\n      0.11877625435590744,\n      0.06872579455375671,\n      -0.11617772281169891,\n      0.03612040355801582,\n      -0.16681629419326782,\n      -0.08907241374254227,\n      0.1376401036977768,\n      -0.1855916678905487,\n      -0.3528212904930115,\n      0.11137473583221436,\n      -0.29772117733955383,\n      0.041235391050577164,\n      -0.18815580010414124,\n      -0.2076304852962494,\n      -0.2096562534570694,\n      0.07762166112661362,\n      0.12353263050317764,\n      -0.15356886386871338,\n      0.11867258697748184,\n      -0.03339965641498566,\n      -0.02726588025689125,\n      -0.07425384968519211,\n      -0.40078026056289673,\n      -0.06500399857759476,\n      -0.018184185028076172\n    ],\n    [\n      0.4918694794178009,\n      -0.09184308350086212,\n      -0.10593931376934052,\n      0.0768250897526741,\n      -0.0030091716907918453,\n      -0.00347901601344347,\n      0.13406826555728912,\n      -0.343121737241745,\n      0.2106088250875473,\n      0.06864454597234726,\n      0.04999275133013725,\n      0.15562224388122559,\n      0.20771604776382446,\n      0.08296357840299606,\n      0.15503042936325073,\n      -0.058727867901325226,\n      -0.06747199594974518,\n      0.44509634375572205,\n      -0.24313364923000336,\n      0.013474739156663418,\n      0.250759094953537,\n      0.1218719407916069,\n      0.17055590450763702,\n      -0.11442520469427109,\n      0.15263603627681732,\n      0.1005808487534523,\n      -0.17982400953769684,\n      -0.1079341247677803,\n      0.022308915853500366,\n      0.2890250086784363,\n      0.11258335411548615,\n      0.08401180058717728,\n      0.22220231592655182,\n      0.026849795132875443,\n      0.2102389633655548,\n      -0.3291676342487335,\n      -0.04932258278131485,\n      0.028696002438664436,\n      -0.1139938086271286,\n      0.08923022449016571,\n      0.11143812537193298,\n      -0.19215844571590424,\n      -0.0960032045841217,\n      0.06268410384654999,\n      0.15016791224479675,\n      0.14517700672149658,\n      0.17379812896251678,\n      -0.14905989170074463,\n      0.0925498902797699,\n      0.16406987607479095\n    ],\n    [\n      -0.15951792895793915,\n      -0.007223795633763075,\n      -0.07752205431461334,\n      0.09382535517215729,\n      -0.23242107033729553,\n      -0.024078169837594032,\n      0.03434081748127937,\n      -0.18549948930740356,\n      -0.28660717606544495,\n      -0.011796466074883938,\n      0.06759588420391083,\n      0.04711402207612991,\n      0.21265476942062378,\n      0.07893088459968567,\n      -0.07329941540956497,\n      -0.0003827899054158479,\n      0.036843545734882355,\n      0.12369068711996078,\n      -0.07981054484844208,\n      0.12736926972866058,\n      0.11926992982625961,\n      0.2016214281320572,\n      0.1330837905406952,\n      0.1702556163072586,\n      0.10162252187728882,\n      -0.0017799680354073644,\n      -0.10989182442426682,\n      0.16986148059368134,\n      -0.15263406932353973,\n      0.25951847434043884,\n      -0.247427299618721,\n      0.15109029412269592,\n      -0.0517875999212265,\n      0.02621280774474144,\n      -0.045267101377248764,\n      0.0070148734375834465,\n      -0.047489624470472336,\n      -0.059242431074380875,\n      0.22468526661396027,\n      0.12084975093603134,\n      0.0976051390171051,\n      -0.1380992829799652,\n      0.0878472775220871,\n      0.14596424996852875,\n      -0.010403472930192947,\n      -0.06502129882574081,\n      0.36701586842536926,\n      0.020695682615041733,\n      -0.06184825301170349,\n      0.07155085355043411\n    ],\n    [\n      0.2087855041027069,\n      -0.0199139267206192,\n      -0.06089893728494644,\n      0.07436578720808029,\n      0.31761813163757324,\n      0.06599210947751999,\n      -0.17799785733222961,\n      -0.09821417182683945,\n      0.02775050327181816,\n      0.19589640200138092,\n      0.1551983654499054,\n      -0.25496116280555725,\n      0.13187161087989807,\n      -0.00028760218992829323,\n      -0.16217488050460815,\n      0.040808726102113724,\n      0.04300685599446297,\n      0.39812031388282776,\n      -0.1642824411392212,\n      0.04874267801642418,\n      0.4351644217967987,\n      0.005490140523761511,\n      0.030446527525782585,\n      -0.09475720673799515,\n      0.1953277587890625,\n      -0.12730465829372406,\n      0.032956112176179886,\n      -0.12319114804267883,\n      0.03994337469339371,\n      0.15683980286121368,\n      0.08801518380641937,\n      -0.003703215392306447,\n      0.1831882894039154,\n      -0.1301453560590744,\n      -0.07606719434261322,\n      -0.10256574302911758,\n      -0.033814314752817154,\n      0.07428374141454697,\n      -0.041929930448532104,\n      0.06758321821689606,\n      0.08299064636230469,\n      -0.13507287204265594,\n      -0.24681401252746582,\n      -0.344871461391449,\n      0.35337647795677185,\n      -0.11147906631231308,\n      -0.02963596023619175,\n      -0.0301114022731781,\n      0.15648680925369263,\n      0.017035039141774178\n    ],\n    [\n      0.44142788648605347,\n      -0.17016609013080597,\n      0.12772385776042938,\n      0.044604890048503876,\n      -0.22768718004226685,\n      -0.1453944742679596,\n      -0.042871683835983276,\n      -0.11301378905773163,\n      -0.11929617822170258,\n      0.22946609556674957,\n      -0.043765485286712646,\n      -0.05390739440917969,\n      -0.08954637497663498,\n      0.1325882375240326,\n      0.1234724223613739,\n      0.053074657917022705,\n      -0.05969734862446785,\n      0.2860318422317505,\n      -0.5046888589859009,\n      -0.31630849838256836,\n      0.14760549366474152,\n      -0.06949001550674438,\n      0.027566837146878242,\n      -0.10003694146871567,\n      0.12448225170373917,\n      0.04333040118217468,\n      -0.05965850502252579,\n      -0.15553496778011322,\n      0.02826617658138275,\n      0.1082470640540123,\n      0.19224810600280762,\n      -0.10652778297662735,\n      0.08423031866550446,\n      0.15128283202648163,\n      -0.15132185816764832,\n      -0.13632525503635406,\n      -0.1481262445449829,\n      0.02162802964448929,\n      0.025891367346048355,\n      0.06150549650192261,\n      -0.0630982518196106,\n      -0.23347315192222595,\n      -0.26198911666870117,\n      0.23944908380508423,\n      0.24080048501491547,\n      -0.03831356391310692,\n      -0.129680797457695,\n      -0.2708808183670044,\n      -0.019898125901818275,\n      0.09635769575834274\n    ],\n    [\n      0.2883129119873047,\n      -0.00033231411362066865,\n      -0.14333918690681458,\n      0.0027264473028481007,\n      0.08414026349782944,\n      -0.14833234250545502,\n      -0.2521386444568634,\n      0.5477294921875,\n      -0.15390153229236603,\n      0.23126620054244995,\n      0.16510625183582306,\n      0.09454213082790375,\n      -0.17145995795726776,\n      0.09817563742399216,\n      -0.07554641366004944,\n      -0.09583626687526703,\n      -0.049381062388420105,\n      0.05284225568175316,\n      -0.09252213686704636,\n      -0.038141362369060516,\n      0.09995747357606888,\n      -0.07350622862577438,\n      0.04530133306980133,\n      -0.4251772463321686,\n      -0.1961507499217987,\n      0.20023901760578156,\n      0.21056537330150604,\n      -0.13529072701931,\n      0.11467635631561279,\n      0.07648726552724838,\n      0.18133360147476196,\n      0.12148788571357727,\n      -0.12067530304193497,\n      -0.03906947001814842,\n      -0.10686229914426804,\n      -0.1341876983642578,\n      -0.10319197922945023,\n      -0.05365218222141266,\n      -0.15175290405750275,\n      0.09125947952270508,\n      -0.3190040588378906,\n      -0.09059609472751617,\n      -0.1516568958759308,\n      -0.1431613564491272,\n      0.04905145242810249,\n      -0.14715681970119476,\n      -0.1301688849925995,\n      0.006801411043852568,\n      -0.07760694622993469,\n      0.1368001252412796\n    ],\n    [\n      0.33423224091529846,\n      0.13605475425720215,\n      -0.02915332466363907,\n      -0.028331242501735687,\n      -0.30691882967948914,\n      0.0032548648305237293,\n      -0.2424255758523941,\n      0.5375502109527588,\n      0.004788287449628115,\n      0.2022036761045456,\n      -0.013269887305796146,\n      -0.1904379427433014,\n      0.12699724733829498,\n      -0.17354032397270203,\n      0.18590517342090607,\n      0.06954517960548401,\n      0.27602246403694153,\n      0.46906840801239014,\n      -0.26483869552612305,\n      -0.355812668800354,\n      0.026267193257808685,\n      -0.036936745047569275,\n      0.2121741622686386,\n      -0.19964365661144257,\n      0.08106034249067307,\n      -0.015218924731016159,\n      -0.27891048789024353,\n      -0.011229321360588074,\n      -0.02360214665532112,\n      0.14044399559497833,\n      0.28141075372695923,\n      0.12026042491197586,\n      0.1467159539461136,\n      -0.4454598128795624,\n      -0.17936798930168152,\n      0.11587929725646973,\n      -0.01262320764362812,\n      0.042570553719997406,\n      0.003709460375830531,\n      0.11387676000595093,\n      0.19526329636573792,\n      -0.10242725163698196,\n      -0.2666325867176056,\n      0.16418322920799255,\n      0.0797736868262291,\n      -0.0342470183968544,\n      -0.04926137998700142,\n      -0.1146811917424202,\n      0.14232321083545685,\n      0.17017289996147156\n    ],\n    [\n      0.08314882963895798,\n      -0.1248893216252327,\n      0.20731104910373688,\n      0.13559015095233917,\n      -0.2658231854438782,\n      -0.09928513318300247,\n      -0.20124199986457825,\n      -0.14620287716388702,\n      -0.19338081777095795,\n      0.07336747646331787,\n      -0.15520362555980682,\n      0.053512994199991226,\n      0.07616175711154938,\n      -0.03844364732503891,\n      -0.008948740549385548,\n      -0.12949402630329132,\n      -0.1308450847864151,\n      0.07293438166379929,\n      0.1003585234284401,\n      0.05132546275854111,\n      -0.28823697566986084,\n      0.014056080020964146,\n      -0.13427507877349854,\n      0.17190219461917877,\n      -0.19973620772361755,\n      -0.14813287556171417,\n      -0.10173112154006958,\n      -0.1594512015581131,\n      -0.16314280033111572,\n      0.04950450733304024,\n      -0.020614635199308395,\n      -0.116559699177742,\n      0.0011130026541650295,\n      -0.38257378339767456,\n      -0.1708449423313141,\n      -0.34400635957717896,\n      -0.1498088836669922,\n      -0.06846565008163452,\n      -0.0789477601647377,\n      0.0696069598197937,\n      -0.05060147866606712,\n      0.04938124120235443,\n      0.3675074577331543,\n      0.1759907454252243,\n      0.11800655722618103,\n      0.20126916468143463,\n      -0.09478605538606644,\n      0.1608993411064148,\n      -0.039053868502378464,\n      -0.057955000549554825\n    ],\n    [\n      0.24940168857574463,\n      0.024716345593333244,\n      -0.01064287032932043,\n      0.10983604937791824,\n      -0.08989269286394119,\n      -0.0168409775942564,\n      -0.024853738024830818,\n      -0.11167365312576294,\n      -0.07460541278123856,\n      0.0812225416302681,\n      -0.13712090253829956,\n      -0.23787911236286163,\n      0.22650805115699768,\n      -0.11958148330450058,\n      -0.21595288813114166,\n      0.05114639550447464,\n      0.1849711537361145,\n      0.20117618143558502,\n      0.05201767385005951,\n      0.08090420812368393,\n      -0.15167132019996643,\n      -0.224803626537323,\n      0.021680297330021858,\n      0.0014701841864734888,\n      0.18945318460464478,\n      0.09805719554424286,\n      -0.2690085470676422,\n      -0.015536712482571602,\n      -0.09493754059076309,\n      0.15802371501922607,\n      0.18224197626113892,\n      0.13034725189208984,\n      0.13653135299682617,\n      -0.01833777315914631,\n      0.08046599477529526,\n      -0.038190703839063644,\n      -0.15217414498329163,\n      -0.1143726035952568,\n      0.007245555054396391,\n      -0.2143642008304596,\n      0.304427832365036,\n      -0.09111102670431137,\n      0.4003005027770996,\n      0.18526403605937958,\n      0.21273145079612732,\n      0.0451832078397274,\n      0.029577195644378662,\n      0.03207048028707504,\n      0.09221290796995163,\n      0.08803737908601761\n    ],\n    [\n      0.29041457176208496,\n      -0.10560005903244019,\n      0.23842264711856842,\n      0.049158282577991486,\n      -0.11112026870250702,\n      -0.2621012330055237,\n      -0.2504193186759949,\n      -0.13479140400886536,\n      -0.2696118950843811,\n      0.09992913901805878,\n      0.023210952058434486,\n      -0.24596023559570312,\n      -0.00140807603020221,\n      0.03183615580201149,\n      0.031108450144529343,\n      0.11323458701372147,\n      -0.02234663814306259,\n      0.058662597090005875,\n      -0.25454989075660706,\n      0.02584359049797058,\n      -0.11215590685606003,\n      0.13465985655784607,\n      0.10515443980693817,\n      -0.2874284088611603,\n      -0.12020211666822433,\n      -0.20797930657863617,\n      0.026537051424384117,\n      -0.20870186388492584,\n      -0.04331200569868088,\n      -0.3615126311779022,\n      -0.04004049301147461,\n      0.0975424274802208,\n      -0.06836909055709839,\n      -0.3277927339076996,\n      -0.13517211377620697,\n      -0.22592289745807648,\n      -0.11006645113229752,\n      -0.0032526005525141954,\n      -0.0773773118853569,\n      0.10061437636613846,\n      0.010048025287687778,\n      -0.1269938051700592,\n      0.004480374976992607,\n      0.06235353276133537,\n      0.04450024664402008,\n      0.09737991541624069,\n      0.022044764831662178,\n      -0.011076991446316242,\n      0.054227955639362335,\n      -0.008573085069656372\n    ],\n    [\n      -0.16040374338626862,\n      -0.06114489957690239,\n      0.12635934352874756,\n      -0.14880263805389404,\n      -0.16752640902996063,\n      -0.12518231570720673,\n      -0.0006771627231501043,\n      0.37593400478363037,\n      -0.09838363528251648,\n      0.0419880710542202,\n      0.1510785073041916,\n      0.10905633866786957,\n      0.19017399847507477,\n      -0.40076425671577454,\n      0.08915375918149948,\n      -0.13313236832618713,\n      -0.34835460782051086,\n      -0.5785733461380005,\n      0.07277288287878036,\n      -0.07876788824796677,\n      0.2918541729450226,\n      0.11966429650783539,\n      -0.2647276818752289,\n      -0.2800191640853882,\n      0.15854182839393616,\n      0.024245968088507652,\n      0.1329628974199295,\n      -0.016254717484116554,\n      -0.0931701511144638,\n      -0.3193225860595703,\n      -0.017677489668130875,\n      0.02941056899726391,\n      -0.253005713224411,\n      -0.02810906432569027,\n      0.005164540372788906,\n      -0.13285379111766815,\n      -0.08353272825479507,\n      -0.508655309677124,\n      -0.2231859415769577,\n      -0.13639305531978607,\n      0.1899433583021164,\n      0.09522166848182678,\n      -0.1960235834121704,\n      0.06340235471725464,\n      0.0664910301566124,\n      0.09744038432836533,\n      0.078181691467762,\n      0.11668366938829422,\n      0.241763174533844,\n      0.22097806632518768\n    ],\n    [\n      0.3017946183681488,\n      -0.1198500394821167,\n      0.1548229455947876,\n      -0.15195973217487335,\n      0.14815428853034973,\n      -0.06595226377248764,\n      -0.3147523105144501,\n      -0.1471778154373169,\n      -0.24870392680168152,\n      0.06098450347781181,\n      0.12074140459299088,\n      -0.007696377579122782,\n      0.04571383818984032,\n      0.07298709452152252,\n      0.18322430551052094,\n      -0.10519193857908249,\n      -0.13468484580516815,\n      0.2478276491165161,\n      -0.09015729278326035,\n      0.08221790194511414,\n      -0.1372561901807785,\n      0.020856134593486786,\n      0.037283655256032944,\n      -0.02025933377444744,\n      0.057599954307079315,\n      -0.003302742959931493,\n      0.1699594259262085,\n      -0.08505775779485703,\n      -0.009895509108901024,\n      -0.05968786031007767,\n      0.062292151153087616,\n      0.10433639585971832,\n      0.10751616209745407,\n      -0.23986482620239258,\n      -0.13334591686725616,\n      -0.2720738351345062,\n      0.012007981538772583,\n      0.012661553919315338,\n      -0.08866359293460846,\n      0.03473375737667084,\n      -0.03483934327960014,\n      0.10010943561792374,\n      -0.021141190081834793,\n      0.24139593541622162,\n      0.24196147918701172,\n      -0.05551838502287865,\n      -0.11912603676319122,\n      0.020430000498890877,\n      -0.10019668936729431,\n      0.04078010097146034\n    ],\n    [\n      0.5870478749275208,\n      0.00779021717607975,\n      -0.11320974677801132,\n      0.06622015684843063,\n      0.07122370600700378,\n      0.19227290153503418,\n      0.0035636175889521837,\n      0.5882098078727722,\n      -0.27869096398353577,\n      0.04254445806145668,\n      0.1324610412120819,\n      -0.04675908014178276,\n      -0.00768587039783597,\n      -0.01167207583785057,\n      0.1787206083536148,\n      -0.0002018546947510913,\n      -0.1321600377559662,\n      0.24567192792892456,\n      -0.0672755241394043,\n      -0.13180294632911682,\n      0.038084808737039566,\n      0.2076774388551712,\n      0.07342799752950668,\n      0.030923131853342056,\n      0.1260622888803482,\n      -0.09775038808584213,\n      -0.18005838990211487,\n      -0.035990532487630844,\n      0.25140005350112915,\n      0.1796887367963791,\n      -0.11971073597669601,\n      -0.19657078385353088,\n      0.19197458028793335,\n      -0.31020113825798035,\n      0.09794861823320389,\n      -0.17317645251750946,\n      0.2755277454853058,\n      -0.003415801329538226,\n      -0.008205201476812363,\n      0.046629250049591064,\n      -0.04311418533325195,\n      -0.34845948219299316,\n      -0.0030574931297451258,\n      -0.17758315801620483,\n      -0.06326738744974136,\n      0.014798959717154503,\n      0.037625569850206375,\n      -0.08245965838432312,\n      -0.049996230751276016,\n      -0.03922571614384651\n    ],\n    [\n      0.08515572547912598,\n      -0.11467069387435913,\n      0.10115674883127213,\n      -0.11073830723762512,\n      -0.10406842082738876,\n      0.03338400274515152,\n      -0.2712315618991852,\n      0.2279064953327179,\n      -0.08287211507558823,\n      -0.12040182948112488,\n      0.0408104807138443,\n      -0.06652224063873291,\n      -0.03393569961190224,\n      -0.10349512845277786,\n      -0.028895724564790726,\n      -0.04804296791553497,\n      0.06171615049242973,\n      0.06644205003976822,\n      0.35540854930877686,\n      -0.15119101107120514,\n      -0.17789623141288757,\n      0.0855090394616127,\n      -0.07658164203166962,\n      0.04587625339627266,\n      0.024479638785123825,\n      0.020439686253666878,\n      -0.13573084771633148,\n      -0.15643833577632904,\n      -0.09690357744693756,\n      -0.02219380810856819,\n      0.26205846667289734,\n      -0.06834767013788223,\n      0.3016897141933441,\n      -0.5582737326622009,\n      0.11449244618415833,\n      -0.15883749723434448,\n      -0.039247769862413406,\n      -0.07305337488651276,\n      -0.002842735266312957,\n      -0.05074580758810043,\n      -0.0026718543376773596,\n      0.07321933656930923,\n      0.521507740020752,\n      0.0696089118719101,\n      0.24921709299087524,\n      -0.27430567145347595,\n      0.17649798095226288,\n      -0.12017422914505005,\n      -0.03522875905036926,\n      0.04715048521757126\n    ],\n    [\n      -0.1822836846113205,\n      0.11848115175962448,\n      -0.04700479283928871,\n      -0.010047624818980694,\n      -0.29059267044067383,\n      -0.3747742474079132,\n      0.011109182611107826,\n      -0.1056012287735939,\n      -0.024866236373782158,\n      0.06527488678693771,\n      -0.23528096079826355,\n      -0.09579160064458847,\n      -0.231195867061615,\n      0.08028413355350494,\n      -0.22594717144966125,\n      -0.030241526663303375,\n      -0.1516144722700119,\n      0.10015640407800674,\n      -0.2608824074268341,\n      -0.1776014268398285,\n      -0.4393840432167053,\n      0.0637788474559784,\n      -0.24133963882923126,\n      0.014482926577329636,\n      -0.32913991808891296,\n      0.02716226503252983,\n      0.09165160357952118,\n      -0.057755373418331146,\n      0.015312086790800095,\n      -0.2531438171863556,\n      -0.040660396218299866,\n      0.012526340782642365,\n      -0.3437550663948059,\n      0.62122642993927,\n      0.09040891379117966,\n      -0.1374223381280899,\n      0.022293521091341972,\n      0.06816938519477844,\n      -0.11226323992013931,\n      0.009107580408453941,\n      -0.33887434005737305,\n      0.08534333854913712,\n      -0.10124995559453964,\n      -0.1468956172466278,\n      -0.15353929996490479,\n      -0.33957552909851074,\n      0.005826576147228479,\n      -0.29976755380630493,\n      -0.01407485082745552,\n      0.0317748486995697\n    ],\n    [\n      0.09166327118873596,\n      0.03289487212896347,\n      -0.139746755361557,\n      0.1303493231534958,\n      -0.17339347302913666,\n      0.17778578400611877,\n      -0.18673346936702728,\n      -0.018685469403862953,\n      -0.00851132906973362,\n      -0.07381439954042435,\n      0.064118392765522,\n      -0.08115498721599579,\n      -0.04439648240804672,\n      0.21443727612495422,\n      -0.015121251344680786,\n      0.08191933482885361,\n      0.05939842760562897,\n      0.30890506505966187,\n      -0.10591588914394379,\n      0.15977197885513306,\n      0.04449164494872093,\n      -0.024260910227894783,\n      0.0406954362988472,\n      0.14523673057556152,\n      -0.17629699409008026,\n      -0.08216912299394608,\n      -0.1999635249376297,\n      0.07586797326803207,\n      0.07032043486833572,\n      -0.06628930568695068,\n      -0.12206640839576721,\n      0.1966317743062973,\n      -0.07387222349643707,\n      -0.0013125049881637096,\n      0.06050170212984085,\n      -0.05150045454502106,\n      -0.04127884283661842,\n      0.16159114241600037,\n      -0.0867728441953659,\n      0.03310075029730797,\n      -0.05644728243350983,\n      0.0013766648480668664,\n      -0.13697277009487152,\n      -0.023659929633140564,\n      -0.15656796097755432,\n      -0.2974202334880829,\n      -0.023178240284323692,\n      0.0248294435441494,\n      -0.04291268438100815,\n      -0.10460331290960312\n    ],\n    [\n      0.16451585292816162,\n      -0.03438059613108635,\n      0.1244988888502121,\n      -0.2089402675628662,\n      0.12933969497680664,\n      0.09339024126529694,\n      -0.5855752229690552,\n      -0.07907573133707047,\n      0.09198266267776489,\n      0.05522492155432701,\n      -0.10142120718955994,\n      0.14510729908943176,\n      -0.20699845254421234,\n      0.08367811143398285,\n      -0.12306031584739685,\n      -0.12221096456050873,\n      -0.06037618964910507,\n      -0.3017503023147583,\n      -0.01355031318962574,\n      -0.08374058455228806,\n      -0.19435372948646545,\n      0.22658997774124146,\n      0.02195676974952221,\n      -0.10799811035394669,\n      -0.107126384973526,\n      0.10097914189100266,\n      0.31019505858421326,\n      -0.13161207735538483,\n      -0.0473252534866333,\n      -0.09271815419197083,\n      0.2778792679309845,\n      -0.14702555537223816,\n      0.1028856411576271,\n      -0.36516648530960083,\n      -0.14637058973312378,\n      -0.3970029354095459,\n      -0.19333359599113464,\n      0.10176335275173187,\n      -0.09265122562646866,\n      -0.07807255536317825,\n      -0.21392077207565308,\n      0.16817204654216766,\n      0.20754750072956085,\n      0.25349727272987366,\n      0.37720897793769836,\n      -0.1251739114522934,\n      -0.09632284194231033,\n      -0.08449608087539673,\n      0.00899370014667511,\n      -0.10880989581346512\n    ],\n    [\n      -0.15304513275623322,\n      -0.04434039071202278,\n      0.10911855846643448,\n      -0.038442213088274,\n      -0.2936473786830902,\n      -0.07726847380399704,\n      -0.317326158285141,\n      -0.010614432394504547,\n      -0.03157941997051239,\n      0.07415685802698135,\n      -0.086046002805233,\n      0.24007484316825867,\n      -0.06052098795771599,\n      -0.15410634875297546,\n      -0.023081541061401367,\n      0.05301712080836296,\n      -0.1608847677707672,\n      -0.010307321324944496,\n      0.05978773906826973,\n      -0.19505611062049866,\n      0.33950427174568176,\n      -0.004316131584346294,\n      0.07971420139074326,\n      -0.010115822777152061,\n      0.007038942072540522,\n      0.030627379193902016,\n      0.04735594242811203,\n      -0.055714476853609085,\n      -0.06589308381080627,\n      -0.04678899049758911,\n      -0.10717178136110306,\n      0.17998860776424408,\n      -0.15760698914527893,\n      0.22493785619735718,\n      0.08159781247377396,\n      0.0062845610082149506,\n      0.1825694888830185,\n      -0.11101041734218597,\n      -0.04082624241709709,\n      -0.009182256646454334,\n      -0.08027560263872147,\n      0.08834117650985718,\n      -0.1333509087562561,\n      0.12721000611782074,\n      0.11876539140939713,\n      -0.06471730023622513,\n      0.2231539934873581,\n      -0.22390083968639374,\n      0.0068480269983410835,\n      -0.03191039711236954\n    ],\n    [\n      0.1124860867857933,\n      0.22718499600887299,\n      -0.07131465524435043,\n      -0.2092176377773285,\n      -0.18577763438224792,\n      -0.07969044893980026,\n      -0.3536510169506073,\n      0.3589199185371399,\n      -0.07150860875844955,\n      0.013217882253229618,\n      0.18876300752162933,\n      -0.17236053943634033,\n      -0.11854895204305649,\n      0.01433949451893568,\n      -0.07604948431253433,\n      -0.0007762892637401819,\n      0.056693483144044876,\n      0.18572105467319489,\n      0.007185687776654959,\n      0.16476060450077057,\n      0.0807541087269783,\n      0.05344479903578758,\n      0.13423439860343933,\n      -0.19946633279323578,\n      0.16162429749965668,\n      0.11180640757083893,\n      0.01888786070048809,\n      -0.043442320078611374,\n      -0.12265961617231369,\n      -0.08380565792322159,\n      0.1369338184595108,\n      0.09744205325841904,\n      0.14496281743049622,\n      -0.21565194427967072,\n      -0.2307431846857071,\n      -0.09302755445241928,\n      -0.09341201186180115,\n      0.08006782829761505,\n      -0.13593187928199768,\n      -0.14095436036586761,\n      -0.1287253499031067,\n      -0.35891300439834595,\n      -0.3785974681377411,\n      -0.056865498423576355,\n      0.1247904896736145,\n      0.11534511297941208,\n      -0.17580533027648926,\n      0.04527664929628372,\n      0.13525208830833435,\n      0.1921493262052536\n    ],\n    [\n      0.2889766991138458,\n      -0.023808594793081284,\n      0.07719437777996063,\n      -0.09809968620538712,\n      0.18984396755695343,\n      0.1497836410999298,\n      -0.10063008219003677,\n      0.09885230660438538,\n      -0.07150963693857193,\n      0.07257303595542908,\n      0.05604938045144081,\n      -0.08806443959474564,\n      0.1799667328596115,\n      0.13982053101062775,\n      -0.09497137367725372,\n      0.14458967745304108,\n      -0.2714150846004486,\n      -0.058554381132125854,\n      0.10361140221357346,\n      -0.18219584226608276,\n      0.20501549541950226,\n      -0.03963577002286911,\n      0.08855432271957397,\n      -0.14908425509929657,\n      0.13318051397800446,\n      -0.13488203287124634,\n      -0.0036320544313639402,\n      -0.07370943576097488,\n      -0.05272303894162178,\n      0.14075715839862823,\n      0.051034439355134964,\n      -0.0006345458677969873,\n      0.1254344880580902,\n      -0.44069579243659973,\n      0.00535795371979475,\n      -0.08576153963804245,\n      -0.08122538775205612,\n      0.1807173639535904,\n      -0.07250164449214935,\n      0.008780675940215588,\n      0.06480268388986588,\n      -0.18945179879665375,\n      0.24042269587516785,\n      0.08522750437259674,\n      0.2737363576889038,\n      -0.28629401326179504,\n      0.23894314467906952,\n      0.08383630961179733,\n      0.1950281858444214,\n      0.13871823251247406\n    ],\n    [\n      0.23252500593662262,\n      0.026101235300302505,\n      0.06040855869650841,\n      -0.2567649781703949,\n      0.03792092576622963,\n      0.25613874197006226,\n      -0.08945444971323013,\n      0.48772525787353516,\n      -0.10282876342535019,\n      -0.17440105974674225,\n      -0.029254784807562828,\n      -0.12603536248207092,\n      -0.2003365159034729,\n      -0.09396003931760788,\n      0.08492594957351685,\n      -0.14805381000041962,\n      0.017884330824017525,\n      -0.021402180194854736,\n      -0.06585776805877686,\n      -0.10397377610206604,\n      -0.13548444211483002,\n      0.11477797478437424,\n      0.10102792829275131,\n      0.26886460185050964,\n      -0.17871281504631042,\n      -0.23332206904888153,\n      -0.05589296296238899,\n      -0.2012551724910736,\n      0.020768949761986732,\n      -0.032010212540626526,\n      -0.20591561496257782,\n      -0.28669896721839905,\n      0.2098354548215866,\n      -0.009611019864678383,\n      -0.07075996696949005,\n      -0.14981421828269958,\n      -0.17287901043891907,\n      -0.04925311356782913,\n      -0.1835131049156189,\n      -0.062022242695093155,\n      -0.17206130921840668,\n      0.08515449613332748,\n      -0.1059260442852974,\n      0.10563120245933533,\n      -0.06231343746185303,\n      -0.14898599684238434,\n      -0.10152725875377655,\n      -0.43200501799583435,\n      -0.04724784940481186,\n      -0.16685062646865845\n    ],\n    [\n      0.5031941533088684,\n      0.09172769635915756,\n      -0.014922316186130047,\n      -0.028131142258644104,\n      0.11934980750083923,\n      0.28395721316337585,\n      0.036394137889146805,\n      0.1998194456100464,\n      -0.20192484557628632,\n      -0.14633506536483765,\n      0.02640669420361519,\n      -0.022488312795758247,\n      0.072130486369133,\n      0.03868991136550903,\n      0.04932060092687607,\n      -0.045582447201013565,\n      0.005045830737799406,\n      -0.12820108234882355,\n      -0.04500775411725044,\n      0.1518886387348175,\n      0.26024237275123596,\n      -0.2181539237499237,\n      0.11115559935569763,\n      -0.15675516426563263,\n      -0.030691029503941536,\n      0.01748749613761902,\n      -0.16511328518390656,\n      0.2737245261669159,\n      -0.13149811327457428,\n      -0.25186625123023987,\n      0.10031213611364365,\n      -0.31338292360305786,\n      0.08425170183181763,\n      -0.16065917909145355,\n      -0.03827944025397301,\n      -0.1989392191171646,\n      -0.05144273117184639,\n      0.11890798807144165,\n      0.07482045888900757,\n      -0.020772382616996765,\n      0.1267835646867752,\n      0.6120129823684692,\n      0.07377228140830994,\n      -0.13923422992229462,\n      -0.05722372978925705,\n      -0.39800146222114563,\n      0.04091900959610939,\n      0.0695667490363121,\n      -0.21238800883293152,\n      -0.012051491066813469\n    ],\n    [\n      0.19333204627037048,\n      0.0458720326423645,\n      -0.04028969630599022,\n      0.20974576473236084,\n      -0.30047404766082764,\n      0.023881543427705765,\n      0.231315478682518,\n      -0.334807425737381,\n      0.22518306970596313,\n      0.10825835913419724,\n      -0.044438861310482025,\n      -0.10490444302558899,\n      -0.06580216437578201,\n      0.11181963980197906,\n      0.07344821095466614,\n      0.1004733145236969,\n      0.07790057361125946,\n      -0.09642831981182098,\n      -0.41057151556015015,\n      -0.2277158796787262,\n      0.012000317685306072,\n      0.06582014262676239,\n      -0.0963224470615387,\n      0.10442311316728592,\n      -0.10906115174293518,\n      -0.06235705316066742,\n      -0.07611753791570663,\n      -0.03609146550297737,\n      -0.04279308021068573,\n      -0.03413715958595276,\n      -0.002363058738410473,\n      0.06544260680675507,\n      -0.020451916381716728,\n      -0.13197165727615356,\n      0.03877728804945946,\n      0.06253300607204437,\n      -0.054499413818120956,\n      0.05485551804304123,\n      -0.06945515424013138,\n      -0.05654872581362724,\n      -0.35323935747146606,\n      0.027126189321279526,\n      -0.47594118118286133,\n      -0.30037999153137207,\n      -0.1167319193482399,\n      0.04441120848059654,\n      0.03415733948349953,\n      0.036278560757637024,\n      -0.12739530205726624,\n      0.13004659116268158\n    ],\n    [\n      -0.19531531631946564,\n      -0.06908230483531952,\n      -0.07997646182775497,\n      0.18082554638385773,\n      -0.03913739323616028,\n      0.3620392978191376,\n      -0.25715160369873047,\n      0.3623271882534027,\n      0.3385258913040161,\n      -0.10079128295183182,\n      0.07242380082607269,\n      -0.03175617381930351,\n      -0.1428028792142868,\n      -0.12019437551498413,\n      0.17039963603019714,\n      0.04258439317345619,\n      0.10656259208917618,\n      0.49076393246650696,\n      -0.1070040762424469,\n      0.04707799106836319,\n      -0.06308545917272568,\n      0.009104114957153797,\n      -0.022145889699459076,\n      -0.5462942719459534,\n      0.04475608468055725,\n      -0.006547586992383003,\n      -0.20924407243728638,\n      -0.05515274032950401,\n      0.029223470017313957,\n      0.10708140581846237,\n      -0.34568893909454346,\n      0.06828770041465759,\n      0.04967491328716278,\n      -0.0714239850640297,\n      -0.11579643189907074,\n      -0.2800796926021576,\n      0.27578005194664,\n      -0.12725789844989777,\n      -0.09610455483198166,\n      0.1855897605419159,\n      -0.05853331461548805,\n      -0.2883037328720093,\n      0.11204751580953598,\n      -0.053892411291599274,\n      -0.21651248633861542,\n      -0.014042823575437069,\n      0.1405620574951172,\n      0.04716895520687103,\n      0.05377839505672455,\n      -0.09346083551645279\n    ],\n    [\n      0.38722315430641174,\n      0.07805776596069336,\n      -0.01699468307197094,\n      0.09417575597763062,\n      0.12787756323814392,\n      -0.21627578139305115,\n      -0.1031956598162651,\n      0.09149119257926941,\n      -0.012079233303666115,\n      0.24742703139781952,\n      0.23472772538661957,\n      0.09376515448093414,\n      -0.09540009498596191,\n      -0.09115155041217804,\n      0.03954261913895607,\n      -0.04487447440624237,\n      0.21316634118556976,\n      0.10701959580183029,\n      -0.2184303253889084,\n      -0.15556873381137848,\n      0.315602570772171,\n      -0.11679258197546005,\n      0.09910526126623154,\n      -0.20800825953483582,\n      0.14748075604438782,\n      0.1530010849237442,\n      -0.16046838462352753,\n      -0.015338634140789509,\n      0.1674281358718872,\n      -0.13022302091121674,\n      0.16147583723068237,\n      0.1275448054075241,\n      -0.07973726838827133,\n      -0.18106280267238617,\n      -0.04503801837563515,\n      -0.18186162412166595,\n      -0.06958714127540588,\n      0.10882542282342911,\n      -0.13745202124118805,\n      -0.14480236172676086,\n      -0.10987883061170578,\n      -0.07267477363348007,\n      0.17072544991970062,\n      -0.033689916133880615,\n      0.0985187515616417,\n      0.051551200449466705,\n      0.290989488363266,\n      -0.06552910804748535,\n      0.1533345729112625,\n      0.29161331057548523\n    ],\n    [\n      -0.1806090921163559,\n      -0.041738081723451614,\n      -0.08328647166490555,\n      0.031262896955013275,\n      0.052669983357191086,\n      -0.024099813774228096,\n      0.11653532087802887,\n      -0.1316749006509781,\n      0.23747195303440094,\n      -0.06056877598166466,\n      -0.04396016150712967,\n      -0.34092724323272705,\n      -0.03117508627474308,\n      -0.23683127760887146,\n      0.08053101599216461,\n      -0.16742601990699768,\n      -0.08569341897964478,\n      0.23417606949806213,\n      -0.0745312049984932,\n      -0.061512358486652374,\n      -0.23650838434696198,\n      -0.14180172979831696,\n      0.2175980508327484,\n      0.04212731868028641,\n      0.11697442084550858,\n      0.07637447118759155,\n      -0.04949411004781723,\n      -0.012892872095108032,\n      0.16023294627666473,\n      -0.2109481245279312,\n      -0.15935631096363068,\n      0.015166888944804668,\n      -0.10519524663686752,\n      -0.06561078876256943,\n      0.017096571624279022,\n      -0.11447957903146744,\n      -0.08708715438842773,\n      -0.11744873970746994,\n      -0.0681682601571083,\n      -0.030574090778827667,\n      0.009261874482035637,\n      -0.08160953223705292,\n      -0.07823522388935089,\n      -0.135882169008255,\n      0.006310142111033201,\n      -0.02442656084895134,\n      0.17197945713996887,\n      -0.10990486294031143,\n      0.05150023475289345,\n      -0.06341389566659927\n    ],\n    [\n      -0.195855975151062,\n      0.08068233728408813,\n      0.0030696892645210028,\n      -0.16477930545806885,\n      -0.186775341629982,\n      0.04576663300395012,\n      -0.11910846084356308,\n      0.26427438855171204,\n      0.078265480697155,\n      -0.040831308811903,\n      -0.08857382833957672,\n      -0.017643751576542854,\n      -0.02189110405743122,\n      0.04620446637272835,\n      0.0972280502319336,\n      -0.20093266665935516,\n      -0.09430713951587677,\n      -0.08537602424621582,\n      -0.10734760016202927,\n      0.18437807261943817,\n      -0.36455708742141724,\n      -0.531965970993042,\n      0.12938083708286285,\n      0.018176771700382233,\n      -0.07023512572050095,\n      -0.12785735726356506,\n      0.08273214846849442,\n      -0.23998332023620605,\n      -0.12417591363191605,\n      0.000614242919255048,\n      -0.0017246110364794731,\n      0.010793485678732395,\n      0.04279060289263725,\n      0.1751609593629837,\n      -0.06606066226959229,\n      0.0001909426791826263,\n      0.12249313294887543,\n      -0.08414557576179504,\n      -0.09383226931095123,\n      -0.04962225258350372,\n      -0.26251712441444397,\n      0.038789208978414536,\n      -0.1602933406829834,\n      0.06584656238555908,\n      -0.024701738730072975,\n      0.06873393803834915,\n      0.13535602390766144,\n      0.49162381887435913,\n      -0.21223457157611847,\n      -0.014465038664638996\n    ]\n  ],\n  "hidden_bias": [\n    -0.07363396883010864,\n    0.15875248610973358,\n    -0.31080272793769836,\n    0.4160917103290558,\n    -0.1923835277557373,\n    -0.39965683221817017,\n    -0.41103920340538025,\n    -0.3894195258617401,\n    -7.536208431702107e-05,\n    -0.1733192354440689,\n    -0.24702410399913788,\n    0.3272821009159088,\n    -0.09365704655647278,\n    0.2931138277053833,\n    -0.5084273815155029,\n    0.10435609519481659,\n    -0.5507968664169312,\n    0.006588201969861984,\n    -0.4079420566558838,\n    -0.07518008351325989,\n    -0.6805596947669983,\n    0.2022731602191925,\n    -0.2460373044013977,\n    0.021723633632063866,\n    -0.0280121061950922,\n    0.0022320691496133804,\n    0.15329201519489288,\n    -0.044834014028310776,\n    -0.46889400482177734,\n    -0.46889418363571167,\n    0.4265376031398773,\n    -0.054780445992946625,\n    -0.3652142584323883,\n    -0.04137824848294258,\n    -0.17552237212657928,\n    -0.5894663333892822,\n    -0.10497724264860153,\n    -0.03403361886739731,\n    0.19322218000888824,\n    0.15876100957393646,\n    3.490285234875046e-05,\n    -0.34467196464538574,\n    -0.08032781630754471,\n    -0.025370456278324127,\n    -0.17398665845394135,\n    0.23623165488243103,\n    -0.052148185670375824,\n    0.3995191156864166,\n    -0.5714481472969055,\n    0.06075944006443024,\n    0.058783840388059616,\n    0.021865632385015488,\n    0.010065016336739063,\n    0.12408261001110077,\n    0.2505068778991699,\n    -0.1436658799648285,\n    -0.11590539664030075,\n    -0.048637475818395615,\n    -0.19469740986824036,\n    -0.5156285762786865,\n    -0.5128587484359741,\n    -0.1043086126446724,\n    -0.15549486875534058,\n    0.0815107598900795,\n    -0.24105210602283478,\n    0.08099883049726486,\n    -0.07653999328613281,\n    -0.15701040625572205,\n    -0.18135838210582733,\n    0.1722155064344406,\n    -0.17045363783836365,\n    -0.3754845857620239,\n    0.07494650781154633,\n    -0.2460373342037201,\n    -0.6093577742576599,\n    -0.10449647158384323,\n    0.05796008184552193,\n    0.13516440987586975,\n    -0.45864686369895935,\n    0.05954965576529503,\n    -0.15716740489006042,\n    -0.23915575444698334,\n    -0.40096554160118103,\n    -0.42271456122398376,\n    -0.16029639542102814,\n    -0.36560800671577454,\n    0.016936423256993294,\n    0.06611008197069168,\n    0.08838945627212524,\n    -0.23134665191173553,\n    -0.30877095460891724,\n    -0.014620626345276833,\n    -0.06004570052027702,\n    0.05538565665483475,\n    0.3052288889884949,\n    0.011457676067948341,\n    0.09725625813007355,\n    -0.02606082521378994,\n    -0.0816371962428093,\n    -0.21787966787815094,\n    -0.26396700739860535,\n    -0.031790461391210556,\n    -0.3920860290527344,\n    -0.23965300619602203,\n    -0.2274298369884491,\n    0.03271228447556496,\n    -0.09817010909318924,\n    -0.4397488534450531,\n    -0.012852567248046398,\n    -0.0033693185541778803,\n    0.034227509051561356,\n    0.25260746479034424,\n    -0.007464599795639515,\n    -0.10096864402294159,\n    0.06392548233270645,\n    0.0438653938472271,\n    -0.3298985958099365,\n    -0.05068805813789368,\n    0.036723461002111435,\n    -0.24871012568473816,\n    -0.024762196466326714,\n    0.010812989436089993,\n    0.16012732684612274,\n    0.040202707052230835,\n    -0.279644638299942,\n    -0.21797990798950195,\n    -0.08397180587053299,\n    0.10978516936302185\n  ],\n  "output_weight": [\n    0.053299739956855774,\n    0.0494362898170948,\n    0.02690199762582779,\n    -0.047990765422582626,\n    0.040582988411188126,\n    -0.03784165531396866,\n    0.04511450603604317,\n    -0.07545795291662216,\n    -0.002202283125370741,\n    0.03954862803220749,\n    0.05160662531852722,\n    0.03631819039583206,\n    -0.031084004789590836,\n    0.02090883068740368,\n    -0.06359715759754181,\n    0.05548921972513199,\n    0.06748080998659134,\n    0.03749121353030205,\n    0.03215166926383972,\n    0.0816577598452568,\n    0.08679162710905075,\n    -0.04332210123538971,\n    0.03411639854311943,\n    0.042879052460193634,\n    0.030467163771390915,\n    -0.059896983206272125,\n    0.03509872034192085,\n    -0.050147850066423416,\n    -0.07421360164880753,\n    -0.1025557890534401,\n    -0.03147843852639198,\n    0.06394430994987488,\n    -0.08079995959997177,\n    -0.03650537505745888,\n    0.019579509273171425,\n    0.10835205018520355,\n    0.02303464338183403,\n    -0.05667070299386978,\n    0.0368136465549469,\n    -0.05140988528728485,\n    0.026948681101202965,\n    0.04371671378612518,\n    -0.05931580439209938,\n    -0.032683826982975006,\n    -0.04506976157426834,\n    -0.031200947239995003,\n    0.05555320531129837,\n    0.027660418301820755,\n    -0.1036989837884903,\n    -0.06191730126738548,\n    -0.07116928696632385,\n    -0.04779083654284477,\n    0.05672765150666237,\n    0.0010311643127352,\n    0.017832256853580475,\n    0.029157957062125206,\n    -0.06199685111641884,\n    0.0060336957685649395,\n    -0.05632350966334343,\n    0.0770607590675354,\n    0.027620520442724228,\n    0.019664529711008072,\n    0.01515794638544321,\n    0.032883480191230774,\n    0.16692355275154114,\n    0.043998125940561295,\n    -0.057851117104291916,\n    0.03831307962536812,\n    0.04244029521942139,\n    0.06962776929140091,\n    -0.053438764065504074,\n    -0.0882093757390976,\n    0.04637227579951286,\n    0.09443821012973785,\n    0.0447738878428936,\n    -0.053458321839571,\n    -0.043681081384420395,\n    0.024863719940185547,\n    0.12547588348388672,\n    0.042731788009405136,\n    -0.045985106378793716,\n    -0.08141477406024933,\n    0.061577752232551575,\n    -0.058092325925827026,\n    -0.05070123076438904,\n    -0.09170954674482346,\n    -0.03490627557039261,\n    -0.0749359130859375,\n    0.04207119718194008,\n    -0.025956960394978523,\n    0.06794538348913193,\n    0.0921277180314064,\n    -0.05313635244965553,\n    -0.01880647800862789,\n    0.05800596997141838,\n    -0.007990390062332153,\n    0.04025733098387718,\n    -0.07399393618106842,\n    0.12803290784358978,\n    0.07994353026151657,\n    0.115820974111557,\n    0.0534600093960762,\n    -0.08279577642679214,\n    0.04745028540492058,\n    -0.04567165672779083,\n    -0.08125541359186172,\n    -0.04450319707393646,\n    -0.13607074320316315,\n    -0.06773985177278519,\n    -0.019400037825107574,\n    -0.041931040585041046,\n    0.03649004176259041,\n    -0.02885163389146328,\n    0.061769574880599976,\n    -0.08730905503034592,\n    -0.05523056536912918,\n    0.022345323115587234,\n    -0.06804019957780838,\n    0.05832691863179207,\n    -0.0267379991710186,\n    -0.04387293756008148,\n    0.043983589857816696,\n    0.07383667677640915,\n    -0.05783320218324661,\n    0.06562008708715439,\n    -0.041579559445381165,\n    -0.019261376932263374,\n    -0.03969300538301468\n  ],\n  "output_bias": 0.0469956798478961,\n  "training": {\n    "method": "paired counterfactual residual policy distillation",\n    "seed": 20261021,\n    "loss_type": "ranking",\n    "regularization": 0.01,\n    "best_epoch": 29,\n    "validation_preference_loss": 0.4398029148578644,\n    "baseline_preference_loss": 0.5722861886024475,\n    "training_games": 2400,\n    "validation_games": 600,\n    "source_dataset": {\n      "method": "paired one-step policy improvement, phase-stratified whole-game reservoirs",\n      "input_features": [\n        "batch_size",\n        "truthful",\n        "bluff",\n        "bluff_pile",\n        "bluff_call_rate",\n        "bluff_finish",\n        "remaining_groups",\n        "remaining_largest_group",\n        "claimed_rank_held",\n        "bias",\n        "revealed_bluff_rate",\n        "rank_held",\n        "claim_count",\n        "pile_size",\n        "opponent_finishing",\n        "own_small_hand",\n        "opponent_small_hand",\n        "samples",\n        "bluff_any_call_probability",\n        "bluff_impossible_for_caller",\n        "truthful_turns_removed",\n        "bluff_known_pile_fraction",\n        "truthful_finish",\n        "bluff_no_impossibility_call_possible",\n        "placer_global_revealed_bluff_rate",\n        "known_outside_rank_fraction",\n        "known_pile_fraction",\n        "claim_fraction_of_original_hand",\n        "finishing_large_batch",\n        "known_placer_rank_before_claim",\n        "bluff_call_probability_batch",\n        "bluff_call_probability_pile",\n        "bluff_call_probability_finish",\n        "bluff_nearest_opponent_finish",\n        "truthful_call_probability_batch",\n        "bluff_remaining_truthful_turns",\n        "global_revealed_bluff_finishing",\n        "count_revealed_bluff_pile",\n        "global_revealed_bluff_pile",\n        "count_revealed_bluff_small_own_hand",\n        "global_revealed_bluff_count",\n        "opponent_hand_pace_relative_to_own",\n        "bluff_pooled_caller_probability",\n        "bluff_pooled_caller_uncertainty",\n        "bluff_specific_caller_uncertainty",\n        "bluff_sparse_own_outcomes",\n        "truthful_pooled_caller_probability_batch",\n        "bluff_pooled_caller_probability_pile",\n        "global_revealed_bluff_uncertainty",\n        "count_revealed_bluff_uncertainty"\n      ],\n      "feature_policy": "v5",\n      "games": 3000,\n      "states": 12000,\n      "actions": 66269,\n      "trials": 4,\n      "samples_per_phase": 2,\n      "max_actions": 10,\n      "seed_start": 64000000,\n      "incumbent": {\n        "format": "triad-bluff-residual-mlp-v1",\n        "features": [\n          "batch_size",\n          "truthful",\n          "bluff",\n          "bluff_pile",\n          "bluff_call_rate",\n          "bluff_finish",\n          "remaining_groups",\n          "remaining_largest_group",\n          "claimed_rank_held",\n          "bias",\n          "revealed_bluff_rate",\n          "rank_held",\n          "claim_count",\n          "pile_size",\n          "opponent_finishing",\n          "own_small_hand",\n          "opponent_small_hand",\n          "samples",\n          "bluff_any_call_probability",\n          "bluff_impossible_for_caller",\n          "truthful_turns_removed",\n          "bluff_known_pile_fraction",\n          "truthful_finish",\n          "bluff_no_impossibility_call_possible",\n          "placer_global_revealed_bluff_rate",\n          "known_outside_rank_fraction",\n          "known_pile_fraction",\n          "claim_fraction_of_original_hand",\n          "finishing_large_batch",\n          "known_placer_rank_before_claim",\n          "bluff_call_probability_batch",\n          "bluff_call_probability_pile",\n          "bluff_call_probability_finish",\n          "bluff_nearest_opponent_finish",\n          "truthful_call_probability_batch",\n          "bluff_remaining_truthful_turns",\n          "global_revealed_bluff_finishing",\n          "count_revealed_bluff_pile",\n          "global_revealed_bluff_pile",\n          "count_revealed_bluff_small_own_hand",\n          "global_revealed_bluff_count",\n          "opponent_hand_pace_relative_to_own",\n          "bluff_pooled_caller_probability",\n          "bluff_pooled_caller_uncertainty",\n          "bluff_specific_caller_uncertainty",\n          "bluff_sparse_own_outcomes",\n          "truthful_pooled_caller_probability_batch",\n          "bluff_pooled_caller_probability_pile",\n          "global_revealed_bluff_uncertainty",\n          "count_revealed_bluff_uncertainty"\n        ],\n        "base_weights": [\n          0.11889829486608505,\n          1.4551690816879272,\n          -0.8614801168441772,\n          0.12563253939151764,\n          -2.55124568939209,\n          -2.6591808795928955,\n          0.4780709743499756,\n          1.6731796264648438,\n          0.8349596261978149,\n          -0.9919783473014832,\n          1.7952346801757812,\n          -2.062161922454834,\n          -1.0631221532821655,\n          -1.4293476343154907,\n          -2.5744378566741943,\n          -1.0431323051452637,\n          0.9109983444213867,\n          1.1962584257125854,\n          -2.227351665496826,\n          -2.295670509338379,\n          0.8897631168365479,\n          -0.3832564353942871,\n          1.4892596006393433,\n          2.4285433292388916,\n          0.8895594477653503,\n          -0.11950855702161789,\n          -0.31598660349845886,\n          0.15420950949192047,\n          0.7154812216758728,\n          -1.7789300680160522,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0,\n          0.0\n        ],\n        "feature_mean": [\n          2.130405902862549,\n          0.2556110918521881,\n          0.5673422813415527,\n          0.1414773166179657,\n          0.20922617614269257,\n          0.044980913400650024,\n          0.22203241288661957,\n          0.4362170994281769,\n          0.304308146238327,\n          0.08852332085371017,\n          0.012027798220515251,\n          0.005555760581046343,\n          0.05282813683152199,\n          0.03254607692360878,\n          0.002083986531943083,\n          0.015454696491360664,\n          0.015533286146819592,\n          0.002624368527904153,\n          0.28184229135513306,\n          0.06834737956523895,\n          1.5457278490066528,\n          0.023674404248595238,\n          0.010973202995955944,\n          0.09815023094415665,\n          0.015177818946540356,\n          0.007621304597705603,\n          0.008625890128314495,\n          0.020141232758760452,\n          0.0009267285349778831,\n          0.006892831530421972,\n          0.21310904622077942,\n          0.05682198330760002,\n          0.020169535651803017,\n          0.12666580080986023,\n          0.05104479193687439,\n          0.13413675129413605,\n          0.00024044152814894915,\n          0.0043762726709246635,\n          0.005239620339125395,\n          0.0022578288335353136,\n          0.00981908105313778,\n          -0.10914473235607147,\n          0.2997722029685974,\n          0.0871744155883789,\n          0.12125638127326965,\n          0.23309296369552612,\n          0.05302620306611061,\n          0.044763777405023575,\n          0.011351659893989563,\n          0.011535020545125008\n        ],\n        "feature_scale": [\n          1.3542357683181763,\n          0.43625375628471375,\n          0.4955291450023651,\n          0.32548826932907104,\n          0.23007379472255707,\n          0.20728932321071625,\n          0.20233742892742157,\n          0.3010976016521454,\n          0.2976628541946411,\n          0.28402191400527954,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.18657560646533966,\n          0.15651710331439972,\n          0.10000000149011612,\n          0.12336929887533188,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.2647395730018616,\n          0.2523006200790405,\n          1.334463357925415,\n          0.10000000149011612,\n          0.1041841208934784,\n          0.29747799038887024,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.21329738199710846,\n          0.12353038787841797,\n          0.10000000149011612,\n          0.18938058614730835,\n          0.10285001248121262,\n          0.1730906069278717,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.9756709337234497,\n          0.3055936396121979,\n          0.10000000149011612,\n          0.10987535119056702,\n          0.4228348731994629,\n          0.11198851466178894,\n          0.10000000149011612,\n          0.10000000149011612,\n          0.10000000149011612\n        ],\n        "hidden_weight": [\n          [\n            -0.19053113460540771,\n            0.15571263432502747,\n            -0.1011505052447319,\n            0.19476446509361267,\n            0.27500608563423157,\n            0.15307991206645966,\n            0.3815036416053772,\n            0.4381793439388275,\n            -0.22956830263137817,\n            0.027035346254706383,\n            -0.021941421553492546,\n            0.004426548257470131,\n            0.04070810601115227,\n            -0.05979225039482117,\n            0.0010170531459152699,\n            -0.08468963205814362,\n            0.03122447244822979,\n            0.04913467541337013,\n            0.017885252833366394,\n            0.07352987676858902,\n            0.07204055041074753,\n            -0.6565521359443665,\n            0.23869308829307556,\n            -0.137938991189003,\n            0.22352753579616547,\n            0.07389921694993973,\n            -0.021464945748448372,\n            -0.07632283121347427,\n            -0.0017953533679246902,\n            -0.15402238070964813,\n            -0.2351987063884735,\n            0.2819085419178009,\n            -0.04682714119553566,\n            0.3633670508861542,\n            0.1471453309059143,\n            -0.030505336821079254,\n            0.11275077611207962,\n            -0.04455846548080444,\n            -0.19067446887493134,\n            -0.02123730443418026,\n            0.15277132391929626,\n            -0.0441616028547287,\n            0.19499239325523376,\n            0.005214580800384283,\n            -0.16005682945251465,\n            0.41382917761802673,\n            -0.3361380696296692,\n            0.03683564066886902,\n            0.17921903729438782,\n            0.09538088738918304\n          ],\n          [\n            -0.34688928723335266,\n            -0.0341823436319828,\n            0.08536931127309799,\n            -0.15115806460380554,\n            0.2503475546836853,\n            -0.015907486900687218,\n            0.6435881853103638,\n            0.10033245384693146,\n            0.2746313512325287,\n            -0.13110876083374023,\n            -0.2760956585407257,\n            0.004355510231107473,\n            0.11898820102214813,\n            -0.0032388579566031694,\n            0.1219732016324997,\n            -0.09869767725467682,\n            -0.027204176411032677,\n            -0.41678473353385925,\n            0.26750725507736206,\n            -0.011986590921878815,\n            0.05444285273551941,\n            0.28975367546081543,\n            0.05204622447490692,\n            -0.5399481058120728,\n            -0.10123363882303238,\n            0.03145816922187805,\n            0.08416414260864258,\n            0.2687167227268219,\n            0.013803120702505112,\n            0.27541759610176086,\n            -0.3447941243648529,\n            0.18976999819278717,\n            0.16411860287189484,\n            -0.0076088146306574345,\n            -0.14338168501853943,\n            0.16006532311439514,\n            -0.009708402678370476,\n            0.07059653103351593,\n            -0.1445803940296173,\n            -0.09821895509958267,\n            -0.19294242560863495,\n            0.016644466668367386,\n            0.3165804147720337,\n            -0.35061442852020264,\n            0.046103350818157196,\n            -0.020881682634353638,\n            -0.44616249203681946,\n            -0.184276282787323,\n            -0.04051925987005234,\n            -0.076309435069561\n          ],\n          [\n            -0.43836283683776855,\n            -0.03827773407101631,\n            0.02417719177901745,\n            0.0012864257441833615,\n            -0.18788781762123108,\n            0.05437812581658363,\n            -0.048713840544223785,\n            0.1781936138868332,\n            0.36789923906326294,\n            0.15997500717639923,\n            0.08273530751466751,\n            0.11687079071998596,\n            -0.03200904652476311,\n            -0.10258307307958603,\n            -0.044549256563186646,\n            -0.07501626014709473,\n            0.10152506083250046,\n            0.42922788858413696,\n            0.24868355691432953,\n            0.25955644249916077,\n            0.2103431671857834,\n            -0.06754159182310104,\n            0.10614077001810074,\n            -0.3674386739730835,\n            0.03684944659471512,\n            -0.011080002412199974,\n            -0.17129723727703094,\n            -0.07273056358098984,\n            0.012710507027804852,\n            0.0012870589271187782,\n            0.040439359843730927,\n            0.007053954526782036,\n            -0.1431591808795929,\n            0.7334985733032227,\n            -0.020398594439029694,\n            0.07002000510692596,\n            -0.1414639800786972,\n            -0.014410188421607018,\n            0.0591629222035408,\n            -0.02386775054037571,\n            0.1051902174949646,\n            0.06887765228748322,\n            -0.14827775955200195,\n            0.2279590368270874,\n            -0.06324269622564316,\n            0.19206000864505768,\n            -0.15030698478221893,\n            0.04369090497493744,\n            -0.09575971961021423,\n            0.10184834897518158\n          ],\n          [\n            -0.8009151220321655,\n            0.042741164565086365,\n            0.2561957538127899,\n            -0.03245321288704872,\n            0.2126467525959015,\n            -0.3626656234264374,\n            -0.08744355291128159,\n            0.252923846244812,\n            -0.2406282126903534,\n            -0.2533353567123413,\n            -0.3336438238620758,\n            -0.49595001339912415,\n            0.03433312103152275,\n            0.1379004865884781,\n            -0.5763046741485596,\n            -0.4408527612686157,\n            -0.17515550553798676,\n            0.08869050443172455,\n            0.26932889223098755,\n            -0.33251070976257324,\n            -0.2426517903804779,\n            0.0012413441436365247,\n            0.11090754717588425,\n            0.05123414471745491,\n            -0.24921815097332,\n            -0.4750996530056,\n            0.48253777623176575,\n            -0.027838952839374542,\n            -0.38995078206062317,\n            -0.31328147649765015,\n            -0.3602568507194519,\n            -0.19446098804473877,\n            -0.543244481086731,\n            -0.11715645343065262,\n            0.13799040019512177,\n            -0.1073513776063919,\n            -0.37774690985679626,\n            0.022098375484347343,\n            -0.13915301859378815,\n            -0.4281318783760071,\n            -0.2979890704154968,\n            0.10360624641180038,\n            0.3499918580055237,\n            0.13586987555027008,\n            0.13379111886024475,\n            -0.06283661723136902,\n            0.1657276302576065,\n            0.3291209638118744,\n            -0.15946771204471588,\n            -0.14842531085014343\n          ],\n          [\n            -0.4666611850261688,\n            -0.21241676807403564,\n            0.1910749077796936,\n            0.06911717355251312,\n            0.23509050905704498,\n            0.31656408309936523,\n            0.2708587646484375,\n            0.49206769466400146,\n            0.24235671758651733,\n            0.026647981256246567,\n            0.1187470406293869,\n            -0.14169251918792725,\n            -0.08341462910175323,\n            0.12279897183179855,\n            0.05708643049001694,\n            -0.23100155591964722,\n            0.0009216552716679871,\n            -0.23335391283035278,\n            -0.03357965499162674,\n            0.3808215260505676,\n            0.23694391548633575,\n            -0.15176722407341003,\n            0.14328934252262115,\n            -0.06307930499315262,\n            0.1969938725233078,\n            0.33162984251976013,\n            -0.10945877432823181,\n            0.34658071398735046,\n            0.11664565652608871,\n            0.10710567981004715,\n            -0.3654142916202545,\n            0.28199678659439087,\n            0.089230976998806,\n            0.3477048873901367,\n            0.03435347229242325,\n            0.17952503263950348,\n            -0.005079449154436588,\n            0.3451724946498871,\n            0.2539703845977783,\n            0.31034937500953674,\n            0.21049605309963226,\n            0.5691021680831909,\n            -0.04177477955818176,\n            0.14757639169692993,\n            0.46376973390579224,\n            0.05747527629137039,\n            0.18095451593399048,\n            -0.25173884630203247,\n            0.18668290972709656,\n            0.14818932116031647\n          ],\n          [\n            -0.7069453001022339,\n            0.10696735233068466,\n            -0.1955098658800125,\n            0.04555068910121918,\n            0.3874591588973999,\n            0.20683079957962036,\n            -0.446833997964859,\n            0.29962068796157837,\n            -0.034959275275468826,\n            0.20622512698173523,\n            0.12322287261486053,\n            0.025812890380620956,\n            0.03312835097312927,\n            -0.08387616276741028,\n            0.015942800790071487,\n            0.16793064773082733,\n            0.23285695910453796,\n            0.048181504011154175,\n            0.04697374999523163,\n            0.07852131873369217,\n            -0.1740342527627945,\n            -0.21954193711280823,\n            0.04958309605717659,\n            -0.4439425766468048,\n            0.10293136537075043,\n            -0.044991955161094666,\n            0.09410923719406128,\n            -0.0233745314180851,\n            -0.08290598541498184,\n            0.06586933881044388,\n            -0.13631898164749146,\n            0.054769959300756454,\n            0.2936391532421112,\n            -0.4733392000198364,\n            0.025694748386740685,\n            -0.30140426754951477,\n            0.10703320801258087,\n            -0.00779837416484952,\n            0.07616543024778366,\n            -0.020140083506703377,\n            0.06963186711072922,\n            -0.03516196087002754,\n            0.28908097743988037,\n            -0.1399356573820114,\n            0.448266863822937,\n            0.4235385060310364,\n            -0.3639507591724396,\n            -0.09979131817817688,\n            0.018416909500956535,\n            0.01727212220430374\n          ],\n          [\n            -0.03547343611717224,\n            0.042599186301231384,\n            -0.13445666432380676,\n            0.27867192029953003,\n            -0.1933528035879135,\n            0.14748847484588623,\n            0.23424869775772095,\n            0.6576681137084961,\n            -0.2924177944660187,\n            0.0798158124089241,\n            0.10613710433244705,\n            0.015439657494425774,\n            0.14775849878787994,\n            0.09118485450744629,\n            -0.12247393280267715,\n            0.22114013135433197,\n            0.07602106779813766,\n            0.29813095927238464,\n            -0.31005844473838806,\n            0.1560666412115097,\n            -0.2705908715724945,\n            -0.09668433666229248,\n            0.14817866683006287,\n            0.12874020636081696,\n            0.23493953049182892,\n            0.06945419311523438,\n            -0.07678230106830597,\n            0.11114998906850815,\n            0.059269193559885025,\n            0.13170039653778076,\n            -0.32500356435775757,\n            0.21914531290531158,\n            0.2580072581768036,\n            -0.18756340444087982,\n            0.60267573595047,\n            0.09367770701646805,\n            0.002875487320125103,\n            0.031517673283815384,\n            -0.012696266174316406,\n            0.12384146451950073,\n            0.10011958330869675,\n            -0.13896508514881134,\n            -0.06378152966499329,\n            0.12267380952835083,\n            0.07059703767299652,\n            -0.03849215805530548,\n            0.3489634692668915,\n            0.17285385727882385,\n            0.34713345766067505,\n            0.10579388588666916\n          ],\n          [\n            -0.3740462064743042,\n            -0.14343155920505524,\n            0.13711591064929962,\n            0.21937614679336548,\n            0.805578887462616,\n            -0.48606735467910767,\n            -0.6440969109535217,\n            0.008069561794400215,\n            0.03596563637256622,\n            0.06841330230236053,\n            0.09124680608510971,\n            0.20227183401584625,\n            -0.1476958841085434,\n            -0.10490380972623825,\n            -0.19745688140392303,\n            -0.3149966597557068,\n            0.0642014816403389,\n            0.09790126234292984,\n            -0.3586054742336273,\n            -0.14411096274852753,\n            -0.3387010395526886,\n            -0.461421936750412,\n            0.012739426456391811,\n            0.07183930277824402,\n            0.07319998741149902,\n            0.06467419862747192,\n            -0.02730027213692665,\n            0.0177640188485384,\n            0.009762308560311794,\n            0.18767277896404266,\n            -0.25299814343452454,\n            0.004556243307888508,\n            -0.4331885874271393,\n            0.6903507709503174,\n            0.09927846491336823,\n            0.11865057051181793,\n            -0.19630247354507446,\n            0.03672018647193909,\n            0.006273719482123852,\n            -0.27636194229125977,\n            0.10056620836257935,\n            -0.2750764787197113,\n            -0.2743605375289917,\n            0.11487728357315063,\n            -0.21548666059970856,\n            -0.0849001407623291,\n            0.1967046856880188,\n            0.3868160545825958,\n            -0.11838560551404953,\n            0.10246492177248001\n          ],\n          [\n            -0.15618306398391724,\n            -0.0239772517234087,\n            0.04755749553442001,\n            0.010224930010735989,\n            0.11524950712919235,\n            -0.014770879410207272,\n            -0.06294459104537964,\n            -0.031985167413949966,\n            0.032739587128162384,\n            0.09298791736364365,\n            -0.11481371521949768,\n            -0.15526078641414642,\n            -0.022801987826824188,\n            -0.15966346859931946,\n            0.05545642971992493,\n            0.21264657378196716,\n            -0.08094839006662369,\n            0.4150295555591583,\n            0.038763172924518585,\n            0.14816109836101532,\n            -0.10293320566415787,\n            0.05836347118020058,\n            -0.0792263001203537,\n            -0.32350730895996094,\n            0.1665249615907669,\n            -0.04150903970003128,\n            -0.05239039659500122,\n            -0.08471309393644333,\n            0.121266208589077,\n            -0.0014270353130996227,\n            -0.05903973802924156,\n            -0.02083747088909149,\n            0.06164424121379852,\n            0.12489137798547745,\n            -0.10239112377166748,\n            0.11750168353319168,\n            -0.14621612429618835,\n            0.04299991577863693,\n            0.04783204570412636,\n            0.15346753597259521,\n            0.014543632045388222,\n            0.10683470964431763,\n            -0.03323371335864067,\n            -0.003854575101286173,\n            -0.19214366376399994,\n            -0.03526030108332634,\n            -0.04858594015240669,\n            0.10026992112398148,\n            0.007083422504365444,\n            -0.1050834059715271\n          ],\n          [\n            -0.2340119630098343,\n            -0.1311962902545929,\n            0.012483648024499416,\n            0.08511599898338318,\n            -0.044495388865470886,\n            0.20780839025974274,\n            0.20153303444385529,\n            0.2624434232711792,\n            0.32059332728385925,\n            0.08335617929697037,\n            0.04421668499708176,\n            -0.015539539977908134,\n            0.007245282176882029,\n            0.18704229593276978,\n            -0.26119303703308105,\n            -0.43681997060775757,\n            0.03169839456677437,\n            -0.3552597463130951,\n            0.1179923489689827,\n            -0.10485481470823288,\n            0.23826444149017334,\n            -0.25279805064201355,\n            0.058726225048303604,\n            -0.31730422377586365,\n            0.014462308958172798,\n            -0.00023079095990397036,\n            0.1388983428478241,\n            0.12865278124809265,\n            -0.12762616574764252,\n            0.005441761109977961,\n            -0.1515406370162964,\n            -0.051951657980680466,\n            0.17133884131908417,\n            0.685310423374176,\n            -0.00957696232944727,\n            0.038983482867479324,\n            -0.1642482876777649,\n            0.1801123023033142,\n            0.03890753909945488,\n            -0.14680218696594238,\n            0.06145242601633072,\n            1.042846918106079,\n            -0.042383112013339996,\n            -0.15215317904949188,\n            -0.24692203104496002,\n            0.18917889893054962,\n            -0.14350120723247528,\n            0.26828792691230774,\n            0.23018144071102142,\n            0.10932588577270508\n          ],\n          [\n            0.3322123885154724,\n            -0.18821783363819122,\n            0.0765753984451294,\n            0.23367373645305634,\n            0.3780375123023987,\n            0.1865251213312149,\n            0.3168398439884186,\n            -0.8197890520095825,\n            0.11141742765903473,\n            -0.03586093708872795,\n            -0.0836518257856369,\n            0.19181115925312042,\n            0.22029082477092743,\n            -0.024909770116209984,\n            -0.21364426612854004,\n            0.04330916702747345,\n            -0.02714921161532402,\n            -0.6437059640884399,\n            0.10146825015544891,\n            0.2402789294719696,\n            0.27162525057792664,\n            -0.2091243714094162,\n            0.011973648332059383,\n            -0.018197134137153625,\n            -0.09150644391775131,\n            0.1643635779619217,\n            -0.02910521812736988,\n            0.11285676062107086,\n            -0.033838752657175064,\n            -0.22987478971481323,\n            0.13096605241298676,\n            0.12056934833526611,\n            0.024511314928531647,\n            -0.2536150813102722,\n            -0.19454994797706604,\n            0.34864115715026855,\n            -0.10269317030906677,\n            0.15696042776107788,\n            -0.32188767194747925,\n            -0.09561910480260849,\n            0.060371577739715576,\n            0.08191504329442978,\n            -0.8268782496452332,\n            -0.2580358386039734,\n            0.3688259422779083,\n            -0.48169219493865967,\n            -0.049497392028570175,\n            0.29896190762519836,\n            -0.026430204510688782,\n            0.008099472150206566\n          ],\n          [\n            -0.029509885236620903,\n            -0.34242621064186096,\n            0.16561579704284668,\n            -0.010497413575649261,\n            0.04179583117365837,\n            0.10720138251781464,\n            0.04319336637854576,\n            0.02871839515864849,\n            0.023956257849931717,\n            -0.01933148317039013,\n            -0.16761919856071472,\n            -0.09857246279716492,\n            -0.22773438692092896,\n            0.13983239233493805,\n            -0.2308560311794281,\n            -0.2406291514635086,\n            -0.06087779253721237,\n            0.06693939119577408,\n            0.17468194663524628,\n            0.1563374251127243,\n            -0.1310565173625946,\n            0.49474406242370605,\n            -0.15331414341926575,\n            -0.32666221261024475,\n            -0.20615436136722565,\n            -0.21729524433612823,\n            0.20773786306381226,\n            -0.3822798728942871,\n            -0.04398162290453911,\n            -0.1281927078962326,\n            0.007586438674479723,\n            -0.611744225025177,\n            -0.05213972553610802,\n            -0.09962411969900131,\n            -0.6330392956733704,\n            0.23169295489788055,\n            -0.2639612555503845,\n            0.13328424096107483,\n            -0.06009092554450035,\n            -0.1239374503493309,\n            -0.3297577500343323,\n            0.10535461455583572,\n            0.23223823308944702,\n            -0.09761741012334824,\n            -0.3502937853336334,\n            -0.18727360665798187,\n            -0.29612648487091064,\n            -0.056469522416591644,\n            -0.10573189705610275,\n            -0.14346741139888763\n          ],\n          [\n            -0.26724305748939514,\n            0.08889330178499222,\n            0.008113402873277664,\n            -0.27317148447036743,\n            0.2011149823665619,\n            -0.04121546074748039,\n            -0.22013381123542786,\n            -0.797607421875,\n            0.36875659227371216,\n            -0.058561183512210846,\n            0.08153904974460602,\n            -0.007255932316184044,\n            -0.0984342098236084,\n            0.057313233613967896,\n            -0.0735824853181839,\n            -0.26956233382225037,\n            0.017771027982234955,\n            -0.08869937062263489,\n            -0.005857986398041248,\n            -0.25606125593185425,\n            -0.09762841463088989,\n            -0.33106791973114014,\n            -0.06652376800775528,\n            -0.12843550741672516,\n            -0.08270307630300522,\n            -0.046221524477005005,\n            -0.09148037433624268,\n            0.07982458919286728,\n            0.03203823044896126,\n            0.016645997762680054,\n            0.039965804666280746,\n            0.2062731832265854,\n            0.11458174884319305,\n            0.18539603054523468,\n            -0.022299231961369514,\n            -0.49218472838401794,\n            -0.09138567000627518,\n            -0.004584163893014193,\n            0.05923668295145035,\n            -0.29008257389068604,\n            0.10724376887083054,\n            -0.07637747377157211,\n            -0.23378099501132965,\n            0.05910341441631317,\n            0.203413188457489,\n            -0.09736248105764389,\n            0.2546629011631012,\n            0.09849297255277634,\n            -0.24182625114917755,\n            -0.07155266404151917\n          ],\n          [\n            -0.3199852705001831,\n            -0.04902562126517296,\n            -0.11015060544013977,\n            0.13555565476417542,\n            -0.1786220222711563,\n            0.09948953241109848,\n            0.14066071808338165,\n            -0.21072794497013092,\n            -0.5030348896980286,\n            -0.08439727872610092,\n            -0.2530125081539154,\n            -0.3614713251590729,\n            -0.05258870869874954,\n            -0.10137499868869781,\n            -0.22220778465270996,\n            0.32201874256134033,\n            0.0799582377076149,\n            0.15970627963542938,\n            0.0795905590057373,\n            0.27575740218162537,\n            -0.044520117342472076,\n            -0.18729595839977264,\n            -0.11038057506084442,\n            -0.3810097873210907,\n            -0.4145146608352661,\n            -0.056077029556035995,\n            -0.14889715611934662,\n            -0.11386126279830933,\n            -0.4162250757217407,\n            -0.043115030974149704,\n            -0.08273240923881531,\n            -0.021875593811273575,\n            0.010167443193495274,\n            0.3572275638580322,\n            0.09942388534545898,\n            0.3671188950538635,\n            -0.3396758437156677,\n            -0.25534388422966003,\n            -0.4077712297439575,\n            0.06318295001983643,\n            -0.32920631766319275,\n            -0.07883904874324799,\n            -0.010043187066912651,\n            0.07424140721559525,\n            0.05924276262521744,\n            -0.2450912594795227,\n            0.04750494286417961,\n            0.019476082175970078,\n            -0.2556335926055908,\n            -0.15605509281158447\n          ],\n          [\n            0.6340359449386597,\n            0.04145973548293114,\n            -0.07292081415653229,\n            0.07251635938882828,\n            -0.08202191442251205,\n            0.04293441027402878,\n            0.5576285719871521,\n            0.23475027084350586,\n            0.44962626695632935,\n            0.006248448044061661,\n            0.06210228055715561,\n            -0.2050963193178177,\n            0.03397884964942932,\n            -0.022366419434547424,\n            0.08456797897815704,\n            0.0763671025633812,\n            0.10751130431890488,\n            -0.08908049017190933,\n            -0.44752663373947144,\n            -0.2895398437976837,\n            0.33557626605033875,\n            -0.272480845451355,\n            0.09224913269281387,\n            0.35606077313423157,\n            0.2651411294937134,\n            0.4373738169670105,\n            0.3549002408981323,\n            0.08648732304573059,\n            0.14063279330730438,\n            -0.06875208020210266,\n            0.1601409614086151,\n            0.041062165051698685,\n            0.04832049459218979,\n            0.1375073343515396,\n            -0.16349495947360992,\n            -0.20357580482959747,\n            0.04089963436126709,\n            -0.06695489585399628,\n            0.12484778463840485,\n            0.03442877158522606,\n            0.15138018131256104,\n            -0.3351605534553528,\n            -0.2105487734079361,\n            0.008606968447566032,\n            -0.06378739327192307,\n            -0.318213552236557,\n            0.036649588495492935,\n            0.0718257948756218,\n            0.05229468643665314,\n            0.10798918455839157\n          ],\n          [\n            -0.4880310297012329,\n            -0.290319561958313,\n            0.18820735812187195,\n            -0.06892693042755127,\n            0.20368099212646484,\n            -0.26760879158973694,\n            0.31209075450897217,\n            -0.593944787979126,\n            0.498654305934906,\n            0.10167057812213898,\n            0.14594915509223938,\n            0.10385118424892426,\n            -0.30846941471099854,\n            -0.16828879714012146,\n            -0.1857483834028244,\n            -0.4659593105316162,\n            -0.09148602187633514,\n            0.9138979911804199,\n            0.05556275323033333,\n            -0.04497753083705902,\n            -0.07722266763448715,\n            -0.08538047969341278,\n            -0.1086912751197815,\n            -0.18396438658237457,\n            -0.0110703120008111,\n            -0.2346513569355011,\n            0.6007078886032104,\n            -0.4091869592666626,\n            -0.10134308785200119,\n            0.08310243487358093,\n            -0.33675816655158997,\n            0.280028373003006,\n            -0.03388960659503937,\n            -0.2295314520597458,\n            -0.2214173972606659,\n            0.1346883475780487,\n            -0.15148119628429413,\n            -0.11658339947462082,\n            -0.3361017405986786,\n            -0.3612913191318512,\n            -0.09712904691696167,\n            0.024362584576010704,\n            -0.09650429338216782,\n            -0.24035996198654175,\n            0.10614266246557236,\n            0.09804969280958176,\n            -0.5469701886177063,\n            -0.13870614767074585,\n            -0.027438627555966377,\n            0.026141872629523277\n          ],\n          [\n            0.07229895144701004,\n            0.03486057370901108,\n            -0.19355346262454987,\n            0.29655587673187256,\n            0.13598886132240295,\n            0.10626787692308426,\n            0.09095804393291473,\n            0.31509214639663696,\n            -0.13798213005065918,\n            0.23979517817497253,\n            0.276653528213501,\n            -0.03847532346844673,\n            0.32585328817367554,\n            0.05657418817281723,\n            0.09893937408924103,\n            0.145035058259964,\n            0.029836229979991913,\n            0.2778482437133789,\n            -0.35207322239875793,\n            0.08971316367387772,\n            0.4157799780368805,\n            0.028854133561253548,\n            -0.10814930498600006,\n            0.055448174476623535,\n            0.22193656861782074,\n            -0.0042461915872991085,\n            0.030554668977856636,\n            0.16198888421058655,\n            -0.035144224762916565,\n            0.05954234302043915,\n            -0.6285920143127441,\n            0.18410919606685638,\n            0.1915920525789261,\n            -0.5593177080154419,\n            -0.1998709887266159,\n            -0.3633144795894623,\n            -0.005100379232317209,\n            0.22080577909946442,\n            0.07437482476234436,\n            0.16885313391685486,\n            0.27040478587150574,\n            -0.29884520173072815,\n            0.18099215626716614,\n            0.2769649624824524,\n            0.051309648901224136,\n            0.05014529824256897,\n            0.4624188542366028,\n            0.24487924575805664,\n            0.14120706915855408,\n            0.2307230830192566\n          ],\n          [\n            -0.28400999307632446,\n            -0.31002405285835266,\n            -0.012761787511408329,\n            0.14118428528308868,\n            0.4426450729370117,\n            0.1171816810965538,\n            0.10751694440841675,\n            -0.6219719052314758,\n            0.035196222364902496,\n            -0.1460052877664566,\n            -0.022354193031787872,\n            -0.04214084893465042,\n            -0.04056141525506973,\n            -0.0316203348338604,\n            0.058338381350040436,\n            -0.07202573865652084,\n            -0.16142955422401428,\n            -0.12001245468854904,\n            0.3406381607055664,\n            0.0032692679669708014,\n            -0.36121946573257446,\n            -0.22993430495262146,\n            -0.06961605697870255,\n            -0.808266282081604,\n            -0.10942716896533966,\n            -0.10159999877214432,\n            -0.10182453691959381,\n            -0.23560519516468048,\n            -0.14857430756092072,\n            0.0991126224398613,\n            0.08501067012548447,\n            0.12716858088970184,\n            -0.10174532979726791,\n            0.01189819723367691,\n            -0.0013789650984108448,\n            0.2006131261587143,\n            0.03138815984129906,\n            -0.18153460323810577,\n            -0.03544573113322258,\n            -0.20842579007148743,\n            -0.1990332305431366,\n            0.017326844856142998,\n            0.08995715528726578,\n            -0.29164257645606995,\n            -0.054569680243730545,\n            0.07508371770381927,\n            0.1298971027135849,\n            0.11354754865169525,\n            -0.11951977014541626,\n            -0.007311002351343632\n          ],\n          [\n            -0.313959002494812,\n            0.02367100678384304,\n            -0.23750454187393188,\n            0.19636744260787964,\n            -0.4092922508716583,\n            0.3455292284488678,\n            0.1998608410358429,\n            0.22046808898448944,\n            -0.35031044483184814,\n            0.03594677150249481,\n            0.12130099534988403,\n            0.04169800877571106,\n            0.2498067170381546,\n            -0.0372464656829834,\n            0.1514752060174942,\n            -0.011637207120656967,\n            0.17202650010585785,\n            0.10219906270503998,\n            -0.44123736023902893,\n            0.20308686792850494,\n            -0.13556955754756927,\n            0.0338115394115448,\n            0.07485858350992203,\n            -0.441572904586792,\n            0.19151411950588226,\n            0.06156781688332558,\n            -0.09438305348157883,\n            -0.032099466770887375,\n            0.12792031466960907,\n            -0.09593029320240021,\n            -0.14163430035114288,\n            0.21665973961353302,\n            0.2212689220905304,\n            -0.09448543190956116,\n            -0.09020856767892838,\n            -0.09307733178138733,\n            -0.11562037467956543,\n            0.21669070422649384,\n            0.2042226493358612,\n            0.10017234086990356,\n            0.0006136708543635905,\n            0.2547128200531006,\n            -0.24404467642307281,\n            -0.17294526100158691,\n            -0.012948770076036453,\n            0.18893244862556458,\n            -0.06888674944639206,\n            0.09324324131011963,\n            0.1519479602575302,\n            0.11566980183124542\n          ],\n          [\n            -0.4943947196006775,\n            -0.37880223989486694,\n            0.12686514854431152,\n            -0.028622223064303398,\n            0.17918363213539124,\n            -0.13050785660743713,\n            0.2026994228363037,\n            -0.17935694754123688,\n            0.4380630552768707,\n            -0.11766725033521652,\n            -0.129194974899292,\n            -0.04541458934545517,\n            0.1604277342557907,\n            -0.1607377976179123,\n            -0.04632454365491867,\n            -0.011155758053064346,\n            -0.13659162819385529,\n            -1.0188244581222534,\n            0.37685921788215637,\n            -0.18514999747276306,\n            0.07884564250707626,\n            0.1309960037469864,\n            -0.02880169078707695,\n            -0.28296878933906555,\n            -0.004682627506554127,\n            -0.030437076464295387,\n            -0.015656590461730957,\n            0.21033556759357452,\n            -0.1351899802684784,\n            -0.2936747372150421,\n            -0.44261977076530457,\n            0.15120938420295715,\n            -0.21807026863098145,\n            -0.34597519040107727,\n            -0.0027242463547736406,\n            0.11175087839365005,\n            -0.13817018270492554,\n            -0.11875613778829575,\n            -0.05769561976194382,\n            0.12040862441062927,\n            0.00770455552265048,\n            0.07549260556697845,\n            -0.2092660665512085,\n            -0.1579517424106598,\n            0.35385575890541077,\n            -0.2981831431388855,\n            -0.14543303847312927,\n            -0.41573572158813477,\n            -0.1754993051290512,\n            -0.25732412934303284\n          ],\n          [\n            -0.5843906998634338,\n            0.023006102070212364,\n            -0.2868063449859619,\n            0.19544805586338043,\n            -0.2703016400337219,\n            -0.4782143831253052,\n            0.15871690213680267,\n            -0.5789566040039062,\n            0.27165892720222473,\n            0.16037949919700623,\n            0.15952976047992706,\n            -0.4202682077884674,\n            -0.061652787029743195,\n            0.007442873436957598,\n            -0.14205460250377655,\n            0.12480668723583221,\n            0.10434304177761078,\n            0.33649593591690063,\n            0.006205507088452578,\n            0.37024521827697754,\n            -0.1162598580121994,\n            0.2969197630882263,\n            -0.1422189474105835,\n            -0.04484979435801506,\n            0.33930695056915283,\n            -0.3059912323951721,\n            -0.26080235838890076,\n            -0.08195023238658905,\n            -0.2389908730983734,\n            0.2225254625082016,\n            -0.020228303968906403,\n            0.08015476167201996,\n            -0.20816384255886078,\n            0.5272433757781982,\n            -0.24563837051391602,\n            -0.231959730386734,\n            -0.4278993308544159,\n            -0.19001170992851257,\n            -0.06070360913872719,\n            0.01738264597952366,\n            -0.032029006630182266,\n            -0.32012397050857544,\n            0.7366903424263,\n            -0.35473912954330444,\n            0.04068528115749359,\n            0.10522628575563431,\n            0.4234182834625244,\n            0.4338505268096924,\n            0.2428039014339447,\n            0.15285803377628326\n          ],\n          [\n            0.32526499032974243,\n            0.0700555145740509,\n            0.16584300994873047,\n            0.14409296214580536,\n            -0.12014655023813248,\n            -0.1813012957572937,\n            0.23767665028572083,\n            -0.5436683297157288,\n            0.04718820005655289,\n            -0.05421098321676254,\n            0.0914592370390892,\n            -0.03039310686290264,\n            -0.21307942271232605,\n            -0.21871115267276764,\n            0.24239540100097656,\n            -0.1697438806295395,\n            0.15428747236728668,\n            -1.0806773900985718,\n            -0.14046365022659302,\n            -0.26631084084510803,\n            0.0949084609746933,\n            0.576619029045105,\n            0.02062864601612091,\n            -0.3112156391143799,\n            0.029294926673173904,\n            0.30035170912742615,\n            0.19895866513252258,\n            -0.058695610612630844,\n            -0.013687225058674812,\n            0.09398889541625977,\n            -0.00045071024214848876,\n            0.04743901267647743,\n            -0.22651807963848114,\n            -0.48898157477378845,\n            -0.22358489036560059,\n            0.39652758836746216,\n            0.3221066892147064,\n            -0.03893650323152542,\n            0.18594510853290558,\n            -0.13303305208683014,\n            -0.03647095710039139,\n            0.07256828993558884,\n            -0.13758498430252075,\n            0.11072687059640884,\n            0.29907891154289246,\n            -0.04181970655918121,\n            -0.07504820823669434,\n            0.08294575661420822,\n            -0.058195117861032486,\n            0.07449038326740265\n          ],\n          [\n            -0.5233381986618042,\n            0.0008672281401231885,\n            -0.03805981203913689,\n            0.21171528100967407,\n            0.23875588178634644,\n            0.04219716042280197,\n            0.5243920683860779,\n            0.24311195313930511,\n            0.1120820939540863,\n            0.14307865500450134,\n            0.09491016715765,\n            0.18101878464221954,\n            0.08173248171806335,\n            0.11200929433107376,\n            -0.04997217655181885,\n            0.13585078716278076,\n            0.03232493996620178,\n            0.08581267297267914,\n            -0.07864118367433548,\n            0.1779915690422058,\n            -0.058080434799194336,\n            0.03136714547872543,\n            0.11727762222290039,\n            -0.5342357158660889,\n            0.10215844959020615,\n            0.031690146774053574,\n            -0.12930142879486084,\n            0.03609846532344818,\n            -0.02158038690686226,\n            0.051236532628536224,\n            -0.36258170008659363,\n            0.31467580795288086,\n            0.03029535710811615,\n            -0.24494363367557526,\n            0.02350180596113205,\n            -0.026698840782046318,\n            0.05107489600777626,\n            0.12503203749656677,\n            0.02067023143172264,\n            0.1583503931760788,\n            0.07459188997745514,\n            -0.08375407755374908,\n            -0.13252781331539154,\n            0.08350452780723572,\n            0.028414392843842506,\n            0.24695615470409393,\n            0.07464083284139633,\n            -0.0393829308450222,\n            0.16417503356933594,\n            0.03753133490681648\n          ],\n          [\n            -0.25898006558418274,\n            0.025537917390465736,\n            0.05677321180701256,\n            -0.03887172043323517,\n            0.3491106331348419,\n            0.005391890183091164,\n            0.14687418937683105,\n            -0.7723749876022339,\n            0.34060534834861755,\n            -0.04562126472592354,\n            -0.00858503021299839,\n            -0.0998515784740448,\n            -0.1542772501707077,\n            -0.0470099039375782,\n            -0.12248252332210541,\n            -0.15803958475589752,\n            -0.09450267255306244,\n            0.09489660710096359,\n            0.18596765398979187,\n            0.10402277112007141,\n            -0.28702250123023987,\n            -0.33180755376815796,\n            -0.20281799137592316,\n            -0.06972038745880127,\n            -0.03837069496512413,\n            -0.02239038050174713,\n            -0.07939201593399048,\n            0.040755368769168854,\n            -0.04678730294108391,\n            -0.01703268103301525,\n            0.20631742477416992,\n            0.2041403204202652,\n            -0.11710700392723083,\n            0.5981282591819763,\n            -0.09409008175134659,\n            0.1276732236146927,\n            -0.05574595183134079,\n            -0.08588626980781555,\n            0.07564660161733627,\n            0.09508813172578812,\n            -0.07919439673423767,\n            0.07502157241106033,\n            -0.13839483261108398,\n            -0.08790223300457001,\n            0.18147775530815125,\n            0.014002826064825058,\n            0.33050379157066345,\n            0.15746422111988068,\n            -0.0012312111211940646,\n            -0.01717819832265377\n          ],\n          [\n            -0.049337729811668396,\n            -0.25081348419189453,\n            -0.09281174093484879,\n            0.007727284450083971,\n            0.19251953065395355,\n            0.013646545819938183,\n            -0.09981174021959305,\n            -0.45988261699676514,\n            -0.09433955699205399,\n            -0.02321835234761238,\n            0.00871738325804472,\n            -0.2401668131351471,\n            -0.29249924421310425,\n            0.033145297318696976,\n            0.08899916708469391,\n            0.06436201184988022,\n            0.22819466888904572,\n            0.8446778059005737,\n            -0.0030743919778615236,\n            -0.06640619784593582,\n            0.004932293202728033,\n            -0.47813647985458374,\n            -0.01945245824754238,\n            -0.6641075611114502,\n            -0.10394883900880814,\n            -0.23606987297534943,\n            -0.04755371809005737,\n            -0.3163021504878998,\n            0.0927969366312027,\n            -0.1444014608860016,\n            -0.15632379055023193,\n            0.1263875514268875,\n            -0.02288808859884739,\n            0.23840370774269104,\n            -0.4262140989303589,\n            0.03808695450425148,\n            -0.2587064206600189,\n            -0.10681041330099106,\n            -0.27944910526275635,\n            -0.012667844071984291,\n            -0.04578428342938423,\n            0.07030569761991501,\n            0.0029238443821668625,\n            -0.2608352601528168,\n            -0.1478152722120285,\n            0.193078875541687,\n            -0.046339958906173706,\n            0.35504114627838135,\n            -0.16476699709892273,\n            0.10348972678184509\n          ],\n          [\n            -1.026081919670105,\n            0.02744203247129917,\n            0.09528150409460068,\n            0.3015398681163788,\n            0.19705185294151306,\n            0.11942197382450104,\n            0.06681142002344131,\n            0.0752897709608078,\n            0.21040531992912292,\n            0.10917877405881882,\n            0.17000837624073029,\n            -0.4611847996711731,\n            0.08706481754779816,\n            -0.34832972288131714,\n            0.04933488368988037,\n            -0.03971042484045029,\n            -0.13147708773612976,\n            -0.04960452765226364,\n            0.14727677404880524,\n            0.4749727249145508,\n            0.20106777548789978,\n            0.049444835633039474,\n            -0.04854841157793999,\n            0.207233726978302,\n            -0.049859073013067245,\n            0.19830238819122314,\n            -0.13150238990783691,\n            -0.1400122493505478,\n            -0.07341067492961884,\n            -0.14692482352256775,\n            -0.3291547894477844,\n            -0.18517127633094788,\n            0.26848304271698,\n            -0.28378555178642273,\n            0.013217049650847912,\n            0.2950667142868042,\n            0.026300951838493347,\n            -0.06482067704200745,\n            -0.15643450617790222,\n            0.08075056970119476,\n            -0.15444286167621613,\n            0.09519573301076889,\n            0.26808351278305054,\n            -0.19517827033996582,\n            0.12917156517505646,\n            -0.1733582466840744,\n            -0.15164172649383545,\n            0.24183572828769684,\n            -0.054356083273887634,\n            -0.08053381741046906\n          ],\n          [\n            0.2168618142604828,\n            -0.20145684480667114,\n            0.19428397715091705,\n            -0.7246829867362976,\n            -0.2696896195411682,\n            0.13537590205669403,\n            0.16436220705509186,\n            0.05112558975815773,\n            -0.45426303148269653,\n            -0.019151555374264717,\n            -0.20496296882629395,\n            -0.27425745129585266,\n            -0.1241728886961937,\n            -0.04212280362844467,\n            -0.20894964039325714,\n            -0.2541345953941345,\n            -0.11864986270666122,\n            -0.22503788769245148,\n            0.41987642645835876,\n            -0.025292381644248962,\n            -0.19198943674564362,\n            0.36040639877319336,\n            -0.06132618337869644,\n            0.049134623259305954,\n            -0.27169284224510193,\n            -0.13051527738571167,\n            -0.1766655296087265,\n            -0.25108131766319275,\n            -0.11183443665504456,\n            -0.2144920825958252,\n            0.4685809314250946,\n            -0.21427913010120392,\n            -0.035761263221502304,\n            0.2656055688858032,\n            -0.16916987299919128,\n            0.24911807477474213,\n            -0.01632564887404442,\n            0.014032711274921894,\n            -0.20321834087371826,\n            -0.026093054562807083,\n            -0.20576433837413788,\n            0.1716836392879486,\n            -0.38304486870765686,\n            0.174935981631279,\n            -0.03438805788755417,\n            0.14550459384918213,\n            -0.057037413120269775,\n            -0.08195150643587112,\n            -0.0025741150602698326,\n            -0.16669988632202148\n          ],\n          [\n            0.23142458498477936,\n            -0.07066720724105835,\n            0.17238746583461761,\n            0.29519540071487427,\n            -0.19976314902305603,\n            0.04449260234832764,\n            0.27449196577072144,\n            0.02509736828505993,\n            -0.2874855101108551,\n            -0.048564452677965164,\n            -0.0011683930642902851,\n            -0.36487382650375366,\n            0.1721363067626953,\n            -0.33491894602775574,\n            0.25553280115127563,\n            -0.20792138576507568,\n            0.0665774717926979,\n            -0.41052478551864624,\n            -0.4131990969181061,\n            -0.018855014815926552,\n            0.3250945508480072,\n            -0.0759248211979866,\n            0.026377229019999504,\n            -0.6710410714149475,\n            -0.013546440750360489,\n            0.06632640957832336,\n            -0.3221264183521271,\n            0.2857840955257416,\n            -0.14754991233348846,\n            -0.03256852552294731,\n            -0.14526675641536713,\n            0.09109043329954147,\n            0.15627263486385345,\n            -0.21006415784358978,\n            -0.18224649131298065,\n            -0.06908787786960602,\n            0.2840081751346588,\n            -0.20591585338115692,\n            -0.2587043344974518,\n            -0.15888836979866028,\n            0.21626555919647217,\n            0.22565510869026184,\n            -0.6376015543937683,\n            0.25216415524482727,\n            0.4058956801891327,\n            -0.07704833894968033,\n            -0.19777317345142365,\n            0.10293310135602951,\n            0.054722681641578674,\n            0.006313395220786333\n          ],\n          [\n            0.10213359445333481,\n            0.09516975283622742,\n            -0.05086249113082886,\n            -0.6481035947799683,\n            -0.01004539243876934,\n            0.20505502820014954,\n            0.4609759449958801,\n            -0.5429484844207764,\n            -0.4750175476074219,\n            -0.03546210750937462,\n            -0.034196190536022186,\n            -0.3148801326751709,\n            0.1869203746318817,\n            -0.31235817074775696,\n            -0.1983237862586975,\n            0.22295261919498444,\n            -0.14915519952774048,\n            -0.1021757423877716,\n            -0.011998548172414303,\n            0.11987872421741486,\n            -0.03372648358345032,\n            -0.10986010730266571,\n            0.21892975270748138,\n            0.20584993064403534,\n            0.023383690044283867,\n            -0.024683650583028793,\n            -0.2989889085292816,\n            0.098348468542099,\n            -0.17126165330410004,\n            0.03279683366417885,\n            -0.2868504524230957,\n            -0.038162991404533386,\n            -0.2008214294910431,\n            -0.7928566932678223,\n            -0.3392408490180969,\n            0.16560392081737518,\n            0.4668349623680115,\n            -0.136211559176445,\n            -0.044954363256692886,\n            0.3486177921295166,\n            -0.04184315726161003,\n            0.13062907755374908,\n            -0.019696103408932686,\n            0.1324857473373413,\n            0.34107741713523865,\n            0.09835230559110641,\n            -0.01589975319802761,\n            -0.389157235622406,\n            -0.10209089517593384,\n            0.14743943512439728\n          ],\n          [\n            -0.45345762372016907,\n            0.29428210854530334,\n            -0.2090853899717331,\n            0.18112409114837646,\n            -0.285805344581604,\n            0.251838743686676,\n            0.28611043095588684,\n            0.43320441246032715,\n            0.3690531849861145,\n            -0.09351328015327454,\n            0.0426195003092289,\n            0.34546083211898804,\n            0.1765691488981247,\n            -0.0577813945710659,\n            -0.02116667665541172,\n            -0.09047780930995941,\n            -0.06086895242333412,\n            0.08966435492038727,\n            0.10113221406936646,\n            0.05189082771539688,\n            0.5182246565818787,\n            -0.22100670635700226,\n            0.14233289659023285,\n            -0.16081075370311737,\n            0.22065068781375885,\n            -0.04606064409017563,\n            -0.41266435384750366,\n            0.22386234998703003,\n            -0.17480097711086273,\n            0.04691757261753082,\n            -0.2166384607553482,\n            0.03212357684969902,\n            0.18761563301086426,\n            0.8363586664199829,\n            -0.1254202127456665,\n            0.4455370008945465,\n            0.042582012712955475,\n            0.16312097012996674,\n            0.3006892800331116,\n            -0.2718108594417572,\n            0.3281973600387573,\n            0.19491545855998993,\n            -0.18445897102355957,\n            -0.20376981794834137,\n            -0.2981553077697754,\n            0.1237783282995224,\n            -0.016201067715883255,\n            0.1662474125623703,\n            -0.10616820305585861,\n            -0.06032990664243698\n          ],\n          [\n            0.232689768075943,\n            -0.44447171688079834,\n            0.2105448693037033,\n            0.2751316428184509,\n            0.37711894512176514,\n            0.0022084673400968313,\n            0.36341285705566406,\n            -0.12637849152088165,\n            0.12386828660964966,\n            -0.33580708503723145,\n            -0.27682510018348694,\n            -0.19002771377563477,\n            -0.3964892327785492,\n            -0.3058200776576996,\n            -0.3354426920413971,\n            -0.26642197370529175,\n            -0.35868096351623535,\n            -0.21975313127040863,\n            0.18821504712104797,\n            -0.0631081610918045,\n            0.3153245747089386,\n            0.228838250041008,\n            -0.42645883560180664,\n            -0.3963789641857147,\n            -0.22596678137779236,\n            -0.39264315366744995,\n            -0.3945613205432892,\n            -0.18992874026298523,\n            -0.23827767372131348,\n            -0.4351137578487396,\n            0.14640508592128754,\n            0.2626323699951172,\n            0.05601993203163147,\n            0.17204931378364563,\n            -0.4240603744983673,\n            0.24811267852783203,\n            -0.3620324432849884,\n            -0.2059416025876999,\n            -0.38578566908836365,\n            -0.3957683742046356,\n            -0.37723562121391296,\n            0.43035218119621277,\n            0.36406394839286804,\n            0.19895757734775543,\n            0.36992719769477844,\n            -0.06417641043663025,\n            -0.24774883687496185,\n            0.3493569493293762,\n            -0.308956503868103,\n            -0.29132598638534546\n          ],\n          [\n            0.023923726752400398,\n            -0.2246474176645279,\n            0.05102603882551193,\n            -0.05412611365318298,\n            0.6693800091743469,\n            0.15467675030231476,\n            0.2673344016075134,\n            0.4409731328487396,\n            0.218567356467247,\n            -0.027865683659911156,\n            0.14328183233737946,\n            0.28424784541130066,\n            0.29170235991477966,\n            -0.21605022251605988,\n            -0.21891926229000092,\n            -0.20754027366638184,\n            -0.06515808403491974,\n            -0.45035234093666077,\n            0.34284478425979614,\n            -0.011246146634221077,\n            -0.565503716468811,\n            0.5686787366867065,\n            0.1872691512107849,\n            0.22016026079654694,\n            -0.054653093218803406,\n            0.14171342551708221,\n            0.25563979148864746,\n            0.33700719475746155,\n            -0.11873351037502289,\n            -0.41830453276634216,\n            -0.006842146161943674,\n            -0.05337260663509369,\n            0.0794617310166359,\n            0.4183383882045746,\n            -0.1572146862745285,\n            0.11208879202604294,\n            -0.1934702843427658,\n            0.06295482814311981,\n            -0.2973204255104065,\n            -0.24784885346889496,\n            0.21634313464164734,\n            -0.06762319058179855,\n            -0.06652755290269852,\n            0.34584861993789673,\n            0.15847043693065643,\n            -0.4551164507865906,\n            0.2029569298028946,\n            -0.18294475972652435,\n            0.16255755722522736,\n            0.11765027046203613\n          ],\n          [\n            0.14456596970558167,\n            0.23278626799583435,\n            -0.1887637823820114,\n            0.3462575674057007,\n            0.04277247190475464,\n            -0.3237273097038269,\n            -0.001447975286282599,\n            -0.11434292048215866,\n            0.1814809888601303,\n            -0.0034667898435145617,\n            0.20990003645420074,\n            0.13823778927326202,\n            0.07381826639175415,\n            -0.07411574572324753,\n            -0.004729114938527346,\n            0.05571029707789421,\n            0.09177210181951523,\n            0.18506179749965668,\n            -0.27223801612854004,\n            -0.08937279134988785,\n            -0.1868707537651062,\n            -0.5123598575592041,\n            0.06821846216917038,\n            -0.061055004596710205,\n            0.08442848920822144,\n            0.04776329547166824,\n            0.190947026014328,\n            0.022976884618401527,\n            -0.08878858387470245,\n            0.061760641634464264,\n            0.27804508805274963,\n            0.1212826669216156,\n            -0.0813269391655922,\n            -0.787715494632721,\n            0.032655902206897736,\n            -0.14507944881916046,\n            -0.0301596038043499,\n            -0.01838349923491478,\n            -0.004320202860981226,\n            0.1694989651441574,\n            0.0779040977358818,\n            -0.038277775049209595,\n            0.9786800742149353,\n            0.21260057389736176,\n            0.26448294520378113,\n            0.364663690328598,\n            0.09841179847717285,\n            0.6356847882270813,\n            -0.08115611970424652,\n            -0.01330023817718029\n          ],\n          [\n            -0.07180869579315186,\n            0.10873803496360779,\n            -0.32323765754699707,\n            -0.09696871787309647,\n            0.11804728955030441,\n            0.11073006689548492,\n            -0.22454044222831726,\n            -0.6546792387962341,\n            0.3506810963153839,\n            0.11831827461719513,\n            0.05820620432496071,\n            -0.09232763200998306,\n            0.08361367136240005,\n            -0.15400183200836182,\n            0.06318897008895874,\n            -0.12331978976726532,\n            -0.21589307487010956,\n            0.1472158432006836,\n            0.03838353604078293,\n            -0.05095091834664345,\n            0.31625866889953613,\n            -0.460136741399765,\n            0.09728768467903137,\n            -0.5579326152801514,\n            -0.09198839217424393,\n            0.019890688359737396,\n            0.10566385835409164,\n            -0.21438159048557281,\n            -0.16338294744491577,\n            -0.1149531677365303,\n            0.14449386298656464,\n            0.3294515013694763,\n            0.1777021735906601,\n            -0.0473497100174427,\n            -0.05907529592514038,\n            -0.33387672901153564,\n            0.07428231835365295,\n            -0.06336923688650131,\n            -0.06709934771060944,\n            0.231694757938385,\n            0.15215325355529785,\n            -0.1876620352268219,\n            -0.03445301577448845,\n            -0.28921061754226685,\n            -0.11027219146490097,\n            -0.27413409948349,\n            -0.14854860305786133,\n            0.13366009294986725,\n            -0.10490302741527557,\n            0.03806979954242706\n          ],\n          [\n            -0.542121410369873,\n            -0.09144406765699387,\n            0.06516508758068085,\n            0.1444881111383438,\n            -0.06035938858985901,\n            -0.18947480618953705,\n            0.18588626384735107,\n            0.20526662468910217,\n            0.20760725438594818,\n            0.09160985052585602,\n            0.07479687035083771,\n            -0.0952373743057251,\n            0.014119399711489677,\n            0.12985841929912567,\n            -0.025724386796355247,\n            -0.08784901350736618,\n            0.001289311796426773,\n            0.055360712110996246,\n            -0.08706086874008179,\n            0.08450601994991302,\n            0.14436344802379608,\n            -0.014574314467608929,\n            0.13801687955856323,\n            -0.5366510152816772,\n            0.1834602653980255,\n            0.06558286398649216,\n            -0.15803036093711853,\n            0.12210821360349655,\n            -0.05941721424460411,\n            -0.0416509173810482,\n            -0.5011538863182068,\n            0.18266136944293976,\n            0.013269172050058842,\n            0.09535439312458038,\n            -0.11875949800014496,\n            0.044719498604536057,\n            -0.06809011846780777,\n            0.17947758734226227,\n            -0.026595115661621094,\n            0.0039175753481686115,\n            0.03470650687813759,\n            0.11196630448102951,\n            0.04789390787482262,\n            -0.02402217872440815,\n            -0.031177038326859474,\n            0.181244894862175,\n            -0.20908616483211517,\n            -0.04465540125966072,\n            0.01718798093497753,\n            0.008290152065455914\n          ],\n          [\n            -0.7346735000610352,\n            -0.658176600933075,\n            0.20461003482341766,\n            0.07439437508583069,\n            -0.13856419920921326,\n            0.07014510780572891,\n            -0.24387702345848083,\n            0.11632256209850311,\n            0.8128634691238403,\n            -0.04128701612353325,\n            -0.21861222386360168,\n            -0.09075810015201569,\n            0.023668358102440834,\n            0.18390357494354248,\n            -0.04334521293640137,\n            -0.13654561340808868,\n            0.07880350947380066,\n            -0.5477664470672607,\n            0.3222714066505432,\n            0.20942415297031403,\n            0.24103082716464996,\n            0.0010369870578870177,\n            0.07903929054737091,\n            -0.7326748967170715,\n            -0.018193775787949562,\n            0.14304082095623016,\n            -0.5774063467979431,\n            0.04663686826825142,\n            -0.38176771998405457,\n            -0.09514370560646057,\n            -0.2693597674369812,\n            -0.1760627031326294,\n            0.10530556738376617,\n            -0.0075754462741315365,\n            -0.2862960696220398,\n            -0.12947608530521393,\n            -0.340509295463562,\n            0.31726163625717163,\n            0.16130729019641876,\n            0.4510969817638397,\n            -0.1292114555835724,\n            0.5841554403305054,\n            -0.1322399079799652,\n            0.135287806391716,\n            -0.2676231265068054,\n            0.038900963962078094,\n            0.28827574849128723,\n            0.1366054266691208,\n            0.17256863415241241,\n            -0.053452786058187485\n          ],\n          [\n            -0.3961889147758484,\n            -0.46040841937065125,\n            0.2196914255619049,\n            0.05046434327960014,\n            -0.33147358894348145,\n            0.2684059143066406,\n            0.0474977120757103,\n            -0.017679188400506973,\n            0.26660850644111633,\n            -0.09793423116207123,\n            0.1641201674938202,\n            0.10482963919639587,\n            0.22193235158920288,\n            0.10818701237440109,\n            -0.10824785381555557,\n            -0.019321439787745476,\n            -0.014227773062884808,\n            -0.020141875371336937,\n            -0.13751612603664398,\n            0.05705909803509712,\n            -0.22966720163822174,\n            -0.18884046375751495,\n            0.20037731528282166,\n            -0.29322007298469543,\n            0.0446658581495285,\n            0.07818875461816788,\n            -0.19028626382350922,\n            0.13077665865421295,\n            0.08423267304897308,\n            0.008018983528017998,\n            -0.519278883934021,\n            0.21920281648635864,\n            0.05410536378622055,\n            0.2532445192337036,\n            -0.20290900766849518,\n            0.22744540870189667,\n            -0.1427524983882904,\n            0.13327495753765106,\n            -0.05179793760180473,\n            0.006791737861931324,\n            0.09608040004968643,\n            0.20773813128471375,\n            0.24394957721233368,\n            0.1281476765871048,\n            -0.026160025969147682,\n            0.03990846127271652,\n            -0.24915944039821625,\n            0.0083148293197155,\n            0.2173263132572174,\n            -0.05482521653175354\n          ],\n          [\n            -0.46897947788238525,\n            0.1939813494682312,\n            -0.032292917370796204,\n            0.31936466693878174,\n            -0.13258635997772217,\n            0.08417251706123352,\n            0.06938806176185608,\n            -0.5968083143234253,\n            0.15931163728237152,\n            0.025160523131489754,\n            0.10951870679855347,\n            -0.06244724616408348,\n            -0.028845395892858505,\n            0.0747387632727623,\n            0.10975819081068039,\n            0.024257972836494446,\n            0.14591236412525177,\n            0.18429380655288696,\n            -0.33624523878097534,\n            0.33360517024993896,\n            -0.24695807695388794,\n            -0.4387824237346649,\n            0.0842091292142868,\n            -0.03296291083097458,\n            0.14091207087039948,\n            -0.004438221920281649,\n            0.06744446605443954,\n            -0.06671259552240372,\n            -0.0782000795006752,\n            0.03303195908665657,\n            -0.25879067182540894,\n            0.09065626561641693,\n            0.12398450076580048,\n            0.9408542513847351,\n            0.03293769806623459,\n            0.09764484316110611,\n            -0.0036100498400628567,\n            -0.09164755046367645,\n            0.10436680167913437,\n            0.0855497345328331,\n            0.10866373777389526,\n            0.0606394000351429,\n            -0.15805990993976593,\n            0.04706179350614548,\n            -0.18564823269844055,\n            0.32456475496292114,\n            0.013767565600574017,\n            0.2760608196258545,\n            -0.08011768013238907,\n            0.15603870153427124\n          ],\n          [\n            0.06530220061540604,\n            -0.1938244104385376,\n            -0.06791539490222931,\n            0.1684546023607254,\n            0.588080108165741,\n            -0.05347514525055885,\n            -0.027186203747987747,\n            -0.6285409927368164,\n            -0.03389909118413925,\n            -0.07645681500434875,\n            -0.1571633219718933,\n            0.3467240035533905,\n            -0.08898109942674637,\n            -0.1689881533384323,\n            0.024456139653921127,\n            -0.25249725580215454,\n            -0.31211522221565247,\n            0.08157378435134888,\n            -0.0979478731751442,\n            -0.12501373887062073,\n            -0.25109243392944336,\n            -0.15055960416793823,\n            -0.13126854598522186,\n            0.07118936628103256,\n            0.01431370247155428,\n            0.10147762298583984,\n            0.08022072166204453,\n            -0.18964862823486328,\n            -0.10489154607057571,\n            -0.31562647223472595,\n            -0.20392489433288574,\n            0.09224116057157516,\n            -0.06302554905414581,\n            0.40176886320114136,\n            -0.17208777368068695,\n            0.05694785341620445,\n            -0.13730579614639282,\n            -0.10423211753368378,\n            -0.15937544405460358,\n            -0.046164944767951965,\n            0.04765899106860161,\n            0.0626683160662651,\n            -0.42633363604545593,\n            -0.019328445196151733,\n            0.23325718939304352,\n            0.0571424625813961,\n            -0.1589375138282776,\n            0.5786423087120056,\n            0.02490766905248165,\n            -0.13667245209217072\n          ],\n          [\n            -0.28771018981933594,\n            0.11562525480985641,\n            0.13038471341133118,\n            0.05946330353617668,\n            0.23543591797351837,\n            0.24393689632415771,\n            -0.227728009223938,\n            -0.2421497106552124,\n            0.25773248076438904,\n            0.009055891074240208,\n            0.042949821799993515,\n            -0.13164646923542023,\n            0.01785208471119404,\n            0.032690197229385376,\n            0.08964651823043823,\n            -0.2653658986091614,\n            -0.2937764823436737,\n            -0.04746067896485329,\n            -0.0561523400247097,\n            -0.6565371155738831,\n            0.12242861092090607,\n            0.2353164255619049,\n            -0.11213594675064087,\n            0.10395673662424088,\n            -0.02287227101624012,\n            -0.11630058288574219,\n            0.10504776239395142,\n            -0.0014501643599942327,\n            -0.007164937909692526,\n            -0.022275321185588837,\n            -0.1653028130531311,\n            0.022580260410904884,\n            0.2609707713127136,\n            -0.7206534147262573,\n            -0.2215133160352707,\n            -0.5918362140655518,\n            -0.0035360658075660467,\n            -0.1185218021273613,\n            0.07292405515909195,\n            -0.2265658974647522,\n            -0.05880394205451012,\n            0.17773087322711945,\n            0.5721856355667114,\n            0.14834612607955933,\n            -0.011880542151629925,\n            0.16113513708114624,\n            -0.2819710969924927,\n            0.14223423600196838,\n            -0.11073827743530273,\n            -0.14627468585968018\n          ],\n          [\n            0.040821321308612823,\n            -0.20160777866840363,\n            -0.05570583418011665,\n            0.004754761699587107,\n            -0.5616325736045837,\n            -0.05090862140059471,\n            0.1670302450656891,\n            -0.19239507615566254,\n            -0.48343104124069214,\n            -0.17791041731834412,\n            -0.22411267459392548,\n            -0.047452736645936966,\n            -0.1549903005361557,\n            -0.11515803635120392,\n            -0.08002974838018417,\n            -0.11467361450195312,\n            -0.1606924831867218,\n            -0.21562559902668,\n            0.23508183658123016,\n            0.15265709161758423,\n            -0.2348959594964981,\n            0.008862434886395931,\n            -0.04552668705582619,\n            -0.2892385721206665,\n            -0.0220014750957489,\n            -0.07863172143697739,\n            -0.0968034639954567,\n            -0.028949622064828873,\n            -0.12328556925058365,\n            -0.17934443056583405,\n            -0.11296600848436356,\n            -0.11576605588197708,\n            0.022461242973804474,\n            0.3400169312953949,\n            -0.2392335832118988,\n            -0.17141087353229523,\n            -0.13378208875656128,\n            -0.21247243881225586,\n            -0.1951640099287033,\n            -0.15454521775245667,\n            0.04504670202732086,\n            0.09864577651023865,\n            0.13875499367713928,\n            0.3617118299007416,\n            -0.05337328463792801,\n            0.32986173033714294,\n            0.32204610109329224,\n            0.26012060046195984,\n            -0.01748303696513176,\n            -0.1285756379365921\n          ],\n          [\n            -0.36107540130615234,\n            -0.23316925764083862,\n            -0.0005261289770714939,\n            0.23076489567756653,\n            0.43435776233673096,\n            -0.07070977240800858,\n            -0.3080746531486511,\n            0.44530999660491943,\n            -0.21137793362140656,\n            -0.003932056948542595,\n            -0.01840481534600258,\n            0.14063875377178192,\n            -0.08314060419797897,\n            -0.16481882333755493,\n            0.004022188019007444,\n            -0.07609113305807114,\n            -0.03963321074843407,\n            0.6156153082847595,\n            0.11877110600471497,\n            -0.5687031745910645,\n            -0.35158517956733704,\n            -0.36379000544548035,\n            0.23329587280750275,\n            0.23173342645168304,\n            0.300642192363739,\n            0.08068472146987915,\n            -0.08311239629983902,\n            -0.2587467133998871,\n            0.12286464869976044,\n            -0.05061759427189827,\n            0.015050970017910004,\n            0.31135866045951843,\n            -0.14364568889141083,\n            0.2513780891895294,\n            -0.0556335411965847,\n            -0.27911168336868286,\n            -0.12411349266767502,\n            -0.02942589484155178,\n            -0.05207731947302818,\n            0.2911379337310791,\n            0.0683738961815834,\n            0.11167559027671814,\n            -0.09840988367795944,\n            0.12265720218420029,\n            -0.08435030281543732,\n            -0.16036835312843323,\n            -0.11171884834766388,\n            0.33031022548675537,\n            -0.09275522083044052,\n            -0.035640649497509\n          ],\n          [\n            0.3510441184043884,\n            -0.035518430173397064,\n            0.1440327763557434,\n            -0.006552477832883596,\n            -0.19958235323429108,\n            -0.08374685049057007,\n            0.2199011594057083,\n            -0.811407208442688,\n            -0.04289533570408821,\n            0.13828562200069427,\n            0.07134454697370529,\n            -0.5139862298965454,\n            0.020516257733106613,\n            -0.03293876349925995,\n            0.17269454896450043,\n            0.21152111887931824,\n            0.16204194724559784,\n            0.23592056334018707,\n            -0.16266214847564697,\n            0.0677967369556427,\n            0.31394630670547485,\n            0.14992383122444153,\n            0.0345202200114727,\n            -0.4013928771018982,\n            0.03696933388710022,\n            -0.1786099076271057,\n            -0.07471805810928345,\n            0.2389640063047409,\n            0.05816379189491272,\n            0.05141192674636841,\n            0.09221195429563522,\n            0.07856535166501999,\n            0.0026714038103818893,\n            -0.36936143040657043,\n            -0.18525612354278564,\n            -0.5446624159812927,\n            0.1133439689874649,\n            -0.09709855169057846,\n            -0.10296037048101425,\n            0.26518115401268005,\n            0.16124682128429413,\n            -0.005081284791231155,\n            -0.6138672828674316,\n            0.16156505048274994,\n            0.3464154005050659,\n            0.1769765019416809,\n            -0.38684895634651184,\n            -0.5983367562294006,\n            -0.026411525905132294,\n            0.07318852841854095\n          ],\n          [\n            -0.7498357892036438,\n            0.21063445508480072,\n            -0.20563901960849762,\n            0.12044817209243774,\n            -0.04553386941552162,\n            0.2572336792945862,\n            0.20674782991409302,\n            -0.06195385009050369,\n            0.18859727680683136,\n            0.007550694048404694,\n            -0.08987760543823242,\n            -0.027995649725198746,\n            0.05885208025574684,\n            -0.07259200513362885,\n            -0.06576356291770935,\n            -0.07104557752609253,\n            -0.0628618597984314,\n            0.4630751609802246,\n            0.12426687031984329,\n            -0.11534039676189423,\n            0.09168915450572968,\n            -0.2538411319255829,\n            -0.030915604904294014,\n            0.08649176359176636,\n            -0.14069928228855133,\n            0.0226445235311985,\n            0.20883843302726746,\n            0.042885906994342804,\n            0.0958562046289444,\n            -0.0032906290143728256,\n            -0.2657560110092163,\n            0.041179344058036804,\n            0.3829485774040222,\n            0.07227688282728195,\n            0.09059078246355057,\n            0.13119883835315704,\n            -0.09330447763204575,\n            -0.33125561475753784,\n            -0.2283739298582077,\n            0.09303531050682068,\n            -0.18821220099925995,\n            0.06556814163923264,\n            0.3424741327762604,\n            0.1392085701227188,\n            -0.03409351781010628,\n            0.2728283703327179,\n            0.14966948330402374,\n            0.3203960657119751,\n            -0.17096266150474548,\n            -0.010720270685851574\n          ],\n          [\n            0.274734228849411,\n            0.07605774700641632,\n            0.05234622582793236,\n            0.1289052963256836,\n            0.27108338475227356,\n            -0.13771401345729828,\n            0.19547876715660095,\n            -0.1534997969865799,\n            0.4250090718269348,\n            0.015761446207761765,\n            0.1377006471157074,\n            -0.05734637379646301,\n            -0.014811448752880096,\n            -0.21452313661575317,\n            -0.08140499144792557,\n            0.08328989148139954,\n            0.09046290814876556,\n            -0.19140201807022095,\n            0.07533324509859085,\n            0.14998066425323486,\n            0.24585434794425964,\n            0.15308083593845367,\n            -0.1229090616106987,\n            0.18071803450584412,\n            0.25310906767845154,\n            -0.24070844054222107,\n            -0.010410037823021412,\n            0.06309834122657776,\n            -0.1297897845506668,\n            -0.16657744348049164,\n            0.37421631813049316,\n            0.2173442393541336,\n            -0.2538306415081024,\n            -0.8775382041931152,\n            -0.07950269430875778,\n            -0.15970011055469513,\n            0.10895555466413498,\n            0.008314157836139202,\n            0.14638255536556244,\n            0.072003573179245,\n            0.6190196871757507,\n            0.02677847445011139,\n            0.2699536085128784,\n            -0.19142070412635803,\n            -0.2592925727367401,\n            -0.2565824091434479,\n            -0.262987345457077,\n            -0.1489066481590271,\n            -0.0432773195207119,\n            -0.08126294612884521\n          ],\n          [\n            -0.6639822125434875,\n            0.0880877748131752,\n            -0.005751533899456263,\n            -0.19255366921424866,\n            0.3057144582271576,\n            -0.08999431878328323,\n            -0.019433338195085526,\n            -0.1559668332338333,\n            -0.15749216079711914,\n            -0.011301230639219284,\n            -0.11821027845144272,\n            -0.30145376920700073,\n            0.006112752482295036,\n            -0.1560450941324234,\n            -0.5127982497215271,\n            0.40875670313835144,\n            0.13342377543449402,\n            -0.04711974039673805,\n            0.27859559655189514,\n            -0.25916099548339844,\n            -0.10510764271020889,\n            -0.041911106556653976,\n            0.037033870816230774,\n            -0.20566396415233612,\n            0.05327753722667694,\n            -0.1296263337135315,\n            -0.32234781980514526,\n            0.0389278419315815,\n            -0.40813302993774414,\n            -0.02087150141596794,\n            -0.19934184849262238,\n            -0.07899387180805206,\n            -0.25175032019615173,\n            -0.2088051736354828,\n            0.13817232847213745,\n            -0.04268897697329521,\n            -0.42443978786468506,\n            -0.4097912311553955,\n            -0.04090137407183647,\n            -0.15951122343540192,\n            0.04449190944433212,\n            0.21488967537879944,\n            0.12114228308200836,\n            0.09004763513803482,\n            -0.006397236604243517,\n            -0.07948800921440125,\n            0.19370833039283752,\n            0.683956503868103,\n            -0.11428768187761307,\n            -0.10317038744688034\n          ],\n          [\n            -0.5914733409881592,\n            -0.4672893285751343,\n            0.12114158272743225,\n            0.00020536499505396932,\n            -0.2679958939552307,\n            0.26662468910217285,\n            0.1092640832066536,\n            -0.21656334400177002,\n            0.17207543551921844,\n            0.08223824948072433,\n            0.001051013940013945,\n            0.0034190842416137457,\n            0.13344737887382507,\n            -0.5096179842948914,\n            -0.006541278213262558,\n            0.08032097667455673,\n            -0.005550340749323368,\n            -0.04771978780627251,\n            -0.21063093841075897,\n            0.052590880542993546,\n            0.2812643349170685,\n            0.15709182620048523,\n            -0.13493511080741882,\n            -0.48999109864234924,\n            0.15144944190979004,\n            -0.10234226286411285,\n            0.2640151381492615,\n            0.18310357630252838,\n            0.17227308452129364,\n            0.111046701669693,\n            -0.5058703422546387,\n            0.07171519845724106,\n            0.02478756196796894,\n            -0.045091744512319565,\n            -0.2756809890270233,\n            -0.28741583228111267,\n            -0.22533218562602997,\n            -0.33518844842910767,\n            -0.46407124400138855,\n            0.41094958782196045,\n            0.031681012362241745,\n            0.19160377979278564,\n            0.22435687482357025,\n            0.21299022436141968,\n            0.07534463703632355,\n            0.3006431758403778,\n            0.4345487356185913,\n            0.170183464884758,\n            0.19991670548915863,\n            0.0319303423166275\n          ],\n          [\n            -0.21427439153194427,\n            -0.20036062598228455,\n            0.07794656604528427,\n            -0.14284713566303253,\n            0.25140538811683655,\n            0.3854665160179138,\n            0.46422120928764343,\n            0.36879846453666687,\n            -0.362587034702301,\n            -0.17855657637119293,\n            -0.19077561795711517,\n            -0.2539205253124237,\n            -0.1934506893157959,\n            -0.2961663603782654,\n            -0.08020146191120148,\n            -0.1223592534661293,\n            -0.09612061083316803,\n            -0.08487864583730698,\n            0.030782759189605713,\n            0.13439634442329407,\n            -0.016591235995292664,\n            -0.23079997301101685,\n            -0.3025470972061157,\n            0.2626320719718933,\n            -0.09485653042793274,\n            -0.09040284156799316,\n            -0.31883135437965393,\n            -0.09886917471885681,\n            0.06538306176662445,\n            -0.15228867530822754,\n            -0.1643630415201187,\n            -0.31876298785209656,\n            0.13331247866153717,\n            0.09882406890392303,\n            -0.07007796317338943,\n            0.28392279148101807,\n            -0.2196052223443985,\n            -0.3272660970687866,\n            -0.08426976948976517,\n            -0.20013859868049622,\n            -0.3208335340023041,\n            0.014314942993223667,\n            -0.2237250953912735,\n            0.12013121694326401,\n            -0.0037638030480593443,\n            0.3212618827819824,\n            -0.2214321345090866,\n            0.1737927347421646,\n            -0.24851001799106598,\n            -0.2322719395160675\n          ],\n          [\n            -0.11294197291135788,\n            0.22274744510650635,\n            -0.2330183982849121,\n            -0.7324397563934326,\n            0.4676364064216614,\n            -0.07979191094636917,\n            0.7319750785827637,\n            0.12972930073738098,\n            0.021024459972977638,\n            0.36666375398635864,\n            0.23946106433868408,\n            0.02072736620903015,\n            0.2554115653038025,\n            -0.04969428852200508,\n            -0.3307410478591919,\n            0.18114912509918213,\n            0.028270814567804337,\n            0.015179718844592571,\n            -0.12330888956785202,\n            -0.06808768212795258,\n            0.11298701167106628,\n            0.3111850619316101,\n            0.15453025698661804,\n            -0.036064568907022476,\n            -0.15355663001537323,\n            -0.08854296803474426,\n            -0.1556462049484253,\n            0.13265253603458405,\n            -0.1301099956035614,\n            0.1300511360168457,\n            0.05666785687208176,\n            0.05629177764058113,\n            0.02590491622686386,\n            0.4094301164150238,\n            -0.044398609548807144,\n            0.452364057302475,\n            0.3546171188354492,\n            -0.13299521803855896,\n            -0.1892523318529129,\n            0.25428497791290283,\n            0.04711531475186348,\n            0.021106703206896782,\n            0.6154316067695618,\n            -0.22931945323944092,\n            -0.16165129840373993,\n            -0.11139032244682312,\n            0.09658266603946686,\n            -0.1307058334350586,\n            -0.049249663949012756,\n            0.32638129591941833\n          ],\n          [\n            -0.46581366658210754,\n            0.03137717396020889,\n            -0.1218806728720665,\n            0.09832312911748886,\n            -0.009367070160806179,\n            -0.11927109211683273,\n            0.7112864851951599,\n            -0.734951376914978,\n            -0.07221429795026779,\n            -0.011268037371337414,\n            0.04569047689437866,\n            0.04369886964559555,\n            -0.032484911382198334,\n            0.09794636815786362,\n            -0.32328906655311584,\n            0.1170458123087883,\n            0.12185964733362198,\n            0.19422496855258942,\n            0.17055435478687286,\n            -0.003093091771006584,\n            -0.5446919202804565,\n            0.12102081626653671,\n            0.027012798935174942,\n            0.17547154426574707,\n            0.055674947798252106,\n            -0.13497957587242126,\n            0.17319534718990326,\n            -0.07802064716815948,\n            -0.19643627107143402,\n            0.04205043241381645,\n            0.213498055934906,\n            0.07426552474498749,\n            0.02582940086722374,\n            0.451744019985199,\n            -0.022335831075906754,\n            0.3766533136367798,\n            -0.4091492295265198,\n            -0.015331956557929516,\n            -0.05918412283062935,\n            -0.06600836664438248,\n            0.14513570070266724,\n            -0.035239510238170624,\n            0.40115639567375183,\n            -0.20591595768928528,\n            -0.23105399310588837,\n            -0.07920513302087784,\n            -0.16204708814620972,\n            0.0754992663860321,\n            0.07525797933340073,\n            0.09106867760419846\n          ],\n          [\n            0.33886078000068665,\n            -0.15836355090141296,\n            0.11545507609844208,\n            -0.307790070772171,\n            -0.3583011329174042,\n            -0.4114040434360504,\n            -0.17187818884849548,\n            -0.5699397325515747,\n            0.1770302802324295,\n            0.05155625194311142,\n            0.25689783692359924,\n            -0.07145075500011444,\n            -0.39431020617485046,\n            0.13408873975276947,\n            -0.047830283641815186,\n            -0.3556029498577118,\n            -0.1249995082616806,\n            0.33150044083595276,\n            0.05355924740433693,\n            0.0580989271402359,\n            -0.15124715864658356,\n            0.1805446445941925,\n            -0.27744537591934204,\n            -0.05419614911079407,\n            -0.14440882205963135,\n            0.1770683228969574,\n            0.24545757472515106,\n            -0.026452191174030304,\n            -0.008723985403776169,\n            0.038196153938770294,\n            0.03161676228046417,\n            0.1373031884431839,\n            -0.4143252372741699,\n            0.3969142436981201,\n            -0.06792732328176498,\n            -0.1232592761516571,\n            -0.16646811366081238,\n            0.06960832327604294,\n            0.03154929354786873,\n            -0.123887799680233,\n            -0.46353864669799805,\n            -0.3144764304161072,\n            -0.42628389596939087,\n            0.5393332242965698,\n            0.2493637055158615,\n            -0.12768633663654327,\n            -0.0760553702712059,\n            -0.5405345559120178,\n            -0.08851554989814758,\n            0.03100821003317833\n          ],\n          [\n            -0.9735671877861023,\n            -0.04137355089187622,\n            0.35126975178718567,\n            -0.10205425322055817,\n            -0.025996064767241478,\n            -0.0580744668841362,\n            -0.1881583034992218,\n            0.16081131994724274,\n            0.13075855374336243,\n            0.07091133296489716,\n            -0.006204509641975164,\n            0.11568878591060638,\n            -0.08243140578269958,\n            0.07040645182132721,\n            0.09325511008501053,\n            0.028905700892210007,\n            0.15887966752052307,\n            0.10187280178070068,\n            0.15174755454063416,\n            0.35554617643356323,\n            -0.3071753978729248,\n            -0.1548262983560562,\n            0.1684771478176117,\n            0.27995574474334717,\n            0.003711925121024251,\n            -0.06057242304086685,\n            0.02376418001949787,\n            -0.05478191003203392,\n            -0.1576126664876938,\n            -0.2858216464519501,\n            -0.2798856496810913,\n            0.2505143880844116,\n            0.23728787899017334,\n            -0.2534143328666687,\n            -0.2322133481502533,\n            -0.5443998575210571,\n            -0.04497148469090462,\n            0.07133816182613373,\n            0.0013389517553150654,\n            0.08474922180175781,\n            0.08366569876670837,\n            -0.0008644083864055574,\n            -0.08997055888175964,\n            0.12731218338012695,\n            0.24673467874526978,\n            -0.10185850411653519,\n            0.3059315085411072,\n            -0.06534984707832336,\n            -0.04846072942018509,\n            -0.01469405833631754\n          ],\n          [\n            0.4039386510848999,\n            -0.13471591472625732,\n            0.027628807350993156,\n            -0.034723374992609024,\n            0.17345017194747925,\n            0.08883119374513626,\n            -0.609667956829071,\n            -0.6436949968338013,\n            0.182982936501503,\n            0.042495593428611755,\n            -0.024463780224323273,\n            0.05978408828377724,\n            0.03751043230295181,\n            -0.029150333255529404,\n            0.009013879112899303,\n            -0.08349311351776123,\n            -0.22251936793327332,\n            -0.14870254695415497,\n            0.14848963916301727,\n            -0.20075109601020813,\n            0.16211113333702087,\n            -0.7619535326957703,\n            0.18761400878429413,\n            0.2754133343696594,\n            -0.023325514048337936,\n            -0.2032531052827835,\n            0.08467560261487961,\n            -0.2319343388080597,\n            -0.05421333387494087,\n            -0.14198003709316254,\n            -0.12662072479724884,\n            0.1968347579240799,\n            -0.03838423267006874,\n            0.3028755784034729,\n            0.20501698553562164,\n            0.24259786307811737,\n            0.010480734519660473,\n            -0.13123713433742523,\n            0.09066292643547058,\n            0.013406556099653244,\n            -0.14184661209583282,\n            -0.008003261871635914,\n            -0.49010083079338074,\n            -0.42149728536605835,\n            0.05045846849679947,\n            0.16863785684108734,\n            -0.34566766023635864,\n            0.38461145758628845,\n            0.04986937716603279,\n            0.01650077849626541\n          ],\n          [\n            -0.12281874567270279,\n            -0.06224335357546806,\n            0.01965164951980114,\n            0.036817919462919235,\n            0.01696583442389965,\n            0.08764713257551193,\n            0.025815490633249283,\n            -0.17511172592639923,\n            -0.013873397372663021,\n            -0.0496794767677784,\n            -0.01390866655856371,\n            0.2167406976222992,\n            -0.016514839604496956,\n            -0.13962915539741516,\n            -0.12483185529708862,\n            -0.055758003145456314,\n            -0.11738940328359604,\n            0.02215925231575966,\n            0.11241894215345383,\n            0.1515226513147354,\n            -0.053121067583560944,\n            -0.027850231155753136,\n            -0.030644873157143593,\n            -0.014299597591161728,\n            0.09084217250347137,\n            -0.006216452457010746,\n            -0.2358647584915161,\n            -0.15631726384162903,\n            0.015916109085083008,\n            -0.16677819192409515,\n            -0.01726796291768551,\n            -0.10054439306259155,\n            0.026055416092276573,\n            0.3712286651134491,\n            -0.3131727874279022,\n            0.18314386904239655,\n            0.05813062936067581,\n            -0.014103861525654793,\n            -0.04529847577214241,\n            -0.09722699970006943,\n            0.11625651270151138,\n            0.18894419074058533,\n            -0.26961812376976013,\n            -0.05369789898395538,\n            -0.11172289401292801,\n            0.0071302978321909904,\n            -0.09918048977851868,\n            0.24546149373054504,\n            -0.06613007187843323,\n            -0.05349305644631386\n          ],\n          [\n            -0.10706257820129395,\n            -0.014513627626001835,\n            -0.01650029793381691,\n            -0.022345444187521935,\n            -0.2375548928976059,\n            0.14926046133041382,\n            0.05824356526136398,\n            -0.13587728142738342,\n            -0.5190252065658569,\n            -0.23528948426246643,\n            -0.19684772193431854,\n            0.08985518664121628,\n            -0.09019802510738373,\n            -0.16498494148254395,\n            -0.16400064527988434,\n            -0.17665381729602814,\n            -0.1495702862739563,\n            0.04063289985060692,\n            0.018178608268499374,\n            0.0679873526096344,\n            -0.16714423894882202,\n            -0.036891527473926544,\n            -0.10445253551006317,\n            -0.3927890360355377,\n            -0.17704632878303528,\n            -0.1166594922542572,\n            0.012740793637931347,\n            -0.14439982175827026,\n            -0.1996820718050003,\n            0.10451915115118027,\n            -0.03469640761613846,\n            -0.0004551775346044451,\n            -0.07949463278055191,\n            -0.039038173854351044,\n            -0.15445175766944885,\n            -0.1380963772535324,\n            -0.1047859638929367,\n            0.033395104110240936,\n            -0.1391402930021286,\n            -0.11430581659078598,\n            -0.16138480603694916,\n            -0.013521285727620125,\n            0.16289760172367096,\n            0.06026143580675125,\n            -0.08880285918712616,\n            0.27378812432289124,\n            -0.7299013137817383,\n            0.12981179356575012,\n            -0.21602566540241241,\n            -0.03437846899032593\n          ],\n          [\n            -0.07186418026685715,\n            -0.055654894560575485,\n            0.023081880062818527,\n            -0.014171147719025612,\n            0.3718494772911072,\n            0.15714910626411438,\n            0.5212790966033936,\n            -0.057541623711586,\n            -0.1241057738661766,\n            0.03125312924385071,\n            -0.07598546147346497,\n            -0.046080268919467926,\n            0.2779327929019928,\n            -0.012316126376390457,\n            -0.2595909833908081,\n            -0.1340722143650055,\n            0.05720413103699684,\n            -0.5329489707946777,\n            0.11871278285980225,\n            0.2923130691051483,\n            -0.11652947962284088,\n            0.443386971950531,\n            0.11399097740650177,\n            -0.39724382758140564,\n            0.0910022184252739,\n            0.07382900267839432,\n            -0.02331758663058281,\n            0.2986155152320862,\n            -0.17429904639720917,\n            -0.02758517861366272,\n            -0.005745948292315006,\n            -0.06663986295461655,\n            -0.03176482766866684,\n            0.5119363069534302,\n            0.04511744901537895,\n            0.4174656867980957,\n            -0.2300683706998825,\n            -0.01011301577091217,\n            0.21833020448684692,\n            -0.052029334008693695,\n            0.1804760843515396,\n            0.4034663438796997,\n            0.2590266168117523,\n            -0.2084108591079712,\n            0.06783833354711533,\n            -0.13150788843631744,\n            0.14485538005828857,\n            0.017271699383854866,\n            -0.05304720252752304,\n            -0.057500772178173065\n          ],\n          [\n            -0.6604385375976562,\n            -0.06986501812934875,\n            0.15083815157413483,\n            0.2219560742378235,\n            0.3681388199329376,\n            -0.24392996728420258,\n            -0.2625875174999237,\n            0.9658703804016113,\n            -0.06555946171283722,\n            -0.04877181351184845,\n            -0.09979092329740524,\n            0.19463306665420532,\n            -0.27455735206604004,\n            0.13402371108531952,\n            -0.37881940603256226,\n            0.10094164311885834,\n            0.13876503705978394,\n            -0.447234183549881,\n            -0.1345466673374176,\n            0.12663979828357697,\n            -0.24984702467918396,\n            -0.035014744848012924,\n            -0.039633940905332565,\n            -0.210527703166008,\n            0.20277711749076843,\n            0.2858479917049408,\n            0.08572372794151306,\n            0.036232221871614456,\n            -0.36387720704078674,\n            0.06315044313669205,\n            -0.30623769760131836,\n            -0.06741265207529068,\n            -0.30438026785850525,\n            0.09858912974596024,\n            0.21076439321041107,\n            0.40742477774620056,\n            -0.43515026569366455,\n            -0.157973974943161,\n            0.3636869788169861,\n            -0.24309775233268738,\n            -0.09181814640760422,\n            0.2706977427005768,\n            0.36064156889915466,\n            -0.036217812448740005,\n            -0.07993815094232559,\n            0.07931309938430786,\n            0.45539867877960205,\n            0.07486847043037415,\n            -0.024540740996599197,\n            -0.03841240331530571\n          ],\n          [\n            -0.307781845331192,\n            -0.034045323729515076,\n            -0.0005269395187497139,\n            -0.19310402870178223,\n            -0.026454007253050804,\n            0.18268531560897827,\n            -0.1673271805047989,\n            -0.09658391773700714,\n            -0.11760496348142624,\n            0.10150157660245895,\n            0.11351918429136276,\n            -0.13518212735652924,\n            -0.09026109427213669,\n            -0.018844613805413246,\n            -0.14594362676143646,\n            -0.09376624971628189,\n            0.06628540903329849,\n            0.16020053625106812,\n            -0.08841736614704132,\n            0.3251096308231354,\n            -0.06996630132198334,\n            -0.07898686081171036,\n            -0.010989238508045673,\n            0.1558353155851364,\n            -0.056911613792181015,\n            -0.23423999547958374,\n            0.18207210302352905,\n            -0.22104330360889435,\n            -0.017040472477674484,\n            -0.3210408091545105,\n            -0.15926975011825562,\n            -0.19055554270744324,\n            0.19050413370132446,\n            -0.10608169436454773,\n            -0.02338770218193531,\n            -0.12002378702163696,\n            -0.17946694791316986,\n            -0.14818662405014038,\n            -0.16118058562278748,\n            -0.08301952481269836,\n            0.09531703591346741,\n            0.13841426372528076,\n            -0.060536179691553116,\n            -0.032108038663864136,\n            0.12043174356222153,\n            0.047239068895578384,\n            -0.01471246499568224,\n            -0.08050220459699631,\n            0.07315975427627563,\n            -0.046004388481378555\n          ],\n          [\n            -0.28982484340667725,\n            -0.08965249359607697,\n            0.10088396072387695,\n            0.044090624898672104,\n            0.376592218875885,\n            -0.08558747172355652,\n            0.3574191927909851,\n            -0.7516403198242188,\n            0.22832489013671875,\n            0.05686384066939354,\n            -0.07454803586006165,\n            -0.24541698396205902,\n            -0.14552736282348633,\n            0.14130418002605438,\n            0.048876725137233734,\n            -0.12436597049236298,\n            0.15423622727394104,\n            0.20786859095096588,\n            0.2391236424446106,\n            0.2524341642856598,\n            -0.6358815431594849,\n            0.03661400079727173,\n            -0.08577727526426315,\n            -0.4446631073951721,\n            0.15542654693126678,\n            -0.15350627899169922,\n            -0.0007989830919541419,\n            0.07778619229793549,\n            0.030498310923576355,\n            -0.1517181098461151,\n            -0.061767566949129105,\n            0.07415859401226044,\n            0.10067164897918701,\n            0.6278400421142578,\n            -0.2585182189941406,\n            0.05506706237792969,\n            0.08913308382034302,\n            -0.08907200396060944,\n            0.11066343635320663,\n            -0.4256574809551239,\n            -0.09891491383314133,\n            0.10912012308835983,\n            -0.26256752014160156,\n            -0.2248126119375229,\n            0.03860471025109291,\n            0.09886275976896286,\n            0.17226631939411163,\n            -0.025798451155424118,\n            -0.09317366033792496,\n            -0.05684979632496834\n          ],\n          [\n            -0.37888529896736145,\n            -0.16662314534187317,\n            0.07877650856971741,\n            0.33243048191070557,\n            0.33421242237091064,\n            0.04229206219315529,\n            0.06698255240917206,\n            0.5107118487358093,\n            -0.19182775914669037,\n            -0.17274397611618042,\n            0.11606128513813019,\n            -0.03831028565764427,\n            0.06799456477165222,\n            -0.23126263916492462,\n            0.17431028187274933,\n            0.1489490568637848,\n            0.162456214427948,\n            -0.2882015109062195,\n            -0.2679458260536194,\n            0.0428243950009346,\n            -0.39375025033950806,\n            -0.2358098179101944,\n            0.29235535860061646,\n            0.15713715553283691,\n            0.1423434019088745,\n            0.25796201825141907,\n            0.2916123569011688,\n            0.13723468780517578,\n            0.15418045222759247,\n            -0.2943059206008911,\n            -0.4850558638572693,\n            0.3461304306983948,\n            0.1502530425786972,\n            -0.43227624893188477,\n            0.12890367209911346,\n            0.26796016097068787,\n            -0.11738845705986023,\n            0.1978311687707901,\n            -0.19749420881271362,\n            0.06924550980329514,\n            0.07652833312749863,\n            0.4045135974884033,\n            0.571014404296875,\n            0.07547017186880112,\n            -0.2980058193206787,\n            0.02181682363152504,\n            -0.3753744959831238,\n            0.25161999464035034,\n            -0.08938631415367126,\n            -0.03211525082588196\n          ],\n          [\n            -0.49257224798202515,\n            -0.09388156235218048,\n            0.0012350549222901464,\n            0.18059146404266357,\n            0.0011243124026805162,\n            -0.20821326971054077,\n            0.20082233846187592,\n            0.4148622155189514,\n            -0.06775477528572083,\n            -0.058037225157022476,\n            -0.10307089239358902,\n            -0.10048248618841171,\n            -0.05047061666846275,\n            0.12638187408447266,\n            -0.4556204080581665,\n            -0.46558642387390137,\n            -0.0772886872291565,\n            -0.6759362816810608,\n            -0.18581464886665344,\n            0.11267050355672836,\n            -0.15583279728889465,\n            -0.16388441622257233,\n            0.23114919662475586,\n            -0.3857767581939697,\n            -0.06667602807283401,\n            0.01988230086863041,\n            0.21008752286434174,\n            0.27807095646858215,\n            -0.41078582406044006,\n            -0.022135812789201736,\n            -0.3471928834915161,\n            0.20695865154266357,\n            -0.08093701303005219,\n            -0.22231993079185486,\n            -0.08899680525064468,\n            0.06807203590869904,\n            -0.25230348110198975,\n            -0.0014619325520470738,\n            0.08896080404520035,\n            -0.3568987548351288,\n            0.11678487062454224,\n            1.2827746868133545,\n            -0.014346606098115444,\n            0.08429359644651413,\n            -0.0919218435883522,\n            0.07523822784423828,\n            -0.04129438474774361,\n            0.11294715851545334,\n            0.24544107913970947,\n            -0.042737267911434174\n          ],\n          [\n            -0.48072704672813416,\n            -0.18810005486011505,\n            0.0374588705599308,\n            -0.05727287009358406,\n            -0.012722119688987732,\n            0.14065587520599365,\n            -0.003408705350011587,\n            0.1122223362326622,\n            0.340884804725647,\n            -0.15227937698364258,\n            -0.005189506337046623,\n            -0.04472450166940689,\n            -0.09916462749242783,\n            -0.040019359439611435,\n            0.16836167871952057,\n            0.1395205408334732,\n            -0.1028418242931366,\n            -0.30095595121383667,\n            0.48140761256217957,\n            0.21626786887645721,\n            0.09874799847602844,\n            0.05301765352487564,\n            -0.005831585265696049,\n            -0.436678022146225,\n            0.0865410789847374,\n            0.0398264117538929,\n            -0.16981326043605804,\n            0.038346730172634125,\n            -0.23007746040821075,\n            0.2817489802837372,\n            -0.09297100454568863,\n            0.11058783531188965,\n            -0.00528382183983922,\n            0.4117512106895447,\n            -0.06859137117862701,\n            0.03556973114609718,\n            -0.05617225542664528,\n            -0.08823215961456299,\n            0.14528052508831024,\n            0.020949963480234146,\n            0.12183349579572678,\n            -0.1382322609424591,\n            -0.13707837462425232,\n            0.0687415823340416,\n            -0.14030469954013824,\n            0.15818864107131958,\n            0.19207841157913208,\n            -0.033355679363012314,\n            0.000742139876820147,\n            0.12967880070209503\n          ],\n          [\n            0.3177640438079834,\n            0.03588578850030899,\n            -0.0939568430185318,\n            0.12949220836162567,\n            0.33398666977882385,\n            0.08892231434583664,\n            -0.31378835439682007,\n            -0.21445046365261078,\n            0.10949963331222534,\n            -0.20236003398895264,\n            -0.15028421580791473,\n            0.10194536298513412,\n            0.0360504575073719,\n            0.13285522162914276,\n            -0.0156784076243639,\n            -0.11452782899141312,\n            -0.10350515693426132,\n            -0.6414148807525635,\n            -0.047703564167022705,\n            0.1961110234260559,\n            -0.44564664363861084,\n            0.03627097234129906,\n            -0.18321439623832703,\n            -0.17838998138904572,\n            -0.13331592082977295,\n            0.1563485711812973,\n            -0.35932809114456177,\n            0.2402045577764511,\n            0.11359019577503204,\n            -0.3508605360984802,\n            0.16375738382339478,\n            0.08338475227355957,\n            -0.006938163191080093,\n            0.1444127857685089,\n            -0.05632803216576576,\n            0.03932121396064758,\n            0.10900809615850449,\n            0.07841342687606812,\n            0.005295031704008579,\n            -0.15297219157218933,\n            0.081606924533844,\n            0.8683913946151733,\n            -0.3636198043823242,\n            0.00883118249475956,\n            0.2409229576587677,\n            -0.35125502943992615,\n            -0.005632631480693817,\n            0.14590124785900116,\n            0.13412189483642578,\n            -0.20834799110889435\n          ],\n          [\n            0.0017151302890852094,\n            -0.305098295211792,\n            0.12967051565647125,\n            0.022289346903562546,\n            -0.8472644090652466,\n            0.03573717549443245,\n            0.3002815842628479,\n            -0.27122384309768677,\n            -0.4793834686279297,\n            -0.1459033042192459,\n            -0.20546512305736542,\n            -0.06842319667339325,\n            -0.08808693289756775,\n            0.07788810133934021,\n            -0.0466180220246315,\n            -0.10641411691904068,\n            0.10383602976799011,\n            -0.07716105133295059,\n            -0.06759563088417053,\n            0.3445044755935669,\n            0.3319224417209625,\n            0.1663070172071457,\n            0.17388753592967987,\n            -0.10807190835475922,\n            -0.15468557178974152,\n            -0.029798904433846474,\n            0.16725780069828033,\n            -0.048754703253507614,\n            -0.1269257366657257,\n            0.2151537537574768,\n            0.035467348992824554,\n            0.03292301297187805,\n            0.052482858300209045,\n            0.11522477865219116,\n            -0.4186718165874481,\n            0.07275981456041336,\n            0.07813485711812973,\n            -0.15219397842884064,\n            -0.1822328418493271,\n            -0.040962252765893936,\n            0.016940148547291756,\n            -0.07688132673501968,\n            0.023374469950795174,\n            0.32189542055130005,\n            0.1331385225057602,\n            -0.15390019118785858,\n            0.4464007318019867,\n            -0.3328644931316376,\n            -0.19362740218639374,\n            -0.08363735675811768\n          ]\n        ],\n        "hidden_bias": [\n          -0.0910271480679512,\n          0.07835296541452408,\n          -0.2877812087535858,\n          0.536487877368927,\n          -0.17930996417999268,\n          -0.4639956057071686,\n          -0.3537112772464752,\n          -0.20710493624210358,\n          -0.012712390162050724,\n          -0.05994794890284538,\n          -0.35677650570869446,\n          0.2527293562889099,\n          -0.01686500385403633,\n          0.23655222356319427,\n          -0.47310569882392883,\n          0.045338720083236694,\n          -0.6184362173080444,\n          -0.04151638597249985,\n          -0.4233519434928894,\n          -0.0897008553147316,\n          -0.6524513363838196,\n          0.09297261387109756,\n          -0.29338133335113525,\n          0.05763170123100281,\n          -0.07583652436733246,\n          -0.04625171050429344,\n          0.09338902682065964,\n          -0.1357330083847046,\n          -0.6487828493118286,\n          -0.41045859456062317,\n          0.29976391792297363,\n          0.009535040706396103,\n          -0.41884297132492065,\n          -0.04188937693834305,\n          -0.1835736483335495,\n          -0.5297504663467407,\n          -0.12637421488761902,\n          0.023102406412363052,\n          0.19732075929641724,\n          0.12233787775039673,\n          -0.039070453494787216,\n          -0.22084969282150269,\n          -0.23959358036518097,\n          -0.033917754888534546,\n          -0.23289041221141815,\n          0.26288798451423645,\n          -0.11982260644435883,\n          0.30594393610954285,\n          -0.6035798788070679,\n          0.015034908428788185,\n          0.07078143209218979,\n          0.08575760573148727,\n          0.04642843082547188,\n          0.11488718539476395,\n          0.13566158711910248,\n          -0.16577203571796417,\n          0.02040993794798851,\n          -0.0511690191924572,\n          -0.17587392032146454,\n          -0.5083130598068237,\n          -0.38909322023391724,\n          -0.11265306919813156,\n          -0.039799291640520096,\n          -0.036090556532144547\n        ],\n        "output_weight": [\n          0.2131989598274231,\n          0.1977451592683792,\n          0.10760799050331116,\n          -0.1919630616903305,\n          0.1623319536447525,\n          -0.15136662125587463,\n          0.18045802414417267,\n          -0.30183181166648865,\n          -0.008809132501482964,\n          0.15819451212882996,\n          0.2064265012741089,\n          0.14527276158332825,\n          -0.12433601915836334,\n          0.08363532274961472,\n          -0.25438863039016724,\n          0.22195687890052795,\n          0.26992323994636536,\n          0.1499648541212082,\n          0.1286066770553589,\n          0.3266310393810272,\n          0.347166508436203,\n          -0.17328840494155884,\n          0.13646559417247772,\n          0.17151620984077454,\n          0.12186865508556366,\n          -0.2395879328250885,\n          0.1403948813676834,\n          -0.20059140026569366,\n          -0.2968544065952301,\n          -0.4102231562137604,\n          -0.12591375410556793,\n          0.2557772397994995,\n          -0.3231998383998871,\n          -0.1460215002298355,\n          0.0783180370926857,\n          0.4334082007408142,\n          0.09213857352733612,\n          -0.22668281197547913,\n          0.1472545862197876,\n          -0.2056395411491394,\n          0.10779472440481186,\n          0.17486685514450073,\n          -0.23726321756839752,\n          -0.13073530793190002,\n          -0.18027904629707336,\n          -0.12480378895998001,\n          0.22221282124519348,\n          0.11064167320728302,\n          -0.4147959351539612,\n          -0.24766920506954193,\n          -0.2846771478652954,\n          -0.1911633461713791,\n          0.22691060602664948,\n          0.0041246572509408,\n          0.0713290274143219,\n          0.11663182824850082,\n          -0.24798740446567535,\n          0.024134783074259758,\n          -0.22529403865337372,\n          0.3082430362701416,\n          0.11048208177089691,\n          0.07865811884403229,\n          0.06063178554177284,\n          0.1315339207649231\n        ],\n        "output_bias": 0.1393638551235199,\n        "training": {\n          "method": "paired counterfactual residual policy distillation",\n          "seed": 20261020,\n          "loss_type": "ranking",\n          "regularization": 0.01,\n          "best_epoch": 97,\n          "validation_preference_loss": 0.46406126022338867,\n          "baseline_preference_loss": 0.7314271330833435,\n          "training_games": 2400,\n          "validation_games": 600,\n          "source_dataset": {\n            "method": "paired one-step policy improvement, phase-stratified whole-game reservoirs",\n            "input_features": [\n              "batch_size",\n              "truthful",\n              "bluff",\n              "bluff_pile",\n              "bluff_call_rate",\n              "bluff_finish",\n              "remaining_groups",\n              "remaining_largest_group",\n              "claimed_rank_held",\n              "bias",\n              "revealed_bluff_rate",\n              "rank_held",\n              "claim_count",\n              "pile_size",\n              "opponent_finishing",\n              "own_small_hand",\n              "opponent_small_hand",\n              "samples",\n              "bluff_any_call_probability",\n              "bluff_impossible_for_caller",\n              "truthful_turns_removed",\n              "bluff_known_pile_fraction",\n              "truthful_finish",\n              "bluff_no_impossibility_call_possible",\n              "placer_global_revealed_bluff_rate",\n              "known_outside_rank_fraction",\n              "known_pile_fraction",\n              "claim_fraction_of_original_hand",\n              "finishing_large_batch",\n              "known_placer_rank_before_claim",\n              "bluff_call_probability_batch",\n              "bluff_call_probability_pile",\n              "bluff_call_probability_finish",\n              "bluff_nearest_opponent_finish",\n              "truthful_call_probability_batch",\n              "bluff_remaining_truthful_turns",\n              "global_revealed_bluff_finishing",\n              "count_revealed_bluff_pile",\n              "global_revealed_bluff_pile",\n              "count_revealed_bluff_small_own_hand",\n              "global_revealed_bluff_count",\n              "opponent_hand_pace_relative_to_own",\n              "bluff_pooled_caller_probability",\n              "bluff_pooled_caller_uncertainty",\n              "bluff_specific_caller_uncertainty",\n              "bluff_sparse_own_outcomes",\n              "truthful_pooled_caller_probability_batch",\n              "bluff_pooled_caller_probability_pile",\n              "global_revealed_bluff_uncertainty",\n              "count_revealed_bluff_uncertainty"\n            ],\n            "feature_policy": "v5",\n            "games": 3000,\n            "states": 12000,\n            "actions": 67757,\n            "trials": 4,\n            "samples_per_phase": 2,\n            "max_actions": 10,\n            "seed_start": 50000000,\n            "incumbent": {\n              "format": "triad-bluff-linear-v2",\n              "features": [\n                "batch_size",\n                "truthful",\n                "bluff",\n                "bluff_pile",\n                "bluff_call_rate",\n                "bluff_finish",\n                "remaining_groups",\n                "remaining_largest_group",\n                "claimed_rank_held",\n                "bias",\n                "revealed_bluff_rate",\n                "rank_held",\n                "claim_count",\n                "pile_size",\n                "opponent_finishing",\n                "own_small_hand",\n                "opponent_small_hand",\n                "samples",\n                "bluff_any_call_probability",\n                "bluff_impossible_for_caller",\n                "truthful_turns_removed",\n                "bluff_known_pile_fraction",\n                "truthful_finish",\n                "bluff_no_impossibility_call_possible",\n                "placer_global_revealed_bluff_rate",\n                "known_outside_rank_fraction",\n                "known_pile_fraction",\n                "claim_fraction_of_original_hand",\n                "finishing_large_batch",\n                "known_placer_rank_before_claim"\n              ],\n              "weights": [\n                0.4755931924211193,\n                5.820676470779726,\n                -3.445920468550993,\n                0.5025301512894197,\n                -10.204982725387449,\n                -10.63672367105182,\n                1.9122839363624424,\n                6.692718730179745,\n                3.3398384247884447,\n                -3.967913294968817,\n                7.18093879068249,\n                -8.248647532051631,\n                -4.252488827029067,\n                -5.717390427387336,\n                -10.297751467694116,\n                -4.172529040138095,\n                3.643993344291045,\n                4.785033532440487,\n                -8.90940656773177,\n                -9.18268199666696,\n                3.5590525036076692,\n                -1.533025684746697,\n                5.957038223067108,\n                9.714173152584005,\n                3.5582377937169647,\n                -0.47803421898358756,\n                -1.2639464162072949,\n                0.6168380283723138,\n                2.861924827690982,\n                -7.115720155350792\n              ],\n              "training": {\n                "method": "archive-league cross-entropy policy optimization",\n                "seed": 20261016,\n                "generations": 30,\n                "population": 32,\n                "train_blocks": 8,\n                "train_horizon": 200,\n                "evaluation_horizon": 1000,\n                "validation_blocks": 64,\n                "selected_validation_points": 4.0859375,\n                "incumbent_validation_points": 3.62109375,\n                "sdk_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6"\n              }\n            },\n            "opponent_pool": [\n              {\n                "kind": "v2",\n                "weights": [\n                  0.4755931924211193,\n                  5.820676470779726,\n                  -3.445920468550993,\n                  0.5025301512894197,\n                  -10.204982725387449,\n                  -10.63672367105182,\n                  1.9122839363624424,\n                  6.692718730179745,\n                  3.3398384247884447,\n                  -3.967913294968817,\n                  7.18093879068249,\n                  -8.248647532051631,\n                  -4.252488827029067,\n                  -5.717390427387336,\n                  -10.297751467694116,\n                  -4.172529040138095,\n                  3.643993344291045,\n                  4.785033532440487,\n                  -8.90940656773177,\n                  -9.18268199666696,\n                  3.5590525036076692,\n                  -1.533025684746697,\n                  5.957038223067108,\n                  9.714173152584005,\n                  3.5582377937169647,\n                  -0.47803421898358756,\n                  -1.2639464162072949,\n                  0.6168380283723138,\n                  2.861924827690982,\n                  -7.115720155350792\n                ]\n              },\n              {\n                "kind": "v1",\n                "weights": [\n                  4.351868833686492,\n                  5.6322712663248415,\n                  -2.6260873498626167,\n                  0.7063932700439369,\n                  -10.84185815552825,\n                  -8.300750779890834,\n                  -0.7486711333834275,\n                  1.2260583122844886,\n                  -1.5575700886723332,\n                  -4.558178606791033,\n                  8.25750430388175,\n                  0.43873901259948533,\n                  -1.3556124209301408,\n                  -1.1333529834957075,\n                  0.14766041336362645,\n                  -1.5734330345566885,\n                  -0.4730921181099686,\n                  -0.5811911507935986\n                ]\n              },\n              {\n                "kind": "starter"\n              },\n              {\n                "kind": "competitive"\n              },\n              {\n                "kind": "random"\n              },\n              {\n                "kind": "aggressive"\n              },\n              {\n                "kind": "heuristic"\n              },\n              {\n                "kind": "bluffer",\n                "bluff_rate": 0.1,\n                "call_rate": 0.0\n              },\n              {\n                "kind": "bluffer",\n                "bluff_rate": 0.1,\n                "call_rate": 0.35\n              },\n              {\n                "kind": "bluffer",\n                "bluff_rate": 0.35,\n                "call_rate": 0.1\n              },\n              {\n                "kind": "bluffer",\n                "bluff_rate": 0.65,\n                "call_rate": 0.35\n              },\n              {\n                "kind": "bluffer",\n                "bluff_rate": 0.95,\n                "call_rate": 0.0\n              },\n              {\n                "kind": "bluffer",\n                "bluff_rate": 0.95,\n                "call_rate": 0.35\n              }\n            ],\n            "sdk_game_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6",\n            "hidden_state_use": "training returns only; dealt hands fixed within state, common future actor seeds across actions",\n            "future_draw_pile_shuffle": false,\n            "own_hand_size_histogram": {\n              "2": 640,\n              "3": 1392,\n              "4": 1021,\n              "5": 1355,\n              "6": 783,\n              "7": 1373,\n              "8": 606,\n              "9": 1081,\n              "10": 720,\n              "11": 1005,\n              "12": 98,\n              "13": 1443,\n              "14": 51,\n              "15": 65,\n              "16": 65,\n              "17": 41,\n              "18": 46,\n              "19": 31,\n              "20": 35,\n              "21": 27,\n              "22": 14,\n              "23": 21,\n              "24": 19,\n              "25": 14,\n              "26": 9,\n              "27": 8,\n              "28": 5,\n              "29": 4,\n              "30": 4,\n              "31": 5,\n              "32": 3,\n              "34": 3,\n              "35": 4,\n              "37": 1,\n              "38": 2,\n              "39": 1,\n              "40": 1,\n              "41": 1,\n              "43": 2,\n              "46": 1\n            },\n            "phase_histogram": {\n              "PLACE": 6000,\n              "CHALLENGE": 6000\n            },\n            "seconds": 2063.4\n          },\n          "base_score_factor": 0.25,\n          "hidden_state_use": "training labels only; policy input is own/public features",\n          "residual_scale": 0.5\n        }\n      },\n      "opponent_pool": [\n        {\n          "kind": "mlp",\n          "model": {\n            "format": "triad-bluff-residual-mlp-v1",\n            "features": [\n              "batch_size",\n              "truthful",\n              "bluff",\n              "bluff_pile",\n              "bluff_call_rate",\n              "bluff_finish",\n              "remaining_groups",\n              "remaining_largest_group",\n              "claimed_rank_held",\n              "bias",\n              "revealed_bluff_rate",\n              "rank_held",\n              "claim_count",\n              "pile_size",\n              "opponent_finishing",\n              "own_small_hand",\n              "opponent_small_hand",\n              "samples",\n              "bluff_any_call_probability",\n              "bluff_impossible_for_caller",\n              "truthful_turns_removed",\n              "bluff_known_pile_fraction",\n              "truthful_finish",\n              "bluff_no_impossibility_call_possible",\n              "placer_global_revealed_bluff_rate",\n              "known_outside_rank_fraction",\n              "known_pile_fraction",\n              "claim_fraction_of_original_hand",\n              "finishing_large_batch",\n              "known_placer_rank_before_claim",\n              "bluff_call_probability_batch",\n              "bluff_call_probability_pile",\n              "bluff_call_probability_finish",\n              "bluff_nearest_opponent_finish",\n              "truthful_call_probability_batch",\n              "bluff_remaining_truthful_turns",\n              "global_revealed_bluff_finishing",\n              "count_revealed_bluff_pile",\n              "global_revealed_bluff_pile",\n              "count_revealed_bluff_small_own_hand",\n              "global_revealed_bluff_count",\n              "opponent_hand_pace_relative_to_own",\n              "bluff_pooled_caller_probability",\n              "bluff_pooled_caller_uncertainty",\n              "bluff_specific_caller_uncertainty",\n              "bluff_sparse_own_outcomes",\n              "truthful_pooled_caller_probability_batch",\n              "bluff_pooled_caller_probability_pile",\n              "global_revealed_bluff_uncertainty",\n              "count_revealed_bluff_uncertainty"\n            ],\n            "base_weights": [\n              0.11889829486608505,\n              1.4551690816879272,\n              -0.8614801168441772,\n              0.12563253939151764,\n              -2.55124568939209,\n              -2.6591808795928955,\n              0.4780709743499756,\n              1.6731796264648438,\n              0.8349596261978149,\n              -0.9919783473014832,\n              1.7952346801757812,\n              -2.062161922454834,\n              -1.0631221532821655,\n              -1.4293476343154907,\n              -2.5744378566741943,\n              -1.0431323051452637,\n              0.9109983444213867,\n              1.1962584257125854,\n              -2.227351665496826,\n              -2.295670509338379,\n              0.8897631168365479,\n              -0.3832564353942871,\n              1.4892596006393433,\n              2.4285433292388916,\n              0.8895594477653503,\n              -0.11950855702161789,\n              -0.31598660349845886,\n              0.15420950949192047,\n              0.7154812216758728,\n              -1.7789300680160522,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0,\n              0.0\n            ],\n            "feature_mean": [\n              2.130405902862549,\n              0.2556110918521881,\n              0.5673422813415527,\n              0.1414773166179657,\n              0.20922617614269257,\n              0.044980913400650024,\n              0.22203241288661957,\n              0.4362170994281769,\n              0.304308146238327,\n              0.08852332085371017,\n              0.012027798220515251,\n              0.005555760581046343,\n              0.05282813683152199,\n              0.03254607692360878,\n              0.002083986531943083,\n              0.015454696491360664,\n              0.015533286146819592,\n              0.002624368527904153,\n              0.28184229135513306,\n              0.06834737956523895,\n              1.5457278490066528,\n              0.023674404248595238,\n              0.010973202995955944,\n              0.09815023094415665,\n              0.015177818946540356,\n              0.007621304597705603,\n              0.008625890128314495,\n              0.020141232758760452,\n              0.0009267285349778831,\n              0.006892831530421972,\n              0.21310904622077942,\n              0.05682198330760002,\n              0.020169535651803017,\n              0.12666580080986023,\n              0.05104479193687439,\n              0.13413675129413605,\n              0.00024044152814894915,\n              0.0043762726709246635,\n              0.005239620339125395,\n              0.0022578288335353136,\n              0.00981908105313778,\n              -0.10914473235607147,\n              0.2997722029685974,\n              0.0871744155883789,\n              0.12125638127326965,\n              0.23309296369552612,\n              0.05302620306611061,\n              0.044763777405023575,\n              0.011351659893989563,\n              0.011535020545125008\n            ],\n            "feature_scale": [\n              1.3542357683181763,\n              0.43625375628471375,\n              0.4955291450023651,\n              0.32548826932907104,\n              0.23007379472255707,\n              0.20728932321071625,\n              0.20233742892742157,\n              0.3010976016521454,\n              0.2976628541946411,\n              0.28402191400527954,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.18657560646533966,\n              0.15651710331439972,\n              0.10000000149011612,\n              0.12336929887533188,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.2647395730018616,\n              0.2523006200790405,\n              1.334463357925415,\n              0.10000000149011612,\n              0.1041841208934784,\n              0.29747799038887024,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.21329738199710846,\n              0.12353038787841797,\n              0.10000000149011612,\n              0.18938058614730835,\n              0.10285001248121262,\n              0.1730906069278717,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.9756709337234497,\n              0.3055936396121979,\n              0.10000000149011612,\n              0.10987535119056702,\n              0.4228348731994629,\n              0.11198851466178894,\n              0.10000000149011612,\n              0.10000000149011612,\n              0.10000000149011612\n            ],\n            "hidden_weight": [\n              [\n                -0.19053113460540771,\n                0.15571263432502747,\n                -0.1011505052447319,\n                0.19476446509361267,\n                0.27500608563423157,\n                0.15307991206645966,\n                0.3815036416053772,\n                0.4381793439388275,\n                -0.22956830263137817,\n                0.027035346254706383,\n                -0.021941421553492546,\n                0.004426548257470131,\n                0.04070810601115227,\n                -0.05979225039482117,\n                0.0010170531459152699,\n                -0.08468963205814362,\n                0.03122447244822979,\n                0.04913467541337013,\n                0.017885252833366394,\n                0.07352987676858902,\n                0.07204055041074753,\n                -0.6565521359443665,\n                0.23869308829307556,\n                -0.137938991189003,\n                0.22352753579616547,\n                0.07389921694993973,\n                -0.021464945748448372,\n                -0.07632283121347427,\n                -0.0017953533679246902,\n                -0.15402238070964813,\n                -0.2351987063884735,\n                0.2819085419178009,\n                -0.04682714119553566,\n                0.3633670508861542,\n                0.1471453309059143,\n                -0.030505336821079254,\n                0.11275077611207962,\n                -0.04455846548080444,\n                -0.19067446887493134,\n                -0.02123730443418026,\n                0.15277132391929626,\n                -0.0441616028547287,\n                0.19499239325523376,\n                0.005214580800384283,\n                -0.16005682945251465,\n                0.41382917761802673,\n                -0.3361380696296692,\n                0.03683564066886902,\n                0.17921903729438782,\n                0.09538088738918304\n              ],\n              [\n                -0.34688928723335266,\n                -0.0341823436319828,\n                0.08536931127309799,\n                -0.15115806460380554,\n                0.2503475546836853,\n                -0.015907486900687218,\n                0.6435881853103638,\n                0.10033245384693146,\n                0.2746313512325287,\n                -0.13110876083374023,\n                -0.2760956585407257,\n                0.004355510231107473,\n                0.11898820102214813,\n                -0.0032388579566031694,\n                0.1219732016324997,\n                -0.09869767725467682,\n                -0.027204176411032677,\n                -0.41678473353385925,\n                0.26750725507736206,\n                -0.011986590921878815,\n                0.05444285273551941,\n                0.28975367546081543,\n                0.05204622447490692,\n                -0.5399481058120728,\n                -0.10123363882303238,\n                0.03145816922187805,\n                0.08416414260864258,\n                0.2687167227268219,\n                0.013803120702505112,\n                0.27541759610176086,\n                -0.3447941243648529,\n                0.18976999819278717,\n                0.16411860287189484,\n                -0.0076088146306574345,\n                -0.14338168501853943,\n                0.16006532311439514,\n                -0.009708402678370476,\n                0.07059653103351593,\n                -0.1445803940296173,\n                -0.09821895509958267,\n                -0.19294242560863495,\n                0.016644466668367386,\n                0.3165804147720337,\n                -0.35061442852020264,\n                0.046103350818157196,\n                -0.020881682634353638,\n                -0.44616249203681946,\n                -0.184276282787323,\n                -0.04051925987005234,\n                -0.076309435069561\n              ],\n              [\n                -0.43836283683776855,\n                -0.03827773407101631,\n                0.02417719177901745,\n                0.0012864257441833615,\n                -0.18788781762123108,\n                0.05437812581658363,\n                -0.048713840544223785,\n                0.1781936138868332,\n                0.36789923906326294,\n                0.15997500717639923,\n                0.08273530751466751,\n                0.11687079071998596,\n                -0.03200904652476311,\n                -0.10258307307958603,\n                -0.044549256563186646,\n                -0.07501626014709473,\n                0.10152506083250046,\n                0.42922788858413696,\n                0.24868355691432953,\n                0.25955644249916077,\n                0.2103431671857834,\n                -0.06754159182310104,\n                0.10614077001810074,\n                -0.3674386739730835,\n                0.03684944659471512,\n                -0.011080002412199974,\n                -0.17129723727703094,\n                -0.07273056358098984,\n                0.012710507027804852,\n                0.0012870589271187782,\n                0.040439359843730927,\n                0.007053954526782036,\n                -0.1431591808795929,\n                0.7334985733032227,\n                -0.020398594439029694,\n                0.07002000510692596,\n                -0.1414639800786972,\n                -0.014410188421607018,\n                0.0591629222035408,\n                -0.02386775054037571,\n                0.1051902174949646,\n                0.06887765228748322,\n                -0.14827775955200195,\n                0.2279590368270874,\n                -0.06324269622564316,\n                0.19206000864505768,\n                -0.15030698478221893,\n                0.04369090497493744,\n                -0.09575971961021423,\n                0.10184834897518158\n              ],\n              [\n                -0.8009151220321655,\n                0.042741164565086365,\n                0.2561957538127899,\n                -0.03245321288704872,\n                0.2126467525959015,\n                -0.3626656234264374,\n                -0.08744355291128159,\n                0.252923846244812,\n                -0.2406282126903534,\n                -0.2533353567123413,\n                -0.3336438238620758,\n                -0.49595001339912415,\n                0.03433312103152275,\n                0.1379004865884781,\n                -0.5763046741485596,\n                -0.4408527612686157,\n                -0.17515550553798676,\n                0.08869050443172455,\n                0.26932889223098755,\n                -0.33251070976257324,\n                -0.2426517903804779,\n                0.0012413441436365247,\n                0.11090754717588425,\n                0.05123414471745491,\n                -0.24921815097332,\n                -0.4750996530056,\n                0.48253777623176575,\n                -0.027838952839374542,\n                -0.38995078206062317,\n                -0.31328147649765015,\n                -0.3602568507194519,\n                -0.19446098804473877,\n                -0.543244481086731,\n                -0.11715645343065262,\n                0.13799040019512177,\n                -0.1073513776063919,\n                -0.37774690985679626,\n                0.022098375484347343,\n                -0.13915301859378815,\n                -0.4281318783760071,\n                -0.2979890704154968,\n                0.10360624641180038,\n                0.3499918580055237,\n                0.13586987555027008,\n                0.13379111886024475,\n                -0.06283661723136902,\n                0.1657276302576065,\n                0.3291209638118744,\n                -0.15946771204471588,\n                -0.14842531085014343\n              ],\n              [\n                -0.4666611850261688,\n                -0.21241676807403564,\n                0.1910749077796936,\n                0.06911717355251312,\n                0.23509050905704498,\n                0.31656408309936523,\n                0.2708587646484375,\n                0.49206769466400146,\n                0.24235671758651733,\n                0.026647981256246567,\n                0.1187470406293869,\n                -0.14169251918792725,\n                -0.08341462910175323,\n                0.12279897183179855,\n                0.05708643049001694,\n                -0.23100155591964722,\n                0.0009216552716679871,\n                -0.23335391283035278,\n                -0.03357965499162674,\n                0.3808215260505676,\n                0.23694391548633575,\n                -0.15176722407341003,\n                0.14328934252262115,\n                -0.06307930499315262,\n                0.1969938725233078,\n                0.33162984251976013,\n                -0.10945877432823181,\n                0.34658071398735046,\n                0.11664565652608871,\n                0.10710567981004715,\n                -0.3654142916202545,\n                0.28199678659439087,\n                0.089230976998806,\n                0.3477048873901367,\n                0.03435347229242325,\n                0.17952503263950348,\n                -0.005079449154436588,\n                0.3451724946498871,\n                0.2539703845977783,\n                0.31034937500953674,\n                0.21049605309963226,\n                0.5691021680831909,\n                -0.04177477955818176,\n                0.14757639169692993,\n                0.46376973390579224,\n                0.05747527629137039,\n                0.18095451593399048,\n                -0.25173884630203247,\n                0.18668290972709656,\n                0.14818932116031647\n              ],\n              [\n                -0.7069453001022339,\n                0.10696735233068466,\n                -0.1955098658800125,\n                0.04555068910121918,\n                0.3874591588973999,\n                0.20683079957962036,\n                -0.446833997964859,\n                0.29962068796157837,\n                -0.034959275275468826,\n                0.20622512698173523,\n                0.12322287261486053,\n                0.025812890380620956,\n                0.03312835097312927,\n                -0.08387616276741028,\n                0.015942800790071487,\n                0.16793064773082733,\n                0.23285695910453796,\n                0.048181504011154175,\n                0.04697374999523163,\n                0.07852131873369217,\n                -0.1740342527627945,\n                -0.21954193711280823,\n                0.04958309605717659,\n                -0.4439425766468048,\n                0.10293136537075043,\n                -0.044991955161094666,\n                0.09410923719406128,\n                -0.0233745314180851,\n                -0.08290598541498184,\n                0.06586933881044388,\n                -0.13631898164749146,\n                0.054769959300756454,\n                0.2936391532421112,\n                -0.4733392000198364,\n                0.025694748386740685,\n                -0.30140426754951477,\n                0.10703320801258087,\n                -0.00779837416484952,\n                0.07616543024778366,\n                -0.020140083506703377,\n                0.06963186711072922,\n                -0.03516196087002754,\n                0.28908097743988037,\n                -0.1399356573820114,\n                0.448266863822937,\n                0.4235385060310364,\n                -0.3639507591724396,\n                -0.09979131817817688,\n                0.018416909500956535,\n                0.01727212220430374\n              ],\n              [\n                -0.03547343611717224,\n                0.042599186301231384,\n                -0.13445666432380676,\n                0.27867192029953003,\n                -0.1933528035879135,\n                0.14748847484588623,\n                0.23424869775772095,\n                0.6576681137084961,\n                -0.2924177944660187,\n                0.0798158124089241,\n                0.10613710433244705,\n                0.015439657494425774,\n                0.14775849878787994,\n                0.09118485450744629,\n                -0.12247393280267715,\n                0.22114013135433197,\n                0.07602106779813766,\n                0.29813095927238464,\n                -0.31005844473838806,\n                0.1560666412115097,\n                -0.2705908715724945,\n                -0.09668433666229248,\n                0.14817866683006287,\n                0.12874020636081696,\n                0.23493953049182892,\n                0.06945419311523438,\n                -0.07678230106830597,\n                0.11114998906850815,\n                0.059269193559885025,\n                0.13170039653778076,\n                -0.32500356435775757,\n                0.21914531290531158,\n                0.2580072581768036,\n                -0.18756340444087982,\n                0.60267573595047,\n                0.09367770701646805,\n                0.002875487320125103,\n                0.031517673283815384,\n                -0.012696266174316406,\n                0.12384146451950073,\n                0.10011958330869675,\n                -0.13896508514881134,\n                -0.06378152966499329,\n                0.12267380952835083,\n                0.07059703767299652,\n                -0.03849215805530548,\n                0.3489634692668915,\n                0.17285385727882385,\n                0.34713345766067505,\n                0.10579388588666916\n              ],\n              [\n                -0.3740462064743042,\n                -0.14343155920505524,\n                0.13711591064929962,\n                0.21937614679336548,\n                0.805578887462616,\n                -0.48606735467910767,\n                -0.6440969109535217,\n                0.008069561794400215,\n                0.03596563637256622,\n                0.06841330230236053,\n                0.09124680608510971,\n                0.20227183401584625,\n                -0.1476958841085434,\n                -0.10490380972623825,\n                -0.19745688140392303,\n                -0.3149966597557068,\n                0.0642014816403389,\n                0.09790126234292984,\n                -0.3586054742336273,\n                -0.14411096274852753,\n                -0.3387010395526886,\n                -0.461421936750412,\n                0.012739426456391811,\n                0.07183930277824402,\n                0.07319998741149902,\n                0.06467419862747192,\n                -0.02730027213692665,\n                0.0177640188485384,\n                0.009762308560311794,\n                0.18767277896404266,\n                -0.25299814343452454,\n                0.004556243307888508,\n                -0.4331885874271393,\n                0.6903507709503174,\n                0.09927846491336823,\n                0.11865057051181793,\n                -0.19630247354507446,\n                0.03672018647193909,\n                0.006273719482123852,\n                -0.27636194229125977,\n                0.10056620836257935,\n                -0.2750764787197113,\n                -0.2743605375289917,\n                0.11487728357315063,\n                -0.21548666059970856,\n                -0.0849001407623291,\n                0.1967046856880188,\n                0.3868160545825958,\n                -0.11838560551404953,\n                0.10246492177248001\n              ],\n              [\n                -0.15618306398391724,\n                -0.0239772517234087,\n                0.04755749553442001,\n                0.010224930010735989,\n                0.11524950712919235,\n                -0.014770879410207272,\n                -0.06294459104537964,\n                -0.031985167413949966,\n                0.032739587128162384,\n                0.09298791736364365,\n                -0.11481371521949768,\n                -0.15526078641414642,\n                -0.022801987826824188,\n                -0.15966346859931946,\n                0.05545642971992493,\n                0.21264657378196716,\n                -0.08094839006662369,\n                0.4150295555591583,\n                0.038763172924518585,\n                0.14816109836101532,\n                -0.10293320566415787,\n                0.05836347118020058,\n                -0.0792263001203537,\n                -0.32350730895996094,\n                0.1665249615907669,\n                -0.04150903970003128,\n                -0.05239039659500122,\n                -0.08471309393644333,\n                0.121266208589077,\n                -0.0014270353130996227,\n                -0.05903973802924156,\n                -0.02083747088909149,\n                0.06164424121379852,\n                0.12489137798547745,\n                -0.10239112377166748,\n                0.11750168353319168,\n                -0.14621612429618835,\n                0.04299991577863693,\n                0.04783204570412636,\n                0.15346753597259521,\n                0.014543632045388222,\n                0.10683470964431763,\n                -0.03323371335864067,\n                -0.003854575101286173,\n                -0.19214366376399994,\n                -0.03526030108332634,\n                -0.04858594015240669,\n                0.10026992112398148,\n                0.007083422504365444,\n                -0.1050834059715271\n              ],\n              [\n                -0.2340119630098343,\n                -0.1311962902545929,\n                0.012483648024499416,\n                0.08511599898338318,\n                -0.044495388865470886,\n                0.20780839025974274,\n                0.20153303444385529,\n                0.2624434232711792,\n                0.32059332728385925,\n                0.08335617929697037,\n                0.04421668499708176,\n                -0.015539539977908134,\n                0.007245282176882029,\n                0.18704229593276978,\n                -0.26119303703308105,\n                -0.43681997060775757,\n                0.03169839456677437,\n                -0.3552597463130951,\n                0.1179923489689827,\n                -0.10485481470823288,\n                0.23826444149017334,\n                -0.25279805064201355,\n                0.058726225048303604,\n                -0.31730422377586365,\n                0.014462308958172798,\n                -0.00023079095990397036,\n                0.1388983428478241,\n                0.12865278124809265,\n                -0.12762616574764252,\n                0.005441761109977961,\n                -0.1515406370162964,\n                -0.051951657980680466,\n                0.17133884131908417,\n                0.685310423374176,\n                -0.00957696232944727,\n                0.038983482867479324,\n                -0.1642482876777649,\n                0.1801123023033142,\n                0.03890753909945488,\n                -0.14680218696594238,\n                0.06145242601633072,\n                1.042846918106079,\n                -0.042383112013339996,\n                -0.15215317904949188,\n                -0.24692203104496002,\n                0.18917889893054962,\n                -0.14350120723247528,\n                0.26828792691230774,\n                0.23018144071102142,\n                0.10932588577270508\n              ],\n              [\n                0.3322123885154724,\n                -0.18821783363819122,\n                0.0765753984451294,\n                0.23367373645305634,\n                0.3780375123023987,\n                0.1865251213312149,\n                0.3168398439884186,\n                -0.8197890520095825,\n                0.11141742765903473,\n                -0.03586093708872795,\n                -0.0836518257856369,\n                0.19181115925312042,\n                0.22029082477092743,\n                -0.024909770116209984,\n                -0.21364426612854004,\n                0.04330916702747345,\n                -0.02714921161532402,\n                -0.6437059640884399,\n                0.10146825015544891,\n                0.2402789294719696,\n                0.27162525057792664,\n                -0.2091243714094162,\n                0.011973648332059383,\n                -0.018197134137153625,\n                -0.09150644391775131,\n                0.1643635779619217,\n                -0.02910521812736988,\n                0.11285676062107086,\n                -0.033838752657175064,\n                -0.22987478971481323,\n                0.13096605241298676,\n                0.12056934833526611,\n                0.024511314928531647,\n                -0.2536150813102722,\n                -0.19454994797706604,\n                0.34864115715026855,\n                -0.10269317030906677,\n                0.15696042776107788,\n                -0.32188767194747925,\n                -0.09561910480260849,\n                0.060371577739715576,\n                0.08191504329442978,\n                -0.8268782496452332,\n                -0.2580358386039734,\n                0.3688259422779083,\n                -0.48169219493865967,\n                -0.049497392028570175,\n                0.29896190762519836,\n                -0.026430204510688782,\n                0.008099472150206566\n              ],\n              [\n                -0.029509885236620903,\n                -0.34242621064186096,\n                0.16561579704284668,\n                -0.010497413575649261,\n                0.04179583117365837,\n                0.10720138251781464,\n                0.04319336637854576,\n                0.02871839515864849,\n                0.023956257849931717,\n                -0.01933148317039013,\n                -0.16761919856071472,\n                -0.09857246279716492,\n                -0.22773438692092896,\n                0.13983239233493805,\n                -0.2308560311794281,\n                -0.2406291514635086,\n                -0.06087779253721237,\n                0.06693939119577408,\n                0.17468194663524628,\n                0.1563374251127243,\n                -0.1310565173625946,\n                0.49474406242370605,\n                -0.15331414341926575,\n                -0.32666221261024475,\n                -0.20615436136722565,\n                -0.21729524433612823,\n                0.20773786306381226,\n                -0.3822798728942871,\n                -0.04398162290453911,\n                -0.1281927078962326,\n                0.007586438674479723,\n                -0.611744225025177,\n                -0.05213972553610802,\n                -0.09962411969900131,\n                -0.6330392956733704,\n                0.23169295489788055,\n                -0.2639612555503845,\n                0.13328424096107483,\n                -0.06009092554450035,\n                -0.1239374503493309,\n                -0.3297577500343323,\n                0.10535461455583572,\n                0.23223823308944702,\n                -0.09761741012334824,\n                -0.3502937853336334,\n                -0.18727360665798187,\n                -0.29612648487091064,\n                -0.056469522416591644,\n                -0.10573189705610275,\n                -0.14346741139888763\n              ],\n              [\n                -0.26724305748939514,\n                0.08889330178499222,\n                0.008113402873277664,\n                -0.27317148447036743,\n                0.2011149823665619,\n                -0.04121546074748039,\n                -0.22013381123542786,\n                -0.797607421875,\n                0.36875659227371216,\n                -0.058561183512210846,\n                0.08153904974460602,\n                -0.007255932316184044,\n                -0.0984342098236084,\n                0.057313233613967896,\n                -0.0735824853181839,\n                -0.26956233382225037,\n                0.017771027982234955,\n                -0.08869937062263489,\n                -0.005857986398041248,\n                -0.25606125593185425,\n                -0.09762841463088989,\n                -0.33106791973114014,\n                -0.06652376800775528,\n                -0.12843550741672516,\n                -0.08270307630300522,\n                -0.046221524477005005,\n                -0.09148037433624268,\n                0.07982458919286728,\n                0.03203823044896126,\n                0.016645997762680054,\n                0.039965804666280746,\n                0.2062731832265854,\n                0.11458174884319305,\n                0.18539603054523468,\n                -0.022299231961369514,\n                -0.49218472838401794,\n                -0.09138567000627518,\n                -0.004584163893014193,\n                0.05923668295145035,\n                -0.29008257389068604,\n                0.10724376887083054,\n                -0.07637747377157211,\n                -0.23378099501132965,\n                0.05910341441631317,\n                0.203413188457489,\n                -0.09736248105764389,\n                0.2546629011631012,\n                0.09849297255277634,\n                -0.24182625114917755,\n                -0.07155266404151917\n              ],\n              [\n                -0.3199852705001831,\n                -0.04902562126517296,\n                -0.11015060544013977,\n                0.13555565476417542,\n                -0.1786220222711563,\n                0.09948953241109848,\n                0.14066071808338165,\n                -0.21072794497013092,\n                -0.5030348896980286,\n                -0.08439727872610092,\n                -0.2530125081539154,\n                -0.3614713251590729,\n                -0.05258870869874954,\n                -0.10137499868869781,\n                -0.22220778465270996,\n                0.32201874256134033,\n                0.0799582377076149,\n                0.15970627963542938,\n                0.0795905590057373,\n                0.27575740218162537,\n                -0.044520117342472076,\n                -0.18729595839977264,\n                -0.11038057506084442,\n                -0.3810097873210907,\n                -0.4145146608352661,\n                -0.056077029556035995,\n                -0.14889715611934662,\n                -0.11386126279830933,\n                -0.4162250757217407,\n                -0.043115030974149704,\n                -0.08273240923881531,\n                -0.021875593811273575,\n                0.010167443193495274,\n                0.3572275638580322,\n                0.09942388534545898,\n                0.3671188950538635,\n                -0.3396758437156677,\n                -0.25534388422966003,\n                -0.4077712297439575,\n                0.06318295001983643,\n                -0.32920631766319275,\n                -0.07883904874324799,\n                -0.010043187066912651,\n                0.07424140721559525,\n                0.05924276262521744,\n                -0.2450912594795227,\n                0.04750494286417961,\n                0.019476082175970078,\n                -0.2556335926055908,\n                -0.15605509281158447\n              ],\n              [\n                0.6340359449386597,\n                0.04145973548293114,\n                -0.07292081415653229,\n                0.07251635938882828,\n                -0.08202191442251205,\n                0.04293441027402878,\n                0.5576285719871521,\n                0.23475027084350586,\n                0.44962626695632935,\n                0.006248448044061661,\n                0.06210228055715561,\n                -0.2050963193178177,\n                0.03397884964942932,\n                -0.022366419434547424,\n                0.08456797897815704,\n                0.0763671025633812,\n                0.10751130431890488,\n                -0.08908049017190933,\n                -0.44752663373947144,\n                -0.2895398437976837,\n                0.33557626605033875,\n                -0.272480845451355,\n                0.09224913269281387,\n                0.35606077313423157,\n                0.2651411294937134,\n                0.4373738169670105,\n                0.3549002408981323,\n                0.08648732304573059,\n                0.14063279330730438,\n                -0.06875208020210266,\n                0.1601409614086151,\n                0.041062165051698685,\n                0.04832049459218979,\n                0.1375073343515396,\n                -0.16349495947360992,\n                -0.20357580482959747,\n                0.04089963436126709,\n                -0.06695489585399628,\n                0.12484778463840485,\n                0.03442877158522606,\n                0.15138018131256104,\n                -0.3351605534553528,\n                -0.2105487734079361,\n                0.008606968447566032,\n                -0.06378739327192307,\n                -0.318213552236557,\n                0.036649588495492935,\n                0.0718257948756218,\n                0.05229468643665314,\n                0.10798918455839157\n              ],\n              [\n                -0.4880310297012329,\n                -0.290319561958313,\n                0.18820735812187195,\n                -0.06892693042755127,\n                0.20368099212646484,\n                -0.26760879158973694,\n                0.31209075450897217,\n                -0.593944787979126,\n                0.498654305934906,\n                0.10167057812213898,\n                0.14594915509223938,\n                0.10385118424892426,\n                -0.30846941471099854,\n                -0.16828879714012146,\n                -0.1857483834028244,\n                -0.4659593105316162,\n                -0.09148602187633514,\n                0.9138979911804199,\n                0.05556275323033333,\n                -0.04497753083705902,\n                -0.07722266763448715,\n                -0.08538047969341278,\n                -0.1086912751197815,\n                -0.18396438658237457,\n                -0.0110703120008111,\n                -0.2346513569355011,\n                0.6007078886032104,\n                -0.4091869592666626,\n                -0.10134308785200119,\n                0.08310243487358093,\n                -0.33675816655158997,\n                0.280028373003006,\n                -0.03388960659503937,\n                -0.2295314520597458,\n                -0.2214173972606659,\n                0.1346883475780487,\n                -0.15148119628429413,\n                -0.11658339947462082,\n                -0.3361017405986786,\n                -0.3612913191318512,\n                -0.09712904691696167,\n                0.024362584576010704,\n                -0.09650429338216782,\n                -0.24035996198654175,\n                0.10614266246557236,\n                0.09804969280958176,\n                -0.5469701886177063,\n                -0.13870614767074585,\n                -0.027438627555966377,\n                0.026141872629523277\n              ],\n              [\n                0.07229895144701004,\n                0.03486057370901108,\n                -0.19355346262454987,\n                0.29655587673187256,\n                0.13598886132240295,\n                0.10626787692308426,\n                0.09095804393291473,\n                0.31509214639663696,\n                -0.13798213005065918,\n                0.23979517817497253,\n                0.276653528213501,\n                -0.03847532346844673,\n                0.32585328817367554,\n                0.05657418817281723,\n                0.09893937408924103,\n                0.145035058259964,\n                0.029836229979991913,\n                0.2778482437133789,\n                -0.35207322239875793,\n                0.08971316367387772,\n                0.4157799780368805,\n                0.028854133561253548,\n                -0.10814930498600006,\n                0.055448174476623535,\n                0.22193656861782074,\n                -0.0042461915872991085,\n                0.030554668977856636,\n                0.16198888421058655,\n                -0.035144224762916565,\n                0.05954234302043915,\n                -0.6285920143127441,\n                0.18410919606685638,\n                0.1915920525789261,\n                -0.5593177080154419,\n                -0.1998709887266159,\n                -0.3633144795894623,\n                -0.005100379232317209,\n                0.22080577909946442,\n                0.07437482476234436,\n                0.16885313391685486,\n                0.27040478587150574,\n                -0.29884520173072815,\n                0.18099215626716614,\n                0.2769649624824524,\n                0.051309648901224136,\n                0.05014529824256897,\n                0.4624188542366028,\n                0.24487924575805664,\n                0.14120706915855408,\n                0.2307230830192566\n              ],\n              [\n                -0.28400999307632446,\n                -0.31002405285835266,\n                -0.012761787511408329,\n                0.14118428528308868,\n                0.4426450729370117,\n                0.1171816810965538,\n                0.10751694440841675,\n                -0.6219719052314758,\n                0.035196222364902496,\n                -0.1460052877664566,\n                -0.022354193031787872,\n                -0.04214084893465042,\n                -0.04056141525506973,\n                -0.0316203348338604,\n                0.058338381350040436,\n                -0.07202573865652084,\n                -0.16142955422401428,\n                -0.12001245468854904,\n                0.3406381607055664,\n                0.0032692679669708014,\n                -0.36121946573257446,\n                -0.22993430495262146,\n                -0.06961605697870255,\n                -0.808266282081604,\n                -0.10942716896533966,\n                -0.10159999877214432,\n                -0.10182453691959381,\n                -0.23560519516468048,\n                -0.14857430756092072,\n                0.0991126224398613,\n                0.08501067012548447,\n                0.12716858088970184,\n                -0.10174532979726791,\n                0.01189819723367691,\n                -0.0013789650984108448,\n                0.2006131261587143,\n                0.03138815984129906,\n                -0.18153460323810577,\n                -0.03544573113322258,\n                -0.20842579007148743,\n                -0.1990332305431366,\n                0.017326844856142998,\n                0.08995715528726578,\n                -0.29164257645606995,\n                -0.054569680243730545,\n                0.07508371770381927,\n                0.1298971027135849,\n                0.11354754865169525,\n                -0.11951977014541626,\n                -0.007311002351343632\n              ],\n              [\n                -0.313959002494812,\n                0.02367100678384304,\n                -0.23750454187393188,\n                0.19636744260787964,\n                -0.4092922508716583,\n                0.3455292284488678,\n                0.1998608410358429,\n                0.22046808898448944,\n                -0.35031044483184814,\n                0.03594677150249481,\n                0.12130099534988403,\n                0.04169800877571106,\n                0.2498067170381546,\n                -0.0372464656829834,\n                0.1514752060174942,\n                -0.011637207120656967,\n                0.17202650010585785,\n                0.10219906270503998,\n                -0.44123736023902893,\n                0.20308686792850494,\n                -0.13556955754756927,\n                0.0338115394115448,\n                0.07485858350992203,\n                -0.441572904586792,\n                0.19151411950588226,\n                0.06156781688332558,\n                -0.09438305348157883,\n                -0.032099466770887375,\n                0.12792031466960907,\n                -0.09593029320240021,\n                -0.14163430035114288,\n                0.21665973961353302,\n                0.2212689220905304,\n                -0.09448543190956116,\n                -0.09020856767892838,\n                -0.09307733178138733,\n                -0.11562037467956543,\n                0.21669070422649384,\n                0.2042226493358612,\n                0.10017234086990356,\n                0.0006136708543635905,\n                0.2547128200531006,\n                -0.24404467642307281,\n                -0.17294526100158691,\n                -0.012948770076036453,\n                0.18893244862556458,\n                -0.06888674944639206,\n                0.09324324131011963,\n                0.1519479602575302,\n                0.11566980183124542\n              ],\n              [\n                -0.4943947196006775,\n                -0.37880223989486694,\n                0.12686514854431152,\n                -0.028622223064303398,\n                0.17918363213539124,\n                -0.13050785660743713,\n                0.2026994228363037,\n                -0.17935694754123688,\n                0.4380630552768707,\n                -0.11766725033521652,\n                -0.129194974899292,\n                -0.04541458934545517,\n                0.1604277342557907,\n                -0.1607377976179123,\n                -0.04632454365491867,\n                -0.011155758053064346,\n                -0.13659162819385529,\n                -1.0188244581222534,\n                0.37685921788215637,\n                -0.18514999747276306,\n                0.07884564250707626,\n                0.1309960037469864,\n                -0.02880169078707695,\n                -0.28296878933906555,\n                -0.004682627506554127,\n                -0.030437076464295387,\n                -0.015656590461730957,\n                0.21033556759357452,\n                -0.1351899802684784,\n                -0.2936747372150421,\n                -0.44261977076530457,\n                0.15120938420295715,\n                -0.21807026863098145,\n                -0.34597519040107727,\n                -0.0027242463547736406,\n                0.11175087839365005,\n                -0.13817018270492554,\n                -0.11875613778829575,\n                -0.05769561976194382,\n                0.12040862441062927,\n                0.00770455552265048,\n                0.07549260556697845,\n                -0.2092660665512085,\n                -0.1579517424106598,\n                0.35385575890541077,\n                -0.2981831431388855,\n                -0.14543303847312927,\n                -0.41573572158813477,\n                -0.1754993051290512,\n                -0.25732412934303284\n              ],\n              [\n                -0.5843906998634338,\n                0.023006102070212364,\n                -0.2868063449859619,\n                0.19544805586338043,\n                -0.2703016400337219,\n                -0.4782143831253052,\n                0.15871690213680267,\n                -0.5789566040039062,\n                0.27165892720222473,\n                0.16037949919700623,\n                0.15952976047992706,\n                -0.4202682077884674,\n                -0.061652787029743195,\n                0.007442873436957598,\n                -0.14205460250377655,\n                0.12480668723583221,\n                0.10434304177761078,\n                0.33649593591690063,\n                0.006205507088452578,\n                0.37024521827697754,\n                -0.1162598580121994,\n                0.2969197630882263,\n                -0.1422189474105835,\n                -0.04484979435801506,\n                0.33930695056915283,\n                -0.3059912323951721,\n                -0.26080235838890076,\n                -0.08195023238658905,\n                -0.2389908730983734,\n                0.2225254625082016,\n                -0.020228303968906403,\n                0.08015476167201996,\n                -0.20816384255886078,\n                0.5272433757781982,\n                -0.24563837051391602,\n                -0.231959730386734,\n                -0.4278993308544159,\n                -0.19001170992851257,\n                -0.06070360913872719,\n                0.01738264597952366,\n                -0.032029006630182266,\n                -0.32012397050857544,\n                0.7366903424263,\n                -0.35473912954330444,\n                0.04068528115749359,\n                0.10522628575563431,\n                0.4234182834625244,\n                0.4338505268096924,\n                0.2428039014339447,\n                0.15285803377628326\n              ],\n              [\n                0.32526499032974243,\n                0.0700555145740509,\n                0.16584300994873047,\n                0.14409296214580536,\n                -0.12014655023813248,\n                -0.1813012957572937,\n                0.23767665028572083,\n                -0.5436683297157288,\n                0.04718820005655289,\n                -0.05421098321676254,\n                0.0914592370390892,\n                -0.03039310686290264,\n                -0.21307942271232605,\n                -0.21871115267276764,\n                0.24239540100097656,\n                -0.1697438806295395,\n                0.15428747236728668,\n                -1.0806773900985718,\n                -0.14046365022659302,\n                -0.26631084084510803,\n                0.0949084609746933,\n                0.576619029045105,\n                0.02062864601612091,\n                -0.3112156391143799,\n                0.029294926673173904,\n                0.30035170912742615,\n                0.19895866513252258,\n                -0.058695610612630844,\n                -0.013687225058674812,\n                0.09398889541625977,\n                -0.00045071024214848876,\n                0.04743901267647743,\n                -0.22651807963848114,\n                -0.48898157477378845,\n                -0.22358489036560059,\n                0.39652758836746216,\n                0.3221066892147064,\n                -0.03893650323152542,\n                0.18594510853290558,\n                -0.13303305208683014,\n                -0.03647095710039139,\n                0.07256828993558884,\n                -0.13758498430252075,\n                0.11072687059640884,\n                0.29907891154289246,\n                -0.04181970655918121,\n                -0.07504820823669434,\n                0.08294575661420822,\n                -0.058195117861032486,\n                0.07449038326740265\n              ],\n              [\n                -0.5233381986618042,\n                0.0008672281401231885,\n                -0.03805981203913689,\n                0.21171528100967407,\n                0.23875588178634644,\n                0.04219716042280197,\n                0.5243920683860779,\n                0.24311195313930511,\n                0.1120820939540863,\n                0.14307865500450134,\n                0.09491016715765,\n                0.18101878464221954,\n                0.08173248171806335,\n                0.11200929433107376,\n                -0.04997217655181885,\n                0.13585078716278076,\n                0.03232493996620178,\n                0.08581267297267914,\n                -0.07864118367433548,\n                0.1779915690422058,\n                -0.058080434799194336,\n                0.03136714547872543,\n                0.11727762222290039,\n                -0.5342357158660889,\n                0.10215844959020615,\n                0.031690146774053574,\n                -0.12930142879486084,\n                0.03609846532344818,\n                -0.02158038690686226,\n                0.051236532628536224,\n                -0.36258170008659363,\n                0.31467580795288086,\n                0.03029535710811615,\n                -0.24494363367557526,\n                0.02350180596113205,\n                -0.026698840782046318,\n                0.05107489600777626,\n                0.12503203749656677,\n                0.02067023143172264,\n                0.1583503931760788,\n                0.07459188997745514,\n                -0.08375407755374908,\n                -0.13252781331539154,\n                0.08350452780723572,\n                0.028414392843842506,\n                0.24695615470409393,\n                0.07464083284139633,\n                -0.0393829308450222,\n                0.16417503356933594,\n                0.03753133490681648\n              ],\n              [\n                -0.25898006558418274,\n                0.025537917390465736,\n                0.05677321180701256,\n                -0.03887172043323517,\n                0.3491106331348419,\n                0.005391890183091164,\n                0.14687418937683105,\n                -0.7723749876022339,\n                0.34060534834861755,\n                -0.04562126472592354,\n                -0.00858503021299839,\n                -0.0998515784740448,\n                -0.1542772501707077,\n                -0.0470099039375782,\n                -0.12248252332210541,\n                -0.15803958475589752,\n                -0.09450267255306244,\n                0.09489660710096359,\n                0.18596765398979187,\n                0.10402277112007141,\n                -0.28702250123023987,\n                -0.33180755376815796,\n                -0.20281799137592316,\n                -0.06972038745880127,\n                -0.03837069496512413,\n                -0.02239038050174713,\n                -0.07939201593399048,\n                0.040755368769168854,\n                -0.04678730294108391,\n                -0.01703268103301525,\n                0.20631742477416992,\n                0.2041403204202652,\n                -0.11710700392723083,\n                0.5981282591819763,\n                -0.09409008175134659,\n                0.1276732236146927,\n                -0.05574595183134079,\n                -0.08588626980781555,\n                0.07564660161733627,\n                0.09508813172578812,\n                -0.07919439673423767,\n                0.07502157241106033,\n                -0.13839483261108398,\n                -0.08790223300457001,\n                0.18147775530815125,\n                0.014002826064825058,\n                0.33050379157066345,\n                0.15746422111988068,\n                -0.0012312111211940646,\n                -0.01717819832265377\n              ],\n              [\n                -0.049337729811668396,\n                -0.25081348419189453,\n                -0.09281174093484879,\n                0.007727284450083971,\n                0.19251953065395355,\n                0.013646545819938183,\n                -0.09981174021959305,\n                -0.45988261699676514,\n                -0.09433955699205399,\n                -0.02321835234761238,\n                0.00871738325804472,\n                -0.2401668131351471,\n                -0.29249924421310425,\n                0.033145297318696976,\n                0.08899916708469391,\n                0.06436201184988022,\n                0.22819466888904572,\n                0.8446778059005737,\n                -0.0030743919778615236,\n                -0.06640619784593582,\n                0.004932293202728033,\n                -0.47813647985458374,\n                -0.01945245824754238,\n                -0.6641075611114502,\n                -0.10394883900880814,\n                -0.23606987297534943,\n                -0.04755371809005737,\n                -0.3163021504878998,\n                0.0927969366312027,\n                -0.1444014608860016,\n                -0.15632379055023193,\n                0.1263875514268875,\n                -0.02288808859884739,\n                0.23840370774269104,\n                -0.4262140989303589,\n                0.03808695450425148,\n                -0.2587064206600189,\n                -0.10681041330099106,\n                -0.27944910526275635,\n                -0.012667844071984291,\n                -0.04578428342938423,\n                0.07030569761991501,\n                0.0029238443821668625,\n                -0.2608352601528168,\n                -0.1478152722120285,\n                0.193078875541687,\n                -0.046339958906173706,\n                0.35504114627838135,\n                -0.16476699709892273,\n                0.10348972678184509\n              ],\n              [\n                -1.026081919670105,\n                0.02744203247129917,\n                0.09528150409460068,\n                0.3015398681163788,\n                0.19705185294151306,\n                0.11942197382450104,\n                0.06681142002344131,\n                0.0752897709608078,\n                0.21040531992912292,\n                0.10917877405881882,\n                0.17000837624073029,\n                -0.4611847996711731,\n                0.08706481754779816,\n                -0.34832972288131714,\n                0.04933488368988037,\n                -0.03971042484045029,\n                -0.13147708773612976,\n                -0.04960452765226364,\n                0.14727677404880524,\n                0.4749727249145508,\n                0.20106777548789978,\n                0.049444835633039474,\n                -0.04854841157793999,\n                0.207233726978302,\n                -0.049859073013067245,\n                0.19830238819122314,\n                -0.13150238990783691,\n                -0.1400122493505478,\n                -0.07341067492961884,\n                -0.14692482352256775,\n                -0.3291547894477844,\n                -0.18517127633094788,\n                0.26848304271698,\n                -0.28378555178642273,\n                0.013217049650847912,\n                0.2950667142868042,\n                0.026300951838493347,\n                -0.06482067704200745,\n                -0.15643450617790222,\n                0.08075056970119476,\n                -0.15444286167621613,\n                0.09519573301076889,\n                0.26808351278305054,\n                -0.19517827033996582,\n                0.12917156517505646,\n                -0.1733582466840744,\n                -0.15164172649383545,\n                0.24183572828769684,\n                -0.054356083273887634,\n                -0.08053381741046906\n              ],\n              [\n                0.2168618142604828,\n                -0.20145684480667114,\n                0.19428397715091705,\n                -0.7246829867362976,\n                -0.2696896195411682,\n                0.13537590205669403,\n                0.16436220705509186,\n                0.05112558975815773,\n                -0.45426303148269653,\n                -0.019151555374264717,\n                -0.20496296882629395,\n                -0.27425745129585266,\n                -0.1241728886961937,\n                -0.04212280362844467,\n                -0.20894964039325714,\n                -0.2541345953941345,\n                -0.11864986270666122,\n                -0.22503788769245148,\n                0.41987642645835876,\n                -0.025292381644248962,\n                -0.19198943674564362,\n                0.36040639877319336,\n                -0.06132618337869644,\n                0.049134623259305954,\n                -0.27169284224510193,\n                -0.13051527738571167,\n                -0.1766655296087265,\n                -0.25108131766319275,\n                -0.11183443665504456,\n                -0.2144920825958252,\n                0.4685809314250946,\n                -0.21427913010120392,\n                -0.035761263221502304,\n                0.2656055688858032,\n                -0.16916987299919128,\n                0.24911807477474213,\n                -0.01632564887404442,\n                0.014032711274921894,\n                -0.20321834087371826,\n                -0.026093054562807083,\n                -0.20576433837413788,\n                0.1716836392879486,\n                -0.38304486870765686,\n                0.174935981631279,\n                -0.03438805788755417,\n                0.14550459384918213,\n                -0.057037413120269775,\n                -0.08195150643587112,\n                -0.0025741150602698326,\n                -0.16669988632202148\n              ],\n              [\n                0.23142458498477936,\n                -0.07066720724105835,\n                0.17238746583461761,\n                0.29519540071487427,\n                -0.19976314902305603,\n                0.04449260234832764,\n                0.27449196577072144,\n                0.02509736828505993,\n                -0.2874855101108551,\n                -0.048564452677965164,\n                -0.0011683930642902851,\n                -0.36487382650375366,\n                0.1721363067626953,\n                -0.33491894602775574,\n                0.25553280115127563,\n                -0.20792138576507568,\n                0.0665774717926979,\n                -0.41052478551864624,\n                -0.4131990969181061,\n                -0.018855014815926552,\n                0.3250945508480072,\n                -0.0759248211979866,\n                0.026377229019999504,\n                -0.6710410714149475,\n                -0.013546440750360489,\n                0.06632640957832336,\n                -0.3221264183521271,\n                0.2857840955257416,\n                -0.14754991233348846,\n                -0.03256852552294731,\n                -0.14526675641536713,\n                0.09109043329954147,\n                0.15627263486385345,\n                -0.21006415784358978,\n                -0.18224649131298065,\n                -0.06908787786960602,\n                0.2840081751346588,\n                -0.20591585338115692,\n                -0.2587043344974518,\n                -0.15888836979866028,\n                0.21626555919647217,\n                0.22565510869026184,\n                -0.6376015543937683,\n                0.25216415524482727,\n                0.4058956801891327,\n                -0.07704833894968033,\n                -0.19777317345142365,\n                0.10293310135602951,\n                0.054722681641578674,\n                0.006313395220786333\n              ],\n              [\n                0.10213359445333481,\n                0.09516975283622742,\n                -0.05086249113082886,\n                -0.6481035947799683,\n                -0.01004539243876934,\n                0.20505502820014954,\n                0.4609759449958801,\n                -0.5429484844207764,\n                -0.4750175476074219,\n                -0.03546210750937462,\n                -0.034196190536022186,\n                -0.3148801326751709,\n                0.1869203746318817,\n                -0.31235817074775696,\n                -0.1983237862586975,\n                0.22295261919498444,\n                -0.14915519952774048,\n                -0.1021757423877716,\n                -0.011998548172414303,\n                0.11987872421741486,\n                -0.03372648358345032,\n                -0.10986010730266571,\n                0.21892975270748138,\n                0.20584993064403534,\n                0.023383690044283867,\n                -0.024683650583028793,\n                -0.2989889085292816,\n                0.098348468542099,\n                -0.17126165330410004,\n                0.03279683366417885,\n                -0.2868504524230957,\n                -0.038162991404533386,\n                -0.2008214294910431,\n                -0.7928566932678223,\n                -0.3392408490180969,\n                0.16560392081737518,\n                0.4668349623680115,\n                -0.136211559176445,\n                -0.044954363256692886,\n                0.3486177921295166,\n                -0.04184315726161003,\n                0.13062907755374908,\n                -0.019696103408932686,\n                0.1324857473373413,\n                0.34107741713523865,\n                0.09835230559110641,\n                -0.01589975319802761,\n                -0.389157235622406,\n                -0.10209089517593384,\n                0.14743943512439728\n              ],\n              [\n                -0.45345762372016907,\n                0.29428210854530334,\n                -0.2090853899717331,\n                0.18112409114837646,\n                -0.285805344581604,\n                0.251838743686676,\n                0.28611043095588684,\n                0.43320441246032715,\n                0.3690531849861145,\n                -0.09351328015327454,\n                0.0426195003092289,\n                0.34546083211898804,\n                0.1765691488981247,\n                -0.0577813945710659,\n                -0.02116667665541172,\n                -0.09047780930995941,\n                -0.06086895242333412,\n                0.08966435492038727,\n                0.10113221406936646,\n                0.05189082771539688,\n                0.5182246565818787,\n                -0.22100670635700226,\n                0.14233289659023285,\n                -0.16081075370311737,\n                0.22065068781375885,\n                -0.04606064409017563,\n                -0.41266435384750366,\n                0.22386234998703003,\n                -0.17480097711086273,\n                0.04691757261753082,\n                -0.2166384607553482,\n                0.03212357684969902,\n                0.18761563301086426,\n                0.8363586664199829,\n                -0.1254202127456665,\n                0.4455370008945465,\n                0.042582012712955475,\n                0.16312097012996674,\n                0.3006892800331116,\n                -0.2718108594417572,\n                0.3281973600387573,\n                0.19491545855998993,\n                -0.18445897102355957,\n                -0.20376981794834137,\n                -0.2981553077697754,\n                0.1237783282995224,\n                -0.016201067715883255,\n                0.1662474125623703,\n                -0.10616820305585861,\n                -0.06032990664243698\n              ],\n              [\n                0.232689768075943,\n                -0.44447171688079834,\n                0.2105448693037033,\n                0.2751316428184509,\n                0.37711894512176514,\n                0.0022084673400968313,\n                0.36341285705566406,\n                -0.12637849152088165,\n                0.12386828660964966,\n                -0.33580708503723145,\n                -0.27682510018348694,\n                -0.19002771377563477,\n                -0.3964892327785492,\n                -0.3058200776576996,\n                -0.3354426920413971,\n                -0.26642197370529175,\n                -0.35868096351623535,\n                -0.21975313127040863,\n                0.18821504712104797,\n                -0.0631081610918045,\n                0.3153245747089386,\n                0.228838250041008,\n                -0.42645883560180664,\n                -0.3963789641857147,\n                -0.22596678137779236,\n                -0.39264315366744995,\n                -0.3945613205432892,\n                -0.18992874026298523,\n                -0.23827767372131348,\n                -0.4351137578487396,\n                0.14640508592128754,\n                0.2626323699951172,\n                0.05601993203163147,\n                0.17204931378364563,\n                -0.4240603744983673,\n                0.24811267852783203,\n                -0.3620324432849884,\n                -0.2059416025876999,\n                -0.38578566908836365,\n                -0.3957683742046356,\n                -0.37723562121391296,\n                0.43035218119621277,\n                0.36406394839286804,\n                0.19895757734775543,\n                0.36992719769477844,\n                -0.06417641043663025,\n                -0.24774883687496185,\n                0.3493569493293762,\n                -0.308956503868103,\n                -0.29132598638534546\n              ],\n              [\n                0.023923726752400398,\n                -0.2246474176645279,\n                0.05102603882551193,\n                -0.05412611365318298,\n                0.6693800091743469,\n                0.15467675030231476,\n                0.2673344016075134,\n                0.4409731328487396,\n                0.218567356467247,\n                -0.027865683659911156,\n                0.14328183233737946,\n                0.28424784541130066,\n                0.29170235991477966,\n                -0.21605022251605988,\n                -0.21891926229000092,\n                -0.20754027366638184,\n                -0.06515808403491974,\n                -0.45035234093666077,\n                0.34284478425979614,\n                -0.011246146634221077,\n                -0.565503716468811,\n                0.5686787366867065,\n                0.1872691512107849,\n                0.22016026079654694,\n                -0.054653093218803406,\n                0.14171342551708221,\n                0.25563979148864746,\n                0.33700719475746155,\n                -0.11873351037502289,\n                -0.41830453276634216,\n                -0.006842146161943674,\n                -0.05337260663509369,\n                0.0794617310166359,\n                0.4183383882045746,\n                -0.1572146862745285,\n                0.11208879202604294,\n                -0.1934702843427658,\n                0.06295482814311981,\n                -0.2973204255104065,\n                -0.24784885346889496,\n                0.21634313464164734,\n                -0.06762319058179855,\n                -0.06652755290269852,\n                0.34584861993789673,\n                0.15847043693065643,\n                -0.4551164507865906,\n                0.2029569298028946,\n                -0.18294475972652435,\n                0.16255755722522736,\n                0.11765027046203613\n              ],\n              [\n                0.14456596970558167,\n                0.23278626799583435,\n                -0.1887637823820114,\n                0.3462575674057007,\n                0.04277247190475464,\n                -0.3237273097038269,\n                -0.001447975286282599,\n                -0.11434292048215866,\n                0.1814809888601303,\n                -0.0034667898435145617,\n                0.20990003645420074,\n                0.13823778927326202,\n                0.07381826639175415,\n                -0.07411574572324753,\n                -0.004729114938527346,\n                0.05571029707789421,\n                0.09177210181951523,\n                0.18506179749965668,\n                -0.27223801612854004,\n                -0.08937279134988785,\n                -0.1868707537651062,\n                -0.5123598575592041,\n                0.06821846216917038,\n                -0.061055004596710205,\n                0.08442848920822144,\n                0.04776329547166824,\n                0.190947026014328,\n                0.022976884618401527,\n                -0.08878858387470245,\n                0.061760641634464264,\n                0.27804508805274963,\n                0.1212826669216156,\n                -0.0813269391655922,\n                -0.787715494632721,\n                0.032655902206897736,\n                -0.14507944881916046,\n                -0.0301596038043499,\n                -0.01838349923491478,\n                -0.004320202860981226,\n                0.1694989651441574,\n                0.0779040977358818,\n                -0.038277775049209595,\n                0.9786800742149353,\n                0.21260057389736176,\n                0.26448294520378113,\n                0.364663690328598,\n                0.09841179847717285,\n                0.6356847882270813,\n                -0.08115611970424652,\n                -0.01330023817718029\n              ],\n              [\n                -0.07180869579315186,\n                0.10873803496360779,\n                -0.32323765754699707,\n                -0.09696871787309647,\n                0.11804728955030441,\n                0.11073006689548492,\n                -0.22454044222831726,\n                -0.6546792387962341,\n                0.3506810963153839,\n                0.11831827461719513,\n                0.05820620432496071,\n                -0.09232763200998306,\n                0.08361367136240005,\n                -0.15400183200836182,\n                0.06318897008895874,\n                -0.12331978976726532,\n                -0.21589307487010956,\n                0.1472158432006836,\n                0.03838353604078293,\n                -0.05095091834664345,\n                0.31625866889953613,\n                -0.460136741399765,\n                0.09728768467903137,\n                -0.5579326152801514,\n                -0.09198839217424393,\n                0.019890688359737396,\n                0.10566385835409164,\n                -0.21438159048557281,\n                -0.16338294744491577,\n                -0.1149531677365303,\n                0.14449386298656464,\n                0.3294515013694763,\n                0.1777021735906601,\n                -0.0473497100174427,\n                -0.05907529592514038,\n                -0.33387672901153564,\n                0.07428231835365295,\n                -0.06336923688650131,\n                -0.06709934771060944,\n                0.231694757938385,\n                0.15215325355529785,\n                -0.1876620352268219,\n                -0.03445301577448845,\n                -0.28921061754226685,\n                -0.11027219146490097,\n                -0.27413409948349,\n                -0.14854860305786133,\n                0.13366009294986725,\n                -0.10490302741527557,\n                0.03806979954242706\n              ],\n              [\n                -0.542121410369873,\n                -0.09144406765699387,\n                0.06516508758068085,\n                0.1444881111383438,\n                -0.06035938858985901,\n                -0.18947480618953705,\n                0.18588626384735107,\n                0.20526662468910217,\n                0.20760725438594818,\n                0.09160985052585602,\n                0.07479687035083771,\n                -0.0952373743057251,\n                0.014119399711489677,\n                0.12985841929912567,\n                -0.025724386796355247,\n                -0.08784901350736618,\n                0.001289311796426773,\n                0.055360712110996246,\n                -0.08706086874008179,\n                0.08450601994991302,\n                0.14436344802379608,\n                -0.014574314467608929,\n                0.13801687955856323,\n                -0.5366510152816772,\n                0.1834602653980255,\n                0.06558286398649216,\n                -0.15803036093711853,\n                0.12210821360349655,\n                -0.05941721424460411,\n                -0.0416509173810482,\n                -0.5011538863182068,\n                0.18266136944293976,\n                0.013269172050058842,\n                0.09535439312458038,\n                -0.11875949800014496,\n                0.044719498604536057,\n                -0.06809011846780777,\n                0.17947758734226227,\n                -0.026595115661621094,\n                0.0039175753481686115,\n                0.03470650687813759,\n                0.11196630448102951,\n                0.04789390787482262,\n                -0.02402217872440815,\n                -0.031177038326859474,\n                0.181244894862175,\n                -0.20908616483211517,\n                -0.04465540125966072,\n                0.01718798093497753,\n                0.008290152065455914\n              ],\n              [\n                -0.7346735000610352,\n                -0.658176600933075,\n                0.20461003482341766,\n                0.07439437508583069,\n                -0.13856419920921326,\n                0.07014510780572891,\n                -0.24387702345848083,\n                0.11632256209850311,\n                0.8128634691238403,\n                -0.04128701612353325,\n                -0.21861222386360168,\n                -0.09075810015201569,\n                0.023668358102440834,\n                0.18390357494354248,\n                -0.04334521293640137,\n                -0.13654561340808868,\n                0.07880350947380066,\n                -0.5477664470672607,\n                0.3222714066505432,\n                0.20942415297031403,\n                0.24103082716464996,\n                0.0010369870578870177,\n                0.07903929054737091,\n                -0.7326748967170715,\n                -0.018193775787949562,\n                0.14304082095623016,\n                -0.5774063467979431,\n                0.04663686826825142,\n                -0.38176771998405457,\n                -0.09514370560646057,\n                -0.2693597674369812,\n                -0.1760627031326294,\n                0.10530556738376617,\n                -0.0075754462741315365,\n                -0.2862960696220398,\n                -0.12947608530521393,\n                -0.340509295463562,\n                0.31726163625717163,\n                0.16130729019641876,\n                0.4510969817638397,\n                -0.1292114555835724,\n                0.5841554403305054,\n                -0.1322399079799652,\n                0.135287806391716,\n                -0.2676231265068054,\n                0.038900963962078094,\n                0.28827574849128723,\n                0.1366054266691208,\n                0.17256863415241241,\n                -0.053452786058187485\n              ],\n              [\n                -0.3961889147758484,\n                -0.46040841937065125,\n                0.2196914255619049,\n                0.05046434327960014,\n                -0.33147358894348145,\n                0.2684059143066406,\n                0.0474977120757103,\n                -0.017679188400506973,\n                0.26660850644111633,\n                -0.09793423116207123,\n                0.1641201674938202,\n                0.10482963919639587,\n                0.22193235158920288,\n                0.10818701237440109,\n                -0.10824785381555557,\n                -0.019321439787745476,\n                -0.014227773062884808,\n                -0.020141875371336937,\n                -0.13751612603664398,\n                0.05705909803509712,\n                -0.22966720163822174,\n                -0.18884046375751495,\n                0.20037731528282166,\n                -0.29322007298469543,\n                0.0446658581495285,\n                0.07818875461816788,\n                -0.19028626382350922,\n                0.13077665865421295,\n                0.08423267304897308,\n                0.008018983528017998,\n                -0.519278883934021,\n                0.21920281648635864,\n                0.05410536378622055,\n                0.2532445192337036,\n                -0.20290900766849518,\n                0.22744540870189667,\n                -0.1427524983882904,\n                0.13327495753765106,\n                -0.05179793760180473,\n                0.006791737861931324,\n                0.09608040004968643,\n                0.20773813128471375,\n                0.24394957721233368,\n                0.1281476765871048,\n                -0.026160025969147682,\n                0.03990846127271652,\n                -0.24915944039821625,\n                0.0083148293197155,\n                0.2173263132572174,\n                -0.05482521653175354\n              ],\n              [\n                -0.46897947788238525,\n                0.1939813494682312,\n                -0.032292917370796204,\n                0.31936466693878174,\n                -0.13258635997772217,\n                0.08417251706123352,\n                0.06938806176185608,\n                -0.5968083143234253,\n                0.15931163728237152,\n                0.025160523131489754,\n                0.10951870679855347,\n                -0.06244724616408348,\n                -0.028845395892858505,\n                0.0747387632727623,\n                0.10975819081068039,\n                0.024257972836494446,\n                0.14591236412525177,\n                0.18429380655288696,\n                -0.33624523878097534,\n                0.33360517024993896,\n                -0.24695807695388794,\n                -0.4387824237346649,\n                0.0842091292142868,\n                -0.03296291083097458,\n                0.14091207087039948,\n                -0.004438221920281649,\n                0.06744446605443954,\n                -0.06671259552240372,\n                -0.0782000795006752,\n                0.03303195908665657,\n                -0.25879067182540894,\n                0.09065626561641693,\n                0.12398450076580048,\n                0.9408542513847351,\n                0.03293769806623459,\n                0.09764484316110611,\n                -0.0036100498400628567,\n                -0.09164755046367645,\n                0.10436680167913437,\n                0.0855497345328331,\n                0.10866373777389526,\n                0.0606394000351429,\n                -0.15805990993976593,\n                0.04706179350614548,\n                -0.18564823269844055,\n                0.32456475496292114,\n                0.013767565600574017,\n                0.2760608196258545,\n                -0.08011768013238907,\n                0.15603870153427124\n              ],\n              [\n                0.06530220061540604,\n                -0.1938244104385376,\n                -0.06791539490222931,\n                0.1684546023607254,\n                0.588080108165741,\n                -0.05347514525055885,\n                -0.027186203747987747,\n                -0.6285409927368164,\n                -0.03389909118413925,\n                -0.07645681500434875,\n                -0.1571633219718933,\n                0.3467240035533905,\n                -0.08898109942674637,\n                -0.1689881533384323,\n                0.024456139653921127,\n                -0.25249725580215454,\n                -0.31211522221565247,\n                0.08157378435134888,\n                -0.0979478731751442,\n                -0.12501373887062073,\n                -0.25109243392944336,\n                -0.15055960416793823,\n                -0.13126854598522186,\n                0.07118936628103256,\n                0.01431370247155428,\n                0.10147762298583984,\n                0.08022072166204453,\n                -0.18964862823486328,\n                -0.10489154607057571,\n                -0.31562647223472595,\n                -0.20392489433288574,\n                0.09224116057157516,\n                -0.06302554905414581,\n                0.40176886320114136,\n                -0.17208777368068695,\n                0.05694785341620445,\n                -0.13730579614639282,\n                -0.10423211753368378,\n                -0.15937544405460358,\n                -0.046164944767951965,\n                0.04765899106860161,\n                0.0626683160662651,\n                -0.42633363604545593,\n                -0.019328445196151733,\n                0.23325718939304352,\n                0.0571424625813961,\n                -0.1589375138282776,\n                0.5786423087120056,\n                0.02490766905248165,\n                -0.13667245209217072\n              ],\n              [\n                -0.28771018981933594,\n                0.11562525480985641,\n                0.13038471341133118,\n                0.05946330353617668,\n                0.23543591797351837,\n                0.24393689632415771,\n                -0.227728009223938,\n                -0.2421497106552124,\n                0.25773248076438904,\n                0.009055891074240208,\n                0.042949821799993515,\n                -0.13164646923542023,\n                0.01785208471119404,\n                0.032690197229385376,\n                0.08964651823043823,\n                -0.2653658986091614,\n                -0.2937764823436737,\n                -0.04746067896485329,\n                -0.0561523400247097,\n                -0.6565371155738831,\n                0.12242861092090607,\n                0.2353164255619049,\n                -0.11213594675064087,\n                0.10395673662424088,\n                -0.02287227101624012,\n                -0.11630058288574219,\n                0.10504776239395142,\n                -0.0014501643599942327,\n                -0.007164937909692526,\n                -0.022275321185588837,\n                -0.1653028130531311,\n                0.022580260410904884,\n                0.2609707713127136,\n                -0.7206534147262573,\n                -0.2215133160352707,\n                -0.5918362140655518,\n                -0.0035360658075660467,\n                -0.1185218021273613,\n                0.07292405515909195,\n                -0.2265658974647522,\n                -0.05880394205451012,\n                0.17773087322711945,\n                0.5721856355667114,\n                0.14834612607955933,\n                -0.011880542151629925,\n                0.16113513708114624,\n                -0.2819710969924927,\n                0.14223423600196838,\n                -0.11073827743530273,\n                -0.14627468585968018\n              ],\n              [\n                0.040821321308612823,\n                -0.20160777866840363,\n                -0.05570583418011665,\n                0.004754761699587107,\n                -0.5616325736045837,\n                -0.05090862140059471,\n                0.1670302450656891,\n                -0.19239507615566254,\n                -0.48343104124069214,\n                -0.17791041731834412,\n                -0.22411267459392548,\n                -0.047452736645936966,\n                -0.1549903005361557,\n                -0.11515803635120392,\n                -0.08002974838018417,\n                -0.11467361450195312,\n                -0.1606924831867218,\n                -0.21562559902668,\n                0.23508183658123016,\n                0.15265709161758423,\n                -0.2348959594964981,\n                0.008862434886395931,\n                -0.04552668705582619,\n                -0.2892385721206665,\n                -0.0220014750957489,\n                -0.07863172143697739,\n                -0.0968034639954567,\n                -0.028949622064828873,\n                -0.12328556925058365,\n                -0.17934443056583405,\n                -0.11296600848436356,\n                -0.11576605588197708,\n                0.022461242973804474,\n                0.3400169312953949,\n                -0.2392335832118988,\n                -0.17141087353229523,\n                -0.13378208875656128,\n                -0.21247243881225586,\n                -0.1951640099287033,\n                -0.15454521775245667,\n                0.04504670202732086,\n                0.09864577651023865,\n                0.13875499367713928,\n                0.3617118299007416,\n                -0.05337328463792801,\n                0.32986173033714294,\n                0.32204610109329224,\n                0.26012060046195984,\n                -0.01748303696513176,\n                -0.1285756379365921\n              ],\n              [\n                -0.36107540130615234,\n                -0.23316925764083862,\n                -0.0005261289770714939,\n                0.23076489567756653,\n                0.43435776233673096,\n                -0.07070977240800858,\n                -0.3080746531486511,\n                0.44530999660491943,\n                -0.21137793362140656,\n                -0.003932056948542595,\n                -0.01840481534600258,\n                0.14063875377178192,\n                -0.08314060419797897,\n                -0.16481882333755493,\n                0.004022188019007444,\n                -0.07609113305807114,\n                -0.03963321074843407,\n                0.6156153082847595,\n                0.11877110600471497,\n                -0.5687031745910645,\n                -0.35158517956733704,\n                -0.36379000544548035,\n                0.23329587280750275,\n                0.23173342645168304,\n                0.300642192363739,\n                0.08068472146987915,\n                -0.08311239629983902,\n                -0.2587467133998871,\n                0.12286464869976044,\n                -0.05061759427189827,\n                0.015050970017910004,\n                0.31135866045951843,\n                -0.14364568889141083,\n                0.2513780891895294,\n                -0.0556335411965847,\n                -0.27911168336868286,\n                -0.12411349266767502,\n                -0.02942589484155178,\n                -0.05207731947302818,\n                0.2911379337310791,\n                0.0683738961815834,\n                0.11167559027671814,\n                -0.09840988367795944,\n                0.12265720218420029,\n                -0.08435030281543732,\n                -0.16036835312843323,\n                -0.11171884834766388,\n                0.33031022548675537,\n                -0.09275522083044052,\n                -0.035640649497509\n              ],\n              [\n                0.3510441184043884,\n                -0.035518430173397064,\n                0.1440327763557434,\n                -0.006552477832883596,\n                -0.19958235323429108,\n                -0.08374685049057007,\n                0.2199011594057083,\n                -0.811407208442688,\n                -0.04289533570408821,\n                0.13828562200069427,\n                0.07134454697370529,\n                -0.5139862298965454,\n                0.020516257733106613,\n                -0.03293876349925995,\n                0.17269454896450043,\n                0.21152111887931824,\n                0.16204194724559784,\n                0.23592056334018707,\n                -0.16266214847564697,\n                0.0677967369556427,\n                0.31394630670547485,\n                0.14992383122444153,\n                0.0345202200114727,\n                -0.4013928771018982,\n                0.03696933388710022,\n                -0.1786099076271057,\n                -0.07471805810928345,\n                0.2389640063047409,\n                0.05816379189491272,\n                0.05141192674636841,\n                0.09221195429563522,\n                0.07856535166501999,\n                0.0026714038103818893,\n                -0.36936143040657043,\n                -0.18525612354278564,\n                -0.5446624159812927,\n                0.1133439689874649,\n                -0.09709855169057846,\n                -0.10296037048101425,\n                0.26518115401268005,\n                0.16124682128429413,\n                -0.005081284791231155,\n                -0.6138672828674316,\n                0.16156505048274994,\n                0.3464154005050659,\n                0.1769765019416809,\n                -0.38684895634651184,\n                -0.5983367562294006,\n                -0.026411525905132294,\n                0.07318852841854095\n              ],\n              [\n                -0.7498357892036438,\n                0.21063445508480072,\n                -0.20563901960849762,\n                0.12044817209243774,\n                -0.04553386941552162,\n                0.2572336792945862,\n                0.20674782991409302,\n                -0.06195385009050369,\n                0.18859727680683136,\n                0.007550694048404694,\n                -0.08987760543823242,\n                -0.027995649725198746,\n                0.05885208025574684,\n                -0.07259200513362885,\n                -0.06576356291770935,\n                -0.07104557752609253,\n                -0.0628618597984314,\n                0.4630751609802246,\n                0.12426687031984329,\n                -0.11534039676189423,\n                0.09168915450572968,\n                -0.2538411319255829,\n                -0.030915604904294014,\n                0.08649176359176636,\n                -0.14069928228855133,\n                0.0226445235311985,\n                0.20883843302726746,\n                0.042885906994342804,\n                0.0958562046289444,\n                -0.0032906290143728256,\n                -0.2657560110092163,\n                0.041179344058036804,\n                0.3829485774040222,\n                0.07227688282728195,\n                0.09059078246355057,\n                0.13119883835315704,\n                -0.09330447763204575,\n                -0.33125561475753784,\n                -0.2283739298582077,\n                0.09303531050682068,\n                -0.18821220099925995,\n                0.06556814163923264,\n                0.3424741327762604,\n                0.1392085701227188,\n                -0.03409351781010628,\n                0.2728283703327179,\n                0.14966948330402374,\n                0.3203960657119751,\n                -0.17096266150474548,\n                -0.010720270685851574\n              ],\n              [\n                0.274734228849411,\n                0.07605774700641632,\n                0.05234622582793236,\n                0.1289052963256836,\n                0.27108338475227356,\n                -0.13771401345729828,\n                0.19547876715660095,\n                -0.1534997969865799,\n                0.4250090718269348,\n                0.015761446207761765,\n                0.1377006471157074,\n                -0.05734637379646301,\n                -0.014811448752880096,\n                -0.21452313661575317,\n                -0.08140499144792557,\n                0.08328989148139954,\n                0.09046290814876556,\n                -0.19140201807022095,\n                0.07533324509859085,\n                0.14998066425323486,\n                0.24585434794425964,\n                0.15308083593845367,\n                -0.1229090616106987,\n                0.18071803450584412,\n                0.25310906767845154,\n                -0.24070844054222107,\n                -0.010410037823021412,\n                0.06309834122657776,\n                -0.1297897845506668,\n                -0.16657744348049164,\n                0.37421631813049316,\n                0.2173442393541336,\n                -0.2538306415081024,\n                -0.8775382041931152,\n                -0.07950269430875778,\n                -0.15970011055469513,\n                0.10895555466413498,\n                0.008314157836139202,\n                0.14638255536556244,\n                0.072003573179245,\n                0.6190196871757507,\n                0.02677847445011139,\n                0.2699536085128784,\n                -0.19142070412635803,\n                -0.2592925727367401,\n                -0.2565824091434479,\n                -0.262987345457077,\n                -0.1489066481590271,\n                -0.0432773195207119,\n                -0.08126294612884521\n              ],\n              [\n                -0.6639822125434875,\n                0.0880877748131752,\n                -0.005751533899456263,\n                -0.19255366921424866,\n                0.3057144582271576,\n                -0.08999431878328323,\n                -0.019433338195085526,\n                -0.1559668332338333,\n                -0.15749216079711914,\n                -0.011301230639219284,\n                -0.11821027845144272,\n                -0.30145376920700073,\n                0.006112752482295036,\n                -0.1560450941324234,\n                -0.5127982497215271,\n                0.40875670313835144,\n                0.13342377543449402,\n                -0.04711974039673805,\n                0.27859559655189514,\n                -0.25916099548339844,\n                -0.10510764271020889,\n                -0.041911106556653976,\n                0.037033870816230774,\n                -0.20566396415233612,\n                0.05327753722667694,\n                -0.1296263337135315,\n                -0.32234781980514526,\n                0.0389278419315815,\n                -0.40813302993774414,\n                -0.02087150141596794,\n                -0.19934184849262238,\n                -0.07899387180805206,\n                -0.25175032019615173,\n                -0.2088051736354828,\n                0.13817232847213745,\n                -0.04268897697329521,\n                -0.42443978786468506,\n                -0.4097912311553955,\n                -0.04090137407183647,\n                -0.15951122343540192,\n                0.04449190944433212,\n                0.21488967537879944,\n                0.12114228308200836,\n                0.09004763513803482,\n                -0.006397236604243517,\n                -0.07948800921440125,\n                0.19370833039283752,\n                0.683956503868103,\n                -0.11428768187761307,\n                -0.10317038744688034\n              ],\n              [\n                -0.5914733409881592,\n                -0.4672893285751343,\n                0.12114158272743225,\n                0.00020536499505396932,\n                -0.2679958939552307,\n                0.26662468910217285,\n                0.1092640832066536,\n                -0.21656334400177002,\n                0.17207543551921844,\n                0.08223824948072433,\n                0.001051013940013945,\n                0.0034190842416137457,\n                0.13344737887382507,\n                -0.5096179842948914,\n                -0.006541278213262558,\n                0.08032097667455673,\n                -0.005550340749323368,\n                -0.04771978780627251,\n                -0.21063093841075897,\n                0.052590880542993546,\n                0.2812643349170685,\n                0.15709182620048523,\n                -0.13493511080741882,\n                -0.48999109864234924,\n                0.15144944190979004,\n                -0.10234226286411285,\n                0.2640151381492615,\n                0.18310357630252838,\n                0.17227308452129364,\n                0.111046701669693,\n                -0.5058703422546387,\n                0.07171519845724106,\n                0.02478756196796894,\n                -0.045091744512319565,\n                -0.2756809890270233,\n                -0.28741583228111267,\n                -0.22533218562602997,\n                -0.33518844842910767,\n                -0.46407124400138855,\n                0.41094958782196045,\n                0.031681012362241745,\n                0.19160377979278564,\n                0.22435687482357025,\n                0.21299022436141968,\n                0.07534463703632355,\n                0.3006431758403778,\n                0.4345487356185913,\n                0.170183464884758,\n                0.19991670548915863,\n                0.0319303423166275\n              ],\n              [\n                -0.21427439153194427,\n                -0.20036062598228455,\n                0.07794656604528427,\n                -0.14284713566303253,\n                0.25140538811683655,\n                0.3854665160179138,\n                0.46422120928764343,\n                0.36879846453666687,\n                -0.362587034702301,\n                -0.17855657637119293,\n                -0.19077561795711517,\n                -0.2539205253124237,\n                -0.1934506893157959,\n                -0.2961663603782654,\n                -0.08020146191120148,\n                -0.1223592534661293,\n                -0.09612061083316803,\n                -0.08487864583730698,\n                0.030782759189605713,\n                0.13439634442329407,\n                -0.016591235995292664,\n                -0.23079997301101685,\n                -0.3025470972061157,\n                0.2626320719718933,\n                -0.09485653042793274,\n                -0.09040284156799316,\n                -0.31883135437965393,\n                -0.09886917471885681,\n                0.06538306176662445,\n                -0.15228867530822754,\n                -0.1643630415201187,\n                -0.31876298785209656,\n                0.13331247866153717,\n                0.09882406890392303,\n                -0.07007796317338943,\n                0.28392279148101807,\n                -0.2196052223443985,\n                -0.3272660970687866,\n                -0.08426976948976517,\n                -0.20013859868049622,\n                -0.3208335340023041,\n                0.014314942993223667,\n                -0.2237250953912735,\n                0.12013121694326401,\n                -0.0037638030480593443,\n                0.3212618827819824,\n                -0.2214321345090866,\n                0.1737927347421646,\n                -0.24851001799106598,\n                -0.2322719395160675\n              ],\n              [\n                -0.11294197291135788,\n                0.22274744510650635,\n                -0.2330183982849121,\n                -0.7324397563934326,\n                0.4676364064216614,\n                -0.07979191094636917,\n                0.7319750785827637,\n                0.12972930073738098,\n                0.021024459972977638,\n                0.36666375398635864,\n                0.23946106433868408,\n                0.02072736620903015,\n                0.2554115653038025,\n                -0.04969428852200508,\n                -0.3307410478591919,\n                0.18114912509918213,\n                0.028270814567804337,\n                0.015179718844592571,\n                -0.12330888956785202,\n                -0.06808768212795258,\n                0.11298701167106628,\n                0.3111850619316101,\n                0.15453025698661804,\n                -0.036064568907022476,\n                -0.15355663001537323,\n                -0.08854296803474426,\n                -0.1556462049484253,\n                0.13265253603458405,\n                -0.1301099956035614,\n                0.1300511360168457,\n                0.05666785687208176,\n                0.05629177764058113,\n                0.02590491622686386,\n                0.4094301164150238,\n                -0.044398609548807144,\n                0.452364057302475,\n                0.3546171188354492,\n                -0.13299521803855896,\n                -0.1892523318529129,\n                0.25428497791290283,\n                0.04711531475186348,\n                0.021106703206896782,\n                0.6154316067695618,\n                -0.22931945323944092,\n                -0.16165129840373993,\n                -0.11139032244682312,\n                0.09658266603946686,\n                -0.1307058334350586,\n                -0.049249663949012756,\n                0.32638129591941833\n              ],\n              [\n                -0.46581366658210754,\n                0.03137717396020889,\n                -0.1218806728720665,\n                0.09832312911748886,\n                -0.009367070160806179,\n                -0.11927109211683273,\n                0.7112864851951599,\n                -0.734951376914978,\n                -0.07221429795026779,\n                -0.011268037371337414,\n                0.04569047689437866,\n                0.04369886964559555,\n                -0.032484911382198334,\n                0.09794636815786362,\n                -0.32328906655311584,\n                0.1170458123087883,\n                0.12185964733362198,\n                0.19422496855258942,\n                0.17055435478687286,\n                -0.003093091771006584,\n                -0.5446919202804565,\n                0.12102081626653671,\n                0.027012798935174942,\n                0.17547154426574707,\n                0.055674947798252106,\n                -0.13497957587242126,\n                0.17319534718990326,\n                -0.07802064716815948,\n                -0.19643627107143402,\n                0.04205043241381645,\n                0.213498055934906,\n                0.07426552474498749,\n                0.02582940086722374,\n                0.451744019985199,\n                -0.022335831075906754,\n                0.3766533136367798,\n                -0.4091492295265198,\n                -0.015331956557929516,\n                -0.05918412283062935,\n                -0.06600836664438248,\n                0.14513570070266724,\n                -0.035239510238170624,\n                0.40115639567375183,\n                -0.20591595768928528,\n                -0.23105399310588837,\n                -0.07920513302087784,\n                -0.16204708814620972,\n                0.0754992663860321,\n                0.07525797933340073,\n                0.09106867760419846\n              ],\n              [\n                0.33886078000068665,\n                -0.15836355090141296,\n                0.11545507609844208,\n                -0.307790070772171,\n                -0.3583011329174042,\n                -0.4114040434360504,\n                -0.17187818884849548,\n                -0.5699397325515747,\n                0.1770302802324295,\n                0.05155625194311142,\n                0.25689783692359924,\n                -0.07145075500011444,\n                -0.39431020617485046,\n                0.13408873975276947,\n                -0.047830283641815186,\n                -0.3556029498577118,\n                -0.1249995082616806,\n                0.33150044083595276,\n                0.05355924740433693,\n                0.0580989271402359,\n                -0.15124715864658356,\n                0.1805446445941925,\n                -0.27744537591934204,\n                -0.05419614911079407,\n                -0.14440882205963135,\n                0.1770683228969574,\n                0.24545757472515106,\n                -0.026452191174030304,\n                -0.008723985403776169,\n                0.038196153938770294,\n                0.03161676228046417,\n                0.1373031884431839,\n                -0.4143252372741699,\n                0.3969142436981201,\n                -0.06792732328176498,\n                -0.1232592761516571,\n                -0.16646811366081238,\n                0.06960832327604294,\n                0.03154929354786873,\n                -0.123887799680233,\n                -0.46353864669799805,\n                -0.3144764304161072,\n                -0.42628389596939087,\n                0.5393332242965698,\n                0.2493637055158615,\n                -0.12768633663654327,\n                -0.0760553702712059,\n                -0.5405345559120178,\n                -0.08851554989814758,\n                0.03100821003317833\n              ],\n              [\n                -0.9735671877861023,\n                -0.04137355089187622,\n                0.35126975178718567,\n                -0.10205425322055817,\n                -0.025996064767241478,\n                -0.0580744668841362,\n                -0.1881583034992218,\n                0.16081131994724274,\n                0.13075855374336243,\n                0.07091133296489716,\n                -0.006204509641975164,\n                0.11568878591060638,\n                -0.08243140578269958,\n                0.07040645182132721,\n                0.09325511008501053,\n                0.028905700892210007,\n                0.15887966752052307,\n                0.10187280178070068,\n                0.15174755454063416,\n                0.35554617643356323,\n                -0.3071753978729248,\n                -0.1548262983560562,\n                0.1684771478176117,\n                0.27995574474334717,\n                0.003711925121024251,\n                -0.06057242304086685,\n                0.02376418001949787,\n                -0.05478191003203392,\n                -0.1576126664876938,\n                -0.2858216464519501,\n                -0.2798856496810913,\n                0.2505143880844116,\n                0.23728787899017334,\n                -0.2534143328666687,\n                -0.2322133481502533,\n                -0.5443998575210571,\n                -0.04497148469090462,\n                0.07133816182613373,\n                0.0013389517553150654,\n                0.08474922180175781,\n                0.08366569876670837,\n                -0.0008644083864055574,\n                -0.08997055888175964,\n                0.12731218338012695,\n                0.24673467874526978,\n                -0.10185850411653519,\n                0.3059315085411072,\n                -0.06534984707832336,\n                -0.04846072942018509,\n                -0.01469405833631754\n              ],\n              [\n                0.4039386510848999,\n                -0.13471591472625732,\n                0.027628807350993156,\n                -0.034723374992609024,\n                0.17345017194747925,\n                0.08883119374513626,\n                -0.609667956829071,\n                -0.6436949968338013,\n                0.182982936501503,\n                0.042495593428611755,\n                -0.024463780224323273,\n                0.05978408828377724,\n                0.03751043230295181,\n                -0.029150333255529404,\n                0.009013879112899303,\n                -0.08349311351776123,\n                -0.22251936793327332,\n                -0.14870254695415497,\n                0.14848963916301727,\n                -0.20075109601020813,\n                0.16211113333702087,\n                -0.7619535326957703,\n                0.18761400878429413,\n                0.2754133343696594,\n                -0.023325514048337936,\n                -0.2032531052827835,\n                0.08467560261487961,\n                -0.2319343388080597,\n                -0.05421333387494087,\n                -0.14198003709316254,\n                -0.12662072479724884,\n                0.1968347579240799,\n                -0.03838423267006874,\n                0.3028755784034729,\n                0.20501698553562164,\n                0.24259786307811737,\n                0.010480734519660473,\n                -0.13123713433742523,\n                0.09066292643547058,\n                0.013406556099653244,\n                -0.14184661209583282,\n                -0.008003261871635914,\n                -0.49010083079338074,\n                -0.42149728536605835,\n                0.05045846849679947,\n                0.16863785684108734,\n                -0.34566766023635864,\n                0.38461145758628845,\n                0.04986937716603279,\n                0.01650077849626541\n              ],\n              [\n                -0.12281874567270279,\n                -0.06224335357546806,\n                0.01965164951980114,\n                0.036817919462919235,\n                0.01696583442389965,\n                0.08764713257551193,\n                0.025815490633249283,\n                -0.17511172592639923,\n                -0.013873397372663021,\n                -0.0496794767677784,\n                -0.01390866655856371,\n                0.2167406976222992,\n                -0.016514839604496956,\n                -0.13962915539741516,\n                -0.12483185529708862,\n                -0.055758003145456314,\n                -0.11738940328359604,\n                0.02215925231575966,\n                0.11241894215345383,\n                0.1515226513147354,\n                -0.053121067583560944,\n                -0.027850231155753136,\n                -0.030644873157143593,\n                -0.014299597591161728,\n                0.09084217250347137,\n                -0.006216452457010746,\n                -0.2358647584915161,\n                -0.15631726384162903,\n                0.015916109085083008,\n                -0.16677819192409515,\n                -0.01726796291768551,\n                -0.10054439306259155,\n                0.026055416092276573,\n                0.3712286651134491,\n                -0.3131727874279022,\n                0.18314386904239655,\n                0.05813062936067581,\n                -0.014103861525654793,\n                -0.04529847577214241,\n                -0.09722699970006943,\n                0.11625651270151138,\n                0.18894419074058533,\n                -0.26961812376976013,\n                -0.05369789898395538,\n                -0.11172289401292801,\n                0.0071302978321909904,\n                -0.09918048977851868,\n                0.24546149373054504,\n                -0.06613007187843323,\n                -0.05349305644631386\n              ],\n              [\n                -0.10706257820129395,\n                -0.014513627626001835,\n                -0.01650029793381691,\n                -0.022345444187521935,\n                -0.2375548928976059,\n                0.14926046133041382,\n                0.05824356526136398,\n                -0.13587728142738342,\n                -0.5190252065658569,\n                -0.23528948426246643,\n                -0.19684772193431854,\n                0.08985518664121628,\n                -0.09019802510738373,\n                -0.16498494148254395,\n                -0.16400064527988434,\n                -0.17665381729602814,\n                -0.1495702862739563,\n                0.04063289985060692,\n                0.018178608268499374,\n                0.0679873526096344,\n                -0.16714423894882202,\n                -0.036891527473926544,\n                -0.10445253551006317,\n                -0.3927890360355377,\n                -0.17704632878303528,\n                -0.1166594922542572,\n                0.012740793637931347,\n                -0.14439982175827026,\n                -0.1996820718050003,\n                0.10451915115118027,\n                -0.03469640761613846,\n                -0.0004551775346044451,\n                -0.07949463278055191,\n                -0.039038173854351044,\n                -0.15445175766944885,\n                -0.1380963772535324,\n                -0.1047859638929367,\n                0.033395104110240936,\n                -0.1391402930021286,\n                -0.11430581659078598,\n                -0.16138480603694916,\n                -0.013521285727620125,\n                0.16289760172367096,\n                0.06026143580675125,\n                -0.08880285918712616,\n                0.27378812432289124,\n                -0.7299013137817383,\n                0.12981179356575012,\n                -0.21602566540241241,\n                -0.03437846899032593\n              ],\n              [\n                -0.07186418026685715,\n                -0.055654894560575485,\n                0.023081880062818527,\n                -0.014171147719025612,\n                0.3718494772911072,\n                0.15714910626411438,\n                0.5212790966033936,\n                -0.057541623711586,\n                -0.1241057738661766,\n                0.03125312924385071,\n                -0.07598546147346497,\n                -0.046080268919467926,\n                0.2779327929019928,\n                -0.012316126376390457,\n                -0.2595909833908081,\n                -0.1340722143650055,\n                0.05720413103699684,\n                -0.5329489707946777,\n                0.11871278285980225,\n                0.2923130691051483,\n                -0.11652947962284088,\n                0.443386971950531,\n                0.11399097740650177,\n                -0.39724382758140564,\n                0.0910022184252739,\n                0.07382900267839432,\n                -0.02331758663058281,\n                0.2986155152320862,\n                -0.17429904639720917,\n                -0.02758517861366272,\n                -0.005745948292315006,\n                -0.06663986295461655,\n                -0.03176482766866684,\n                0.5119363069534302,\n                0.04511744901537895,\n                0.4174656867980957,\n                -0.2300683706998825,\n                -0.01011301577091217,\n                0.21833020448684692,\n                -0.052029334008693695,\n                0.1804760843515396,\n                0.4034663438796997,\n                0.2590266168117523,\n                -0.2084108591079712,\n                0.06783833354711533,\n                -0.13150788843631744,\n                0.14485538005828857,\n                0.017271699383854866,\n                -0.05304720252752304,\n                -0.057500772178173065\n              ],\n              [\n                -0.6604385375976562,\n                -0.06986501812934875,\n                0.15083815157413483,\n                0.2219560742378235,\n                0.3681388199329376,\n                -0.24392996728420258,\n                -0.2625875174999237,\n                0.9658703804016113,\n                -0.06555946171283722,\n                -0.04877181351184845,\n                -0.09979092329740524,\n                0.19463306665420532,\n                -0.27455735206604004,\n                0.13402371108531952,\n                -0.37881940603256226,\n                0.10094164311885834,\n                0.13876503705978394,\n                -0.447234183549881,\n                -0.1345466673374176,\n                0.12663979828357697,\n                -0.24984702467918396,\n                -0.035014744848012924,\n                -0.039633940905332565,\n                -0.210527703166008,\n                0.20277711749076843,\n                0.2858479917049408,\n                0.08572372794151306,\n                0.036232221871614456,\n                -0.36387720704078674,\n                0.06315044313669205,\n                -0.30623769760131836,\n                -0.06741265207529068,\n                -0.30438026785850525,\n                0.09858912974596024,\n                0.21076439321041107,\n                0.40742477774620056,\n                -0.43515026569366455,\n                -0.157973974943161,\n                0.3636869788169861,\n                -0.24309775233268738,\n                -0.09181814640760422,\n                0.2706977427005768,\n                0.36064156889915466,\n                -0.036217812448740005,\n                -0.07993815094232559,\n                0.07931309938430786,\n                0.45539867877960205,\n                0.07486847043037415,\n                -0.024540740996599197,\n                -0.03841240331530571\n              ],\n              [\n                -0.307781845331192,\n                -0.034045323729515076,\n                -0.0005269395187497139,\n                -0.19310402870178223,\n                -0.026454007253050804,\n                0.18268531560897827,\n                -0.1673271805047989,\n                -0.09658391773700714,\n                -0.11760496348142624,\n                0.10150157660245895,\n                0.11351918429136276,\n                -0.13518212735652924,\n                -0.09026109427213669,\n                -0.018844613805413246,\n                -0.14594362676143646,\n                -0.09376624971628189,\n                0.06628540903329849,\n                0.16020053625106812,\n                -0.08841736614704132,\n                0.3251096308231354,\n                -0.06996630132198334,\n                -0.07898686081171036,\n                -0.010989238508045673,\n                0.1558353155851364,\n                -0.056911613792181015,\n                -0.23423999547958374,\n                0.18207210302352905,\n                -0.22104330360889435,\n                -0.017040472477674484,\n                -0.3210408091545105,\n                -0.15926975011825562,\n                -0.19055554270744324,\n                0.19050413370132446,\n                -0.10608169436454773,\n                -0.02338770218193531,\n                -0.12002378702163696,\n                -0.17946694791316986,\n                -0.14818662405014038,\n                -0.16118058562278748,\n                -0.08301952481269836,\n                0.09531703591346741,\n                0.13841426372528076,\n                -0.060536179691553116,\n                -0.032108038663864136,\n                0.12043174356222153,\n                0.047239068895578384,\n                -0.01471246499568224,\n                -0.08050220459699631,\n                0.07315975427627563,\n                -0.046004388481378555\n              ],\n              [\n                -0.28982484340667725,\n                -0.08965249359607697,\n                0.10088396072387695,\n                0.044090624898672104,\n                0.376592218875885,\n                -0.08558747172355652,\n                0.3574191927909851,\n                -0.7516403198242188,\n                0.22832489013671875,\n                0.05686384066939354,\n                -0.07454803586006165,\n                -0.24541698396205902,\n                -0.14552736282348633,\n                0.14130418002605438,\n                0.048876725137233734,\n                -0.12436597049236298,\n                0.15423622727394104,\n                0.20786859095096588,\n                0.2391236424446106,\n                0.2524341642856598,\n                -0.6358815431594849,\n                0.03661400079727173,\n                -0.08577727526426315,\n                -0.4446631073951721,\n                0.15542654693126678,\n                -0.15350627899169922,\n                -0.0007989830919541419,\n                0.07778619229793549,\n                0.030498310923576355,\n                -0.1517181098461151,\n                -0.061767566949129105,\n                0.07415859401226044,\n                0.10067164897918701,\n                0.6278400421142578,\n                -0.2585182189941406,\n                0.05506706237792969,\n                0.08913308382034302,\n                -0.08907200396060944,\n                0.11066343635320663,\n                -0.4256574809551239,\n                -0.09891491383314133,\n                0.10912012308835983,\n                -0.26256752014160156,\n                -0.2248126119375229,\n                0.03860471025109291,\n                0.09886275976896286,\n                0.17226631939411163,\n                -0.025798451155424118,\n                -0.09317366033792496,\n                -0.05684979632496834\n              ],\n              [\n                -0.37888529896736145,\n                -0.16662314534187317,\n                0.07877650856971741,\n                0.33243048191070557,\n                0.33421242237091064,\n                0.04229206219315529,\n                0.06698255240917206,\n                0.5107118487358093,\n                -0.19182775914669037,\n                -0.17274397611618042,\n                0.11606128513813019,\n                -0.03831028565764427,\n                0.06799456477165222,\n                -0.23126263916492462,\n                0.17431028187274933,\n                0.1489490568637848,\n                0.162456214427948,\n                -0.2882015109062195,\n                -0.2679458260536194,\n                0.0428243950009346,\n                -0.39375025033950806,\n                -0.2358098179101944,\n                0.29235535860061646,\n                0.15713715553283691,\n                0.1423434019088745,\n                0.25796201825141907,\n                0.2916123569011688,\n                0.13723468780517578,\n                0.15418045222759247,\n                -0.2943059206008911,\n                -0.4850558638572693,\n                0.3461304306983948,\n                0.1502530425786972,\n                -0.43227624893188477,\n                0.12890367209911346,\n                0.26796016097068787,\n                -0.11738845705986023,\n                0.1978311687707901,\n                -0.19749420881271362,\n                0.06924550980329514,\n                0.07652833312749863,\n                0.4045135974884033,\n                0.571014404296875,\n                0.07547017186880112,\n                -0.2980058193206787,\n                0.02181682363152504,\n                -0.3753744959831238,\n                0.25161999464035034,\n                -0.08938631415367126,\n                -0.03211525082588196\n              ],\n              [\n                -0.49257224798202515,\n                -0.09388156235218048,\n                0.0012350549222901464,\n                0.18059146404266357,\n                0.0011243124026805162,\n                -0.20821326971054077,\n                0.20082233846187592,\n                0.4148622155189514,\n                -0.06775477528572083,\n                -0.058037225157022476,\n                -0.10307089239358902,\n                -0.10048248618841171,\n                -0.05047061666846275,\n                0.12638187408447266,\n                -0.4556204080581665,\n                -0.46558642387390137,\n                -0.0772886872291565,\n                -0.6759362816810608,\n                -0.18581464886665344,\n                0.11267050355672836,\n                -0.15583279728889465,\n                -0.16388441622257233,\n                0.23114919662475586,\n                -0.3857767581939697,\n                -0.06667602807283401,\n                0.01988230086863041,\n                0.21008752286434174,\n                0.27807095646858215,\n                -0.41078582406044006,\n                -0.022135812789201736,\n                -0.3471928834915161,\n                0.20695865154266357,\n                -0.08093701303005219,\n                -0.22231993079185486,\n                -0.08899680525064468,\n                0.06807203590869904,\n                -0.25230348110198975,\n                -0.0014619325520470738,\n                0.08896080404520035,\n                -0.3568987548351288,\n                0.11678487062454224,\n                1.2827746868133545,\n                -0.014346606098115444,\n                0.08429359644651413,\n                -0.0919218435883522,\n                0.07523822784423828,\n                -0.04129438474774361,\n                0.11294715851545334,\n                0.24544107913970947,\n                -0.042737267911434174\n              ],\n              [\n                -0.48072704672813416,\n                -0.18810005486011505,\n                0.0374588705599308,\n                -0.05727287009358406,\n                -0.012722119688987732,\n                0.14065587520599365,\n                -0.003408705350011587,\n                0.1122223362326622,\n                0.340884804725647,\n                -0.15227937698364258,\n                -0.005189506337046623,\n                -0.04472450166940689,\n                -0.09916462749242783,\n                -0.040019359439611435,\n                0.16836167871952057,\n                0.1395205408334732,\n                -0.1028418242931366,\n                -0.30095595121383667,\n                0.48140761256217957,\n                0.21626786887645721,\n                0.09874799847602844,\n                0.05301765352487564,\n                -0.005831585265696049,\n                -0.436678022146225,\n                0.0865410789847374,\n                0.0398264117538929,\n                -0.16981326043605804,\n                0.038346730172634125,\n                -0.23007746040821075,\n                0.2817489802837372,\n                -0.09297100454568863,\n                0.11058783531188965,\n                -0.00528382183983922,\n                0.4117512106895447,\n                -0.06859137117862701,\n                0.03556973114609718,\n                -0.05617225542664528,\n                -0.08823215961456299,\n                0.14528052508831024,\n                0.020949963480234146,\n                0.12183349579572678,\n                -0.1382322609424591,\n                -0.13707837462425232,\n                0.0687415823340416,\n                -0.14030469954013824,\n                0.15818864107131958,\n                0.19207841157913208,\n                -0.033355679363012314,\n                0.000742139876820147,\n                0.12967880070209503\n              ],\n              [\n                0.3177640438079834,\n                0.03588578850030899,\n                -0.0939568430185318,\n                0.12949220836162567,\n                0.33398666977882385,\n                0.08892231434583664,\n                -0.31378835439682007,\n                -0.21445046365261078,\n                0.10949963331222534,\n                -0.20236003398895264,\n                -0.15028421580791473,\n                0.10194536298513412,\n                0.0360504575073719,\n                0.13285522162914276,\n                -0.0156784076243639,\n                -0.11452782899141312,\n                -0.10350515693426132,\n                -0.6414148807525635,\n                -0.047703564167022705,\n                0.1961110234260559,\n                -0.44564664363861084,\n                0.03627097234129906,\n                -0.18321439623832703,\n                -0.17838998138904572,\n                -0.13331592082977295,\n                0.1563485711812973,\n                -0.35932809114456177,\n                0.2402045577764511,\n                0.11359019577503204,\n                -0.3508605360984802,\n                0.16375738382339478,\n                0.08338475227355957,\n                -0.006938163191080093,\n                0.1444127857685089,\n                -0.05632803216576576,\n                0.03932121396064758,\n                0.10900809615850449,\n                0.07841342687606812,\n                0.005295031704008579,\n                -0.15297219157218933,\n                0.081606924533844,\n                0.8683913946151733,\n                -0.3636198043823242,\n                0.00883118249475956,\n                0.2409229576587677,\n                -0.35125502943992615,\n                -0.005632631480693817,\n                0.14590124785900116,\n                0.13412189483642578,\n                -0.20834799110889435\n              ],\n              [\n                0.0017151302890852094,\n                -0.305098295211792,\n                0.12967051565647125,\n                0.022289346903562546,\n                -0.8472644090652466,\n                0.03573717549443245,\n                0.3002815842628479,\n                -0.27122384309768677,\n                -0.4793834686279297,\n                -0.1459033042192459,\n                -0.20546512305736542,\n                -0.06842319667339325,\n                -0.08808693289756775,\n                0.07788810133934021,\n                -0.0466180220246315,\n                -0.10641411691904068,\n                0.10383602976799011,\n                -0.07716105133295059,\n                -0.06759563088417053,\n                0.3445044755935669,\n                0.3319224417209625,\n                0.1663070172071457,\n                0.17388753592967987,\n                -0.10807190835475922,\n                -0.15468557178974152,\n                -0.029798904433846474,\n                0.16725780069828033,\n                -0.048754703253507614,\n                -0.1269257366657257,\n                0.2151537537574768,\n                0.035467348992824554,\n                0.03292301297187805,\n                0.052482858300209045,\n                0.11522477865219116,\n                -0.4186718165874481,\n                0.07275981456041336,\n                0.07813485711812973,\n                -0.15219397842884064,\n                -0.1822328418493271,\n                -0.040962252765893936,\n                0.016940148547291756,\n                -0.07688132673501968,\n                0.023374469950795174,\n                0.32189542055130005,\n                0.1331385225057602,\n                -0.15390019118785858,\n                0.4464007318019867,\n                -0.3328644931316376,\n                -0.19362740218639374,\n                -0.08363735675811768\n              ]\n            ],\n            "hidden_bias": [\n              -0.0910271480679512,\n              0.07835296541452408,\n              -0.2877812087535858,\n              0.536487877368927,\n              -0.17930996417999268,\n              -0.4639956057071686,\n              -0.3537112772464752,\n              -0.20710493624210358,\n              -0.012712390162050724,\n              -0.05994794890284538,\n              -0.35677650570869446,\n              0.2527293562889099,\n              -0.01686500385403633,\n              0.23655222356319427,\n              -0.47310569882392883,\n              0.045338720083236694,\n              -0.6184362173080444,\n              -0.04151638597249985,\n              -0.4233519434928894,\n              -0.0897008553147316,\n              -0.6524513363838196,\n              0.09297261387109756,\n              -0.29338133335113525,\n              0.05763170123100281,\n              -0.07583652436733246,\n              -0.04625171050429344,\n              0.09338902682065964,\n              -0.1357330083847046,\n              -0.6487828493118286,\n              -0.41045859456062317,\n              0.29976391792297363,\n              0.009535040706396103,\n              -0.41884297132492065,\n              -0.04188937693834305,\n              -0.1835736483335495,\n              -0.5297504663467407,\n              -0.12637421488761902,\n              0.023102406412363052,\n              0.19732075929641724,\n              0.12233787775039673,\n              -0.039070453494787216,\n              -0.22084969282150269,\n              -0.23959358036518097,\n              -0.033917754888534546,\n              -0.23289041221141815,\n              0.26288798451423645,\n              -0.11982260644435883,\n              0.30594393610954285,\n              -0.6035798788070679,\n              0.015034908428788185,\n              0.07078143209218979,\n              0.08575760573148727,\n              0.04642843082547188,\n              0.11488718539476395,\n              0.13566158711910248,\n              -0.16577203571796417,\n              0.02040993794798851,\n              -0.0511690191924572,\n              -0.17587392032146454,\n              -0.5083130598068237,\n              -0.38909322023391724,\n              -0.11265306919813156,\n              -0.039799291640520096,\n              -0.036090556532144547\n            ],\n            "output_weight": [\n              0.2131989598274231,\n              0.1977451592683792,\n              0.10760799050331116,\n              -0.1919630616903305,\n              0.1623319536447525,\n              -0.15136662125587463,\n              0.18045802414417267,\n              -0.30183181166648865,\n              -0.008809132501482964,\n              0.15819451212882996,\n              0.2064265012741089,\n              0.14527276158332825,\n              -0.12433601915836334,\n              0.08363532274961472,\n              -0.25438863039016724,\n              0.22195687890052795,\n              0.26992323994636536,\n              0.1499648541212082,\n              0.1286066770553589,\n              0.3266310393810272,\n              0.347166508436203,\n              -0.17328840494155884,\n              0.13646559417247772,\n              0.17151620984077454,\n              0.12186865508556366,\n              -0.2395879328250885,\n              0.1403948813676834,\n              -0.20059140026569366,\n              -0.2968544065952301,\n              -0.4102231562137604,\n              -0.12591375410556793,\n              0.2557772397994995,\n              -0.3231998383998871,\n              -0.1460215002298355,\n              0.0783180370926857,\n              0.4334082007408142,\n              0.09213857352733612,\n              -0.22668281197547913,\n              0.1472545862197876,\n              -0.2056395411491394,\n              0.10779472440481186,\n              0.17486685514450073,\n              -0.23726321756839752,\n              -0.13073530793190002,\n              -0.18027904629707336,\n              -0.12480378895998001,\n              0.22221282124519348,\n              0.11064167320728302,\n              -0.4147959351539612,\n              -0.24766920506954193,\n              -0.2846771478652954,\n              -0.1911633461713791,\n              0.22691060602664948,\n              0.0041246572509408,\n              0.0713290274143219,\n              0.11663182824850082,\n              -0.24798740446567535,\n              0.024134783074259758,\n              -0.22529403865337372,\n              0.3082430362701416,\n              0.11048208177089691,\n              0.07865811884403229,\n              0.06063178554177284,\n              0.1315339207649231\n            ],\n            "output_bias": 0.1393638551235199,\n            "training": {\n              "method": "paired counterfactual residual policy distillation",\n              "seed": 20261020,\n              "loss_type": "ranking",\n              "regularization": 0.01,\n              "best_epoch": 97,\n              "validation_preference_loss": 0.46406126022338867,\n              "baseline_preference_loss": 0.7314271330833435,\n              "training_games": 2400,\n              "validation_games": 600,\n              "source_dataset": {\n                "method": "paired one-step policy improvement, phase-stratified whole-game reservoirs",\n                "input_features": [\n                  "batch_size",\n                  "truthful",\n                  "bluff",\n                  "bluff_pile",\n                  "bluff_call_rate",\n                  "bluff_finish",\n                  "remaining_groups",\n                  "remaining_largest_group",\n                  "claimed_rank_held",\n                  "bias",\n                  "revealed_bluff_rate",\n                  "rank_held",\n                  "claim_count",\n                  "pile_size",\n                  "opponent_finishing",\n                  "own_small_hand",\n                  "opponent_small_hand",\n                  "samples",\n                  "bluff_any_call_probability",\n                  "bluff_impossible_for_caller",\n                  "truthful_turns_removed",\n                  "bluff_known_pile_fraction",\n                  "truthful_finish",\n                  "bluff_no_impossibility_call_possible",\n                  "placer_global_revealed_bluff_rate",\n                  "known_outside_rank_fraction",\n                  "known_pile_fraction",\n                  "claim_fraction_of_original_hand",\n                  "finishing_large_batch",\n                  "known_placer_rank_before_claim",\n                  "bluff_call_probability_batch",\n                  "bluff_call_probability_pile",\n                  "bluff_call_probability_finish",\n                  "bluff_nearest_opponent_finish",\n                  "truthful_call_probability_batch",\n                  "bluff_remaining_truthful_turns",\n                  "global_revealed_bluff_finishing",\n                  "count_revealed_bluff_pile",\n                  "global_revealed_bluff_pile",\n                  "count_revealed_bluff_small_own_hand",\n                  "global_revealed_bluff_count",\n                  "opponent_hand_pace_relative_to_own",\n                  "bluff_pooled_caller_probability",\n                  "bluff_pooled_caller_uncertainty",\n                  "bluff_specific_caller_uncertainty",\n                  "bluff_sparse_own_outcomes",\n                  "truthful_pooled_caller_probability_batch",\n                  "bluff_pooled_caller_probability_pile",\n                  "global_revealed_bluff_uncertainty",\n                  "count_revealed_bluff_uncertainty"\n                ],\n                "feature_policy": "v5",\n                "games": 3000,\n                "states": 12000,\n                "actions": 67757,\n                "trials": 4,\n                "samples_per_phase": 2,\n                "max_actions": 10,\n                "seed_start": 50000000,\n                "incumbent": {\n                  "format": "triad-bluff-linear-v2",\n                  "features": [\n                    "batch_size",\n                    "truthful",\n                    "bluff",\n                    "bluff_pile",\n                    "bluff_call_rate",\n                    "bluff_finish",\n                    "remaining_groups",\n                    "remaining_largest_group",\n                    "claimed_rank_held",\n                    "bias",\n                    "revealed_bluff_rate",\n                    "rank_held",\n                    "claim_count",\n                    "pile_size",\n                    "opponent_finishing",\n                    "own_small_hand",\n                    "opponent_small_hand",\n                    "samples",\n                    "bluff_any_call_probability",\n                    "bluff_impossible_for_caller",\n                    "truthful_turns_removed",\n                    "bluff_known_pile_fraction",\n                    "truthful_finish",\n                    "bluff_no_impossibility_call_possible",\n                    "placer_global_revealed_bluff_rate",\n                    "known_outside_rank_fraction",\n                    "known_pile_fraction",\n                    "claim_fraction_of_original_hand",\n                    "finishing_large_batch",\n                    "known_placer_rank_before_claim"\n                  ],\n                  "weights": [\n                    0.4755931924211193,\n                    5.820676470779726,\n                    -3.445920468550993,\n                    0.5025301512894197,\n                    -10.204982725387449,\n                    -10.63672367105182,\n                    1.9122839363624424,\n                    6.692718730179745,\n                    3.3398384247884447,\n                    -3.967913294968817,\n                    7.18093879068249,\n                    -8.248647532051631,\n                    -4.252488827029067,\n                    -5.717390427387336,\n                    -10.297751467694116,\n                    -4.172529040138095,\n                    3.643993344291045,\n                    4.785033532440487,\n                    -8.90940656773177,\n                    -9.18268199666696,\n                    3.5590525036076692,\n                    -1.533025684746697,\n                    5.957038223067108,\n                    9.714173152584005,\n                    3.5582377937169647,\n                    -0.47803421898358756,\n                    -1.2639464162072949,\n                    0.6168380283723138,\n                    2.861924827690982,\n                    -7.115720155350792\n                  ],\n                  "training": {\n                    "method": "archive-league cross-entropy policy optimization",\n                    "seed": 20261016,\n                    "generations": 30,\n                    "population": 32,\n                    "train_blocks": 8,\n                    "train_horizon": 200,\n                    "evaluation_horizon": 1000,\n                    "validation_blocks": 64,\n                    "selected_validation_points": 4.0859375,\n                    "incumbent_validation_points": 3.62109375,\n                    "sdk_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6"\n                  }\n                },\n                "opponent_pool": [\n                  {\n                    "kind": "v2",\n                    "weights": [\n                      0.4755931924211193,\n                      5.820676470779726,\n                      -3.445920468550993,\n                      0.5025301512894197,\n                      -10.204982725387449,\n                      -10.63672367105182,\n                      1.9122839363624424,\n                      6.692718730179745,\n                      3.3398384247884447,\n                      -3.967913294968817,\n                      7.18093879068249,\n                      -8.248647532051631,\n                      -4.252488827029067,\n                      -5.717390427387336,\n                      -10.297751467694116,\n                      -4.172529040138095,\n                      3.643993344291045,\n                      4.785033532440487,\n                      -8.90940656773177,\n                      -9.18268199666696,\n                      3.5590525036076692,\n                      -1.533025684746697,\n                      5.957038223067108,\n                      9.714173152584005,\n                      3.5582377937169647,\n                      -0.47803421898358756,\n                      -1.2639464162072949,\n                      0.6168380283723138,\n                      2.861924827690982,\n                      -7.115720155350792\n                    ]\n                  },\n                  {\n                    "kind": "v1",\n                    "weights": [\n                      4.351868833686492,\n                      5.6322712663248415,\n                      -2.6260873498626167,\n                      0.7063932700439369,\n                      -10.84185815552825,\n                      -8.300750779890834,\n                      -0.7486711333834275,\n                      1.2260583122844886,\n                      -1.5575700886723332,\n                      -4.558178606791033,\n                      8.25750430388175,\n                      0.43873901259948533,\n                      -1.3556124209301408,\n                      -1.1333529834957075,\n                      0.14766041336362645,\n                      -1.5734330345566885,\n                      -0.4730921181099686,\n                      -0.5811911507935986\n                    ]\n                  },\n                  {\n                    "kind": "starter"\n                  },\n                  {\n                    "kind": "competitive"\n                  },\n                  {\n                    "kind": "random"\n                  },\n                  {\n                    "kind": "aggressive"\n                  },\n                  {\n                    "kind": "heuristic"\n                  },\n                  {\n                    "kind": "bluffer",\n                    "bluff_rate": 0.1,\n                    "call_rate": 0.0\n                  },\n                  {\n                    "kind": "bluffer",\n                    "bluff_rate": 0.1,\n                    "call_rate": 0.35\n                  },\n                  {\n                    "kind": "bluffer",\n                    "bluff_rate": 0.35,\n                    "call_rate": 0.1\n                  },\n                  {\n                    "kind": "bluffer",\n                    "bluff_rate": 0.65,\n                    "call_rate": 0.35\n                  },\n                  {\n                    "kind": "bluffer",\n                    "bluff_rate": 0.95,\n                    "call_rate": 0.0\n                  },\n                  {\n                    "kind": "bluffer",\n                    "bluff_rate": 0.95,\n                    "call_rate": 0.35\n                  }\n                ],\n                "sdk_game_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6",\n                "hidden_state_use": "training returns only; dealt hands fixed within state, common future actor seeds across actions",\n                "future_draw_pile_shuffle": false,\n                "own_hand_size_histogram": {\n                  "2": 640,\n                  "3": 1392,\n                  "4": 1021,\n                  "5": 1355,\n                  "6": 783,\n                  "7": 1373,\n                  "8": 606,\n                  "9": 1081,\n                  "10": 720,\n                  "11": 1005,\n                  "12": 98,\n                  "13": 1443,\n                  "14": 51,\n                  "15": 65,\n                  "16": 65,\n                  "17": 41,\n                  "18": 46,\n                  "19": 31,\n                  "20": 35,\n                  "21": 27,\n                  "22": 14,\n                  "23": 21,\n                  "24": 19,\n                  "25": 14,\n                  "26": 9,\n                  "27": 8,\n                  "28": 5,\n                  "29": 4,\n                  "30": 4,\n                  "31": 5,\n                  "32": 3,\n                  "34": 3,\n                  "35": 4,\n                  "37": 1,\n                  "38": 2,\n                  "39": 1,\n                  "40": 1,\n                  "41": 1,\n                  "43": 2,\n                  "46": 1\n                },\n                "phase_histogram": {\n                  "PLACE": 6000,\n                  "CHALLENGE": 6000\n                },\n                "seconds": 2063.4\n              },\n              "base_score_factor": 0.25,\n              "hidden_state_use": "training labels only; policy input is own/public features",\n              "residual_scale": 0.5\n            }\n          }\n        },\n        {\n          "kind": "v2",\n          "weights": [\n            0.4755931924211193,\n            5.820676470779726,\n            -3.445920468550993,\n            0.5025301512894197,\n            -10.204982725387449,\n            -10.63672367105182,\n            1.9122839363624424,\n            6.692718730179745,\n            3.3398384247884447,\n            -3.967913294968817,\n            7.18093879068249,\n            -8.248647532051631,\n            -4.252488827029067,\n            -5.717390427387336,\n            -10.297751467694116,\n            -4.172529040138095,\n            3.643993344291045,\n            4.785033532440487,\n            -8.90940656773177,\n            -9.18268199666696,\n            3.5590525036076692,\n            -1.533025684746697,\n            5.957038223067108,\n            9.714173152584005,\n            3.5582377937169647,\n            -0.47803421898358756,\n            -1.2639464162072949,\n            0.6168380283723138,\n            2.861924827690982,\n            -7.115720155350792\n          ]\n        },\n        {\n          "kind": "starter"\n        },\n        {\n          "kind": "competitive"\n        },\n        {\n          "kind": "random"\n        },\n        {\n          "kind": "aggressive"\n        },\n        {\n          "kind": "heuristic"\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.1,\n          "call_rate": 0.0\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.1,\n          "call_rate": 0.35\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.35,\n          "call_rate": 0.1\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.65,\n          "call_rate": 0.35\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.95,\n          "call_rate": 0.0\n        },\n        {\n          "kind": "bluffer",\n          "bluff_rate": 0.95,\n          "call_rate": 0.35\n        }\n      ],\n      "sdk_game_commit": "96da8042c285af4e4f43655f2ad99592bd49b8e6",\n      "incumbent_policy_kind": "mlp",\n      "hidden_state_use": "training returns only; dealt hands fixed within state, common future actor seeds across actions",\n      "future_draw_pile_shuffle": false,\n      "own_hand_size_histogram": {\n        "1": 280,\n        "2": 1044,\n        "3": 1104,\n        "4": 1063,\n        "5": 1259,\n        "6": 659,\n        "7": 1147,\n        "8": 652,\n        "9": 946,\n        "10": 693,\n        "11": 1054,\n        "12": 100,\n        "13": 1474,\n        "14": 48,\n        "15": 76,\n        "16": 59,\n        "17": 46,\n        "18": 35,\n        "19": 33,\n        "20": 23,\n        "21": 43,\n        "22": 18,\n        "23": 26,\n        "24": 22,\n        "25": 3,\n        "26": 8,\n        "27": 14,\n        "28": 10,\n        "29": 9,\n        "30": 11,\n        "31": 6,\n        "32": 6,\n        "33": 4,\n        "34": 4,\n        "35": 3,\n        "36": 2,\n        "37": 2,\n        "38": 1,\n        "39": 3,\n        "40": 3,\n        "41": 1,\n        "42": 1,\n        "43": 1,\n        "44": 1,\n        "45": 3\n      },\n      "phase_histogram": {\n        "PLACE": 6000,\n        "CHALLENGE": 6000\n      },\n      "seconds": 4373.9\n    },\n    "base_score_factor": 0.25,\n    "frozen_hidden_units": 64,\n    "baseline_policy_kind": "mlp",\n    "hidden_state_use": "training labels only; policy input is own/public features",\n    "residual_scale": 0.25\n  }\n}\n')
summaries=list(Path("/kaggle/input").rglob("fit-summary.json"))
assert len(summaries)==1,"ambiguous candidate export"
summary=summaries[0]
subprocess.run(["python",str(work/"select_bluff_residual_cpu.py"),"--candidates",str(summary.parent),"--linear-backup",str(work/"linear.json"),"--previous-v1",str(work/"previous.json"),"--archive",str(work/"archive-v3.json"),"--archive",str(work/"archive-v4.json"),"--archive",str(work/"archive-ranking.json"),"--validation-blocks","128","--test-blocks","256","--validation-seed-start","88000000","--test-seed-start","89000000","--robust-selection","--robust-calling-styles","--max-selection-regression",".025","--workers","4","--output","/kaggle/working/output"],check=True,timeout=21600)
